# Stage 0 - Runtime and reproducibility

## Goal and scientific context

Before chemistry or diffusion, we need a reproducible process that uses only
GPUs eligible under the user-authorized utilization policy. Stage 0 discovers
the active Git worktree, verifies
that imports come from its `src/` directory, locates the shared project virtual
environment, records source provenance, and establishes device visibility and
random seeds.

### Paper, released repository, and notebook distinction

- **Paper:** Appendix D.1 reports training on 8 NVIDIA A100 GPUs for about 5
  hours and generation on one A100 with 32 CPU cores. Appendix D.2 reports a
  global batch size of 2,048 and 50,000 optimizer steps.
- **Released repository:** the trainer uses the CUDA devices visible to the
  process. It does not define this notebook's shared-server eligibility guard.
- **Notebook-only safety:** the notebook records process count and checks
  utilization, free memory, and compute mode before exposing devices. This
  local guard is not an algorithm from the paper.

An A6000 is not computationally equivalent to an A100. Matching the GPU count
does not reproduce the paper's wall-clock time. The active user authorization
permits a count from one through two without another permission request;
physical IDs are selected dynamically.

## What the next code cell does

The next cell defines `NUM_GPUS` as an integer from 1 through 2. It is a count, not a
physical GPU index and not a list of device IDs. For example,

```python
NUM_GPUS = 2
```

means "require exactly two eligible GPUs." It does not mean physical GPU 2.
The validation rejects zero, values above two, floats, strings, and Booleans.

The cell prints the requested count. It does not inspect, reserve, or initialize
any GPU. Changing the count requires a kernel restart and a run from the top
because CUDA visibility must be fixed before PyTorch is imported.

Stage 0 contains no SAFE encoding, diffusion, BERT, optimizer, or training.

### Comprehension checkpoint

1. Does `NUM_GPUS = 2` select physical GPU 2?
2. What should happen if fewer than two GPUs later pass the guard?
3. Why is the two-GPU authorization distinct from the paper's eight-GPU setup?
4. Why must the kernel be restarted after changing `NUM_GPUS`?


In [ ]:
# USER SETTING: request 1--2 GPUs; physical IDs are selected dynamically.
NUM_GPUS: int = 1

if type(NUM_GPUS) is not int or not 1 <= NUM_GPUS <= 2:
    raise ValueError('NUM_GPUS must be an integer from 1 through 2.')

print(f'Requested GPU count: {NUM_GPUS} (user-authorized hard ceiling: 2)')


## Stage 0.2 - Resolve this worktree and inspect the server

Before any scientific-library import, the next cell asks Git for the active
worktree root and common repository directory. It requires the shared project
`.venv`, prepends this checkout's `src/` to both `sys.path` and `PYTHONPATH`,
imports `genmol`, and proves that `genmol.__file__` lies under that `src/`.
This prevents an editable install from silently importing a different checkout.

It then discovers physical GPU indices from `nvidia-smi`; no fixed inventory or
assumption about physical GPU 0 is encoded. Every discovered card is queried
independently. A card is eligible only if:

- its status, physical index, and UUID can be verified;
- its process inventory can be recorded (a nonempty inventory is allowed);
- it has at least 30,000 MiB free;
- utilization is strictly below 10 percent; and
- compute mode is not prohibited.

For example, a card at 9% utilization with an active process can pass when its
free-memory and compute-mode checks also pass, while a card at exactly 10% is
rejected. Process evidence is observational: the notebook never interrupts or
kills another process. Querying cards independently means a fault on one
physical index does not hide all healthy cards. Eligible cards are ordered by
free memory (most first), utilization, and physical index; the immediate
re-probe must retain each selected card's exact UUID. Caches are redirected into
this worktree's `.cache` directory.

Expected output includes the dynamic root, virtual environment, resolved
`genmol` source, discovered physical indices, one status row per card, and the
eligible count. CUDA visibility remains unchanged.

**Notebook-only scope:** this point-in-time cooperative guard is not an atomic
scheduler reservation. The cell still does not select a device or import
PyTorch.

### Comprehension checkpoint

1. Why verify `genmol.__file__` before importing PyTorch?
2. Why discover physical indices instead of assuming `range(8)`?
3. At the end of this cell, has `CUDA_VISIBLE_DEVICES` been set?
4. Can this snapshot guarantee that another process will not claim a card later?


In [ ]:
from __future__ import annotations

import csv
import io
import os
import subprocess
import sys
from pathlib import Path


def _checked_text_before_runtime(command: list[str], *, cwd: Path) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        capture_output=True,
        text=True,
        check=True,
    )
    return completed.stdout.strip()


START_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = Path(
    _checked_text_before_runtime(
        ['git', 'rev-parse', '--show-toplevel'], cwd=START_DIRECTORY
    )
).resolve()
GIT_COMMON_DIR_TEXT = _checked_text_before_runtime(
    ['git', 'rev-parse', '--git-common-dir'], cwd=PROJECT_ROOT
)
GIT_COMMON_DIR = Path(GIT_COMMON_DIR_TEXT)
if not GIT_COMMON_DIR.is_absolute():
    GIT_COMMON_DIR = (PROJECT_ROOT / GIT_COMMON_DIR).resolve()
SHARED_REPOSITORY_ROOT = GIT_COMMON_DIR.parent
PROJECT_VENV = (SHARED_REPOSITORY_ROOT / '.venv').resolve()
PROJECT_SOURCE_ROOT = (PROJECT_ROOT / 'src').resolve()

assert (PROJECT_ROOT / 'src' / 'genmol').is_dir(), (
    f'Git worktree has no src/genmol package: {PROJECT_ROOT}'
)
assert (PROJECT_ROOT / 'src' / 'genmol' / 'diffusion.py').is_file(), (
    'This checkout does not contain the UDLM diffusion implementation.'
)
assert Path(sys.prefix).resolve() == PROJECT_VENV, (
    f'Wrong Python environment: {Path(sys.prefix).resolve()}. '
    f'Expected the project environment {PROJECT_VENV}.'
)
assert 'torch' not in sys.modules, (
    'PyTorch was imported before GPU isolation. Restart the kernel and run from the top.'
)

# Resolve imports from this Git worktree rather than another editable checkout.
sys.path[:] = [
    str(PROJECT_SOURCE_ROOT),
    *[
        entry
        for entry in sys.path
        if not entry or Path(entry).resolve() != PROJECT_SOURCE_ROOT
    ],
]
existing_pythonpath = os.environ.get('PYTHONPATH', '')
pythonpath_parts = [
    part
    for part in existing_pythonpath.split(os.pathsep)
    if part and Path(part).resolve() != PROJECT_SOURCE_ROOT
]
os.environ['PYTHONPATH'] = os.pathsep.join(
    [str(PROJECT_SOURCE_ROOT), *pythonpath_parts]
)

import genmol as stage0_genmol

GENMOL_SOURCE_PATH = Path(stage0_genmol.__file__).resolve()
assert GENMOL_SOURCE_PATH.is_relative_to(PROJECT_SOURCE_ROOT), (
    f'genmol resolved from {GENMOL_SOURCE_PATH}, not this worktree at '
    f'{PROJECT_SOURCE_ROOT}.'
)

MIN_IDLE_FREE_MEMORY_MIB = 30000
MAX_IDLE_UTILIZATION_PERCENT = 10

preexisting_visibility = os.environ.get('CUDA_VISIBLE_DEVICES')
assert preexisting_visibility in (None, ''), (
    'CUDA_VISIBLE_DEVICES was already set when this kernel started. The notebook will '
    'not override a possible scheduler allocation. Start a fresh unallocated kernel.'
)


def _run_nvidia_smi(gpu_id: int, query: str) -> subprocess.CompletedProcess:
    return subprocess.run(
        [
            'nvidia-smi',
            '-i',
            str(gpu_id),
            query,
            '--format=csv,noheader,nounits',
        ],
        capture_output=True,
        text=True,
        check=False,
    )


def _failure_text(result: subprocess.CompletedProcess) -> str:
    detail = result.stderr.strip() or result.stdout.strip()
    return ' | '.join(detail.splitlines()) if detail else f'exit code {result.returncode}'


def discover_physical_gpu_ids() -> tuple[int, ...]:
    result = subprocess.run(
        [
            'nvidia-smi',
            '--query-gpu=index',
            '--format=csv,noheader,nounits',
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0 or result.stderr.strip():
        raise RuntimeError(f'Could not discover physical GPUs: {_failure_text(result)}')
    try:
        gpu_ids = tuple(int(line.strip()) for line in result.stdout.splitlines() if line.strip())
    except ValueError as error:
        raise RuntimeError('nvidia-smi returned a non-integer physical GPU index') from error
    if not gpu_ids or len(gpu_ids) != len(set(gpu_ids)):
        raise RuntimeError(f'Invalid physical GPU inventory: {gpu_ids}')
    return gpu_ids


PHYSICAL_GPU_IDS = discover_physical_gpu_ids()


def probe_physical_gpu(gpu_id: int) -> dict:
    gpu = {'index': gpu_id, 'eligible': False, 'reasons': []}
    status = _run_nvidia_smi(
        gpu_id,
        '--query-gpu=index,uuid,name,memory.free,utilization.gpu,compute_mode',
    )
    if status.returncode != 0 or status.stderr.strip():
        gpu['reasons'].append(f'status query failed: {_failure_text(status)}')
        return gpu

    rows = [
        [field.strip() for field in row]
        for row in csv.reader(io.StringIO(status.stdout))
        if any(field.strip() for field in row)
    ]
    if len(rows) != 1 or len(rows[0]) != 6:
        gpu['reasons'].append('status query returned an unexpected row')
        return gpu

    index_text, uuid, name, memory_text, utilization_text, compute_mode = rows[0]
    try:
        reported_index = int(index_text)
        free_memory_mib = int(memory_text)
        utilization_percent = int(utilization_text)
    except ValueError:
        gpu['reasons'].append('status query contained an N/A or non-numeric field')
        return gpu
    if reported_index != gpu_id or not uuid.startswith('GPU-'):
        gpu['reasons'].append('status query returned an inconsistent index or UUID')
        return gpu

    gpu.update(
        {
            'uuid': uuid,
            'name': name,
            'free_memory_mib': free_memory_mib,
            'utilization_percent': utilization_percent,
            'compute_mode': compute_mode,
        }
    )

    processes = _run_nvidia_smi(
        gpu_id,
        '--query-compute-apps=pid,used_memory',
    )
    if processes.returncode != 0 or processes.stderr.strip():
        gpu['reasons'].append(f'process query failed: {_failure_text(processes)}')
        return gpu

    process_rows = []
    for row in csv.reader(io.StringIO(processes.stdout)):
        fields = [field.strip() for field in row]
        if not any(fields) or fields[0].lower().startswith('no running'):
            continue
        if len(fields) != 2 or not fields[0].isdigit():
            gpu['reasons'].append('process query returned an unverifiable row')
            return gpu
        process_rows.append({'pid': int(fields[0]), 'used_memory': fields[1]})

    gpu['compute_process_count'] = len(process_rows)
    gpu['compute_processes'] = process_rows
    if free_memory_mib < MIN_IDLE_FREE_MEMORY_MIB:
        gpu['reasons'].append(
            f'{free_memory_mib} MiB free is below {MIN_IDLE_FREE_MEMORY_MIB} MiB'
        )
    if utilization_percent >= MAX_IDLE_UTILIZATION_PERCENT:
        gpu['reasons'].append(
            f'{utilization_percent}% utilization is not below '
            f'{MAX_IDLE_UTILIZATION_PERCENT}%'
        )
    if compute_mode.lower() == 'prohibited':
        gpu['reasons'].append('compute mode is prohibited')

    gpu['eligible'] = not gpu['reasons']
    return gpu


def print_gpu_snapshot(snapshot: list[dict]) -> None:
    print('GPU availability snapshot:')
    for gpu in snapshot:
        if 'name' in gpu:
            details = (
                f"{gpu['name']}, {gpu['free_memory_mib']} MiB free, "
                f"{gpu['utilization_percent']}%, "
                f"compute_processes={gpu.get('compute_process_count', '?')}"
            )
        else:
            details = 'status unavailable'
        state = 'ELIGIBLE' if gpu['eligible'] else 'REJECTED: ' + '; '.join(gpu['reasons'])
        print(f"  physical {gpu['index']}: {details} -> {state}")


def take_gpu_snapshot() -> list[dict]:
    snapshot = [probe_physical_gpu(gpu_id) for gpu_id in PHYSICAL_GPU_IDS]
    print_gpu_snapshot(snapshot)
    return snapshot


def select_idle_gpus(num_gpus: int) -> list[dict]:
    if type(num_gpus) is not int or not 1 <= num_gpus <= 2:
        raise ValueError('NUM_GPUS must be an integer from 1 through 2.')

    snapshot = take_gpu_snapshot()
    eligible = sorted(
        (gpu for gpu in snapshot if gpu['eligible']),
        key=lambda gpu: (
            -gpu['free_memory_mib'],
            gpu['utilization_percent'],
            gpu['index'],
        ),
    )
    if len(eligible) < num_gpus:
        eligible_ids = [gpu['index'] for gpu in eligible]
        raise RuntimeError(
            f'Requested {num_gpus} GPU(s), but only {len(eligible)} passed the guard. '
            f'Currently eligible physical IDs: {eligible_ids}. Wait and rerun.'
        )

    selected = eligible[:num_gpus]
    rechecked = [probe_physical_gpu(gpu['index']) for gpu in selected]
    changed = [gpu for gpu in rechecked if not gpu['eligible']]
    if changed:
        details = {gpu['index']: gpu['reasons'] for gpu in changed}
        raise RuntimeError(
            f'A selected GPU changed state during validation: {details}. '
            'Restart the kernel and run again.'
        )
    identity_changes = [
        {
            'physical_index': initial['index'],
            'initial_uuid': initial['uuid'],
            'final_uuid': final['uuid'],
        }
        for initial, final in zip(selected, rechecked, strict=True)
        if final['uuid'] != initial['uuid']
    ]
    if identity_changes:
        raise RuntimeError(
            f'A selected physical GPU changed UUID: {identity_changes}. '
            'Restart the kernel and run again.'
        )

    return rechecked


os.environ.update(
    {
        'CUDA_DEVICE_ORDER': 'PCI_BUS_ID',
        'HF_HOME': str(PROJECT_ROOT / '.cache' / 'huggingface'),
        'TORCH_HOME': str(PROJECT_ROOT / '.cache' / 'torch'),
        'PIP_CACHE_DIR': str(PROJECT_ROOT / '.cache' / 'pip'),
        'TOKENIZERS_PARALLELISM': 'false',
        'CUBLAS_WORKSPACE_CONFIG': ':4096:8',
    }
)

print(f'Git worktree root: {PROJECT_ROOT}')
print(f'Project virtual environment: {PROJECT_VENV}')
print(f'genmol source: {GENMOL_SOURCE_PATH}')
print(f'Discovered physical GPU IDs: {PHYSICAL_GPU_IDS}')
discovery_snapshot = take_gpu_snapshot()
print(
    'Currently eligible GPU count:',
    sum(gpu['eligible'] for gpu in discovery_snapshot),
)
print('Discovery only: CUDA_VISIBLE_DEVICES has not been set.')


## Stage 0.3 - Select devices, import the runtime, and verify CUDA

The next cell takes a fresh snapshot, requires exactly `NUM_GPUS` eligible cards, and rechecks the chosen cards immediately before use. Their UUIDs are written to `CUDA_VISIBLE_DEVICES`, and only then is PyTorch imported.

CUDA renumbers visible cards. If physical GPUs 2 and 6 are selected:

```text
physical GPU 2 -> logical cuda:0
physical GPU 6 -> logical cuda:1
```

Therefore `cuda:0` means the first selected card, not necessarily physical GPU 0.

The cell seeds Python, NumPy, CPU PyTorch, and every visible CUDA device with seed 7. It requests deterministic algorithms where supported. A tensor with shape `[1024]` is created and summed independently on every logical device, and each reported device name is checked against the selected physical card.

Expected output includes package versions, visible GPU count, physical-to-logical mappings, VRAM, and seed 7. Assertions stop on a CUDA, count, mapping, or tensor-probe failure.

The released configuration can use multiple visible devices, but visibility alone does not parallelize this notebook. DDP or another multi-process strategy must be added in a later training stage.

### Comprehension checkpoint

1. If physical GPU 6 is selected first, what logical name does it receive?
2. Why is PyTorch imported only after the UUID visibility string is set?
3. What does the `[1024]` tensor probe verify?
4. Does seeing two logical devices automatically split BERT or a batch across them?


### Output hygiene

The following runtime cell suppresses exactly two known, non-actionable third-party startup messages: the tqdm missing-widget notice and the wandb pkg_resources deprecation. Other warnings remain visible.


In [ ]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="pkg_resources is deprecated as an API.*")

assert 'torch' not in sys.modules, (
    'PyTorch is already imported. Restart the kernel after changing NUM_GPUS.'
)

selected_gpus = select_idle_gpus(NUM_GPUS)
SELECTED_PHYSICAL_GPU_IDS = [gpu['index'] for gpu in selected_gpus]
SELECTED_GPU_UUIDS = [gpu['uuid'] for gpu in selected_gpus]
os.environ['CUDA_VISIBLE_DEVICES'] = ','.join(SELECTED_GPU_UUIDS)

print('Selected physical GPU IDs:', SELECTED_PHYSICAL_GPU_IDS)
print('CUDA_VISIBLE_DEVICES:', os.environ['CUDA_VISIBLE_DEVICES'])

# Import immediately after the final recheck to minimize the shared-server race window.
import importlib.metadata as metadata
import platform
import random

import numpy as np
import torch
import datasets
import lightning
import rdkit
import transformers

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

assert torch.cuda.is_available(), 'CUDA is not available to PyTorch.'
assert torch.cuda.device_count() == NUM_GPUS, (
    f'Expected {NUM_GPUS} visible GPU(s), found {torch.cuda.device_count()}.'
)

device_summaries = []
for logical_index, physical_gpu in enumerate(selected_gpus):
    device = torch.device(f'cuda:{logical_index}')
    device_name = torch.cuda.get_device_name(device)
    assert device_name == physical_gpu['name'], (
        f"Mapping mismatch for physical GPU {physical_gpu['index']}: {device_name}"
    )
    total_vram_gib = torch.cuda.get_device_properties(device).total_memory / 2**30
    probe = torch.ones(1024, device=device)
    assert probe.sum().item() == 1024.0
    torch.cuda.synchronize(device)
    device_summaries.append(
        {
            'logical_index': logical_index,
            'physical_index': physical_gpu['index'],
            'name': device_name,
            'vram_gib': total_vram_gib,
        }
    )
    del probe

for logical_index in range(NUM_GPUS):
    with torch.cuda.device(logical_index):
        torch.cuda.empty_cache()

versions = {
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda_build': torch.version.cuda,
    'transformers': transformers.__version__,
    'lightning': lightning.__version__,
    'datasets': datasets.__version__,
    'safe-mol': metadata.version('safe-mol'),
    'rdkit': rdkit.__version__,
    'bionemo-moco': metadata.version('bionemo-moco'),
    'genmol': metadata.version('genmol'),
}

for name, version in versions.items():
    print(f'{name:>14}: {version}')
print(f'{"visible_gpus":>14}: {torch.cuda.device_count()}')
for summary in device_summaries:
    print(
        f"physical GPU {summary['physical_index']} -> "
        f"logical cuda:{summary['logical_index']}: "
        f"{summary['name']} ({summary['vram_gib']:.1f} GiB)"
    )
print(f'{"seed":>14}: {SEED}')


## Stage 0.4 - Verify source and dependency provenance

The next cell records the active Git worktree rather than assuming one fixed
directory or branch. It requires:

- the full UDLM implementation base commit `02595d1ecf994bbd432ec67ef21b0d56ed97918d` to be an
  ancestor of the active `HEAD`;
- `HEAD` to equal its local upstream-tracking commit, establishing that the
  recorded commit has been pushed;
- `genmol` to have resolved from this worktree's `src/` directory;
- `python -m pip check` to pass in the shared project `.venv`.

The branch name, tracking ref, full commits, and working-tree status are
recorded. Uncommitted paths are evidence, not silently discarded. The ancestor
check permits later fixes on top of the base implementation; an equality check
against an obsolete notebook commit would make every legitimate update fail.

**Concrete example.** If `HEAD=A` and `origin/branch=A`, the pushed-HEAD check
passes. If the required UDLM base lies in `git log A`, the ancestry check also
passes even when documentation fixes were committed later.

**Difference from released code.** These provenance assertions are notebook
engineering controls, not GenMol or UDLM algorithms.

### Comprehension checkpoint

1. Why is ancestry more appropriate than equality for the implementation base?
2. What does `HEAD == @{upstream}` establish, and what does it not establish?
3. Why record dirty paths rather than pretending they belong to the pushed commit?
4. Does a successful provenance check prove scientific correctness?


In [ ]:
def run_text(command: list[str]) -> str:
    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        check=True,
    )
    return completed.stdout.strip()


UDLM_BASE_COMMIT = '02595d1ecf994bbd432ec67ef21b0d56ed97918d'
actual_commit = run_text(['git', 'rev-parse', 'HEAD'])
branch = run_text(['git', 'branch', '--show-current']) or '(detached HEAD)'
tracking_ref = run_text(
    ['git', 'rev-parse', '--abbrev-ref', '--symbolic-full-name', '@{upstream}']
)
pushed_head_commit = run_text(['git', 'rev-parse', '@{upstream}'])
assert actual_commit == pushed_head_commit, (
    f'HEAD {actual_commit} is not pushed to {tracking_ref} at {pushed_head_commit}.'
)

base_check = subprocess.run(
    ['git', 'merge-base', '--is-ancestor', UDLM_BASE_COMMIT, actual_commit],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=False,
)
assert base_check.returncode == 0, (
    f'Required UDLM base {UDLM_BASE_COMMIT} is not an ancestor of HEAD '
    f'{actual_commit}: {base_check.stderr.strip()}'
)
checked_upstream_commit = pushed_head_commit
working_tree_status = run_text(['git', 'status', '--short'])

pip_check = subprocess.run(
    [sys.executable, '-m', 'pip', 'check'],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=False,
)
assert pip_check.returncode == 0, pip_check.stdout + pip_check.stderr

print('Git worktree:', PROJECT_ROOT)
print('Git branch:', branch)
print('Pushed HEAD:', actual_commit)
print('Tracking ref:', tracking_ref)
print('Required UDLM base ancestor:', UDLM_BASE_COMMIT)
print('Working tree status:', working_tree_status or 'clean')
print('genmol source:', GENMOL_SOURCE_PATH)
print('Dependency check:', pip_check.stdout.strip())
print('CPU cores visible:', os.cpu_count())


## Stage 0 completion gate

The completed runtime path is:

```text
requested count in {1, 2}
    -> dynamic worktree and source resolution
    -> discovered physical GPU inventory and eligibility checks
    -> selected UUIDs
    -> logical CUDA devices
    -> per-device tensor probes
    -> pushed HEAD and UDLM-base ancestry provenance
```

A successful run demonstrates that exactly `NUM_GPUS` eligible cards became
visible, every logical device completed a tensor test, imports came from this
worktree, `HEAD` matches its upstream-tracking commit, and full UDLM base commit
`02595d1ecf994bbd432ec67ef21b0d56ed97918d` is an ancestor. Versions, seeds, branch, tracking ref, and
dirty paths are recorded.

It does not guarantee that a card remains below the utilization threshold, that the local
upstream-tracking ref was refreshed from the network moments ago, or that an
A6000 reproduces A100 timing. It also does not distribute a model or prove
scientific correctness.

### Comprehension checkpoint

1. Is `NUM_GPUS` a count or a physical card ID?
2. If physical GPU 5 is selected first, what logical CUDA name does it receive?
3. Why must `CUDA_VISIBLE_DEVICES` be set before importing PyTorch?
4. Why is base-commit ancestry checked instead of requiring `HEAD` equality?
5. What does a dirty-path record distinguish from the pushed commit?
6. What safety limitation remains after both GPU snapshots pass?

Stop here until the physical-versus-logical mapping and the limits of the
safety and provenance guards are clear.


# Stage 1 - SAFE V1 representation and tokenization

## Goal and paper connection

This stage converts a molecular graph into the exact discrete input used by the paper's GenMol V1 path.

Section 3.2 describes SAFE as a sequence of SMILES-like fragment blocks. BRICS cuts define the blocks, dots separate them, and paired attachment labels preserve cut bonds. Appendix D.2 fixes the model vocabulary at `K = 1880` and the maximum sequence length at 256 positions.

### Why SAFE improves locality without guaranteeing it

A SMILES traversal can place two bonded or spatially nearby atoms far apart in the token sequence. SAFE groups atoms into chemically meaningful fragments, so many relationships inside one fragment become closer in sequence space. This alleviates the problem but does not eliminate it: atoms in different dot-separated fragments may still be far apart, and block order changes token distance even when decoding gives the same molecular graph.

For example, two atoms inside one BRICS fragment are usually represented nearby. If their bond crosses a BRICS cut, their fragments may be separated in the SAFE string. Paired attachment labels preserve how to reconnect the bond; they do not force the two fragments to be adjacent in the sequence.

### BRICS training fragments versus later remasking fragments

This cell uses the BRICS-based representation used for the training examples. A later remasking stage will distinguish two additional fragmentations: R-vocab makes up to three random cuts of non-ring single bonds, while R-remask cuts every non-ring single bond. R-remask therefore creates more dots and smaller fragments, but the attachment labels reconnect them into the same molecule. This is a conceptual preview only; no remasking code runs in this cell.

The complete path is:

```text
molecular graph
    -> BRICS fragment blocks
    -> SAFE V1 string
    -> tokenizer tokens
    -> integer token IDs
    -> K-way categorical representation
```

An atom, a fragment block, a tokenizer token, and a model position are different units. One fragment usually spans several tokenizer tokens.

## What the next code cell does

The cell constructs a SAFE converter with BRICS slicing and `ignore_stereo=True`. Encoding is deterministic for these examples: `canonical=True`, `randomize=False`, and `allow_empty=True`.

It defines four operations:

1. parse a SMILES string with RDKit;
2. remove stereochemistry and compute a canonical graph identity;
3. encode the molecule as SAFE V1;
4. strictly decode SAFE and compare the recovered graph.

Ethanol has no BRICS-cuttable bond and remains one block. Aspirin and celecoxib produce several dot-separated blocks. Lactic acid shows that equality is checked after stereochemistry is removed.

Expected output is a table containing each input, SAFE string, BRICS-cut count, block count, formula, charge, and roundtrip result. Assertions require every strict roundtrip to recover the same stereo-stripped graph, ethanol to remain one block, and at least one example to have several blocks.

**Paper/repository/notebook distinction:** `ignore_stereo=True` mirrors the released user-data preprocessing policy. The notebook uses five deterministic in-memory examples and records the large dataset revision without downloading or auditing the billion-row corpus.

This cell tests graph representation only. It does not test block permutations, tokenizer behavior, diffusion, or model invariance.

### Comprehension checkpoint

1. What information do paired SAFE attachment labels preserve after a BRICS cut?
2. Why is a fragment block not the same unit as a tokenizer token?
3. What molecular equality is tested after `ignore_stereo=True`?
4. Why can ethanol legitimately contain only one SAFE block?
5. What do the roundtrip assertions prove, and what do they not prove?
5. Why can atoms in different SAFE fragments still be far apart in the token sequence?
6. Why can R-remask create more dots and smaller fragments without changing the decoded molecule?


In [ ]:
from importlib.metadata import version as package_version
from itertools import permutations
from pathlib import Path
import hashlib
import os
import random

import pandas as pd
import torch
import torch.nn.functional as F
import safe as sf
from huggingface_hub import hf_hub_download
from rdkit import Chem, RDLogger
from rdkit.Chem import BRICS, rdMolDescriptors
from safe.tokenizer import SAFETokenizer, split as safe_split

RDLogger.DisableLog("rdApp.*")

SAFE_MODE = "paper_v1"
SAFE_MOL_VERSION = "0.1.14"
TOKENIZER_REPO = "datamol-io/safe-gpt"
TOKENIZER_REVISION = "3d5fa0988383e898d5ac5db7cd52bf715bc37061"
TOKENIZER_SHA256 = "0db5f4dbdc7e8ff759e98483759611a426e187ee7f3f0a91edc8800abe7bf140"
SAFE_DATASET_V1_REVISION = "b83175cd7394e7a4027478a35b2f9d1dda3ac62f"
MODEL_VOCAB_SIZE = 1880
MAX_MODEL_POSITIONS = 256

assert package_version("safe-mol") == SAFE_MOL_VERSION
safe_converter = sf.SAFEConverter(slicer="brics", ignore_stereo=True)


def parse_smiles(smiles: str) -> Chem.Mol:
    if not isinstance(smiles, str) or not smiles.strip():
        raise ValueError("SMILES must be a non-empty string.")
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"RDKit could not parse SMILES: {smiles!r}")
    return mol


def canonical_graph(smiles: str) -> str:
    """Canonical graph identity after removing stereo, matching repo preprocessing."""
    mol = Chem.Mol(parse_smiles(smiles))
    Chem.RemoveStereochemistry(mol)
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=False)


def brics_cut_bonds(smiles: str) -> list[tuple[int, int]]:
    mol = parse_smiles(smiles)
    return [tuple(int(i) for i in atom_pair) for atom_pair, _ in BRICS.FindBRICSBonds(mol)]


def smiles_to_safe_v1(smiles: str) -> str:
    parse_smiles(smiles)
    return safe_converter.encoder(
        smiles,
        canonical=True,
        randomize=False,
        allow_empty=True,
    )


def safe_to_canonical_v1(safe_text: str, *, fix: bool = False) -> str:
    """Decode through SAFE directly so fix=False really means no repair."""
    if not isinstance(safe_text, str) or not safe_text.strip():
        raise ValueError("SAFE must be a non-empty string.")
    decoded = sf.decode(
        safe_text, canonical=False, fix=fix, ignore_errors=True
    )
    if not decoded:
        raise ValueError(f"SAFE did not decode: {safe_text!r}")
    return canonical_graph(decoded)


def graph_diagnostics(smiles: str) -> dict:
    mol = parse_smiles(smiles)
    return {
        "formula": rdMolDescriptors.CalcMolFormula(mol),
        "heavy_atoms": mol.GetNumHeavyAtoms(),
        "formal_charge": Chem.GetFormalCharge(mol),
    }


EXAMPLES = [
    ("ethanol", "CCO", "valid molecule with no BRICS-cuttable bond"),
    ("aspirin", "CC(=O)Oc1ccccc1C(=O)O", "ordinary drug-like fragmentation"),
    (
        "celecoxib",
        "Cc1ccc(-c2cc(C(F)(F)F)nn2-c2ccc(S(N)(=O)=O)cc2)cc1",
        "multi-fragment permutation test",
    ),
    ("glycine_zwitterion", "[NH3+]CC(=O)[O-]", "formal-charge preservation"),
    ("lactic_acid_stereo", "C[C@H](O)C(=O)O", "stereo policy demonstration"),
]

representation_rows = []
for name, smiles, purpose in EXAMPLES:
    source_graph = canonical_graph(smiles)
    safe_text = smiles_to_safe_v1(smiles)
    decoded_graph = safe_to_canonical_v1(safe_text, fix=False)
    diag = graph_diagnostics(decoded_graph)
    representation_rows.append(
        {
            "name": name,
            "input_smiles": smiles,
            "safe_v1": safe_text,
            "brics_cuts": len(brics_cut_bonds(smiles)),
            "fragment_blocks": len(safe_text.split(".")),
            "roundtrip_graph_equal": decoded_graph == source_graph,
            "formula": diag["formula"],
            "formal_charge": diag["formal_charge"],
            "stereo_in_input": "@" in smiles,
            "purpose": purpose,
        }
    )

representation_df = pd.DataFrame(representation_rows)
assert representation_df["roundtrip_graph_equal"].all()
assert int(representation_df.loc[representation_df["name"] == "ethanol", "fragment_blocks"].iloc[0]) == 1
assert representation_df["fragment_blocks"].max() > 1

print(
    {
        "SAFE_MODE": SAFE_MODE,
        "safe_mol": package_version("safe-mol"),
        "rdkit": package_version("rdkit"),
        "datamol": package_version("datamol"),
        "dataset_revision_recorded_not_downloaded": SAFE_DATASET_V1_REVISION,
    }
)
display(representation_df)

## Stage 1.2 - Permute complete SAFE fragment blocks

SAFE preserves molecular identity when complete dot-separated blocks are reordered together with their attachment labels. The next cell tests this chemical property.

If a SAFE string has blocks:

```text
F1.F2.F3
```

then `F3.F1.F2` can be a different string that decodes to the same molecular graph. Tokens inside a block are not rearranged, and attachment labels are not deleted.

The current celecoxib example has four blocks, so the code tests all `4! = 24` block orders. For more than six blocks it would use a deterministic sample to avoid a factorial explosion.

Expected output shows several distinct SAFE strings and `same_molecular_graph=True` for every tested order. Assertions require all orders to decode to the same stereo-stripped graph and require more than one distinct string.

This proves chemical identity under complete-block permutation. It does not prove that BERT assigns the same probability to each ordering. BERT later uses absolute position embeddings, so model invariance is a separate empirical question.

### Comprehension checkpoint

1. What must move together when fragment blocks are permuted?
2. Why can the SAFE strings differ while the decoded graph remains equal?
3. Why does the code avoid enumerating every order for many blocks?
4. Does this test establish neural-model permutation invariance?


In [ ]:
celecoxib_row = representation_df.loc[representation_df["name"] == "celecoxib"].iloc[0]
celecoxib_safe = celecoxib_row["safe_v1"]
celecoxib_target = canonical_graph(celecoxib_row["input_smiles"])
celecoxib_blocks = celecoxib_safe.split(".")

if len(celecoxib_blocks) <= 6:
    tested_orders = list(permutations(celecoxib_blocks))
else:
    rng = random.Random(7)
    tested_orders = [tuple(celecoxib_blocks)]
    for _ in range(24):
        order = celecoxib_blocks.copy()
        rng.shuffle(order)
        tested_orders.append(tuple(order))

permutation_rows = []
for order_index, block_order in enumerate(tested_orders):
    permuted_safe = ".".join(block_order)
    decoded_graph = safe_to_canonical_v1(permuted_safe, fix=False)
    permutation_rows.append(
        {
            "order_index": order_index,
            "block_order": block_order,
            "safe_text": permuted_safe,
            "same_molecular_graph": decoded_graph == celecoxib_target,
        }
    )

permutation_df = pd.DataFrame(permutation_rows)
assert len(celecoxib_blocks) > 1
assert permutation_df["same_molecular_graph"].all()
assert permutation_df["safe_text"].nunique() > 1

print(
    f"Tested {len(permutation_df)} complete block orders; "
    f"all decode to the same stereo-stripped molecular graph."
)
display(permutation_df.head(8))

## Stage 1.3 - Load and verify the published SAFE tokenizer

The next cell loads the exact SAFE-GPT tokenizer artifact from a pinned revision and verifies its SHA-256 digest. The tokenizer performs SAFE-aware splitting followed by BPE.

The paper V1 model vocabulary has `K = 1880` entries with these special IDs:

```text
[UNK] = 0
[CLS] = 1
[SEP] = 2
[PAD] = 3
[MASK] = 4
dot    = 11
```

`[CLS]` and `[SEP]` surround each sequence. `[PAD]` fills unused batch positions. `[MASK]` is reserved for diffusion and must not occur in a clean valid sequence.

For ethanol, the clean sequence occupies five positions including `[CLS]` and `[SEP]`. Across the five examples, padding produces integer tensors with shape `[5,55]`.

### Current repository versus this paper-V1 notebook

The current repository's `get_tokenizer()` helper adds the literal tokens `<` and `>` for bracket-SAFE support. This notebook intentionally does not call that augmentation: it pins the paper-V1 artifact and asserts both `tokenizer.vocab_size` and `len(tokenizer)` are exactly 1880. Angle-label V2 syntax is outside this stage.

Expected assertions verify the artifact revision and digest, vocabulary size, special IDs, absence of unknown tokens, exact SAFE-text roundtrip through token IDs, and correct PAD placement. Output includes sequence lengths and a token-level view of the celecoxib blocks.

This cell tokenizes clean SAFE text. It does not corrupt tokens and does not claim that one token equals one atom or one fragment.

### Current repository tokenizer divergence

The current GitHub helper adds < and > after loading the V1 tokenizer. Starting from the paper's 1880-entry base vocabulary, that makes len(tokenizer) = 1882, while the released BERT configuration still declares vocab_size = 1880. This notebook intentionally pins the paper-compatible tokenizer revision and stays at 1880 categories; the two added IDs are outside this Stage 1 model path.

### Comprehension checkpoint

1. Why are `[PAD]` and diffusion `[MASK]` different states?
2. What does each axis of the `[5,55]` integer tensor mean?
3. Why does the notebook assert 1880 entries instead of adding `<` and `>`?
4. What protects the tokenizer artifact from silently changing?
5. Can the presence of dots tell us that each fragment is one tokenizer token?


In [ ]:
hf_cache = Path(os.environ["HF_HOME"]) / "hub"
tokenizer_path = Path(
    hf_hub_download(
        repo_id=TOKENIZER_REPO,
        filename="tokenizer.json",
        revision=TOKENIZER_REVISION,
        cache_dir=str(hf_cache),
    )
)
tokenizer_digest = hashlib.sha256(tokenizer_path.read_bytes()).hexdigest()
assert tokenizer_digest == TOKENIZER_SHA256

safe_tokenizer = SAFETokenizer.load(str(tokenizer_path))
tokenizer = safe_tokenizer.get_pretrained()

expected_special_ids = {
    "unk_token_id": 0,
    "cls_token_id": 1,
    "sep_token_id": 2,
    "pad_token_id": 3,
    "mask_token_id": 4,
}
actual_special_ids = {name: getattr(tokenizer, name) for name in expected_special_ids}

assert tokenizer.vocab_size == MODEL_VOCAB_SIZE
assert len(tokenizer) == MODEL_VOCAB_SIZE
assert actual_special_ids == expected_special_ids
assert tokenizer.convert_tokens_to_ids(".") == 11

safe_texts = representation_df["safe_v1"].tolist()
unpadded = tokenizer(
    safe_texts,
    add_special_tokens=True,
    padding=False,
    truncation=False,
)
model_token_lengths = [len(ids) for ids in unpadded["input_ids"]]

for safe_text, ids in zip(safe_texts, unpadded["input_ids"]):
    assert ids[0] == tokenizer.cls_token_id
    assert ids[-1] == tokenizer.sep_token_id
    assert tokenizer.unk_token_id not in ids
    assert tokenizer.pad_token_id not in ids
    assert tokenizer.mask_token_id not in ids
    decoded_text = tokenizer.decode(
        ids,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    assert decoded_text == safe_text, (decoded_text, safe_text)

token_batch = tokenizer(
    safe_texts,
    add_special_tokens=True,
    padding=True,
    truncation=False,
    return_tensors="pt",
)
token_batch.pop("token_type_ids", None)

padding_positions = token_batch["attention_mask"].eq(0)
assert torch.all(token_batch["input_ids"][padding_positions].eq(tokenizer.pad_token_id))
assert torch.all(token_batch["input_ids"][~padding_positions].ne(tokenizer.pad_token_id))

celecoxib_ids = unpadded["input_ids"][2]
celecoxib_tokens = tokenizer.convert_ids_to_tokens(celecoxib_ids)
fragment_index = None
token_rows = []
for position, (token_text, token_id) in enumerate(zip(celecoxib_tokens, celecoxib_ids)):
    if token_text == "[CLS]":
        fragment_index = 0
        block_label = None
    elif token_text == "[SEP]":
        block_label = None
    elif token_text == ".":
        block_label = fragment_index
        fragment_index += 1
    else:
        block_label = fragment_index
    token_rows.append(
        {
            "position": position,
            "token": token_text,
            "token_id": token_id,
            "fragment_block": block_label,
        }
    )

token_df = pd.DataFrame(token_rows)
representation_df["model_token_length_with_cls_sep"] = model_token_lengths

print(
    {
        "tokenizer_repo": TOKENIZER_REPO,
        "revision": TOKENIZER_REVISION,
        "sha256": tokenizer_digest,
        "vocab_size": tokenizer.vocab_size,
        "special_ids": actual_special_ids,
        "dot_token_id": tokenizer.convert_tokens_to_ids("."),
        "batch_shape": tuple(token_batch["input_ids"].shape),
    }
)
display(representation_df[["name", "fragment_blocks", "model_token_length_with_cls_sep"]])
display(token_df)

## Stage 1.4 - Build categorical tensors and reject unsafe inputs

The next cell defines the batch contract used by the later diffusion stages.

For batch size `B`, padded length `L`, and vocabulary size `K = 1880`:

```text
clean_token_ids: [B,L]       integer category IDs
valid_positions: [B,L]       Boolean non-padding mask
clean_x:         [B,L,K]     Boolean one-hot categories
```

The mathematical relationship is:

```text
clean_x[b,l,k] = 1 exactly when clean_token_ids[b,l] = k
```

The model stores integer IDs; `clean_x` is included to match the paper's categorical notation. For the current examples, `B = 5` and `L = 55`, so `clean_x` has shape `[5,55,1880]`. Every `[B,L]` position must contain exactly one active category. Valid clean positions cannot contain `[PAD]` or `[MASK]`.

The cell also tests failure behavior:

- a 300-block control tokenizes to 601 positions and must be rejected;
- malformed SAFE fails under strict decoding, while optional repair is shown separately;
- V2 angle-label syntax is rejected by the V1 splitter;
- a demonstration replaces one real token by ID 4 while keeping its non-padding attention value equal to 1.

### Released repository versus notebook safety

The current repository collator uses `truncation=True` with `max_length=256`. This notebook deliberately tokenizes without truncation, measures the full length, and rejects anything above 256. The rejection is a notebook safety divergence: silent truncation can split a SAFE attachment-pair structure, so we make the data problem explicit before training. It is not a claim that the released collator rejects overlength rows.

Expected output reports integer and one-hot shapes, valid-position count, special IDs, the overflow message, strict-versus-repair behavior, and a table of safely rejected inputs. No forward diffusion is performed.

### Comprehension checkpoint

1. Why does `clean_x` need a third axis of length 1880?
2. How can integer IDs and one-hot vectors encode the same categorical state?
3. Why is the attention value still 1 at a real position changed to `[MASK]`?
4. Why does this notebook reject overlength input instead of matching repository truncation?
5. Which failure test separates strict decoding from optional repair?


In [ ]:
def collate_safe_v1(safe_sequences: list[str]) -> dict[str, torch.Tensor]:
    if not safe_sequences:
        raise ValueError("At least one SAFE sequence is required.")

    raw = tokenizer(
        safe_sequences,
        add_special_tokens=True,
        padding=False,
        truncation=False,
    )
    lengths = [len(ids) for ids in raw["input_ids"]]
    overflow = {
        index: length
        for index, length in enumerate(lengths)
        if length > MAX_MODEL_POSITIONS
    }
    if overflow:
        raise ValueError(
            f"Tokenized sequences exceed {MAX_MODEL_POSITIONS} positions: {overflow}. "
            "Reject before training; silent truncation can break attachment pairs."
        )

    batch = tokenizer(
        safe_sequences,
        add_special_tokens=True,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )
    batch.pop("token_type_ids", None)
    return batch


model_batch = collate_safe_v1(safe_texts)
clean_token_ids = model_batch["input_ids"]
valid_positions = model_batch["attention_mask"].bool()
clean_x = F.one_hot(clean_token_ids, num_classes=MODEL_VOCAB_SIZE).to(torch.bool)

assert clean_x.shape == (*clean_token_ids.shape, MODEL_VOCAB_SIZE)
assert torch.all(clean_x.sum(dim=-1).eq(1))
assert tokenizer.mask_token_id not in clean_token_ids[valid_positions]
assert torch.all(clean_token_ids[~valid_positions].eq(tokenizer.pad_token_id))

mask_demo_ids = clean_token_ids[0].clone()
first_chemical_position = 1
original_token_id = int(mask_demo_ids[first_chemical_position])
mask_demo_ids[first_chemical_position] = tokenizer.mask_token_id
assert model_batch["attention_mask"][0, first_chemical_position] == 1

long_control = ".".join(["C"] * 300)
try:
    collate_safe_v1([long_control])
except ValueError as exc:
    overflow_message = str(exc)
else:
    raise AssertionError("The >256-position control was not rejected.")


def expected_failure(label: str, operation) -> dict:
    try:
        operation()
    except Exception as exc:
        return {
            "case": label,
            "failed_safely": True,
            "exception": type(exc).__name__,
            "message": str(exc)[:160],
        }
    return {
        "case": label,
        "failed_safely": False,
        "exception": None,
        "message": "Unexpectedly accepted.",
    }


malformed_demo = "C1.C"
strict_malformed_decode = sf.decode(
    malformed_demo, canonical=False, fix=False, ignore_errors=True
)
repaired_malformed_decode = sf.decode(
    malformed_demo, canonical=False, fix=True, ignore_errors=True
)
assert strict_malformed_decode is None
assert repaired_malformed_decode is not None

negative_rows = [
    expected_failure("empty SMILES", lambda: smiles_to_safe_v1("")),
    expected_failure("invalid SMILES", lambda: smiles_to_safe_v1("C1CC")),
    expected_failure(
        "malformed SAFE with unmatched closure",
        lambda: safe_to_canonical_v1("C1.C", fix=False),
    ),
    expected_failure(
        "V2 angle syntax passed to V1 splitter",
        lambda: safe_split("C<1>.O<1>"),
    ),
]
negative_df = pd.DataFrame(negative_rows)
assert negative_df["failed_safely"].all()

print(
    {
        "integer_id_shape": tuple(clean_token_ids.shape),
        "paper_one_hot_shape": tuple(clean_x.shape),
        "K": MODEL_VOCAB_SIZE,
        "valid_positions": int(valid_positions.sum()),
        "padding_id": tokenizer.pad_token_id,
        "mask_id": tokenizer.mask_token_id,
        "mask_demo": {
            "position": first_chemical_position,
            "original_token": tokenizer.convert_ids_to_tokens(original_token_id),
            "replacement": tokenizer.mask_token,
            "attention_remains": int(model_batch["attention_mask"][0, first_chemical_position]),
        },
        "overflow_guard": overflow_message,
        "repair_is_explicit": {
            "input": malformed_demo,
            "strict_decode": strict_malformed_decode,
            "repair_decode": repaired_malformed_decode,
        },
    }
)
display(negative_df)

## Stage 1 completion gate

The completed representation path is:

```text
molecular graph
    -> BRICS blocks
    -> SAFE V1
    -> pinned 1880-entry tokenizer
    -> integer IDs [B,L]
    -> categorical vectors [B,L,1880]
```

A successful run demonstrates that:

- valid molecules strictly roundtrip to the same stereo-stripped graph;
- complete SAFE block permutations preserve chemical identity;
- the pinned tokenizer has 1880 entries and special IDs `UNK/CLS/SEP/PAD/MASK = 0/1/2/3/4`;
- clean SAFE strings roundtrip through token IDs without unknown tokens;
- PAD and diffusion `[MASK]` are distinct;
- integer IDs and one-hot vectors encode the same category;
- malformed, V2-incompatible, and overlength inputs fail before training.

The equality does not preserve stereochemistry because the preprocessing policy uses `ignore_stereo=True`. Chemical block-permutation invariance does not imply BERT probability invariance. The current repository adds `<` and `>` (so its tokenizer length is 1882) and truncates at 256; this paper-V1 notebook keeps 1880 entries and rejects overflow deliberately. The billion-row training dataset has not been downloaded or audited.

### Comprehension checkpoint

1. What is the difference between an atom, a SAFE fragment block, and a tokenizer token?
2. How can two SAFE block orders decode to the same molecular graph?
3. If IDs have shape `[5,55]`, why does the one-hot tensor have shape `[5,55,1880]`?
4. Why are `[PAD]` and `[MASK]` different?
5. Why does the notebook reject sequences above 256 while the current repository collator truncates them?
6. What kind of molecular equality remains after `ignore_stereo=True`?

Stop here until you can narrate the complete graph-to-categorical-input path and its limitations.


# Stage 2 - Forward absorbing-state masked diffusion

## Lesson 2.1 - Define the released noise schedule

### Purpose and intuition

Stage 1 produced clean SAFE token IDs $x=(x^1,\ldots,x^L)$. Stage 2 moves from clean time 0 toward noisy time 1 by replacing tokens with `[MASK]`. Equation (1) defines one position:

$$
q(z_t^l\mid x^l)=\operatorname{Cat}\left(z_t^l;\alpha_t x^l+(1-\alpha_t)m\right).
\tag{1}
$$

$\alpha_t$ is the clean-token survival probability; $1-\alpha_t$ is the masking probability. The released log-linear schedule is

$$
\sigma(t)=-\log(1-(1-\epsilon)t),\qquad
\alpha_t=e^{-\sigma(t)}=1-(1-\epsilon)t,
$$

with $\epsilon=10^{-3}$. At $t=0.75$, $\alpha_t=0.25075$ and $P(\mathrm{MASK})=0.74925$.

### What the code below does and expected evidence

The code pins BioNeMo MoCo 0.0.2.1, validates token tensors and times, implements the schedule, and compares it with `LogLinearExpNoiseTransform`. The table should show decreasing survival and increasing masking. It also separates the paper endpoint $\alpha_1=0$ from the released numerical endpoint $\alpha_1=0.001$. Token ID 4 is `[MASK]`; the paper's $K$-th-category label is only a relabeling.

### Scope

This lesson defines schedule probabilities only. It does not sample $z_t$, reverse diffusion, run BERT, or compute a loss.

### Comprehension checkpoint

1. Is $\alpha_t$ the survival probability or masking probability?
2. What is $P(\mathrm{MASK})$ when $\alpha_t=0.75$?
3. Why does the released process retain probability 0.001 on the clean token at $t=1$?


In [ ]:
from bionemo.moco.distributions.prior import DiscreteMaskedPrior
from bionemo.moco.distributions.time import UniformTimeDistribution
from bionemo.moco.interpolants import MDLM
from bionemo.moco.schedules.noise.continuous_noise_transforms import (
    LogLinearExpNoiseTransform,
)

BIONEMO_MOCO_VERSION = "0.0.2.1"
NOISE_EPS = 1.0e-3
TIME_SAMPLING_EPS = 1.0e-3

assert package_version("bionemo-moco") == BIONEMO_MOCO_VERSION
assert clean_token_ids.dtype == torch.int64
assert clean_token_ids.ndim == 2
assert clean_x.to(torch.uint8).argmax(dim=-1).equal(clean_token_ids)
assert valid_positions.dtype == torch.bool
assert valid_positions.shape == clean_token_ids.shape
assert torch.all(clean_token_ids[~valid_positions].eq(tokenizer.pad_token_id))
assert not torch.any(clean_token_ids[valid_positions].eq(tokenizer.pad_token_id))
assert not torch.any(clean_token_ids[valid_positions].eq(tokenizer.mask_token_id))
assert not torch.any(clean_token_ids[valid_positions].eq(tokenizer.unk_token_id))
assert torch.all(clean_token_ids[:, 0].eq(tokenizer.cls_token_id))

sequence_lengths = model_batch["attention_mask"].sum(dim=1)
last_valid_ids = clean_token_ids.gather(1, (sequence_lengths - 1).unsqueeze(1)).squeeze(1)
assert torch.all(last_valid_ids.eq(tokenizer.sep_token_id))
assert tokenizer.mask_token_id == 4
assert tokenizer.mask_token_id != MODEL_VOCAB_SIZE - 1


def validate_continuous_time(t) -> torch.Tensor:
    time = torch.as_tensor(t, dtype=torch.float64)
    if time.numel() == 0:
        raise ValueError("Time must contain at least one value.")
    if not torch.isfinite(time).all():
        raise ValueError("Time must be finite.")
    if torch.any((time < 0) | (time > 1)):
        raise ValueError("Continuous diffusion time must be in [0, 1].")
    return time


def loglinear_schedule(t, *, eps: float = NOISE_EPS):
    if not 0.0 <= eps < 1.0:
        raise ValueError("Schedule epsilon must satisfy 0 <= eps < 1.")
    time = validate_continuous_time(t)
    sigma = -torch.log1p(-(1.0 - eps) * time)
    alpha = torch.exp(-sigma)
    p_mask = 1.0 - alpha
    return sigma, alpha, p_mask


schedule_times = torch.tensor([0.0, 0.25, 0.50, 0.75, 1.0], dtype=torch.float64)
repo_sigma, repo_alpha, repo_p_mask = loglinear_schedule(schedule_times)
ideal_sigma, ideal_alpha, ideal_p_mask = loglinear_schedule(schedule_times, eps=0.0)

assert torch.allclose(repo_alpha, 1.0 - (1.0 - NOISE_EPS) * schedule_times)
assert torch.all(torch.diff(repo_alpha) <= 0)
assert torch.all(torch.diff(repo_p_mask) >= 0)
assert repo_alpha[0] == 1.0 and repo_p_mask[0] == 0.0
assert torch.isclose(repo_alpha[-1], torch.tensor(NOISE_EPS, dtype=torch.float64))
assert ideal_alpha[0] == 1.0 and ideal_alpha[-1] == 0.0
assert torch.isinf(ideal_sigma[-1])

oracle_schedule = LogLinearExpNoiseTransform()
oracle_sigma = oracle_schedule.calculate_sigma(schedule_times)
assert torch.allclose(repo_sigma, oracle_sigma, atol=1e-12, rtol=1e-12)

schedule_df = pd.DataFrame(
    {
        "t": schedule_times.tolist(),
        "repo_sigma": repo_sigma.tolist(),
        "repo_alpha_survival": repo_alpha.tolist(),
        "repo_p_mask": repo_p_mask.tolist(),
        "ideal_alpha_eps0": ideal_alpha.tolist(),
        "ideal_p_mask_eps0": ideal_p_mask.tolist(),
    }
)

print(
    {
        "bionemo_moco": package_version("bionemo-moco"),
        "mask_id": tokenizer.mask_token_id,
        "paper_Kth_is_only_relabeling": tokenizer.mask_token_id != MODEL_VOCAB_SIZE - 1,
        "noise_eps": NOISE_EPS,
        "time_sampling_eps": TIME_SAMPLING_EPS,
        "repo_terminal_mask_probability": float(repo_p_mask[-1]),
        "paper_ideal_terminal_mask_probability": float(ideal_p_mask[-1]),
    }
)
display(schedule_df)

## Lesson 2.2 - Expand Equation (1) into categorical probabilities

### Purpose and intuition

Equation (1) is a categorical distribution, not a soft token sent to BERT. For an ordinary token $i$,

$$
P(z_t=i\mid x=i)=\alpha_t,\qquad
P(z_t=m\mid x=i)=1-\alpha_t,
$$

and every unrelated token has probability zero. If the input is already $m$, both terms point to $m$, so `[MASK]` is absorbing.

The cumulative transition matrix is

$$
\overline Q_t=\alpha_t I+(1-\alpha_t)\mathbf 1m^\top.
$$

For $s<t$, transitions compose using $\alpha_{t\mid s}=\alpha_t/\alpha_s$.

For example, with clean token `C` and $\alpha_t=0.75$:

```text
P(C)=0.75, P([MASK])=0.25, P(any other token)=0
```

### What the code below does and expected evidence

The code builds a four-category teaching matrix, proves the mask row is a point mass, checks transition composition, and constructs real probability tensors with shape `[B, L, 1880]`. Every row should sum to one, and only the original token and `[MASK]` should have nonzero mass.

### Scope

This lesson constructs probability vectors only. No hard noisy IDs are sampled.

### Expected number of masked tokens

For a length-$L$ sequence, the forward process factorizes across positions, so

$$
\mathbb E[N_{\mathrm{mask}}]=L(1-\alpha_t).
$$

With $L=20$ and $\alpha_t=0.75$, each token is masked with probability $1-0.75=0.25$, and

$$
\mathbb E[N_{\mathrm{mask}}]=20(1-0.75)=20(0.25)=5.
$$

Five is the expected count across repeated samples, not a guarantee that one sampled sequence contains exactly five masks.

### Comprehension checkpoint

1. Can forward corruption change `C` directly into `N`?
2. Why does an input `[MASK]` remain `[MASK]` with probability one?
3. What does the last axis of `[B, L, 1880]` represent?


In [ ]:
def absorbing_transition(alpha: float, num_classes: int, mask_id: int) -> torch.Tensor:
    if not 0.0 <= alpha <= 1.0:
        raise ValueError("Alpha must be in [0, 1].")
    if not 0 <= mask_id < num_classes:
        raise ValueError("Mask ID must be inside the vocabulary.")
    transition = alpha * torch.eye(num_classes, dtype=torch.float64)
    transition[:, mask_id] += 1.0 - alpha
    return transition


toy_alpha = 0.75
toy_mask_id = 1
toy_transition = absorbing_transition(toy_alpha, num_classes=4, mask_id=toy_mask_id)
assert torch.allclose(toy_transition.sum(dim=-1), torch.ones(4, dtype=torch.float64))
assert torch.equal(
    toy_transition[toy_mask_id],
    F.one_hot(torch.tensor(toy_mask_id), num_classes=4).to(torch.float64),
)

alpha_s, alpha_t = 0.80, 0.30
conditional_alpha_t_given_s = alpha_t / alpha_s
assert torch.allclose(
    absorbing_transition(alpha_s, 4, toy_mask_id)
    @ absorbing_transition(conditional_alpha_t_given_s, 4, toy_mask_id),
    absorbing_transition(alpha_t, 4, toy_mask_id),
)


def expand_batch_times(t, batch_size: int, device: torch.device) -> torch.Tensor:
    time = validate_continuous_time(t).to(device)
    if time.ndim == 0:
        return time.expand(batch_size)
    if time.shape != (batch_size,):
        raise ValueError(
            f"Time must be scalar or shape ({batch_size},), received {tuple(time.shape)}."
        )
    return time


def categorical_forward_probabilities(
    x0_ids: torch.Tensor,
    t,
    *,
    eps: float = NOISE_EPS,
) -> torch.Tensor:
    if x0_ids.ndim != 2:
        raise ValueError("Token IDs must have shape [batch, length].")
    if x0_ids.dtype.is_floating_point or x0_ids.dtype == torch.bool:
        raise TypeError("Token IDs must use an integer dtype.")
    if torch.any((x0_ids < 0) | (x0_ids >= MODEL_VOCAB_SIZE)):
        raise ValueError("Token ID is outside the V1 vocabulary.")
    time = expand_batch_times(t, x0_ids.shape[0], x0_ids.device)
    _, alpha, _ = loglinear_schedule(time, eps=eps)
    x0_one_hot = F.one_hot(x0_ids.to(torch.int64), MODEL_VOCAB_SIZE).to(torch.float64)
    mask_one_hot = F.one_hot(
        torch.tensor(tokenizer.mask_token_id, device=x0_ids.device),
        MODEL_VOCAB_SIZE,
    ).to(torch.float64)
    return alpha[:, None, None] * x0_one_hot + (1.0 - alpha[:, None, None]) * mask_one_hot


probe_token_id = tokenizer.convert_tokens_to_ids("C")
probe_ids = torch.full((len(schedule_times), 1), probe_token_id, dtype=torch.int64)
probe_q = categorical_forward_probabilities(probe_ids, schedule_times)
assert torch.allclose(probe_q.sum(dim=-1), torch.ones_like(probe_q[..., 0]))

probe_original_probability = probe_q[:, 0, probe_token_id]
probe_mask_probability = probe_q[:, 0, tokenizer.mask_token_id]
probe_other_probability = 1.0 - probe_original_probability - probe_mask_probability
assert torch.allclose(probe_original_probability, repo_alpha)
assert torch.allclose(probe_mask_probability, repo_p_mask)
assert torch.allclose(probe_other_probability, torch.zeros_like(probe_other_probability), atol=1e-12)

mask_probe_ids = torch.full_like(probe_ids, tokenizer.mask_token_id)
mask_probe_q = categorical_forward_probabilities(mask_probe_ids, schedule_times)
expected_absorbing = F.one_hot(
    mask_probe_ids, MODEL_VOCAB_SIZE
).to(torch.float64)
assert torch.equal(mask_probe_q, expected_absorbing)

full_q = categorical_forward_probabilities(clean_token_ids, schedule_times)
assert full_q.shape == (*clean_token_ids.shape, MODEL_VOCAB_SIZE)
assert torch.allclose(full_q.sum(dim=-1), torch.ones_like(full_q[..., 0]))

probe_probability_df = pd.DataFrame(
    {
        "t": schedule_times.tolist(),
        "clean_token": tokenizer.convert_ids_to_tokens(probe_token_id),
        "P(original)": probe_original_probability.tolist(),
        "P(MASK)": probe_mask_probability.tolist(),
        "P(any_other_token)": probe_other_probability.tolist(),
        "nonzero_categories": (probe_q[:, 0] > 1e-12).sum(dim=-1).tolist(),
    }
)

display(
    pd.DataFrame(
        toy_transition.numpy(),
        index=["state_0", "MASK_state_1", "state_2", "state_3"],
        columns=["to_0", "to_MASK_1", "to_2", "to_3"],
    )
)
display(probe_probability_df)

## Lesson 2.3 - Sample a hard noisy SAFE sequence

### Purpose and intuition

BERT receives integer token IDs, not Equation (1)'s probability vectors. For each stored token, draw $u\sim U[0,1)$ and set

$$
z_t^l=\begin{cases}
m,&u<1-\alpha_t,\\
x^l,&\text{otherwise}.
\end{cases}
$$

For example:

```text
clean: [CLS] C C O [SEP] [PAD]
noisy: [CLS] C [MASK] O [SEP] [PAD]
```

The sampled output keeps shape `[B, L]` and contains only original IDs or `[MASK]`.

### What the code below does and expected evidence

The code implements reproducible Bernoulli masking and compares it exactly with `MDLM.forward_process`. With `corruptible_positions=None`, it reproduces the released raw policy: every stored slot, including `[CLS]`, `[SEP]`, and PAD, is eligible. The molecule-view policy preserves PAD. Both policies must agree at valid positions. A raw PAD slot may change to `[MASK]`, but its unchanged attention value remains zero, so later BERT and loss code ignore it.

### Scope

This lesson samples one marginal state $z_t$. It does not construct a coupled path across several times.

#### Special-token policy

The released forward process uses attention_mask as its eligibility mask. Therefore every non-padding position, including CLS/SEP (the tokenizer's BOS/EOS roles), can be corrupted; only PAD is excluded. The notebook also shows a molecule-view policy that protects boundary tokens because it is easier to reason about and decode. That protection is a deliberate teaching deviation. An exact GitHub training run must use the released all-nonpadding policy.

## Comprehension checkpoint

1. Is a sampled $z_t^l$ a probability vector or one hard ID?
2. Why may raw corruption change a PAD ID without making that position valid?
3. Which special valid tokens does the released path allow masking?


In [ ]:
def sample_forward_absorbing(
    x0_ids: torch.Tensor,
    t,
    *,
    eps: float = NOISE_EPS,
    generator: torch.Generator | None = None,
    uniforms: torch.Tensor | None = None,
    corruptible_positions: torch.Tensor | None = None,
):
    if x0_ids.ndim != 2:
        raise ValueError("Token IDs must have shape [batch, length].")
    if x0_ids.dtype.is_floating_point or x0_ids.dtype == torch.bool:
        raise TypeError("Token IDs must use an integer dtype.")
    if torch.any((x0_ids < 0) | (x0_ids >= MODEL_VOCAB_SIZE)):
        raise ValueError("Token ID is outside the V1 vocabulary.")
    if generator is not None and uniforms is not None:
        raise ValueError("Pass either generator or uniforms, not both.")

    time = expand_batch_times(t, x0_ids.shape[0], x0_ids.device)
    _, _, p_mask = loglinear_schedule(time, eps=eps)

    if uniforms is None:
        uniforms = torch.rand(
            x0_ids.shape,
            dtype=torch.float32,
            device=x0_ids.device,
            generator=generator,
        )
    else:
        if uniforms.shape != x0_ids.shape:
            raise ValueError("Injected uniforms must match the token-ID shape.")
        if not uniforms.dtype.is_floating_point:
            raise TypeError("Injected uniforms must be floating point.")
        if not torch.isfinite(uniforms).all():
            raise ValueError("Injected uniforms must be finite.")
        if torch.any((uniforms < 0) | (uniforms > 1)):
            raise ValueError("Injected uniforms must be in [0, 1].")
        uniforms = uniforms.to(x0_ids.device)

    mask_events = uniforms < p_mask[:, None]

    if corruptible_positions is not None:
        if corruptible_positions.shape != x0_ids.shape:
            raise ValueError("corruptible_positions must match the token-ID shape.")
        if corruptible_positions.dtype != torch.bool:
            raise TypeError("corruptible_positions must be Boolean.")
        mask_events = mask_events & corruptible_positions.to(x0_ids.device)

    xt_ids = torch.where(
        mask_events,
        torch.full_like(x0_ids, tokenizer.mask_token_id),
        x0_ids,
    )
    return xt_ids, mask_events, p_mask


fixed_times = schedule_times.clone()
ours_generator = torch.Generator(device="cpu").manual_seed(20260830)
oracle_generator = torch.Generator(device="cpu").manual_seed(20260830)

xt_from_scratch, _, _ = sample_forward_absorbing(
    clean_token_ids,
    fixed_times,
    generator=ours_generator,
)

oracle_mdlm = MDLM(
    time_distribution=UniformTimeDistribution(),
    prior_distribution=DiscreteMaskedPrior(
        num_classes=MODEL_VOCAB_SIZE,
        mask_dim=tokenizer.mask_token_id,
    ),
    noise_schedule=LogLinearExpNoiseTransform(),
    rng_generator=oracle_generator,
)
xt_bionemo = oracle_mdlm.forward_process(clean_token_ids, fixed_times)
assert torch.equal(xt_from_scratch, xt_bionemo)

shared_uniforms = torch.rand(
    clean_token_ids.shape,
    generator=torch.Generator(device="cpu").manual_seed(7),
)
xt_repo_exact, repo_events, _ = sample_forward_absorbing(
    clean_token_ids,
    fixed_times,
    uniforms=shared_uniforms,
)
xt_molecule_view, molecule_events, _ = sample_forward_absorbing(
    clean_token_ids,
    fixed_times,
    uniforms=shared_uniforms,
    corruptible_positions=valid_positions,
)

assert torch.equal(xt_repo_exact[valid_positions], xt_molecule_view[valid_positions])
assert torch.all(xt_molecule_view[~valid_positions].eq(tokenizer.pad_token_id))
assert torch.all(
    (xt_molecule_view[valid_positions] == clean_token_ids[valid_positions])
    | xt_molecule_view[valid_positions].eq(tokenizer.mask_token_id)
)
assert xt_repo_exact.shape == clean_token_ids.shape
assert xt_repo_exact.dtype == clean_token_ids.dtype
assert xt_repo_exact.device == clean_token_ids.device

special_positions = (
    clean_token_ids.eq(tokenizer.cls_token_id)
    | clean_token_ids.eq(tokenizer.sep_token_id)
) & valid_positions
padding_positions = ~valid_positions

sample_rows = []
for row_index, name in enumerate(representation_df["name"].tolist()):
    valid_count = int(valid_positions[row_index].sum())
    sample_rows.append(
        {
            "molecule": name,
            "t": float(fixed_times[row_index]),
            "analytic_P_mask": float(repo_p_mask[row_index]),
            "valid_positions": valid_count,
            "sampled_valid_masks": int(
                molecule_events[row_index][valid_positions[row_index]].sum()
            ),
            "sampled_valid_fraction": float(
                molecule_events[row_index][valid_positions[row_index]].float().mean()
            ),
            "CLS_SEP_masked": int(
                molecule_events[row_index][special_positions[row_index]].sum()
            ),
            "repo_PAD_slots_changed_to_MASK": int(
                repo_events[row_index][padding_positions[row_index]].sum()
            ),
            "molecule_view_PAD_changes": int(
                molecule_events[row_index][padding_positions[row_index]].sum()
            ),
        }
    )

sampling_df = pd.DataFrame(sample_rows)

celecoxib_row_index = 2
clean_probe_tokens = tokenizer.convert_ids_to_tokens(
    clean_token_ids[celecoxib_row_index].tolist()
)
noisy_probe_tokens = tokenizer.convert_ids_to_tokens(
    xt_molecule_view[celecoxib_row_index].tolist()
)
token_corruption_df = pd.DataFrame(
    {
        "position": list(range(clean_token_ids.shape[1])),
        "clean_token": clean_probe_tokens,
        "z_t_token": noisy_probe_tokens,
        "masked_here": molecule_events[celecoxib_row_index].tolist(),
        "attention": model_batch["attention_mask"][celecoxib_row_index].tolist(),
    }
)

print(
    {
        "from_scratch_equals_bionemo_0_0_2_1": True,
        "raw_repo_policy": "all stored tensor positions are eligible",
        "molecule_view_policy": "attention-mask-valid positions are eligible",
        "attention_mask_is_unchanged": True,
    }
)
display(sampling_df)
display(token_corruption_df)

## Lesson 2.4 - Distinguish marginals from one coupled path

### Purpose and intuition

Independent samples at several times are correct marginals $q(z_t\mid x)$, but they are not necessarily one trajectory. Independent redraws can make a position look masked at one snapshot and visible at a later snapshot.

One absorbing path reuses a single threshold $u^l$ at every time:

$$
M_t^l=\mathbf 1[u^l<1-\alpha_t].
$$

Because $1-\alpha_t$ increases with $t$, mask sets are nested. Equivalently, a token visible at $s<t$ survives to $t$ with probability $\alpha_t/\alpha_s$.

For $u=0.60$ under the released schedule:

```text
t=0.50: P(MASK)=0.4995 -> visible
t=0.75: P(MASK)=0.74925 -> MASK
t=1.00: remains MASK
```

### What the code below does and expected evidence

The code reuses one uniform tensor over nine times for both released and ideal schedules. Every later mask set must contain the earlier set. At $t=0$ the state must be clean; for the ideal $\epsilon=0$ schedule, every valid position must be masked at $t=1$.

### Scope

This is a coupled forward path only. It does not reverse or unmask tokens.

### Comprehension checkpoint

1. Why can independent marginal snapshots appear to unmask later?
2. Why can one threshold-coupled path never unmask as $t$ increases?
3. What is $P(z_t=x\mid z_s=x)$ for $s<t$?


In [ ]:
path_times = torch.linspace(0.0, 1.0, 9, dtype=torch.float64)
path_clean = clean_token_ids[2:3]
path_valid = valid_positions[2:3]
path_uniforms = torch.rand(
    path_clean.shape,
    generator=torch.Generator(device="cpu").manual_seed(2718),
)

repo_path_events = []
ideal_path_events = []
path_rows = []

for path_time in path_times:
    repo_state, repo_event, repo_path_p = sample_forward_absorbing(
        path_clean,
        path_time,
        eps=NOISE_EPS,
        uniforms=path_uniforms,
        corruptible_positions=path_valid,
    )
    ideal_state, ideal_event, ideal_path_p = sample_forward_absorbing(
        path_clean,
        path_time,
        eps=0.0,
        uniforms=path_uniforms,
        corruptible_positions=path_valid,
    )
    repo_path_events.append(repo_event)
    ideal_path_events.append(ideal_event)
    path_rows.append(
        {
            "t": float(path_time),
            "repo_expected_P_mask": float(repo_path_p[0]),
            "repo_observed_mask_count": int(repo_event[path_valid].sum()),
            "ideal_expected_P_mask": float(ideal_path_p[0]),
            "ideal_observed_mask_count": int(ideal_event[path_valid].sum()),
            "valid_positions": int(path_valid.sum()),
        }
    )
    assert torch.all(
        (repo_state[path_valid] == path_clean[path_valid])
        | repo_state[path_valid].eq(tokenizer.mask_token_id)
    )
    assert torch.all(
        (ideal_state[path_valid] == path_clean[path_valid])
        | ideal_state[path_valid].eq(tokenizer.mask_token_id)
    )

for earlier, later in zip(repo_path_events, repo_path_events[1:]):
    assert torch.all(~earlier | later)
for earlier, later in zip(ideal_path_events, ideal_path_events[1:]):
    assert torch.all(~earlier | later)

assert not repo_path_events[0].any()
assert not ideal_path_events[0].any()
assert torch.all(ideal_path_events[-1][path_valid])
assert torch.equal(
    sample_forward_absorbing(
        path_clean,
        0.0,
        uniforms=torch.zeros_like(path_uniforms),
        corruptible_positions=path_valid,
    )[0],
    path_clean,
)

trajectory_df = pd.DataFrame(path_rows)
print("Shared per-position uniforms produce nested, absorbing mask sets.")
display(trajectory_df)

## Lesson 2.5 - Sample one training time per sequence

### Purpose and intuition

Training should cover the continuous interval without clustering all batch times together. For batch row $b\in\{0,\ldots,B-1\}$, the released default draws

$$
t_b=\epsilon_{\rm sample}+(1-\epsilon_{\rm sample})\frac{b+u_b}{B},
\qquad u_b\sim U[0,1).
$$

This is one jittered time in each ordered stratum. The code calls it antithetic sampling; ordered stratified jitter is the literal behavior. For $B=4$, the unscaled strata are `[0,.25)`, `[.25,.50)`, `[.50,.75)`, and `[.75,1)`.

### What the code below does and expected evidence

The code implements the formula, compares exact seeded output with `AntitheticUniformTimeDistribution`, maps each time to $\alpha_t$ and $P(\mathrm{MASK})$, and corrupts one real SAFE batch. It should show one time per sequence with shape `[B]` and exact repository agreement.

Schedule epsilon and time-sampling epsilon are both $10^{-3}$ here but have different roles: one prevents complete terminal destruction; the other avoids sampling time zero for the later $1/t$ loss.

### Scope

This lesson chooses times and forward masks only. It does not compute Equation (3).

### Comprehension checkpoint

1. Is one $t$ sampled per token or per sequence?
2. What coverage advantage do ordered strata provide?
3. Why are schedule epsilon and time-sampling epsilon conceptually different?


In [ ]:
from genmol.utils.utils_moco import AntitheticUniformTimeDistribution


def sample_ordered_stratified_times(
    batch_size: int,
    *,
    sampling_eps: float = TIME_SAMPLING_EPS,
    generator: torch.Generator | None = None,
) -> torch.Tensor:
    if not isinstance(batch_size, int) or batch_size <= 0:
        raise ValueError("batch_size must be a positive integer.")
    if not 0.0 <= sampling_eps < 1.0:
        raise ValueError("Time-sampling epsilon must satisfy 0 <= eps < 1.")
    jitter = torch.rand(batch_size, generator=generator)
    unit_interval_times = (
        jitter / batch_size
        + torch.arange(batch_size, dtype=jitter.dtype) / batch_size
    )
    return (1.0 - sampling_eps) * unit_interval_times + sampling_eps


time_seed = 314159
ours_time_generator = torch.Generator(device="cpu").manual_seed(time_seed)
oracle_time_generator = torch.Generator(device="cpu").manual_seed(time_seed)

stratified_times = sample_ordered_stratified_times(
    8,
    generator=ours_time_generator,
)
repo_time_sampler = AntitheticUniformTimeDistribution(
    sampling_eps=TIME_SAMPLING_EPS
)
repo_stratified_times = repo_time_sampler.sample(
    8,
    rng_generator=oracle_time_generator,
)
assert torch.equal(stratified_times, repo_stratified_times)

unit_times = (stratified_times - TIME_SAMPLING_EPS) / (1.0 - TIME_SAMPLING_EPS)
stratum_ids = torch.floor(unit_times * len(stratified_times)).to(torch.int64)
assert torch.equal(stratum_ids, torch.arange(len(stratified_times)))
assert torch.all((stratified_times >= TIME_SAMPLING_EPS) & (stratified_times < 1.0))

_, stratified_alpha, stratified_p_mask = loglinear_schedule(stratified_times)
time_sampling_df = pd.DataFrame(
    {
        "batch_row": list(range(len(stratified_times))),
        "stratum": stratum_ids.tolist(),
        "sampled_t": stratified_times.tolist(),
        "alpha_survival": stratified_alpha.tolist(),
        "P_mask": stratified_p_mask.tolist(),
    }
)

training_demo_times = sample_ordered_stratified_times(
    clean_token_ids.shape[0],
    generator=torch.Generator(device="cpu").manual_seed(99),
)
_, training_demo_events, training_demo_p = sample_forward_absorbing(
    clean_token_ids,
    training_demo_times,
    generator=torch.Generator(device="cpu").manual_seed(100),
    corruptible_positions=valid_positions,
)

training_demo_df = pd.DataFrame(
    {
        "molecule": representation_df["name"].tolist(),
        "sampled_t": training_demo_times.tolist(),
        "analytic_P_mask": training_demo_p.tolist(),
        "sampled_valid_masks": [
            int(training_demo_events[i][valid_positions[i]].sum())
            for i in range(clean_token_ids.shape[0])
        ],
        "valid_positions": valid_positions.sum(dim=1).tolist(),
    }
)

print(
    {
        "from_scratch_times_equal_repository_default": True,
        "one_time_per_sequence": tuple(training_demo_times.shape),
        "time_interval": f"[{TIME_SAMPLING_EPS}, 1)",
    }
)
display(time_sampling_df)
display(training_demo_df)

## Lesson 2.6 - Validate probabilities statistically and reject unsafe inputs

### Purpose and intuition

A single noisy molecule cannot validate a probability law. For $n$ independent mask events with probability $p$,

$$
E[\widehat p]=p,\qquad
\operatorname{SE}(\widehat p)=\sqrt{\frac{p(1-p)}{n}}.
$$

At $t=0.5$, the released schedule gives $p=0.4995$. A length-20 sequence therefore has $20(0.4995)=9.99$ masks in expectation, not exactly 9.99 masks in every draw.

### What the code below does and expected evidence

The code uses 50,000 sequences of length 20, giving one million token trials per tested time. Empirical frequencies must fall inside a five-standard-error tolerance around Equation (1). Equal seeds must reproduce samples; a different seed must change them. Negative tests reject invalid times, shapes, token IDs, epsilon values, corruptible masks, and conflicting random controls.

### Scope

These are forward-process statistical and API checks. They do not test reverse diffusion or learning.

### Comprehension checkpoint

1. Why is the number of masks random even when $t$ is fixed?
2. What should the empirical mask fraction approach as the number of trials grows?
3. Why should passing both injected uniforms and a generator be rejected?


In [ ]:
monte_carlo_rows = []
n_trials = 50_000
n_positions = 20
mc_times = [0.0, 0.10, 0.50, 0.90, 1.0]
mc_clean = torch.full(
    (n_trials, n_positions),
    probe_token_id,
    dtype=torch.int64,
)

for index, mc_time in enumerate(mc_times):
    mc_xt, mc_events, mc_p = sample_forward_absorbing(
        mc_clean,
        mc_time,
        generator=torch.Generator(device="cpu").manual_seed(500 + index),
    )
    expected_probability = float(mc_p[0])
    empirical_probability = float(mc_events.float().mean())
    standard_error = (
        expected_probability
        * (1.0 - expected_probability)
        / (n_trials * n_positions)
    ) ** 0.5
    tolerance = 5.0 * standard_error + 2.0 / (n_trials * n_positions)
    assert abs(empirical_probability - expected_probability) <= tolerance
    assert torch.all(
        (mc_xt == probe_token_id) | mc_xt.eq(tokenizer.mask_token_id)
    )
    monte_carlo_rows.append(
        {
            "t": mc_time,
            "analytic_P_mask": expected_probability,
            "empirical_P_mask": empirical_probability,
            "absolute_error": abs(empirical_probability - expected_probability),
            "five_sigma_tolerance": tolerance,
            "expected_masks_among_20": n_positions * expected_probability,
            "empirical_mean_masks_among_20": float(
                mc_events.sum(dim=1).to(torch.float64).mean()
            ),
        }
    )

repro_clean = torch.full((64, 32), probe_token_id, dtype=torch.int64)
repro_a = sample_forward_absorbing(
    repro_clean,
    0.5,
    generator=torch.Generator(device="cpu").manual_seed(123),
)[0]
repro_b = sample_forward_absorbing(
    repro_clean,
    0.5,
    generator=torch.Generator(device="cpu").manual_seed(123),
)[0]
repro_c = sample_forward_absorbing(
    repro_clean,
    0.5,
    generator=torch.Generator(device="cpu").manual_seed(124),
)[0]
assert torch.equal(repro_a, repro_b)
assert not torch.equal(repro_a, repro_c)


def stage2_expected_failure(label: str, operation) -> dict:
    try:
        operation()
    except Exception as exc:
        return {
            "case": label,
            "failed_safely": True,
            "exception": type(exc).__name__,
            "message": str(exc)[:160],
        }
    return {
        "case": label,
        "failed_safely": False,
        "exception": None,
        "message": "Unexpectedly accepted.",
    }


bad_uniforms = torch.zeros_like(clean_token_ids, dtype=torch.float32)
bad_uniforms[0, 0] = 1.1
stage2_negative_rows = [
    stage2_expected_failure("time below zero", lambda: loglinear_schedule(-0.01)),
    stage2_expected_failure("time above one", lambda: loglinear_schedule(1.01)),
    stage2_expected_failure("NaN time", lambda: loglinear_schedule(float("nan"))),
    stage2_expected_failure(
        "wrong per-batch time shape",
        lambda: sample_forward_absorbing(clean_token_ids, torch.tensor([0.5, 0.6])),
    ),
    stage2_expected_failure(
        "epsilon outside range",
        lambda: loglinear_schedule(0.5, eps=1.0),
    ),
    stage2_expected_failure(
        "floating token IDs",
        lambda: sample_forward_absorbing(clean_token_ids.to(torch.float32), 0.5),
    ),
    stage2_expected_failure(
        "negative token ID",
        lambda: sample_forward_absorbing(torch.tensor([[-1]], dtype=torch.int64), 0.5),
    ),
    stage2_expected_failure(
        "wrong corruptible mask shape",
        lambda: sample_forward_absorbing(
            clean_token_ids,
            0.5,
            corruptible_positions=valid_positions[:, :-1],
        ),
    ),
    stage2_expected_failure(
        "non-Boolean corruptible mask",
        lambda: sample_forward_absorbing(
            clean_token_ids,
            0.5,
            corruptible_positions=valid_positions.to(torch.int64),
        ),
    ),
    stage2_expected_failure(
        "uniform outside [0, 1]",
        lambda: sample_forward_absorbing(
            clean_token_ids,
            0.5,
            uniforms=bad_uniforms,
        ),
    ),
    stage2_expected_failure(
        "generator and uniforms together",
        lambda: sample_forward_absorbing(
            clean_token_ids,
            0.5,
            generator=torch.Generator(),
            uniforms=torch.zeros_like(clean_token_ids, dtype=torch.float32),
        ),
    ),
]
stage2_negative_df = pd.DataFrame(stage2_negative_rows)
assert stage2_negative_df["failed_safely"].all()

monte_carlo_df = pd.DataFrame(monte_carlo_rows)
print(
    {
        "monte_carlo_draws_per_time": n_trials * n_positions,
        "same_seed_reproducible": True,
        "different_seed_changes_sample": True,
        "expected_failures_caught": len(stage2_negative_df),
    }
)
display(monte_carlo_df)
display(stage2_negative_df)

## Stage 2 completion gate

Stage 2 is complete only if the notebook demonstrates:

- exact Equation (1) probabilities and an absorbing `[MASK]` state;
- the released log-linear schedule and its $\epsilon=10^{-3}$ endpoint;
- hard integer sampling that exactly matches BioNeMo MoCo 0.0.2.1;
- an explicit distinction between independent marginals and one threshold-coupled path;
- repository-matching ordered stratified time samples;
- empirical agreement over one million token trials per tested time;
- explicit PAD behavior and safe rejection of invalid inputs.

Keep the endpoints separate:

```text
paper:      alpha(1)=0
repository: alpha(1)=0.001
```

No reverse kernel, BERT, loss, optimizer, or generation has been implemented.

### Comprehension checkpoint

1. If $\alpha_t=0.75$, what probabilities are assigned to the original token, `[MASK]`, and every unrelated token?
2. Why is an expected count of five masks not a guarantee of exactly five?
3. Why is Equation (1)'s one-hot blend not a soft BERT input?
4. How do independent marginals differ from one coupled absorbing path?
5. Why can raw corruption alter ignored PAD IDs?
6. What differs between the paper and released terminal schedules?

**Stop here until every answer can be explained without running the code. Do not continue to Stage 3 before the forward process is clear.**


# Stage 3 - Reverse posterior and Equation (2)

## Lesson 3.1 - Derive the exact one-token posterior

### Purpose and intuition

Stage 3 moves from noisier time $t$ to cleaner time $s$, where $0\le s\le t\le1$ and $\alpha_s\ge\alpha_t$. Begin with the exact posterior when the clean token $x\ne m$ is known.

Only three forward paths have nonzero probability:

```text
state at s   state at t   joint probability
x            x            alpha_t
x            MASK         alpha_s - alpha_t
MASK         MASK         1 - alpha_s
```

The path `MASK -> x` is impossible. Conditioning on $z_t=m$ gives

$$
P(z_s=x\mid z_t=m,x)=\frac{\alpha_s-\alpha_t}{1-\alpha_t},\qquad
P(z_s=m\mid z_t=m,x)=\frac{1-\alpha_s}{1-\alpha_t}.
$$

If $z_t=x$, it is copied exactly. With $\alpha_s=0.90$, $\alpha_t=0.75$, and clean token `B`, a current mask gives `P(B)=0.60` and `P(MASK)=0.40`.

### What the code below does and expected evidence

The code implements this scalar Bayes posterior, validates support and zero-evidence cases, and checks normalization. The displayed masked row must be `[0, 0.6, 0.4]`; visible `B` must be a point mass.

### Scope

This is an exact clean-conditioned oracle. There is no predicted $x_\theta$, BERT, or loss.

### Comprehension checkpoint

1. Why is the denominator $1-\alpha_t$?
2. Why is `MASK -> clean` impossible in the forward direction?
3. What happens when the current token is already visible?


In [ ]:
import math

def exact_reverse_posterior_token(
    x0_id: int,
    zt_id: int,
    alpha_s: float,
    alpha_t: float,
    *,
    num_classes: int,
    mask_id: int,
) -> torch.Tensor:
    """Exact q(z_s | z_t, x0) for one absorbing-diffusion token."""
    if isinstance(num_classes, bool) or not isinstance(num_classes, int) or num_classes < 2:
        raise ValueError("num_classes must be an integer of at least 2.")
    for name, token_id in {"x0_id": x0_id, "zt_id": zt_id, "mask_id": mask_id}.items():
        if isinstance(token_id, bool) or not isinstance(token_id, int):
            raise TypeError(f"{name} must be an integer token ID.")
        if not 0 <= token_id < num_classes:
            raise ValueError(f"{name} is outside the vocabulary.")
    if x0_id == mask_id:
        raise ValueError("Strict SUBS assumes a clean token is not the mask token.")

    alpha_s = float(alpha_s)
    alpha_t = float(alpha_t)
    if not math.isfinite(alpha_s) or not math.isfinite(alpha_t):
        raise ValueError("alpha_s and alpha_t must be finite.")
    if not (0.0 <= alpha_t <= alpha_s <= 1.0):
        raise ValueError("Require 0 <= alpha_t <= alpha_s <= 1.")

    probs = torch.zeros(num_classes, dtype=torch.float64)
    if zt_id == x0_id:
        if alpha_t == 0.0:
            raise ValueError("Conditioning on a visible token has zero probability when alpha_t=0.")
        probs[x0_id] = 1.0
    elif zt_id == mask_id:
        if alpha_t == 1.0:
            raise ValueError("Conditioning on a mask has zero probability when alpha_t=1.")
        probs[x0_id] = (alpha_s - alpha_t) / (1.0 - alpha_t)
        probs[mask_id] = (1.0 - alpha_s) / (1.0 - alpha_t)
    else:
        raise ValueError("zt is off the forward support {x0, mask}; the exact posterior is undefined.")

    if not torch.isfinite(probs).all() or torch.any(probs < 0):
        raise AssertionError("Posterior probabilities must be finite and nonnegative.")
    if not torch.allclose(probs.sum(), torch.tensor(1.0, dtype=probs.dtype), atol=1e-12, rtol=0):
        raise AssertionError("Posterior probabilities must sum to one.")
    return probs


toy_exact_masked = exact_reverse_posterior_token(
    x0_id=1, zt_id=2, alpha_s=0.90, alpha_t=0.75, num_classes=3, mask_id=2
)
toy_exact_visible = exact_reverse_posterior_token(
    x0_id=1, zt_id=1, alpha_s=0.90, alpha_t=0.75, num_classes=3, mask_id=2
)
toy_exact_table = pd.DataFrame(
    {
        "category": ["A", "B = clean", "[MASK]"],
        "q(z_s | z_t=MASK, x0=B)": toy_exact_masked.numpy(),
        "q(z_s | z_t=B, x0=B)": toy_exact_visible.numpy(),
    }
)
display(toy_exact_table)
assert torch.allclose(toy_exact_masked, torch.tensor([0.0, 0.6, 0.4], dtype=torch.float64))
assert torch.equal(toy_exact_visible, torch.tensor([0.0, 1.0, 0.0], dtype=torch.float64))

## Lesson 3.2 - Check Bayes with an independent matrix oracle

### Purpose and intuition

An independent derivation guards against repeating the same algebraic mistake. Using Stage 2 transition matrices,

$$
q(z_s\mid z_t,x)=
\frac{\overline Q_s[x,z_s]Q_{t\mid s}[z_s,z_t]}
{\sum_j\overline Q_s[x,j]Q_{t\mid s}[j,z_t]},
$$

with $\alpha_{t\mid s}=\alpha_t/\alpha_s$ when $\alpha_s>0$. This enumerates every possible intermediate $z_s$ and normalizes by the evidence.

For the earlier `B` example, matrix Bayes must independently recover `P(B)=0.60` and `P(MASK)=0.40`.

### What the code below does and expected evidence

The code checks vocabulary sizes 2 through 6, every mask and clean-token ID, and a grid of valid alpha pairs. Every positive-evidence matrix result must equal the closed form; both implementations must reject the same zero-evidence cases.

### Scope

This is a mathematical oracle, not the production batch implementation.

### Comprehension checkpoint

1. What quantity appears in the denominator of matrix Bayes?
2. Why is this check more independent than recomputing the closed formula?
3. What should happen when the observed $z_t$ has zero forward probability?


In [ ]:
def matrix_reverse_posterior_token(
    x0_id: int,
    zt_id: int,
    alpha_s: float,
    alpha_t: float,
    *,
    num_classes: int,
    mask_id: int,
) -> torch.Tensor:
    """Independent Bayes oracle using cumulative and conditional transition matrices."""
    if not (0 <= x0_id < num_classes and 0 <= zt_id < num_classes and 0 <= mask_id < num_classes):
        raise ValueError("Token ID is outside the vocabulary.")
    if x0_id == mask_id:
        raise ValueError("The clean token cannot be mask.")
    if not all(math.isfinite(float(a)) for a in (alpha_s, alpha_t)):
        raise ValueError("Alphas must be finite.")
    if not 0.0 <= alpha_t <= alpha_s <= 1.0:
        raise ValueError("Require 0 <= alpha_t <= alpha_s <= 1.")

    qbar_s = absorbing_transition(float(alpha_s), num_classes, mask_id).to(torch.float64)
    if alpha_s > 0.0:
        q_s_to_t = absorbing_transition(float(alpha_t / alpha_s), num_classes, mask_id).to(torch.float64)
    else:
        q_s_to_t = torch.eye(num_classes, dtype=torch.float64)

    joint_over_zs = qbar_s[x0_id] * q_s_to_t[:, zt_id]
    evidence = joint_over_zs.sum()
    if evidence <= 0:
        raise ValueError("The conditioning event has zero forward probability.")
    return joint_over_zs / evidence


alpha_grid = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]
oracle_matches = 0
zero_evidence_cases = 0

for k in range(2, 7):
    for toy_mask_id in range(k):
        for toy_x0_id in range(k):
            if toy_x0_id == toy_mask_id:
                continue
            for alpha_s_value in alpha_grid:
                for alpha_t_value in [a for a in alpha_grid if a <= alpha_s_value]:
                    for toy_zt_id in range(k):
                        closed_error = matrix_error = None
                        try:
                            closed = exact_reverse_posterior_token(
                                toy_x0_id,
                                toy_zt_id,
                                alpha_s_value,
                                alpha_t_value,
                                num_classes=k,
                                mask_id=toy_mask_id,
                            )
                        except ValueError as exc:
                            closed_error = type(exc)
                        try:
                            matrix = matrix_reverse_posterior_token(
                                toy_x0_id,
                                toy_zt_id,
                                alpha_s_value,
                                alpha_t_value,
                                num_classes=k,
                                mask_id=toy_mask_id,
                            )
                        except ValueError as exc:
                            matrix_error = type(exc)

                        if closed_error or matrix_error:
                            assert closed_error is matrix_error
                            zero_evidence_cases += 1
                        else:
                            assert torch.allclose(closed, matrix, atol=1e-12, rtol=0)
                            oracle_matches += 1

print(
    {
        "closed_form_matrix_matches": oracle_matches,
        "zero_evidence_cases_rejected_by_both": zero_evidence_cases,
        "vocabulary_sizes_checked": "2 through 6",
        "every_mask_id_and_clean_id_permutation_checked": True,
    }
)

## Lesson 3.3 - Convert time pairs into stay and reveal weights

### Purpose and intuition

For a currently masked token, Equation (2) separates the reverse step into

$$
w_{\rm stay}=\frac{1-\alpha_s}{1-\alpha_t},\qquad
w_{\rm reveal}=\frac{\alpha_s-\alpha_t}{1-\alpha_t}.
$$

The weights are nonnegative and sum to one. Under the released schedule $\alpha_u=1-(1-\epsilon)u$, they simplify for $t>0$ to

$$
w_{\rm stay}=\frac{s}{t},\qquad
w_{\rm reveal}=\frac{t-s}{t}.
$$

At fixed $t=0.75$, $s=0.25$ gives stay $1/3$ and reveal $2/3$. At $s=t$, the zero-length step stays masked with probability one.

### What the code below does and expected evidence

The code maps several $(s,t)$ pairs through the released schedule, computes both weights, and verifies nonnegativity, normalization, and the simplified time ratios. Every table row must have `stay + reveal = 1`.

### Scope

These weights decide whether to reveal. They do not choose which clean token appears.

### Comprehension checkpoint

1. What happens to reveal probability as $s$ moves closer to zero?
2. Why does $s=t$ make no reverse progress?
3. For $s=0.50,t=0.75$, what are the two weights?


In [ ]:
def reverse_mixture_weights(s, t, *, eps: float = NOISE_EPS):
    """Return stay-mask, reveal, alpha_s, alpha_t for repository times s <= t."""
    s_tensor = validate_continuous_time(s).to(torch.float64)
    t_tensor = validate_continuous_time(t).to(torch.float64)
    s_tensor, t_tensor = torch.broadcast_tensors(s_tensor, t_tensor)
    if torch.any(s_tensor > t_tensor):
        raise ValueError("Reverse time must satisfy s <= t.")

    _, alpha_s, _ = loglinear_schedule(s_tensor, eps=eps)
    _, alpha_t, _ = loglinear_schedule(t_tensor, eps=eps)
    denominator = 1.0 - alpha_t
    zero_denominator = denominator == 0
    safe_denominator = torch.where(zero_denominator, torch.ones_like(denominator), denominator)

    stay_masked = (1.0 - alpha_s) / safe_denominator
    reveal = (alpha_s - alpha_t) / safe_denominator
    stay_masked = torch.where(zero_denominator, torch.ones_like(stay_masked), stay_masked)
    reveal = torch.where(zero_denominator, torch.zeros_like(reveal), reveal)

    if torch.any(stay_masked < -1e-12) or torch.any(reveal < -1e-12):
        raise AssertionError("Reverse mixture weights cannot be negative.")
    if not torch.allclose(stay_masked + reveal, torch.ones_like(stay_masked), atol=1e-12, rtol=0):
        raise AssertionError("Reverse mixture weights must sum to one.")
    return stay_masked, reveal, alpha_s, alpha_t


weight_pairs = torch.tensor(
    [[0.00, 0.75], [0.25, 0.75], [0.50, 0.75], [0.75, 0.75]],
    dtype=torch.float64,
)
stay_values, reveal_values, alpha_s_values, alpha_t_values = reverse_mixture_weights(
    weight_pairs[:, 0], weight_pairs[:, 1]
)
weight_table = pd.DataFrame(
    {
        "s": weight_pairs[:, 0].numpy(),
        "t": weight_pairs[:, 1].numpy(),
        "alpha_s": alpha_s_values.numpy(),
        "alpha_t": alpha_t_values.numpy(),
        "P(stay MASK | MASK at t)": stay_values.numpy(),
        "P(reveal | MASK at t)": reveal_values.numpy(),
    }
)
display(weight_table)

positive_t = weight_pairs[:, 1] > 0
assert torch.allclose(
    stay_values[positive_t],
    weight_pairs[positive_t, 0] / weight_pairs[positive_t, 1],
    atol=1e-12,
    rtol=0,
)
assert torch.allclose(
    reveal_values[positive_t],
    (weight_pairs[positive_t, 1] - weight_pairs[positive_t, 0]) / weight_pairs[positive_t, 1],
    atol=1e-12,
    rtol=0,
)

## Lesson 3.4 - Implement batched Equation (2) with strict SUBS

### Purpose and intuition

During generation, the true clean token is unknown. Equation (2) replaces its one-hot vector with a clean-token prediction $x_\theta^l(\mathbf z_t,t)$:

$$
p_\theta(z_s^l\mid\mathbf z_t)=
\begin{cases}
\delta_{z_t^l},&z_t^l\ne m,\\
w_{\rm stay}m+w_{\rm reveal}x_\theta^l,&z_t^l=m.
\end{cases}
\tag{2}
$$

Strict SUBS requires $x_{\theta,m}^l=0$. The schedule chooses whether to reveal; $x_\theta$ chooses which clean token.

Inputs and output have shapes:

```text
z_t IDs              [B, L]
clean prediction     [B, L, K]
reverse probabilities[B, L, K]
```

If $x_\theta(A)=0.60$, $x_\theta(B)=0.40$, stay $=0.40$, and reveal $=0.60$, a current mask gives `A=0.36`, `B=0.24`, and `MASK=0.40`. A visible `B` stays `B` with probability one.

### What the code below does and expected evidence

The code validates shapes, IDs, devices, alphas, finite normalized predictions, and zero mask mass. Every output row must sum to one and reproduce the concrete values above.

### Scope

$x_\theta$ is still an abstract tensor. No logits-producing network exists yet.

### Comprehension checkpoint

1. Which component chooses whether revelation occurs?
2. Which component chooses `A` versus `B` after revelation?
3. Why must the clean predictor assign zero mass to `[MASK]`?


In [ ]:
_INTEGER_DTYPES = {torch.uint8, torch.int8, torch.int16, torch.int32, torch.int64}


def _expand_stage3_batch_value(value, batch_size: int, device, dtype, name: str) -> torch.Tensor:
    value = torch.as_tensor(value, device=device, dtype=dtype)
    if value.ndim == 0:
        value = value.repeat(batch_size)
    elif value.shape != (batch_size,):
        raise ValueError(f"{name} must be scalar or shape ({batch_size},), received {tuple(value.shape)}.")
    if not torch.isfinite(value).all():
        raise ValueError(f"{name} must be finite.")
    return value


def reverse_posterior_from_alphas(
    zt_ids: torch.Tensor,
    clean_prediction: torch.Tensor,
    alpha_s,
    alpha_t,
    *,
    mask_id: int,
):
    """Equation (2) for a batch: probabilities plus stay/reveal weights."""
    if zt_ids.ndim != 2 or zt_ids.dtype not in _INTEGER_DTYPES:
        raise TypeError("zt_ids must be a rank-2 integer tensor.")
    if not torch.is_floating_point(clean_prediction) or clean_prediction.ndim != 3:
        raise TypeError("clean_prediction must be a rank-3 floating probability tensor.")
    if clean_prediction.shape[:2] != zt_ids.shape:
        raise ValueError("clean_prediction leading dimensions must match zt_ids.")
    if clean_prediction.device != zt_ids.device:
        raise ValueError("zt_ids and clean_prediction must be on the same device.")

    batch_size, sequence_length = zt_ids.shape
    num_classes = clean_prediction.shape[-1]
    if isinstance(mask_id, bool) or not isinstance(mask_id, int) or not 0 <= mask_id < num_classes:
        raise ValueError("mask_id is outside the predicted vocabulary.")
    if torch.any(zt_ids < 0) or torch.any(zt_ids >= num_classes):
        raise ValueError("zt_ids contains an out-of-vocabulary token.")
    if not torch.isfinite(clean_prediction).all():
        raise ValueError("clean_prediction must be finite.")
    if torch.any(clean_prediction < 0):
        raise ValueError("clean_prediction cannot contain negative probabilities.")

    atol = 2e-6 if clean_prediction.dtype in (torch.float16, torch.bfloat16, torch.float32) else 1e-10
    row_sums = clean_prediction.sum(dim=-1)
    if not torch.allclose(row_sums, torch.ones_like(row_sums), atol=atol, rtol=0):
        raise ValueError("Every clean_prediction row must sum to one.")
    if torch.any(clean_prediction[..., mask_id].abs() > atol):
        raise ValueError("Strict SUBS requires zero clean-prediction mass on MASK.")

    alpha_s = _expand_stage3_batch_value(
        alpha_s, batch_size, zt_ids.device, clean_prediction.dtype, "alpha_s"
    )
    alpha_t = _expand_stage3_batch_value(
        alpha_t, batch_size, zt_ids.device, clean_prediction.dtype, "alpha_t"
    )
    if torch.any(alpha_s < 0) or torch.any(alpha_s > 1) or torch.any(alpha_t < 0) or torch.any(alpha_t > 1):
        raise ValueError("Alphas must lie in [0, 1].")
    if torch.any(alpha_t > alpha_s):
        raise ValueError("Reverse order requires alpha_s >= alpha_t.")

    denominator = 1.0 - alpha_t
    masked_now = zt_ids == mask_id
    if torch.any(masked_now & (denominator == 0)[:, None]):
        raise ValueError("A masked z_t has zero forward probability when alpha_t=1.")

    safe_denominator = torch.where(denominator == 0, torch.ones_like(denominator), denominator)
    stay_masked = (1.0 - alpha_s) / safe_denominator
    reveal = (alpha_s - alpha_t) / safe_denominator
    stay_masked = torch.where(denominator == 0, torch.ones_like(stay_masked), stay_masked)
    reveal = torch.where(denominator == 0, torch.zeros_like(reveal), reveal)

    mask_vector = F.one_hot(
        torch.tensor(mask_id, device=zt_ids.device), num_classes=num_classes
    ).to(clean_prediction.dtype)
    masked_mixture = (
        reveal[:, None, None] * clean_prediction
        + stay_masked[:, None, None] * mask_vector
    )
    visible_copy = F.one_hot(zt_ids.to(torch.int64), num_classes=num_classes).to(clean_prediction.dtype)
    probabilities = torch.where(masked_now[..., None], masked_mixture, visible_copy)

    if not torch.isfinite(probabilities).all() or torch.any(probabilities < -atol):
        raise AssertionError("Reverse probabilities must be finite and nonnegative.")
    if not torch.allclose(
        probabilities.sum(dim=-1),
        torch.ones((batch_size, sequence_length), device=zt_ids.device, dtype=clean_prediction.dtype),
        atol=atol,
        rtol=0,
    ):
        raise AssertionError("Every reverse categorical distribution must sum to one.")
    return probabilities, stay_masked, reveal


def reverse_posterior_at_times(
    zt_ids: torch.Tensor,
    clean_prediction: torch.Tensor,
    s,
    t,
    *,
    mask_id: int,
    eps: float = NOISE_EPS,
):
    batch_size = zt_ids.shape[0]
    s_values = _expand_stage3_batch_value(s, batch_size, zt_ids.device, torch.float64, "s")
    t_values = _expand_stage3_batch_value(t, batch_size, zt_ids.device, torch.float64, "t")
    if torch.any(s_values < 0) or torch.any(s_values > 1) or torch.any(t_values < 0) or torch.any(t_values > 1):
        raise ValueError("Times must lie in [0, 1].")
    if torch.any(s_values > t_values):
        raise ValueError("Reverse time must satisfy s <= t.")
    _, alpha_s, _ = loglinear_schedule(s_values, eps=eps)
    _, alpha_t, _ = loglinear_schedule(t_values, eps=eps)
    return reverse_posterior_from_alphas(
        zt_ids,
        clean_prediction,
        alpha_s.to(clean_prediction.dtype),
        alpha_t.to(clean_prediction.dtype),
        mask_id=mask_id,
    )


TOY_K = 6
TOY_MASK_ID = 2
toy_zt = torch.tensor([[TOY_MASK_ID], [5]], dtype=torch.int64)
toy_prediction = torch.zeros((2, 1, TOY_K), dtype=torch.float64)
toy_prediction[:, 0, 0] = 0.60
toy_prediction[:, 0, 5] = 0.40
toy_probs, _, _ = reverse_posterior_from_alphas(
    toy_zt, toy_prediction, alpha_s=0.90, alpha_t=0.75, mask_id=TOY_MASK_ID
)

toy_subs_rows = []
toy_names = ["A", "other-1", "[MASK]", "other-3", "other-4", "B"]
for row_name, row_probs in zip(["currently MASK", "currently visible B"], toy_probs[:, 0]):
    for token_name, probability in zip(toy_names, row_probs.tolist()):
        if probability > 0:
            toy_subs_rows.append({"current state": row_name, "z_s category": token_name, "probability": probability})
display(pd.DataFrame(toy_subs_rows))
assert torch.allclose(toy_probs[0, 0], torch.tensor([0.36, 0.0, 0.40, 0.0, 0.0, 0.24], dtype=torch.float64))
assert torch.equal(toy_probs[1, 0], F.one_hot(torch.tensor(5), TOY_K).to(torch.float64))

## Lesson 3.5 - Sample hard reverse IDs

### Purpose and intuition

Equation (2) returns categorical probabilities. A reverse trajectory needs one hard token ID per position. The canonical repository kernel uses an exponential race:

$$
E_i\sim\operatorname{Exp}(1),\qquad
z_s=\operatorname*{argmax}_i\frac{p_i}{E_i}.
$$

This is exactly a sample from $\operatorname{Cat}(p)$. For probabilities `A=0.36`, `B=0.24`, `MASK=0.40`, one draw returns exactly one of those IDs, not their mixture.

### What the code below does and expected evidence

The code accepts either a generator or injected uniforms, converts uniforms to exponential variables, and returns `[B, L]` sampled IDs plus the probabilities and weights. Equal injected uniforms must reproduce equal samples. Visible positions have point-mass distributions and therefore copy deterministically.

### Scope

This is canonical Equation (2) sampling. It is not the public GenMol confidence sampler.

### Comprehension checkpoint

1. Why can a zero-probability category never win the race?
2. Why does a visible token copy deterministically?
3. What is the difference between a probability tensor and sampled IDs?


In [ ]:
def sample_reverse_from_alphas(
    zt_ids: torch.Tensor,
    clean_prediction: torch.Tensor,
    alpha_s,
    alpha_t,
    *,
    mask_id: int,
    generator: torch.Generator | None = None,
    race_uniforms: torch.Tensor | None = None,
):
    if generator is not None and race_uniforms is not None:
        raise ValueError("Provide either generator or race_uniforms, not both.")

    probabilities, stay_masked, reveal = reverse_posterior_from_alphas(
        zt_ids, clean_prediction, alpha_s, alpha_t, mask_id=mask_id
    )
    if race_uniforms is None:
        race_uniforms = torch.rand(
            probabilities.shape,
            dtype=probabilities.dtype,
            device=probabilities.device,
            generator=generator,
        )
    else:
        if race_uniforms.shape != probabilities.shape:
            raise ValueError("race_uniforms must have the same shape as the probability tensor.")
        if race_uniforms.device != probabilities.device:
            raise ValueError("race_uniforms must be on the same device as the probabilities.")
        race_uniforms = race_uniforms.to(probabilities.dtype)
        if not torch.isfinite(race_uniforms).all():
            raise ValueError("race_uniforms must be finite.")
        if torch.any(race_uniforms < 0) or torch.any(race_uniforms >= 1):
            raise ValueError("race_uniforms must lie in [0, 1).")

    exponential_race = 1e-10 - torch.log(race_uniforms + 1e-10)
    sampled_ids = (probabilities / exponential_race).argmax(dim=-1)
    return sampled_ids, probabilities, stay_masked, reveal


def sample_reverse_at_times(
    zt_ids: torch.Tensor,
    clean_prediction: torch.Tensor,
    s,
    t,
    *,
    mask_id: int,
    eps: float = NOISE_EPS,
    generator: torch.Generator | None = None,
    race_uniforms: torch.Tensor | None = None,
):
    batch_size = zt_ids.shape[0]
    s_values = _expand_stage3_batch_value(s, batch_size, zt_ids.device, torch.float64, "s")
    t_values = _expand_stage3_batch_value(t, batch_size, zt_ids.device, torch.float64, "t")
    if torch.any(s_values < 0) or torch.any(s_values > 1) or torch.any(t_values < 0) or torch.any(t_values > 1):
        raise ValueError("Times must lie in [0, 1].")
    if torch.any(s_values > t_values):
        raise ValueError("Reverse time must satisfy s <= t.")
    _, alpha_s, _ = loglinear_schedule(s_values, eps=eps)
    _, alpha_t, _ = loglinear_schedule(t_values, eps=eps)
    return sample_reverse_from_alphas(
        zt_ids,
        clean_prediction,
        alpha_s.to(clean_prediction.dtype),
        alpha_t.to(clean_prediction.dtype),
        mask_id=mask_id,
        generator=generator,
        race_uniforms=race_uniforms,
    )


same_uniforms = torch.full((2, 1, TOY_K), 0.5, dtype=torch.float64)
deterministic_sample_a = sample_reverse_from_alphas(
    toy_zt, toy_prediction, 0.90, 0.75, mask_id=TOY_MASK_ID, race_uniforms=same_uniforms
)[0]
deterministic_sample_b = sample_reverse_from_alphas(
    toy_zt, toy_prediction, 0.90, 0.75, mask_id=TOY_MASK_ID, race_uniforms=same_uniforms
)[0]
assert torch.equal(deterministic_sample_a, deterministic_sample_b)
assert deterministic_sample_a[1, 0].item() == 5

## Lesson 3.6 - Run an oracle reverse trajectory

### Purpose and intuition

A reverse chain should reveal tokens without changing any token already visible. To isolate kernel correctness from model error, use the known clean one-hot tensor as the oracle prediction, $x_\theta=x$.

The chain uses

```text
1.00 -> 0.75 -> 0.50 -> 0.25 -> 0.00
```

At every step, a mask may remain masked or reveal its exact clean token. For ethanol, the oracle destination is `[CLS] C C O [SEP]`; an incorrect chemical token is impossible under the oracle.

### What the code below does and expected evidence

The code initializes every valid position as `[MASK]`, keeps ignored padding unchanged, applies canonical reverse steps, records mask counts, and asserts carry-over after every transition. Because $\alpha_0=1$, the final step reveals every remaining mask. The final state must equal the complete clean batch exactly.

### Scope

This is an oracle sanity test. It does not demonstrate learned molecular generation.

### Comprehension checkpoint

1. Why can the oracle never reveal the wrong clean token?
2. Can a token change after it becomes visible?
3. Why must all remaining masks reveal at $s=0$?


In [ ]:
oracle_clean_prediction = clean_x.to(torch.float64)
reverse_times = [1.00, 0.75, 0.50, 0.25, 0.00]
reverse_state = torch.where(
    valid_positions,
    torch.full_like(clean_token_ids, tokenizer.mask_token_id),
    clean_token_ids,
)
reverse_generator = torch.Generator(device=reverse_state.device).manual_seed(30303)

trajectory_rows = []
for time_value in reverse_times:
    first_valid_ids = reverse_state[0, valid_positions[0]].tolist()
    trajectory_rows.append(
        {
            "time": time_value,
            "masked valid positions": int(((reverse_state == tokenizer.mask_token_id) & valid_positions).sum()),
            "visible valid positions": int(((reverse_state != tokenizer.mask_token_id) & valid_positions).sum()),
            "first molecule tokens": " ".join(tokenizer.convert_ids_to_tokens(first_valid_ids)),
        }
    )
    if time_value == 0.0:
        break
    next_time = reverse_times[reverse_times.index(time_value) + 1]
    previous_state = reverse_state.clone()
    reverse_state = sample_reverse_at_times(
        reverse_state,
        oracle_clean_prediction,
        s=next_time,
        t=time_value,
        mask_id=tokenizer.mask_token_id,
        generator=reverse_generator,
    )[0]
    already_visible = previous_state != tokenizer.mask_token_id
    assert torch.equal(reverse_state[already_visible], previous_state[already_visible])
    assert torch.all(
        (reverse_state[valid_positions] == clean_token_ids[valid_positions])
        | (reverse_state[valid_positions] == tokenizer.mask_token_id)
    )

assert torch.equal(reverse_state, clean_token_ids)
trajectory_table = pd.DataFrame(trajectory_rows)
display(trajectory_table)
print("Oracle reverse chain ends exactly at the clean batch:", torch.equal(reverse_state, clean_token_ids))

## Lesson 3.7 - Verify Bayes with coupled Monte Carlo paths

### Purpose and intuition

The reverse posterior can be verified by sampling many coupled forward histories and conditioning on $z_t=m$. With $\alpha_s=0.90$ and $\alpha_t=0.75$:

```text
clean at s, clean at t  0.75
clean at s, MASK at t   0.15
MASK at s, MASK at t    0.10
MASK at s, clean at t   0.00
```

The evidence for `MASK at t` is $0.25$, so reveal probability is $0.15/0.25=0.60$ and stay probability is $0.10/0.25=0.40$.

### What the code below does and expected evidence

The code uses 500,000 shared forward thresholds to construct coupled $(z_s,z_t)$ pairs. Joint and conditional empirical frequencies must fall inside six-standard-error tolerances. The impossible mask-to-clean forward path must occur zero times, while conditioned frequencies should approach 0.60 and 0.40.

### Scope

This validates the Bayes derivation statistically. It does not test BERT or chemical quality.

### Comprehension checkpoint

1. Why is the conditioning denominator 0.25?
2. Which two histories remain after conditioning on `MASK at t`?
3. Why must the forward `MASK at s -> clean at t` count be zero?


In [ ]:
MC_DRAWS = 500_000
MC_ALPHA_S = 0.90
MC_ALPHA_T = 0.75
MC_CLEAN_ID = 1
MC_MASK_ID = 2

mc_generator = torch.Generator().manual_seed(330033)
shared_forward_uniform = torch.rand(MC_DRAWS, generator=mc_generator, dtype=torch.float64)
mc_zs = torch.where(
    shared_forward_uniform < MC_ALPHA_S,
    torch.full((MC_DRAWS,), MC_CLEAN_ID, dtype=torch.int64),
    torch.full((MC_DRAWS,), MC_MASK_ID, dtype=torch.int64),
)
mc_zt = torch.where(
    shared_forward_uniform < MC_ALPHA_T,
    torch.full((MC_DRAWS,), MC_CLEAN_ID, dtype=torch.int64),
    torch.full((MC_DRAWS,), MC_MASK_ID, dtype=torch.int64),
)

joint_labels = [
    ("clean at s, clean at t", (mc_zs == MC_CLEAN_ID) & (mc_zt == MC_CLEAN_ID), MC_ALPHA_T),
    ("clean at s, MASK at t", (mc_zs == MC_CLEAN_ID) & (mc_zt == MC_MASK_ID), MC_ALPHA_S - MC_ALPHA_T),
    ("MASK at s, MASK at t", (mc_zs == MC_MASK_ID) & (mc_zt == MC_MASK_ID), 1.0 - MC_ALPHA_S),
    ("MASK at s, clean at t (impossible)", (mc_zs == MC_MASK_ID) & (mc_zt == MC_CLEAN_ID), 0.0),
]
joint_rows = []
for label, event, expected in joint_labels:
    empirical = event.to(torch.float64).mean().item()
    tolerance = 6.0 * math.sqrt(max(expected * (1.0 - expected), 1e-12) / MC_DRAWS) + 2.0 / MC_DRAWS
    assert abs(empirical - expected) <= tolerance
    joint_rows.append(
        {"path": label, "theory": expected, "empirical": empirical, "six-sigma tolerance": tolerance}
    )

masked_at_t = mc_zt == MC_MASK_ID
empirical_reveal_given_mask = (mc_zs[masked_at_t] == MC_CLEAN_ID).to(torch.float64).mean().item()
empirical_stay_given_mask = (mc_zs[masked_at_t] == MC_MASK_ID).to(torch.float64).mean().item()
theory_reveal = (MC_ALPHA_S - MC_ALPHA_T) / (1.0 - MC_ALPHA_T)
theory_stay = (1.0 - MC_ALPHA_S) / (1.0 - MC_ALPHA_T)
conditional_n = int(masked_at_t.sum())
conditional_tolerance = 6.0 * math.sqrt(theory_reveal * theory_stay / conditional_n) + 2.0 / conditional_n
assert abs(empirical_reveal_given_mask - theory_reveal) <= conditional_tolerance
assert abs(empirical_stay_given_mask - theory_stay) <= conditional_tolerance

display(pd.DataFrame(joint_rows))
print(
    {
        "conditioned_paths": conditional_n,
        "theory_reveal_given_MASK": theory_reveal,
        "empirical_reveal_given_MASK": empirical_reveal_given_mask,
        "theory_stay_MASK": theory_stay,
        "empirical_stay_MASK": empirical_stay_given_mask,
    }
)

## Lesson 3.8 - Match canonical `MDLM.step`

### Purpose and intuition

BioNeMo MoCo 0.0.2.1 implements the canonical Equation (2) DDPM kernel in `MDLM.step`. Its masked-branch raw weights are

$$
(\alpha_s-\alpha_t)x_\theta+(1-\alpha_s)m,
$$

whose total is $1-\alpha_t$. Our implementation divides by that total to display probabilities. `MDLM.step` may omit the division before an exponential race because multiplying every category by one positive constant does not change the winner.

### What the code below does and expected evidence

The code masks a fixed valid-position pattern, supplies oracle logits, uses $t=0.80$ and $s=0.30$, and gives both implementations equal random seeds. The from-scratch hard sample must exactly equal `MDLM.step`, normalized rows must sum to one, and repository raw weights must sum to $1-\alpha_t$.

The public GenMol sampler does not call this kernel. It calls `step_confidence`, which is a later decoding heuristic.

### Scope

This lesson validates canonical `MDLM.step` only, not public confidence sampling.

### Comprehension checkpoint

1. Why can normalized and uniformly scaled weights produce the same race winner?
2. What exact repository function is validated here?
3. Which different function does the public GenMol sampler use?


In [ ]:
oracle_xt_stage3 = clean_token_ids.clone()
position_pattern = (torch.arange(clean_token_ids.shape[1])[None, :] % 3) == 1
oracle_xt_stage3[valid_positions & position_pattern] = tokenizer.mask_token_id

oracle_logits_stage3 = torch.full(
    (*clean_token_ids.shape, tokenizer.vocab_size),
    -1.0e9,
    dtype=torch.float32,
)
oracle_logits_stage3.scatter_(-1, clean_token_ids[..., None], 0.0)
oracle_prediction_stage3 = clean_x.to(torch.float32)

oracle_t_stage3 = torch.full((clean_token_ids.shape[0],), 0.80, dtype=torch.float32)
oracle_s_stage3 = torch.full((clean_token_ids.shape[0],), 0.30, dtype=torch.float32)
oracle_dt_stage3 = oracle_t_stage3 - oracle_s_stage3
stage3_seed = 73003

repository_generator = torch.Generator().manual_seed(stage3_seed)
from_scratch_generator = torch.Generator().manual_seed(stage3_seed)
oracle_mdlm.rng_generator = repository_generator
repository_reverse_sample = oracle_mdlm.step(
    oracle_logits_stage3.clone(),
    oracle_t_stage3,
    oracle_xt_stage3,
    oracle_dt_stage3,
)
oracle_mdlm.rng_generator = None

from_scratch_reverse_sample, normalized_reverse_probs, _, _ = sample_reverse_at_times(
    oracle_xt_stage3,
    oracle_prediction_stage3,
    oracle_s_stage3,
    oracle_t_stage3,
    mask_id=tokenizer.mask_token_id,
    generator=from_scratch_generator,
)
assert torch.equal(from_scratch_reverse_sample, repository_reverse_sample)
assert torch.allclose(
    normalized_reverse_probs.sum(dim=-1),
    torch.ones_like(normalized_reverse_probs[..., 0]),
    atol=2e-6,
    rtol=0,
)

_, repository_alpha_s, _ = loglinear_schedule(oracle_s_stage3)
_, repository_alpha_t, _ = loglinear_schedule(oracle_t_stage3)
raw_repository_weight_sum = (
    repository_alpha_s - repository_alpha_t + 1.0 - repository_alpha_s
)
assert torch.allclose(raw_repository_weight_sum, 1.0 - repository_alpha_t)

print(
    {
        "bionemo_moco_version": BIONEMO_MOCO_VERSION,
        "from_scratch_sample_equals_MDLM_step": bool(
            torch.equal(from_scratch_reverse_sample, repository_reverse_sample)
        ),
        "our_probability_rows_sum_to_one": True,
        "repository_raw_weight_sum_is_1_minus_alpha_t": True,
        "public_GenMol_sampler_uses_step_confidence_instead": True,
    }
)

## Lesson 3.9 - Test boundaries and reject zero-evidence conditions

### Purpose and intuition

A conditional posterior exists only when the observed $z_t$ has positive forward probability. Important valid boundaries are:

```text
alpha_s=1.00, alpha_t=0.75, current MASK -> clean with probability 1
alpha_s=0.40, alpha_t=0.00, current MASK -> clean 0.40, MASK 0.60
alpha_s=0.50, alpha_t=0.50, current MASK -> MASK with probability 1
```

Zero-evidence examples include observing `[MASK]` when $\alpha_t=1$, observing the clean token when $\alpha_t=0$, or observing an unrelated ordinary token.

### What the code below does and expected evidence

The code asserts valid boundary results and rejects reversed times, invalid alphas and IDs, malformed tensor shapes, negative or non-normalized clean predictions, predicted mask mass, non-finite values, and invalid race uniforms. Every intended failure must raise clearly rather than produce NaNs or invented probabilities.

### Scope

This lesson validates defensive behavior. It does not repair mathematically undefined inputs.

### Comprehension checkpoint

1. Why is a current mask impossible when $\alpha_t=1$?
2. Why does $\alpha_s=\alpha_t$ make a zero-length reverse step?
3. What should code do when a conditioning event has zero evidence?


In [ ]:
assert torch.equal(
    exact_reverse_posterior_token(1, 2, 1.0, 0.75, num_classes=3, mask_id=2),
    torch.tensor([0.0, 1.0, 0.0], dtype=torch.float64),
)
assert torch.allclose(
    exact_reverse_posterior_token(1, 2, 0.40, 0.0, num_classes=3, mask_id=2),
    torch.tensor([0.0, 0.40, 0.60], dtype=torch.float64),
)
assert torch.equal(
    exact_reverse_posterior_token(1, 2, 0.50, 0.50, num_classes=3, mask_id=2),
    torch.tensor([0.0, 0.0, 1.0], dtype=torch.float64),
)


def stage3_expected_failure(label: str, operation) -> dict:
    try:
        operation()
    except (TypeError, ValueError) as exc:
        return {
            "case": label,
            "failed_safely": True,
            "exception": type(exc).__name__,
            "message": str(exc),
        }
    raise AssertionError(f"{label} should have failed.")


bad_negative_prediction = toy_prediction.clone()
bad_negative_prediction[0, 0, 0] = -0.1
bad_negative_prediction[0, 0, 5] = 1.1
bad_sum_prediction = toy_prediction.clone()
bad_sum_prediction[0, 0, 0] += 0.1
bad_mask_prediction = toy_prediction.clone()
bad_mask_prediction[0, 0, TOY_MASK_ID] = 0.1
bad_mask_prediction[0, 0, 0] -= 0.1
bad_nan_prediction = toy_prediction.clone()
bad_nan_prediction[0, 0, 0] = float("nan")

valid_race_uniforms = torch.full_like(toy_prediction, 0.5)
stage3_failure_rows = [
    stage3_expected_failure(
        "alpha_s below alpha_t",
        lambda: exact_reverse_posterior_token(1, 2, 0.6, 0.7, num_classes=3, mask_id=2),
    ),
    stage3_expected_failure(
        "alpha outside [0, 1]",
        lambda: exact_reverse_posterior_token(1, 2, 1.1, 0.7, num_classes=3, mask_id=2),
    ),
    stage3_expected_failure(
        "non-finite alpha",
        lambda: exact_reverse_posterior_token(1, 2, float("nan"), 0.7, num_classes=3, mask_id=2),
    ),
    stage3_expected_failure(
        "clean token is MASK",
        lambda: exact_reverse_posterior_token(2, 2, 0.9, 0.7, num_classes=3, mask_id=2),
    ),
    stage3_expected_failure(
        "off-support visible token",
        lambda: exact_reverse_posterior_token(1, 0, 0.9, 0.7, num_classes=3, mask_id=2),
    ),
    stage3_expected_failure(
        "MASK observed with alpha_t=1",
        lambda: exact_reverse_posterior_token(1, 2, 1.0, 1.0, num_classes=3, mask_id=2),
    ),
    stage3_expected_failure(
        "visible observed with alpha_t=0",
        lambda: exact_reverse_posterior_token(1, 1, 0.5, 0.0, num_classes=3, mask_id=2),
    ),
    stage3_expected_failure(
        "floating z_t IDs",
        lambda: reverse_posterior_from_alphas(toy_zt.to(torch.float32), toy_prediction, 0.9, 0.75, mask_id=TOY_MASK_ID),
    ),
    stage3_expected_failure(
        "wrong prediction shape",
        lambda: reverse_posterior_from_alphas(toy_zt, toy_prediction[:, :, :-1], 0.9, 0.75, mask_id=TOY_MASK_ID),
    ),
    stage3_expected_failure(
        "negative prediction probability",
        lambda: reverse_posterior_from_alphas(toy_zt, bad_negative_prediction, 0.9, 0.75, mask_id=TOY_MASK_ID),
    ),
    stage3_expected_failure(
        "prediction rows do not sum to one",
        lambda: reverse_posterior_from_alphas(toy_zt, bad_sum_prediction, 0.9, 0.75, mask_id=TOY_MASK_ID),
    ),
    stage3_expected_failure(
        "prediction gives MASK mass",
        lambda: reverse_posterior_from_alphas(toy_zt, bad_mask_prediction, 0.9, 0.75, mask_id=TOY_MASK_ID),
    ),
    stage3_expected_failure(
        "non-finite prediction",
        lambda: reverse_posterior_from_alphas(toy_zt, bad_nan_prediction, 0.9, 0.75, mask_id=TOY_MASK_ID),
    ),
    stage3_expected_failure(
        "masked state at t=0",
        lambda: reverse_posterior_at_times(toy_zt, toy_prediction, 0.0, 0.0, mask_id=TOY_MASK_ID),
    ),
    stage3_expected_failure(
        "s greater than t",
        lambda: reverse_posterior_at_times(toy_zt, toy_prediction, 0.8, 0.2, mask_id=TOY_MASK_ID),
    ),
    stage3_expected_failure(
        "wrong per-batch time shape",
        lambda: reverse_posterior_at_times(toy_zt, toy_prediction, 0.2, torch.tensor([0.5, 0.6, 0.7]), mask_id=TOY_MASK_ID),
    ),
    stage3_expected_failure(
        "wrong race-uniform shape",
        lambda: sample_reverse_from_alphas(
            toy_zt, toy_prediction, 0.9, 0.75, mask_id=TOY_MASK_ID, race_uniforms=torch.full((2, 1, 5), 0.5)
        ),
    ),
    stage3_expected_failure(
        "race uniform outside [0, 1)",
        lambda: sample_reverse_from_alphas(
            toy_zt, toy_prediction, 0.9, 0.75, mask_id=TOY_MASK_ID, race_uniforms=torch.ones_like(toy_prediction)
        ),
    ),
    stage3_expected_failure(
        "generator and uniforms together",
        lambda: sample_reverse_from_alphas(
            toy_zt,
            toy_prediction,
            0.9,
            0.75,
            mask_id=TOY_MASK_ID,
            generator=torch.Generator(),
            race_uniforms=valid_race_uniforms,
        ),
    ),
]

failure_table_stage3 = pd.DataFrame(stage3_failure_rows)
display(failure_table_stage3)
assert failure_table_stage3["failed_safely"].all()
print(
    {
        "expected_failures_caught": len(failure_table_stage3),
        "matrix_oracle_matches": oracle_matches,
        "monte_carlo_draws": MC_DRAWS,
        "repository_sample_match": bool(torch.equal(from_scratch_reverse_sample, repository_reverse_sample)),
    }
)

## Stage 3 completion gate

Stage 3 is complete only if the notebook demonstrates:

- the three possible coupled forward histories and impossible mask-to-clean path;
- the exact Bayes posterior for one known clean token;
- exhaustive agreement with an independent transition-matrix oracle;
- normalized stay/reveal weights from the released schedule;
- batched Equation (2) with strict zero mask prediction;
- deterministic carry-over for visible tokens;
- reproducible hard categorical sampling;
- an oracle reverse chain ending exactly at the clean batch;
- Monte Carlo agreement over 500,000 coupled paths;
- exact sampled agreement with canonical BioNeMo `MDLM.step`;
- explicit separation from public `step_confidence`;
- safe rejection of invalid and zero-evidence conditions.

No BERT, Equation (3) loss, optimizer, confidence sampling, or molecule generation has been implemented.

### Comprehension checkpoint

Assume $\alpha_s=0.90$, $\alpha_t=0.75$, $x_\theta(A)=0.60$, $x_\theta(B)=0.40$, and $x_\theta(m)=0$.

1. Why is the normalization denominator $1-\alpha_t=0.25$?
2. What are the stay and reveal probabilities?
3. What are the final probabilities of `A`, `B`, and `[MASK]`?
4. What happens if the current token is already visible as `A`?
5. Why do released-schedule weights simplify to $s/t$ and $(t-s)/t$?
6. Why does matching `MDLM.step` not yet validate public `step_confidence`?

**Stop here until the posterior, carry-over rule, and canonical-versus-confidence distinction can all be explained. Do not continue to Stage 4 before that point.**


# Stage 4 - Equation (3): the continuous-time denoising loss

## Purpose and intuition

Stage 3 supplied an exact reverse posterior. Stage 4 turns its masked-token KL term into weighted clean-token cross-entropy. For a masked token, the exact and learned reverse laws share the same mask branch, so their KL reduces to $-u\log p_\theta(x\mid z_t,t)$. For a visible token, both reverse laws are the same point mass, so its KL is zero.

Using $p_{\theta,i\ell}$ for the probability of the correct clean token, paper Equation (3) is

$$
L_{\mathrm{NELBO}} = E_q\int_0^1 \frac{\alpha_t\prime}{1-\alpha_t}\sum_\ell \log p_{\theta,i\ell}\,dt.
$$

Because both factors shown are non-positive, define positive cross-entropy weight

$$
w(t)=-\frac{\alpha_t\prime}{1-\alpha_t},\qquad CE_{i\ell}=-\log p_{\theta,i\ell}.
$$

For the pinned log-linear schedule,

$$
\alpha(t)=1-(1-\epsilon)t,\quad P(z_t=MASK)=(1-\epsilon)t,\quad w(t)=\frac{1}{t},
$$

so $P(MASK)w(t)=1-\epsilon$. The large weight near zero compensates for masks being rare.

## Concrete example

At $t=0.25$ and $\epsilon=0.001$, $\alpha=0.75025$, $P(MASK)=0.24975$, $w=4$, and $P(MASK)w=0.999$.

## What the code below computes and expected evidence

The next cell defines loglinear_loss_terms for a time tensor of shape [N]. It returns alpha, mask probability, sigma, its derivative, the positive weight, and their product. The displayed five-row table should show $w(t)=1/t$ and a constant final column equal to 0.999; exact assertions check both identities.

## Scope

This cell implements schedule and loss arithmetic only. It does not build BERT, optimize parameters, or run training. The sampled endpoint is $t\ge 0.001$ because $1/t$ is undefined at zero.

### Comprehension checkpoint

1. Why is Equation (3) non-negative even though it displays no leading minus sign?
2. Derive $w(t)=1/t$ from the pinned alpha schedule.
3. Why does a rare mask at small $t$ receive a large weight?


In [ ]:
STAGE4_NOISE_EPS = float(NOISE_EPS)
assert math.isclose(STAGE4_NOISE_EPS, 1.0e-3)


def loglinear_loss_terms(time: torch.Tensor, *, eps: float = STAGE4_NOISE_EPS) -> dict:
    """Return the repository schedule and the positive Equation (3) weight."""
    if not isinstance(time, torch.Tensor):
        raise TypeError("time must be a torch.Tensor.")
    if not time.is_floating_point():
        raise TypeError("time must use a floating dtype.")
    if not math.isfinite(float(eps)) or not 0.0 < float(eps) < 1.0:
        raise ValueError("eps must be finite and strictly between zero and one.")

    t = time.to(dtype=torch.float64)
    if not bool(torch.isfinite(t).all()):
        raise ValueError("time must be finite.")
    if bool(((t <= 0.0) | (t > 1.0)).any()):
        raise ValueError("Equation (3) is evaluated only for 0 < time <= 1.")

    one_minus_eps = 1.0 - float(eps)
    alpha = 1.0 - one_minus_eps * t
    mask_probability = one_minus_eps * t
    sigma = -torch.log(alpha)
    dsigma_dt = one_minus_eps / alpha

    # This analytic form is stable even when t is small.
    positive_weight = 1.0 / t
    weight_from_moco = dsigma_dt / torch.expm1(sigma)
    if not torch.allclose(
        positive_weight, weight_from_moco, atol=1e-10, rtol=1e-10
    ):
        raise AssertionError("The alpha and sigma forms of the loss weight disagree.")

    return {
        "time": t,
        "alpha": alpha,
        "mask_probability": mask_probability,
        "sigma": sigma,
        "dsigma_dt": dsigma_dt,
        "positive_weight": positive_weight,
        "mask_probability_times_weight": mask_probability * positive_weight,
    }


stage4_times = torch.tensor([0.001, 0.01, 0.25, 0.50, 1.00], dtype=torch.float64)
stage4_terms = loglinear_loss_terms(stage4_times)
stage4_schedule_df = pd.DataFrame(
    {
        "t": stage4_terms["time"].tolist(),
        "alpha_survival": stage4_terms["alpha"].tolist(),
        "P(MASK)": stage4_terms["mask_probability"].tolist(),
        "positive weight w(t)": stage4_terms["positive_weight"].tolist(),
        "P(MASK) * w(t)": stage4_terms["mask_probability_times_weight"].tolist(),
    }
)
assert torch.allclose(
    stage4_terms["positive_weight"], 1.0 / stage4_times, atol=0.0, rtol=0.0
)
assert torch.allclose(
    stage4_terms["mask_probability_times_weight"],
    torch.full_like(stage4_times, 1.0 - STAGE4_NOISE_EPS),
    atol=1e-12,
    rtol=1e-12,
)
display(stage4_schedule_df)


## Strict SUBS creates the clean-token distribution

## Purpose and intuition

Equation (3) needs a distribution over the clean token, not the full reverse-step distribution. Strict SUBS removes the absorbing MASK category and copies every visible current token exactly.

For raw logits with shape [B, L, V] and current token IDs xt with shape [B, L], the returned log-probabilities also have shape [B, L, V]. At every position,

$$
p_\theta(MASK)=0.
$$

If $xt_{i\ell}\ne MASK$, the output is the point mass $\delta_{xt_{i\ell}}$. If $xt_{i\ell}=MASK$, softmax is taken over the other $V-1$ categories.

## Concrete example

With vocabulary [A, MASK, B, C], a visible B becomes [0, 0, 1, 0] regardless of its raw logits. At a masked position, logits for A, B, and C are normalized while MASK stays at probability zero.

## What the code below computes and expected evidence

The next cell defines substitution_log_probs, validates tensor ranks, dtypes, devices, IDs, and finiteness, and performs the transformation out of place. This definition cell should finish silently. The following toy cell will provide visible evidence that MASK probability is zero, visible tokens are copied, and the caller logits are unchanged.

## Scope

This is a clean-token parameterization, not a sampler step. The pinned helper uses a finite -1e6 sentinel and mutates its logits; this pedagogical version uses exact -inf and leaves its input unchanged.

### Comprehension checkpoint

1. Why must the predicted clean-token distribution assign zero probability to MASK?
2. What distribution is returned when the current token is visible?
3. Why can a visible position have zero cross-entropy even if its raw logits prefer another token?


In [ ]:
STAGE4_INTEGER_DTYPES = {
    torch.uint8,
    torch.int8,
    torch.int16,
    torch.int32,
    torch.int64,
}


def substitution_log_probs(
    logits: torch.Tensor,
    xt: torch.Tensor,
    *,
    mask_id: int,
) -> torch.Tensor:
    """Apply strict SUBS and return log-probabilities with shape [B, L, V]."""
    if not isinstance(logits, torch.Tensor) or not isinstance(xt, torch.Tensor):
        raise TypeError("logits and xt must be torch tensors.")
    if logits.ndim != 3:
        raise ValueError("logits must have shape [B, L, V].")
    if not logits.is_floating_point():
        raise TypeError("logits must use a floating dtype.")
    if not bool(torch.isfinite(logits).all()):
        raise ValueError("logits must be finite.")
    if xt.dtype not in STAGE4_INTEGER_DTYPES or xt.ndim != 2:
        raise TypeError("xt must be an integer tensor with shape [B, L].")
    if xt.device != logits.device:
        raise ValueError("xt and logits must be on the same device.")

    batch_size, length, vocabulary_size = logits.shape
    if xt.shape != (batch_size, length):
        raise ValueError("xt shape must equal logits.shape[:2].")
    if type(mask_id) is not int or not 0 <= mask_id < vocabulary_size:
        raise ValueError("mask_id must index the vocabulary.")
    if bool(((xt < 0) | (xt >= vocabulary_size)).any()):
        raise ValueError("xt contains a token ID outside the vocabulary.")

    class_is_mask = (
        torch.arange(vocabulary_size, device=logits.device) == mask_id
    ).view(1, 1, vocabulary_size)
    clean_logits = logits.masked_fill(class_is_mask, -torch.inf)
    learned_log_probs = torch.log_softmax(clean_logits, dim=-1)

    carried_log_probs = torch.full_like(learned_log_probs, -torch.inf).scatter(
        dim=-1,
        index=xt.unsqueeze(-1),
        value=0.0,
    )
    is_visible = xt.ne(mask_id).unsqueeze(-1)
    return torch.where(is_visible, carried_log_probs, learned_log_probs)


## Build the sampled Equation (3) loss

## Purpose and intuition

One sampled time per sequence and one forward-corrupted sequence turn the integral and expectation into a batch estimator. The per-token quantity is

$$
\ell_{i\ell}=valid_{i\ell}\,1[xt_{i\ell}=MASK]\,\frac{1}{t_i}\left[-\log p_\theta(x_{i\ell}\mid xt_i)\right].
$$

Inputs have logits [B, L, V], target [B, L], xt [B, L], time [B], and valid_positions [B, L]. Here valid_positions is the padding or attention mask. It is not the diffusion mask $1[xt=MASK]$.

The reductions are

$$
paper=\frac{1}{B}\sum_i\sum_\ell\ell_{i\ell},\quad
global=\frac{\sum_{i\ell}\ell_{i\ell}}{\sum_{i\ell}valid_{i\ell}},\quad
sequence=\frac{1}{B}\sum_i\frac{\sum_\ell\ell_{i\ell}}{\sum_\ell valid_{i\ell}}.
$$

The released default is global: it divides by all valid tokens, never by the realized number of diffusion masks.

## Concrete example

At $t=0.5$, a masked token with cross-entropy 0.7 contributes $(1/0.5)*0.7=1.4$. A visible or padded position contributes zero.

## What the code below computes and expected evidence

The next cell defines mdlm_token_losses and reduce_mdlm_loss. It validates forward support, gathers correct-token log-probabilities safely around padding, returns every intermediate [B, L] tensor, and implements the three reductions. It should finish silently; the next cells test exact values, padding invariance, and error paths.

## Scope

This cell defines loss arithmetic. It does not sample corruption, call a neural network, backpropagate, or update parameters.

### Comprehension checkpoint

1. What is the difference between valid_positions and the diffusion mask?
2. Which positions can have nonzero token loss?
3. What appears in the denominator of the released default global mean?


In [ ]:
def mdlm_token_losses(
    logits: torch.Tensor,
    target: torch.Tensor,
    xt: torch.Tensor,
    time: torch.Tensor,
    *,
    mask_id: int,
    valid_positions: torch.Tensor | None = None,
) -> dict:
    """Return strict-SUBS Equation (3) terms without choosing a reduction."""
    log_probs = substitution_log_probs(logits, xt, mask_id=mask_id)
    batch_size, length, vocabulary_size = logits.shape

    if not isinstance(target, torch.Tensor):
        raise TypeError("target must be a torch.Tensor.")
    if target.dtype not in STAGE4_INTEGER_DTYPES or target.ndim != 2:
        raise TypeError("target must be an integer tensor with shape [B, L].")
    if target.shape != (batch_size, length):
        raise ValueError("target shape must equal [B, L].")
    if target.device != logits.device:
        raise ValueError("target and logits must be on the same device.")
    if bool(((target < 0) | (target >= vocabulary_size)).any()):
        raise ValueError("target contains a token ID outside the vocabulary.")

    if not isinstance(time, torch.Tensor) or not time.is_floating_point():
        raise TypeError("time must be a floating torch tensor.")
    if time.shape != (batch_size,):
        raise ValueError("time must have exactly shape [B], one value per sequence.")
    if time.device != logits.device:
        raise ValueError("time and logits must be on the same device.")
    if not bool(torch.isfinite(time).all()):
        raise ValueError("time must be finite.")
    if bool(((time <= 0.0) | (time > 1.0)).any()):
        raise ValueError("time must satisfy 0 < t <= 1.")

    if valid_positions is None:
        valid = torch.ones(
            (batch_size, length), dtype=torch.bool, device=logits.device
        )
    else:
        if (
            not isinstance(valid_positions, torch.Tensor)
            or valid_positions.dtype != torch.bool
            or valid_positions.shape != (batch_size, length)
        ):
            raise TypeError("valid_positions must be a bool tensor with shape [B, L].")
        if valid_positions.device != logits.device:
            raise ValueError("valid_positions and logits must be on the same device.")
        valid = valid_positions

    if bool((valid & target.eq(mask_id)).any()):
        raise ValueError("A valid clean target cannot be the absorbing mask token.")
    visible = xt.ne(mask_id)
    if bool((valid & visible & xt.ne(target)).any()):
        raise ValueError("A visible xt must equal its clean target on forward support.")

    # Choose a harmless clean target for ignored padding locations before gather.
    fallback_id = 0 if mask_id != 0 else 1
    fallback = torch.full_like(target, fallback_id)
    ignored_target = torch.where(visible, xt, fallback)
    safe_target = torch.where(valid, target, ignored_target)

    correct_log_probs = log_probs.gather(
        dim=-1, index=safe_target.unsqueeze(-1)
    ).squeeze(-1)
    cross_entropy = -correct_log_probs
    weight = (1.0 / time).to(dtype=logits.dtype).unsqueeze(-1)
    diffusion_mask = valid & xt.eq(mask_id)
    token_losses = torch.where(
        diffusion_mask,
        weight * cross_entropy,
        torch.zeros_like(cross_entropy),
    )

    if not bool(torch.isfinite(token_losses).all()):
        raise FloatingPointError("The weighted token losses must be finite.")
    if bool((token_losses < 0).any()):
        raise FloatingPointError("The weighted token losses must be non-negative.")

    return {
        "log_probs": log_probs,
        "correct_log_probs": correct_log_probs,
        "cross_entropy": cross_entropy,
        "positive_weight": weight,
        "valid_positions": valid,
        "diffusion_mask": diffusion_mask,
        "token_losses": token_losses,
    }


def reduce_mdlm_loss(
    token_losses: torch.Tensor,
    valid_positions: torch.Tensor,
    *,
    mode: str,
) -> torch.Tensor:
    """Apply either the paper reduction or one of the repository reductions."""
    if (
        not isinstance(token_losses, torch.Tensor)
        or token_losses.ndim != 2
        or not token_losses.is_floating_point()
    ):
        raise TypeError("token_losses must be a floating [B, L] tensor.")
    if (
        not isinstance(valid_positions, torch.Tensor)
        or valid_positions.dtype != torch.bool
        or valid_positions.shape != token_losses.shape
    ):
        raise TypeError("valid_positions must be a bool tensor matching token_losses.")
    if valid_positions.device != token_losses.device:
        raise ValueError("valid_positions and token_losses must share a device.")
    if not bool(torch.isfinite(token_losses).all()) or bool((token_losses < 0).any()):
        raise ValueError("token_losses must be finite and non-negative.")

    counts = valid_positions.sum(dim=-1)
    if bool((counts == 0).any()):
        raise ValueError("Every sequence must contain at least one valid token.")
    masked_losses = torch.where(
        valid_positions, token_losses, torch.zeros_like(token_losses)
    )
    per_sequence_sums = masked_losses.sum(dim=-1)

    if mode == "paper_sum_mean":
        return per_sequence_sums.mean()
    if mode == "repo_global_token_mean":
        return per_sequence_sums.sum() / counts.sum()
    if mode == "repo_sequence_mean":
        return (per_sequence_sums / counts).mean()
    raise ValueError(f"Unknown reduction mode: {mode!r}")


## Verify the loss by hand on one masked token

## Purpose and intuition

A tiny example makes every factor visible and tests that an enormous raw MASK logit cannot leak through strict SUBS.

The tensors have target and xt shape [1, 4], logits shape [1, 4, 4], and time shape [1]. The vocabulary is [A, MASK, B, C], with MASK deliberately at ID 1.

## Concrete example

At $t=0.25$, position 1 is masked and its true token B receives probability 0.5. The other three tokens are visible and copied. Therefore

$$
L_{paper}=4[-\log(0.5)]=4\log 2\mathrel{\approx}2.77259.
$$

With four valid positions, both repository token-mean reductions equal $\log 2\mathrel{\approx}0.69315$.

## What the code below computes and expected evidence

The next cell sets the raw MASK logit to 100, applies the loss, and asserts: output MASK probability is exactly zero; visible contributions are zero; only position 1 contributes $4\log 2$; and the input logits remain unchanged. Expect a four-row table and a dictionary containing paper_sum_mean near 2.77259, both repository means near 0.69315, and an equal direct Equation (3) value.

## Scope

This is a deterministic synthetic check of parameterization and arithmetic, not a learned prediction or a molecular example.

### Comprehension checkpoint

1. Why does the raw MASK logit of 100 not affect the final probability?
2. Why is the paper loss four times the global-token mean here?
3. Which single table row should have a nonzero weighted contribution?


In [ ]:
STAGE4_VOCAB = ("A", "[MASK]", "B", "C")
STAGE4_MASK_ID = 1
STAGE4_VOCAB_SIZE = len(STAGE4_VOCAB)

stage4_target = torch.tensor([[0, 2, 3, 0]], dtype=torch.long)
stage4_xt = torch.tensor([[0, STAGE4_MASK_ID, 3, 0]], dtype=torch.long)
stage4_time = torch.tensor([0.25], dtype=torch.float64)
stage4_valid = torch.ones_like(stage4_target, dtype=torch.bool)

# The raw mask logit is intentionally huge. Strict SUBS must still make P(MASK)=0.
stage4_logits = torch.zeros((1, 4, STAGE4_VOCAB_SIZE), dtype=torch.float64)
stage4_logits[..., STAGE4_MASK_ID] = 100.0
stage4_logits[0, 1, 0] = math.log(0.25)
stage4_logits[0, 1, 2] = math.log(0.50)
stage4_logits[0, 1, 3] = math.log(0.25)
stage4_logits_before = stage4_logits.clone()

stage4_loss_parts = mdlm_token_losses(
    stage4_logits,
    stage4_target,
    stage4_xt,
    stage4_time,
    mask_id=STAGE4_MASK_ID,
    valid_positions=stage4_valid,
)
stage4_probs = stage4_loss_parts["log_probs"].exp()
stage4_token_losses = stage4_loss_parts["token_losses"]

stage4_paper_loss = reduce_mdlm_loss(
    stage4_token_losses, stage4_valid, mode="paper_sum_mean"
)
stage4_global_loss = reduce_mdlm_loss(
    stage4_token_losses, stage4_valid, mode="repo_global_token_mean"
)
stage4_sequence_loss = reduce_mdlm_loss(
    stage4_token_losses, stage4_valid, mode="repo_sequence_mean"
)

assert torch.equal(stage4_logits, stage4_logits_before)
assert torch.equal(
    stage4_probs[..., STAGE4_MASK_ID],
    torch.zeros_like(stage4_probs[..., STAGE4_MASK_ID]),
)
assert torch.allclose(
    stage4_probs[0, 1],
    torch.tensor([0.25, 0.0, 0.50, 0.25], dtype=torch.float64),
    atol=1e-12,
    rtol=1e-12,
)
assert torch.allclose(
    stage4_token_losses,
    torch.tensor([[0.0, 4.0 * math.log(2.0), 0.0, 0.0]], dtype=torch.float64),
    atol=1e-12,
    rtol=1e-12,
)
assert math.isclose(float(stage4_paper_loss), 4.0 * math.log(2.0), rel_tol=1e-12)
assert math.isclose(float(stage4_global_loss), math.log(2.0), rel_tol=1e-12)

stage4_toy_df = pd.DataFrame(
    {
        "position": list(range(stage4_target.shape[1])),
        "clean target": [STAGE4_VOCAB[i] for i in stage4_target[0].tolist()],
        "current z_t": [STAGE4_VOCAB[i] for i in stage4_xt[0].tolist()],
        "P(correct after SUBS)": [
            float(stage4_probs[0, i, stage4_target[0, i]])
            for i in range(stage4_target.shape[1])
        ],
        "cross-entropy": stage4_loss_parts["cross_entropy"][0].tolist(),
        "weighted contribution": stage4_token_losses[0].tolist(),
    }
)
display(stage4_toy_df)
print(
    {
        "paper_sum_mean": float(stage4_paper_loss),
        "repo_global_token_mean": float(stage4_global_loss),
        "repo_sequence_mean": float(stage4_sequence_loss),
        "direct_equation_3": (-1.0 / 0.25) * math.log(0.5),
    }
)


## Compare the three reductions

## Purpose and intuition

The per-token mathematics can be identical while aggregation changes scale and how unequal-length sequences are weighted.

For sequence sums $S_i$ and valid lengths $n_i$,

$$
paper=\frac{1}{B}\sum_i S_i,\quad global=\frac{\sum_i S_i}{\sum_i n_i},\quad sequence=\frac{1}{B}\sum_i\frac{S_i}{n_i}.
$$

The released default global mean weights tokens equally, so a longer sequence has more total influence. The sequence mean weights sequences equally.

## Concrete example

For valid lengths [4, 2] and loss sums [2, 6], paper = 4, global = 8/6 = 1.33333, and sequence = ((2/4) + (6/2))/2 = 1.75.

## What the code below computes and expected evidence

The next cell evaluates exactly that example, asserts all three values, then appends invalid padding entries containing the deliberately large value 123. Every result must remain unchanged. Expect a three-row table with values 4.0, 1.33333, and 1.75.

## Scope

This cell changes only aggregation. It does not alter strict SUBS or any per-token loss. Every denominator uses valid-token counts, not diffusion-mask counts.

### Comprehension checkpoint

1. Why are global and sequence means different for lengths [4, 2]?
2. Which reduction matches the token sum displayed in paper Equation (3)?
3. Why must an invalid padded value of 123 leave every result unchanged?


In [ ]:
stage4_hand_losses = torch.tensor(
    [
        [2.0, 0.0, 0.0, 0.0],
        [6.0, 0.0, 0.0, 0.0],
    ],
    dtype=torch.float64,
)
stage4_hand_valid = torch.tensor(
    [
        [True, True, True, True],
        [True, True, False, False],
    ]
)

stage4_reduction_values = {
    mode: float(reduce_mdlm_loss(stage4_hand_losses, stage4_hand_valid, mode=mode))
    for mode in (
        "paper_sum_mean",
        "repo_global_token_mean",
        "repo_sequence_mean",
    )
}
assert stage4_reduction_values == {
    "paper_sum_mean": 4.0,
    "repo_global_token_mean": 8.0 / 6.0,
    "repo_sequence_mean": 1.75,
}

# Appending ignored padding must not change any reduction.
padded_losses = torch.nn.functional.pad(stage4_hand_losses, (0, 3), value=123.0)
padded_valid = torch.nn.functional.pad(stage4_hand_valid, (0, 3), value=False)
for mode, expected in stage4_reduction_values.items():
    actual = float(reduce_mdlm_loss(padded_losses, padded_valid, mode=mode))
    assert math.isclose(actual, expected, rel_tol=0.0, abs_tol=1e-12)

display(
    pd.DataFrame(
        [
            {
                "reduction": mode,
                "value": value,
                "meaning": {
                    "paper_sum_mean": "mean over sequences of token sums",
                    "repo_global_token_mean": "all token losses / all valid tokens",
                    "repo_sequence_mean": "mean of each sequence's valid-token mean",
                }[mode],
            }
            for mode, value in stage4_reduction_values.items()
        ]
    )
)


## Check the rare-mask and large-weight cancellation

## Purpose and intuition

Although $1/t$ grows near zero, a token is masked with probability $(1-\epsilon)t$. Conditional on time,

$$
E\left[\frac{1[zt=MASK]}{t}\mid t\right]=\frac{(1-\epsilon)t}{t}=1-\epsilon.
$$

The estimator is unbiased but can have high variance because rare small-time mask events are heavily weighted.

## Concrete example

If the correct-token probability is always 0.8, its cross-entropy is $-\log(0.8)\mathrel{\approx}0.22314$. The expected weighted value is $0.999*0.22314\mathrel{\approx}0.22292$.

## What the code below computes and expected evidence

The next cell uses a fixed seed to draw 500,000 times from [0.001, 1), samples one mask event per time, and averages $1[MASK]/t$. It asserts agreement with 0.999 within six estimated standard errors. Expect a dictionary showing theory 0.999, a nearby empirical mean, the six-standard-error radius, and matching theoretical and empirical weighted cross-entropies for probability 0.8.

## Scope

This is a one-token Monte Carlo diagnostic. It does not estimate model quality, train BERT, or remove the variance problem.

### Comprehension checkpoint

1. Why does the expected weighted mask indicator not depend on $t$?
2. Why can the empirical estimator still have high variance?
3. How would multiplying by a constant cross-entropy change the expectation?


In [ ]:
STAGE4_MC_DRAWS = 500_000
stage4_mc_generator = torch.Generator(device="cpu").manual_seed(20260830)
stage4_sampling_eps = float(TIME_SAMPLING_EPS)

stage4_mc_times = stage4_sampling_eps + (1.0 - stage4_sampling_eps) * torch.rand(
    STAGE4_MC_DRAWS,
    generator=stage4_mc_generator,
    dtype=torch.float64,
)
stage4_mc_mask_probability = (
    1.0 - STAGE4_NOISE_EPS
) * stage4_mc_times
stage4_mc_is_masked = torch.rand(
    STAGE4_MC_DRAWS,
    generator=stage4_mc_generator,
    dtype=torch.float64,
) < stage4_mc_mask_probability
stage4_mc_estimator = stage4_mc_is_masked.to(torch.float64) / stage4_mc_times

stage4_mc_mean = float(stage4_mc_estimator.mean())
stage4_mc_stderr = float(
    stage4_mc_estimator.std(unbiased=True) / math.sqrt(STAGE4_MC_DRAWS)
)
stage4_mc_theory = 1.0 - STAGE4_NOISE_EPS
assert abs(stage4_mc_mean - stage4_mc_theory) <= 6.0 * stage4_mc_stderr

stage4_constant_correct_probability = 0.8
stage4_constant_ce = -math.log(stage4_constant_correct_probability)
print(
    {
        "draws": STAGE4_MC_DRAWS,
        "E[1{MASK}/t]_theory": stage4_mc_theory,
        "E[1{MASK}/t]_empirical": stage4_mc_mean,
        "six_standard_error_radius": 6.0 * stage4_mc_stderr,
        "expected_weighted_CE_for_p=0.8": stage4_mc_theory * stage4_constant_ce,
        "empirical_weighted_CE_for_p=0.8": stage4_mc_mean * stage4_constant_ce,
    }
)


## Verify raw-logit gradients and input contracts

## Purpose and intuition

Loss should reach the clean-token logits at a diffusion-masked output, while strict SUBS blocks the MASK category and hard-copies visible outputs. This cell differentiates with respect to the raw logits themselves.

For the toy logits [1, 4, 4], the expected pattern is: the MASK-category gradient is zero everywhere; raw-logit rows for visible positions 0, 2, and 3 are all zero; and at least one non-MASK entry in masked row 1 has nonzero gradient.

## Concrete example

At masked position 1, logits for A, B, and C participate in softmax, so some of their gradients are nonzero. The MASK logit is excluded, even though its raw value is 100.

## What the code below computes and expected evidence

The next cell backpropagates the paper-sum toy loss, asserts the gradient pattern, and exercises 11 invalid cases: bad times, epsilon, time shapes, clean MASK targets, violated forward support, an invalid MASK ID, and an all-padding reduction. Expect a dictionary with gradient_only_at_masked_clean_logits true, raw_logits_remained_immutable true, nonterminal_mask_id_tested equal to 1, and expected_failures_caught equal to 11.

## Scope

These are gradients with respect to output logits, not a claim that only one BERT parameter receives gradient. Shared transformer parameters and visible context embeddings can affect the masked output and therefore can receive parameter gradients. No optimizer step occurs here.

### Comprehension checkpoint

1. Why is the raw MASK-logit gradient exactly zero?
2. Why are visible output-logit rows zero while visible context can still influence BERT parameter gradients?
3. Which forward-support error is detected when a visible xt differs from its clean target?


In [ ]:
# Only the masked position's clean-token logits may receive gradient.
stage4_grad_logits = stage4_logits.clone().requires_grad_(True)
stage4_grad_parts = mdlm_token_losses(
    stage4_grad_logits,
    stage4_target,
    stage4_xt,
    stage4_time,
    mask_id=STAGE4_MASK_ID,
    valid_positions=stage4_valid,
)
stage4_grad_loss = reduce_mdlm_loss(
    stage4_grad_parts["token_losses"],
    stage4_valid,
    mode="paper_sum_mean",
)
stage4_grad_loss.backward()
assert stage4_grad_logits.grad is not None
assert torch.equal(
    stage4_grad_logits.grad[..., STAGE4_MASK_ID],
    torch.zeros_like(stage4_grad_logits.grad[..., STAGE4_MASK_ID]),
)
assert torch.equal(
    stage4_grad_logits.grad[0, [0, 2, 3]],
    torch.zeros_like(stage4_grad_logits.grad[0, [0, 2, 3]]),
)
assert bool((stage4_grad_logits.grad[0, 1, [0, 2, 3]].abs() > 0).any())


def stage4_expected_failure(label, callable_):
    try:
        callable_()
    except (TypeError, ValueError, FloatingPointError):
        return label
    raise AssertionError(f"Expected failure was not raised: {label}")


stage4_failure_labels = [
    stage4_expected_failure(
        "time zero",
        lambda: loglinear_loss_terms(torch.tensor([0.0], dtype=torch.float64)),
    ),
    stage4_expected_failure(
        "negative time",
        lambda: loglinear_loss_terms(torch.tensor([-0.1], dtype=torch.float64)),
    ),
    stage4_expected_failure(
        "time above one",
        lambda: loglinear_loss_terms(torch.tensor([1.1], dtype=torch.float64)),
    ),
    stage4_expected_failure(
        "NaN time",
        lambda: loglinear_loss_terms(torch.tensor([float("nan")], dtype=torch.float64)),
    ),
    stage4_expected_failure(
        "invalid epsilon",
        lambda: loglinear_loss_terms(torch.tensor([0.5], dtype=torch.float64), eps=1.0),
    ),
    stage4_expected_failure(
        "scalar loss time",
        lambda: mdlm_token_losses(
            stage4_logits,
            stage4_target,
            stage4_xt,
            torch.tensor(0.25, dtype=torch.float64),
            mask_id=STAGE4_MASK_ID,
            valid_positions=stage4_valid,
        ),
    ),
    stage4_expected_failure(
        "rank-two loss time",
        lambda: mdlm_token_losses(
            stage4_logits,
            stage4_target,
            stage4_xt,
            torch.tensor([[0.25]], dtype=torch.float64),
            mask_id=STAGE4_MASK_ID,
            valid_positions=stage4_valid,
        ),
    ),
    stage4_expected_failure(
        "clean target is MASK",
        lambda: mdlm_token_losses(
            stage4_logits,
            torch.tensor([[0, 1, 3, 0]], dtype=torch.long),
            stage4_xt,
            stage4_time,
            mask_id=STAGE4_MASK_ID,
            valid_positions=stage4_valid,
        ),
    ),
    stage4_expected_failure(
        "visible token differs from target",
        lambda: mdlm_token_losses(
            stage4_logits,
            stage4_target,
            torch.tensor([[2, 1, 3, 0]], dtype=torch.long),
            stage4_time,
            mask_id=STAGE4_MASK_ID,
            valid_positions=stage4_valid,
        ),
    ),
    stage4_expected_failure(
        "invalid mask ID",
        lambda: substitution_log_probs(
            stage4_logits, stage4_xt, mask_id=STAGE4_VOCAB_SIZE
        ),
    ),
    stage4_expected_failure(
        "all-padding reduction",
        lambda: reduce_mdlm_loss(
            torch.zeros((1, 2), dtype=torch.float64),
            torch.zeros((1, 2), dtype=torch.bool),
            mode="repo_global_token_mean",
        ),
    ),
]
print(
    {
        "gradient_only_at_masked_clean_logits": True,
        "raw_logits_remained_immutable": torch.equal(
            stage4_logits, stage4_logits_before
        ),
        "nonterminal_mask_id_tested": STAGE4_MASK_ID,
        "expected_failures_caught": len(stage4_failure_labels),
    }
)


## Match the pinned BioNeMo MoCo loss

## Purpose and intuition

The final oracle checks that our transparent implementation agrees numerically with the exact dependency used by the released repository. MoCo writes the positive weight as

$$
\frac{\sigma\prime(t)}{\exp(\sigma(t))-1}=-\frac{\alpha\prime(t)}{1-\alpha(t)}=\frac{1}{t}.
$$

Its loss argument named mask means valid or non-padding positions. It is not the diffusion mask. With global_mean=True, the released default divides total loss by all valid tokens.

## Concrete example

For the length-4 toy, MoCo global mean and our global mean both equal $\log 2$. Its per-sequence result averaged over the batch also equals $\log 2$, while the paper token-sum result is $4\log 2$.

## What the code below computes and expected evidence

The next cell builds the pinned prior, time distribution, and log-linear noise transform, then calls MDLM.loss with cloned logits because its private SUBS helper mutates its input. Expect a dictionary showing the pinned package version, both agreement flags true, the paper/global factor-four check true, GenMol_BERT_receives_time false, and repo_alpha_at_t_equals_1 equal to 0.001.

## Scope and paper/repository distinctions

This oracle covers the loss only, not the full model or training loop. Three released choices differ from the displayed paper notation: BERT receives tokens and attention_mask but no explicit $t$; the schedule ends at $\alpha(1)=0.001$ rather than zero; and the default reduction is a global valid-token mean rather than a per-sequence token sum.

### Comprehension checkpoint

1. In MDLM.loss, what does the argument named mask represent?
2. Why must the logits be cloned before calling the pinned helper?
3. Which three choices differ from the displayed paper equation?


In [ ]:
from importlib.metadata import version as distribution_version

stage4_repo_prior = DiscreteMaskedPrior(
    num_classes=STAGE4_VOCAB_SIZE,
    mask_dim=STAGE4_MASK_ID,
)
stage4_repo_mdlm = MDLM(
    time_distribution=UniformTimeDistribution(),
    prior_distribution=stage4_repo_prior,
    noise_schedule=LogLinearExpNoiseTransform(eps=STAGE4_NOISE_EPS),
)

# Clone because the library's private SUBS helper mutates its logits argument.
stage4_repo_global_loss = stage4_repo_mdlm.loss(
    stage4_logits.clone(),
    stage4_target,
    stage4_xt,
    stage4_time,
    mask=stage4_valid,
    global_mean=True,
)
stage4_repo_per_sequence_loss = stage4_repo_mdlm.loss(
    stage4_logits.clone(),
    stage4_target,
    stage4_xt,
    stage4_time,
    mask=stage4_valid,
    global_mean=False,
).mean()

assert torch.allclose(
    stage4_repo_global_loss,
    stage4_global_loss,
    atol=1e-10,
    rtol=1e-10,
)
assert torch.allclose(
    stage4_repo_per_sequence_loss,
    stage4_sequence_loss,
    atol=1e-10,
    rtol=1e-10,
)

print(
    {
        "bionemo_moco_version": distribution_version("bionemo-moco"),
        "our_global_mean_matches_MoCo": True,
        "our_sequence_mean_matches_MoCo": True,
        "paper_sum_is_four_times_global_mean_in_this_length_4_example": math.isclose(
            float(stage4_paper_loss),
            4.0 * float(stage4_repo_global_loss),
            rel_tol=1e-12,
        ),
        "GenMol_BERT_receives_time": False,
        "repo_alpha_at_t_equals_1": STAGE4_NOISE_EPS,
    }
)


## Stage 4 completion gate

Stage 4 now connects the Stage 3 posterior to a tested weighted cross-entropy implementation. The next stage introduces the BERT denoiser only after the schedule, strict SUBS, the two masks, reductions, and repository deviations are understood.

### Comprehension checkpoint

1. Starting from $\alpha(t)=1-(1-\epsilon)t$, derive $-\alpha\prime(t)/(1-\alpha(t))=1/t$.
2. If xt is visible, why does strict SUBS copy it and make its loss zero even when the raw logits disagree?
3. Distinguish valid_positions from $1[xt=MASK]$. Which one supplies the released global-mean denominator?
4. In the toy at $t=0.25$ with one masked true token assigned probability 0.5, compute the paper token-sum loss and the global valid-token mean.
5. For valid lengths [4, 2] and loss sums [2, 6], explain why global mean is 8/6 but sequence mean is 1.75.
6. Name the three differences between displayed paper Equation (3) and the released code: explicit time conditioning, the terminal alpha value, and reduction.

## Stop condition

Do not continue to Stage 5 until you can answer all six questions without running the cells and can explain which assertion or printed value supports each answer. If any answer is unclear, stop here and revisit the corresponding Stage 4 pair.


# Stage 5 - The BERT denoiser

## Lesson 5.1 - Build the clean-token predictor

Stages 2 through 4 defined corruption, reverse probabilities, strict SUBS, and the loss while treating `x_theta` as an abstract predictor. The next cell creates that predictor as a randomly initialized Hugging Face `BertForMaskedLM` whose configuration exactly matches `configs/base.yaml`.

For batch size `B`, padded length `L`, vocabulary size `K = 1880`, and hidden width `H = 768`:

```text
noisy SAFE IDs z_t               [B,L]
    -> token and position states [B,L,768]
    -> 12 BERT encoder blocks    [B,L,768]
    -> masked-LM head            [B,L,1880] raw logits
```

The model has 12 attention heads per block, so each head has dimension `768 / 12 = 64`. Its feed-forward width is 3072 and its maximum length is 256. Bidirectional attention allows a query to use valid context on both sides, but absolute position embeddings mean SAFE order still matters.

At each position `i`, the MLM head returns 1880 unnormalized scores. These logits are not probabilities and are not yet the strict-SUBS distribution `x_theta`.

### Paper, released code, and notebook distinction

The paper writes the denoiser as `x_theta(z_t,t)`. The released BERT call has no explicit time embedding: it receives noisy IDs and the padding attention mask. Time still affects learning through the corruption level in `z_t` and the `1/t` loss weight.

The next cell verifies exactly 87,291,992 unique trainable parameters and confirms that the input token embedding and output classification weights share the same storage. It loads no English-BERT or GenMol checkpoint, so all predictions remain random.

The model configuration records `torch_dtype=float32`, while the released Trainer configuration separately requests `bf16` precision. Their effective interaction depends on the framework and runtime. This notebook makes the narrower, observed claim that its later smoke forward and backward run without autocast and produce FP32 logits. It does not reproduce or establish the released full-training precision behavior.

Expected output is an architecture table plus successful assertions for the exact config, parameter count, tied weights, vocabulary, maximum length, PAD ID, and bidirectional encoder mode.

### Comprehension checkpoint

1. Why is the output shape `[B,L,1880]` rather than `[B,1880]`?
2. What is the difference between an MLM logit and a token probability?
3. What does weight tying share?
4. Does the released BERT receive `t` explicitly?
5. Why are random-initialized predictions not evidence of molecular quality?


In [ ]:
from transformers import BertConfig, BertForMaskedLM
from omegaconf import OmegaConf

STAGE5_MODEL_CONFIG = {
    "attention_probs_dropout_prob": 0.1,
    "classifier_dropout": None,
    "hidden_act": "gelu",
    "hidden_dropout_prob": 0.1,
    "hidden_size": 768,
    "initializer_range": 0.02,
    "intermediate_size": 3072,
    "layer_norm_eps": 1.0e-12,
    "max_position_embeddings": 256,
    "model_type": "bert",
    "num_attention_heads": 12,
    "num_hidden_layers": 12,
    "pad_token_id": 3,
    "position_embedding_type": "absolute",
    "torch_dtype": "float32",
    "type_vocab_size": 2,
    "use_cache": True,
    "vocab_size": 1880,
}

released_config = OmegaConf.load(PROJECT_ROOT / "configs" / "base.yaml")
released_model_config = OmegaConf.to_container(
    released_config.model,
    resolve=True,
)
assert released_model_config == STAGE5_MODEL_CONFIG

assert STAGE5_MODEL_CONFIG["vocab_size"] == MODEL_VOCAB_SIZE
assert STAGE5_MODEL_CONFIG["max_position_embeddings"] == MAX_MODEL_POSITIONS
assert STAGE5_MODEL_CONFIG["pad_token_id"] == tokenizer.pad_token_id
assert STAGE5_MODEL_CONFIG["hidden_size"] % STAGE5_MODEL_CONFIG["num_attention_heads"] == 0

STAGE5_SEED = 5005
torch.manual_seed(STAGE5_SEED)
torch.cuda.manual_seed_all(STAGE5_SEED)
stage5_bert_config = BertConfig.from_dict(dict(STAGE5_MODEL_CONFIG))
stage5_denoiser = BertForMaskedLM(stage5_bert_config)

stage5_parameter_count = sum(parameter.numel() for parameter in stage5_denoiser.parameters())
stage5_trainable_count = sum(
    parameter.numel()
    for parameter in stage5_denoiser.parameters()
    if parameter.requires_grad
)
stage5_input_embedding = stage5_denoiser.get_input_embeddings().weight
stage5_output_embedding = stage5_denoiser.get_output_embeddings().weight

assert stage5_parameter_count == 87_291_992
assert stage5_trainable_count == stage5_parameter_count
assert stage5_input_embedding.data_ptr() == stage5_output_embedding.data_ptr()
assert stage5_bert_config.is_decoder is False

stage5_architecture_df = pd.DataFrame(
    [
        ("vocabulary categories K", stage5_bert_config.vocab_size),
        ("maximum positions", stage5_bert_config.max_position_embeddings),
        ("hidden width", stage5_bert_config.hidden_size),
        ("encoder blocks", stage5_bert_config.num_hidden_layers),
        ("attention heads per block", stage5_bert_config.num_attention_heads),
        ("dimensions per head", stage5_bert_config.hidden_size // stage5_bert_config.num_attention_heads),
        ("feed-forward width", stage5_bert_config.intermediate_size),
        ("unique trainable parameters", stage5_parameter_count),
    ],
    columns=["component", "value"],
)

print(
    {
        "random_initialization": True,
        "released_config_exact_match": True,
        "input_output_embedding_weights_tied": True,
        "explicit_time_input": False,
        "parameter_count": stage5_parameter_count,
    }
)
display(stage5_architecture_df)

## Lesson 5.2 - Corrupt one batch and run BERT on the selected GPU

The next cell connects the Stage 2 absorbing forward process to BERT.

It:

1. re-probes the first physical GPU selected in Stage 0, requiring the same UUID and the full utilization/free-memory/compute-mode policy while allowing and recording active process rows;
2. samples one ordered-stratified time per molecule;
3. corrupts the complete padded ID tensor, matching the released raw-tensor path;
4. keeps the original padding attention tensor;
5. moves the model and batch to logical `cuda:0`, meaning the first Stage 0 selection;
6. runs `eval()` with `no_grad()`;
7. asserts output shape, FP32 dtype, device, and finite values.

If Stage 0 exposed several GPUs, this smoke test deliberately uses only the first logical device. Visibility is not DDP.

### Diffusion MASK, padding attention, and loss validity

These are three separate objects:

- A diffusion `[MASK]` is token ID 4 inside `input_ids`. It means the clean token at a real position is hidden.
- Hugging Face's 2-D `attention_mask` has shape `[B,L]` and controls key/context columns.
- `valid_positions` is the Boolean non-padding mask later used by the loss.

Hugging Face expands the 2-D attention tensor across heads and query positions. For query `i` and key `j`:

```text
score(i,j) = q_i dot k_j / sqrt(64) + key_bias(j)

key_bias(j) = 0          when attention_mask[b,j] = 1
key_bias(j) = -infinity  when attention_mask[b,j] = 0
```

Thus value 0 suppresses key column `j` as context for every query. It does not delete query row `j`. BERT still computes a hidden state and 1880 logits at a padded query row, and that row can attend to valid keys.

A real diffusion `[MASK]` keeps attention value 1 because it remains a real sequence position and an available key/context position. A PAD position has attention 0. The later `valid_positions` argument, not the BERT attention operation, removes padded rows from the loss.

### Concrete example

```text
clean IDs:      [CLS] C      N      O [SEP] [PAD]
sampled z_t:    [CLS] C   [MASK]     O [SEP] [PAD]
attention:        1   1      1       1   1     0
loss valid:       1   1      1       1   1     0
loss active:      0   0      1       0   0     0
```

The released corruption function can replace a PAD ID internally because it samples on the padded tensor. Such a replacement may change the padded query row's logits. It cannot change valid query logits because that position is suppressed as a key at every layer, and it cannot contribute directly to loss because `valid_positions` is false there.

For the current data, expected shapes are `input_ids [5,55]` and `logits [5,55,1880]`. The displayed table reports each molecule's time, mask probability, valid-token count, and sampled valid masks.

### Comprehension checkpoint

1. If `attention_mask[b,j] = 0`, what is suppressed?
2. What output can BERT still compute at query row `j`?
3. Why must a real diffusion `[MASK]` position keep attention value 1?
4. Which tensor excludes padded rows from the later loss?
5. Why can logical `cuda:0` refer to a physical GPU other than GPU 0?


In [ ]:
STAGE5_LOGICAL_DEVICE = torch.device("cuda:0")
STAGE5_PHYSICAL_GPU_ID = SELECTED_PHYSICAL_GPU_IDS[0]
stage5_gpu_reprobe = probe_physical_gpu(STAGE5_PHYSICAL_GPU_ID)
stage5_rejection_reasons = list(stage5_gpu_reprobe["reasons"])
if stage5_gpu_reprobe.get("uuid") != SELECTED_GPU_UUIDS[0]:
    stage5_rejection_reasons.append(
        "UUID changed from "
        f"{SELECTED_GPU_UUIDS[0]!r} to {stage5_gpu_reprobe.get('uuid')!r}"
    )
if stage5_rejection_reasons:
    raise RuntimeError(
        f"Physical GPU {STAGE5_PHYSICAL_GPU_ID} no longer passes the Stage 0 "
        f"eligibility and identity policy: {stage5_rejection_reasons}. "
        "Restart the kernel so Stage 0 can select again."
    )
stage5_compute_processes = stage5_gpu_reprobe["compute_processes"]

stage5_batch_size = clean_token_ids.shape[0]
stage5_times_cpu = sample_ordered_stratified_times(
    stage5_batch_size,
    generator=torch.Generator(device="cpu").manual_seed(5501),
)

# No corruptible_positions argument: this mirrors the released raw padded-tensor corruption.
stage5_xt_cpu, stage5_mask_events_cpu, stage5_mask_probability_cpu = sample_forward_absorbing(
    clean_token_ids,
    stage5_times_cpu,
    generator=torch.Generator(device="cpu").manual_seed(5502),
)
stage5_attention_cpu = model_batch["attention_mask"]
stage5_valid_cpu = stage5_attention_cpu.bool()

assert bool((stage5_xt_cpu[stage5_valid_cpu] == tokenizer.mask_token_id).any())
assert torch.all(stage5_attention_cpu[stage5_valid_cpu] == 1)
assert torch.all(stage5_attention_cpu[~stage5_valid_cpu] == 0)
assert torch.all(
    stage5_attention_cpu[
        stage5_valid_cpu & stage5_xt_cpu.eq(tokenizer.mask_token_id)
    ] == 1
)

stage5_denoiser = stage5_denoiser.to(STAGE5_LOGICAL_DEVICE)
stage5_denoiser.eval()
stage5_xt = stage5_xt_cpu.to(STAGE5_LOGICAL_DEVICE)
stage5_attention = stage5_attention_cpu.to(STAGE5_LOGICAL_DEVICE)
stage5_valid = stage5_valid_cpu.to(STAGE5_LOGICAL_DEVICE)

with torch.no_grad():
    stage5_eval_logits = stage5_denoiser(
        input_ids=stage5_xt,
        attention_mask=stage5_attention,
        return_dict=True,
    ).logits

assert stage5_eval_logits.shape == (
    stage5_batch_size,
    clean_token_ids.shape[1],
    MODEL_VOCAB_SIZE,
)
assert stage5_eval_logits.dtype == torch.float32
assert stage5_eval_logits.device == STAGE5_LOGICAL_DEVICE
assert bool(torch.isfinite(stage5_eval_logits).all())

stage5_batch_df = pd.DataFrame(
    {
        "molecule": representation_df["name"].tolist(),
        "t": stage5_times_cpu.tolist(),
        "P(MASK)": stage5_mask_probability_cpu.tolist(),
        "valid tokens": stage5_valid_cpu.sum(dim=1).tolist(),
        "sampled valid masks": [
            int(stage5_mask_events_cpu[row][stage5_valid_cpu[row]].sum())
            for row in range(stage5_batch_size)
        ],
    }
)
print(
    {
        "physical_GPU_selected_by_stage0": STAGE5_PHYSICAL_GPU_ID,
        "PyTorch_logical_device": str(STAGE5_LOGICAL_DEVICE),
        "extra_visible_GPUs_unused_in_this_smoke_test": max(NUM_GPUS - 1, 0),
        "reprobe_free_memory_MiB": stage5_gpu_reprobe["free_memory_mib"],
        "reprobe_utilization_percent": stage5_gpu_reprobe["utilization_percent"],
        "reprobe_compute_process_count": stage5_gpu_reprobe["compute_process_count"],
        "reprobe_compute_processes": stage5_compute_processes,
        "input_shape": tuple(stage5_xt.shape),
        "logit_shape": tuple(stage5_eval_logits.shape),
        "MASK_attention_value": 1,
        "PAD_attention_value": 0,
    }
)
display(stage5_batch_df)


## Lesson 5.3 - Test context paths, padding isolation, and strict SUBS

A correct output shape is not enough. The next cell performs controlled interventions with dropout disabled.

### 1. Padding isolation at valid query rows

The code changes every token ID at an attention-zero PAD position to the ordinary token `C` while leaving the attention tensor unchanged.

```text
A: [CLS] C [SEP] [PAD] [PAD]
B: [CLS] C [SEP]   C     C
attention for both: 1 1 1 0 0
```

The two zero values suppress the last two key columns. They do not remove the last two query rows. Therefore the assertion compares logits only at `stage5_valid` query rows. Valid logits must agree within tolerance; padded-row logits are deliberately not compared and may differ.

### 2. Bidirectional information paths

The code places `[MASK]` near the middle of one valid sequence. It changes one real token on the left, restores it, and then changes one on the right. Both interventions must change the masked query's 1880-logit vector.

Because BERT has no causal left-to-right mask, either side can be context. A nonzero effect from a randomly initialized network proves only that both computation paths are wired. It does not prove learned chemistry or useful denoising.

### 3. Strict SUBS

Raw BERT can score all 1880 categories. Strict SUBS converts the logits into the diffusion model's clean-token distribution:

```text
visible z_t[i]:  copy z_t[i] with probability 1
masked z_t[i]:   normalize learned clean-token scores
all positions:   assign probability 0 to [MASK] as a clean token
```

Expected diagnostics are a near-zero maximum valid-logit change under the PAD-ID intervention, positive left and right context-effect norms, zero clean `[MASK]` probability, and exact copying of visible valid tokens.

### Comprehension checkpoint

1. Why does the padding test index logits with `stage5_valid`?
2. Are padded query-row logits required to remain unchanged?
3. What does a positive right-context effect establish?
4. Under strict SUBS, what distribution is used at a visible `C`?
5. Can the predicted clean distribution assign probability to `[MASK]`?


In [ ]:
# Padding invariance with dropout disabled.
stage5_pad_variant = stage5_xt.clone()
replacement_for_pad = tokenizer.convert_tokens_to_ids("C")
assert replacement_for_pad not in {
    tokenizer.unk_token_id,
    tokenizer.cls_token_id,
    tokenizer.sep_token_id,
    tokenizer.pad_token_id,
    tokenizer.mask_token_id,
}
stage5_pad_variant[~stage5_valid] = replacement_for_pad
with torch.no_grad():
    stage5_pad_variant_logits = stage5_denoiser(
        input_ids=stage5_pad_variant,
        attention_mask=stage5_attention,
        return_dict=True,
    ).logits
stage5_padding_max_difference = float(
    (stage5_eval_logits[stage5_valid] - stage5_pad_variant_logits[stage5_valid])
    .abs()
    .max()
)
assert torch.allclose(
    stage5_eval_logits[stage5_valid],
    stage5_pad_variant_logits[stage5_valid],
    atol=1.0e-6,
    rtol=1.0e-5,
)

# Build one controlled query with visible context on both sides.
stage5_longest_row = int(stage5_valid_cpu.sum(dim=1).argmax())
stage5_context_length = int(stage5_valid_cpu[stage5_longest_row].sum())
stage5_context_base = clean_token_ids[
    stage5_longest_row : stage5_longest_row + 1,
    :stage5_context_length,
].to(STAGE5_LOGICAL_DEVICE)
stage5_context_attention = torch.ones_like(stage5_context_base)
stage5_query_position = stage5_context_length // 2
stage5_left_position = 1
stage5_right_position = stage5_context_length - 2
assert 0 < stage5_left_position < stage5_query_position < stage5_right_position

stage5_context_base[0, stage5_query_position] = tokenizer.mask_token_id
stage5_left_variant = stage5_context_base.clone()
stage5_right_variant = stage5_context_base.clone()

context_replacements = [
    token_id
    for token_id in (
        tokenizer.convert_tokens_to_ids("C"),
        tokenizer.convert_tokens_to_ids("N"),
        tokenizer.convert_tokens_to_ids("O"),
    )
    if token_id not in {
        tokenizer.unk_token_id,
        tokenizer.cls_token_id,
        tokenizer.sep_token_id,
        tokenizer.pad_token_id,
        tokenizer.mask_token_id,
    }
]
stage5_left_variant[0, stage5_left_position] = next(
    token_id
    for token_id in context_replacements
    if token_id != int(stage5_context_base[0, stage5_left_position])
)
stage5_right_variant[0, stage5_right_position] = next(
    token_id
    for token_id in context_replacements
    if token_id != int(stage5_context_base[0, stage5_right_position])
)

with torch.no_grad():
    stage5_base_query_logits = stage5_denoiser(
        stage5_context_base, attention_mask=stage5_context_attention
    ).logits[0, stage5_query_position]
    stage5_left_query_logits = stage5_denoiser(
        stage5_left_variant, attention_mask=stage5_context_attention
    ).logits[0, stage5_query_position]
    stage5_right_query_logits = stage5_denoiser(
        stage5_right_variant, attention_mask=stage5_context_attention
    ).logits[0, stage5_query_position]

stage5_left_context_effect = float(
    torch.linalg.vector_norm(stage5_left_query_logits - stage5_base_query_logits)
)
stage5_right_context_effect = float(
    torch.linalg.vector_norm(stage5_right_query_logits - stage5_base_query_logits)
)
assert stage5_left_context_effect > 0.0
assert stage5_right_context_effect > 0.0

stage5_eval_log_probs = substitution_log_probs(
    stage5_eval_logits,
    stage5_xt,
    mask_id=tokenizer.mask_token_id,
)
stage5_eval_probs = stage5_eval_log_probs.exp()
stage5_visible_valid = stage5_valid & stage5_xt.ne(tokenizer.mask_token_id)
assert torch.equal(
    stage5_eval_probs[..., tokenizer.mask_token_id],
    torch.zeros_like(stage5_eval_probs[..., tokenizer.mask_token_id]),
)
assert torch.equal(
    stage5_eval_probs[stage5_visible_valid].argmax(dim=-1),
    stage5_xt[stage5_visible_valid],
)

print(
    {
        "left_context_changes_masked_query_logits_L2": stage5_left_context_effect,
        "right_context_changes_masked_query_logits_L2": stage5_right_context_effect,
        "max_valid_logit_change_when_only_PAD_ids_change": stage5_padding_max_difference,
        "strict_SUBS_mask_probability": 0.0,
        "strict_SUBS_visible_tokens_are_copied": True,
    }
)

## Lesson 5.4 - Connect BERT to Equation (3) and gradients

The next cell checks one complete differentiable batch:

```text
clean IDs x
    -> sampled t and corrupted IDs z_t
    -> BERT FP32 logits
    -> strict-SUBS log probabilities
    -> weighted masked-token losses
    -> released global valid-token mean
    -> backward gradients
```

This is a training-plumbing test, not an optimizer step or training loop.

For a valid position `i` in sequence `b`, the implemented token term is:

```text
loss[b,i] =
    -(1 / t[b]) * 1[z_t[b,i] = MASK]
    * log x_theta[b,i,x[b,i]]
```

Visible positions contribute zero through the indicator. Padded rows contribute zero through `valid_positions`. The released global-token reduction sums these terms and divides by the total number of valid tokens, not by the number of masked tokens.

For example, if `t = 0.25` and the model assigns probability 0.10 to the true clean token at a masked valid position, its term before batch reduction is:

```text
-(1 / 0.25) * log(0.10) = 9.2103
```

The cell switches to `train()`, so dropout is active, clears old gradients, computes a finite positive scalar loss, and calls `backward()`. Backward stores derivatives in parameter `.grad` fields; it does not change parameter values. The sentinel assertion proves this. Only a later `optimizer.step()` would update weights.

The displayed top-5 table uses a random model. It checks that masked coordinates, target IDs, probabilities, and vocabulary decoding connect correctly. It is a shape and plumbing diagnostic, not a molecule-quality evaluation.

Expected diagnostics include finite nonzero parameter gradients, a global gradient norm, unchanged sentinel weight, scalar loss, and peak allocated GPU memory. Cleanup moves the model to CPU and releases large allocations, but the CUDA context remains until the kernel restarts. This FP32 smoke backward does not reproduce full released training precision or distributed training.

### Comprehension checkpoint

1. Which positions have nonzero Equation (3) terms?
2. Why does `t = 0.25` produce a weight of 4?
3. What does `backward()` create?
4. What additional operation would change parameters?
5. Why is the random top-5 table not a molecular-quality result?


In [ ]:
stage5_target = clean_token_ids.to(STAGE5_LOGICAL_DEVICE)
stage5_time = stage5_times_cpu.to(STAGE5_LOGICAL_DEVICE)
stage5_denoiser.train()
stage5_denoiser.zero_grad(set_to_none=True)

stage5_sentinel_before = (
    stage5_denoiser.get_input_embeddings().weight[0, 0].detach().clone()
)
stage5_train_output = stage5_denoiser(
    input_ids=stage5_xt,
    attention_mask=stage5_attention,
    return_dict=True,
)
stage5_train_logits = stage5_train_output.logits
stage5_loss_parts = mdlm_token_losses(
    stage5_train_logits,
    stage5_target,
    stage5_xt,
    stage5_time,
    mask_id=tokenizer.mask_token_id,
    valid_positions=stage5_valid,
)
stage5_loss = reduce_mdlm_loss(
    stage5_loss_parts["token_losses"],
    stage5_valid,
    mode="repo_global_token_mean",
)

assert stage5_loss.ndim == 0
assert bool(torch.isfinite(stage5_loss))
assert float(stage5_loss) > 0.0
stage5_loss.backward()

stage5_gradient_square_sum = torch.zeros((), device=STAGE5_LOGICAL_DEVICE)
stage5_parameters_with_gradient = 0
for parameter in stage5_denoiser.parameters():
    if parameter.grad is None:
        continue
    if not bool(torch.isfinite(parameter.grad).all()):
        raise FloatingPointError("A BERT parameter gradient is not finite.")
    stage5_parameters_with_gradient += 1
    stage5_gradient_square_sum += parameter.grad.detach().float().square().sum()

stage5_gradient_norm = float(stage5_gradient_square_sum.sqrt())
stage5_sentinel_after_backward = (
    stage5_denoiser.get_input_embeddings().weight[0, 0].detach().clone()
)
assert stage5_parameters_with_gradient > 0
assert math.isfinite(stage5_gradient_norm) and stage5_gradient_norm > 0.0
assert torch.equal(stage5_sentinel_before, stage5_sentinel_after_backward)

stage5_masked_coordinates = torch.nonzero(
    stage5_valid & stage5_xt.eq(tokenizer.mask_token_id),
    as_tuple=False,
)
stage5_prediction_rows = []
for batch_index, position in stage5_masked_coordinates[:3].tolist():
    true_id = int(stage5_target[batch_index, position])
    probabilities = stage5_loss_parts["log_probs"][batch_index, position].exp()
    top_probabilities, top_ids = probabilities.topk(5)
    stage5_prediction_rows.append(
        {
            "molecule": representation_df.iloc[batch_index]["name"],
            "position": position,
            "true token": tokenizer.convert_ids_to_tokens(true_id),
            "P(true)": float(probabilities[true_id]),
            "random-init top-5 tokens": tokenizer.convert_ids_to_tokens(top_ids.tolist()),
            "random-init top-5 probabilities": [
                round(float(value), 6) for value in top_probabilities
            ],
        }
    )

stage5_loss_value = float(stage5_loss.detach())
stage5_peak_gpu_memory_mib = torch.cuda.max_memory_allocated(
    STAGE5_LOGICAL_DEVICE
) / 2**20
display(pd.DataFrame(stage5_prediction_rows))
print(
    {
        "repo_global_token_mean_loss": stage5_loss_value,
        "parameters_with_gradient": stage5_parameters_with_gradient,
        "global_gradient_L2_norm": stage5_gradient_norm,
        "parameter_changed_without_optimizer_step": False,
        "peak_allocated_GPU_memory_MiB": stage5_peak_gpu_memory_mib,
    }
)

# Release the large model, activation, and gradient allocations. The CUDA context
# itself remains until the notebook kernel is restarted.
stage5_denoiser.zero_grad(set_to_none=True)
del stage5_train_output, stage5_train_logits, stage5_loss_parts, stage5_loss
del stage5_eval_logits, stage5_pad_variant_logits, stage5_eval_log_probs, stage5_eval_probs
del stage5_xt, stage5_attention, stage5_valid, stage5_target, stage5_time
del stage5_pad_variant, stage5_context_attention, stage5_masked_coordinates
del stage5_context_base, stage5_left_variant, stage5_right_variant
del stage5_base_query_logits, stage5_left_query_logits, stage5_right_query_logits
del stage5_sentinel_before, stage5_sentinel_after_backward, stage5_gradient_square_sum
del probabilities, top_probabilities, top_ids
stage5_denoiser = stage5_denoiser.to("cpu")
torch.cuda.empty_cache()
print(
    "Stage 5 model location after cleanup:",
    next(stage5_denoiser.parameters()).device,
    "- restart the kernel to release its CUDA context",
)

## Stage 5 completion gate

The complete differentiable path now exists for one batch:

```text
x -> sample t and z_t -> BERT -> logits -> strict SUBS
  -> weighted masked-token loss -> gradients
```

BERT has no explicit time input in the released path. Time acts through the corruption level and the loss weight.

### Position-by-position accounting

```text
position:       [CLS]   C   [MASK]   N   [SEP]  [PAD]
attention key:      1   1       1    1       1      0
BERT query logits: yes yes     yes  yes     yes    yes
strict SUBS:     copy copy predict copy    copy   copy
loss active:        0   0       1    0       0      0
```

For the displayed visible PAD row, strict SUBS copies PAD because SUBS only sees that the current token is visible. The separate valid-position mask excludes the row from loss. Its attention value 0 suppresses it as a key/context source, but BERT still computes its query logits.

### What the diagnostics establish

- The released BERT fields match exactly: 12 blocks, 12 heads, width 768, feed-forward width 3072, 256 positions, and 1880 categories.
- The random model has 87,291,992 unique trainable parameters and tied input/output token weights.
- A noisy `[B,L]` batch produces finite FP32 `[B,L,1880]` logits in this smoke run.
- A real diffusion `[MASK]` remains attention-valid; an attention-zero PAD position cannot serve as key/context.
- Changing only attention-zero PAD IDs leaves logits at valid query rows unchanged; padded query logits are not required to match.
- Left and right valid context both have computation paths to a masked query.
- Strict SUBS copies visible tokens and forbids clean `[MASK]` probability.
- Equation (3) produces a finite scalar and finite nonzero gradients.
- `backward()` alone leaves parameters unchanged.
- The released BERT call has no explicit `t` input.

These diagnostics do not establish learned chemistry, molecule quality, paper-scale optimization, bf16 behavior, multi-GPU DDP, EMA, checkpointing, generation, confidence sampling, remasking, or guidance.

### Comprehension checkpoint

1. What does each axis of `[B,L,1880]` mean?
2. If `attention_mask[b,j] = 0`, what is suppressed and what is still computed?
3. Why does a real diffusion `[MASK]` use attention 1 while PAD uses 0?
4. Where does `t` affect learning if BERT does not receive it explicitly?
5. What does strict SUBS do at visible, masked, and visible-PAD positions?
6. What is the difference between `backward()` and `optimizer.step()`?
7. Why do nonzero context effects and random top-5 predictions not prove learned chemistry?

**Stop here. Do not continue to the next stage until every checkpoint answer and the key-versus-query attention distinction are clear.**


# Stage 6 - From gradients to one AdamW update

## The one new dependency

Stage 5 stopped after <code>backward()</code>. That operation computed gradients and stored them in each parameter's <code>.grad</code> field, but it did not change any parameter. Stage 6 adds exactly one optimizer transaction:

    clear old gradients
      -> sample a fresh legitimate diffusion corruption
      -> BERT forward pass
      -> Equation (3) loss
      -> backward pass
      -> clip the global gradient norm
      -> AdamW step
      -> clear gradients again

This is one update out of the paper's 50,000 updates, or <code>1 / 50000 = 0.002%</code> of the reported optimization steps. It is a mechanism test on the five-sequence teaching batch, not a claim of trained chemistry.

## Paper, released repository, and this lesson

- **Paper, Appendix D.2:** AdamW, learning rate <code>3e-4</code>, <code>beta1 = 0.9</code>, <code>beta2 = 0.999</code>, batch size 2048, and 50,000 steps.
- **Released <code>configs/base.yaml</code>:** <code>eps = 1e-8</code>, <code>weight_decay = 0</code>, and gradient clipping threshold 1.0.
- **Released <code>src/genmol/model.py</code>:** a separate constant schedule with 2,500 warmup steps and a separate EMA update.
- **This lesson:** exact AdamW values plus the repository's norm clipping, but no scheduler or EMA yet. Omitting the scheduler here makes the first optimizer update use the stated base learning rate, so the distinction between <code>backward()</code> and <code>optimizer.step()</code> is directly visible.

The paper does not specify epsilon, weight decay, clipping, a learning-rate scheduler, EMA, precision details, gradient accumulation, or DDP mechanics. Those repository choices must not be mislabeled as paper equations.

## What the code below does

The next cell reads the pinned local configuration instead of copying numbers silently. It rejects configuration drift, constructs the exact optimizer keyword dictionary for this stage, and displays where each value came from. It performs no tensor update and uses no GPU.

### Comprehension checkpoint

1. Which three AdamW values are explicitly stated in the paper?
2. Which values come only from the released configuration?
3. Why is the 2,500-step scheduler deliberately outside this first optimizer lesson?
4. Does one teaching update reproduce the paper's 50,000-step training run?


In [ ]:
from pathlib import Path

import pandas as pd
from omegaconf import OmegaConf

stage6_config_path = Path.cwd().resolve() / "configs" / "base.yaml"
if stage6_config_path.parent.parent.name != "genmolv2":
    raise RuntimeError(f"Unexpected project root: {stage6_config_path.parent.parent}")

stage6_release_config = OmegaConf.load(stage6_config_path)
stage6_optimizer_kwargs = {
    "lr": float(stage6_release_config.optim.lr),
    "betas": (
        float(stage6_release_config.optim.beta1),
        float(stage6_release_config.optim.beta2),
    ),
    "eps": float(stage6_release_config.optim.eps),
    "weight_decay": float(stage6_release_config.optim.weight_decay),
}
stage6_gradient_clip_norm = float(stage6_release_config.trainer.gradient_clip_val)
stage6_global_batch_size = int(stage6_release_config.loader.global_batch_size)
stage6_paper_training_steps = int(stage6_release_config.trainer.max_steps)

stage6_expected_optimizer_kwargs = {
    "lr": 3e-4,
    "betas": (0.9, 0.999),
    "eps": 1e-8,
    "weight_decay": 0.0,
}
assert stage6_optimizer_kwargs == stage6_expected_optimizer_kwargs
assert stage6_gradient_clip_norm == 1.0
assert stage6_global_batch_size == 2048
assert stage6_paper_training_steps == 50_000

stage6_config_df = pd.DataFrame(
    [
        ("learning rate", "3e-4", "paper Appendix D.2 and repository"),
        ("beta1", "0.9", "paper Appendix D.2 and repository"),
        ("beta2", "0.999", "paper Appendix D.2 and repository"),
        ("epsilon", "1e-8", "repository configs/base.yaml"),
        ("weight decay", "0", "repository configs/base.yaml"),
        ("global gradient norm threshold", "1.0", "repository configs/base.yaml"),
        ("reported optimizer steps", "50000", "paper Appendix D.2 and repository"),
    ],
    columns=["quantity", "value", "provenance"],
)

print(
    {
        "optimizer": "torch.optim.AdamW",
        "optimizer_kwargs": stage6_optimizer_kwargs,
        "gradient_clip_norm": stage6_gradient_clip_norm,
        "scheduler_constructed_in_stage6": False,
        "EMA_constructed_in_stage6": False,
    }
)
display(stage6_config_df)


## Lesson 6.2 - AdamW by hand before applying it to BERT

For optimizer step <code>k</code>, AdamW stores exponential moving averages of the gradient and squared gradient:

$$
m_k = \beta_1 m_{k-1} + (1 - \beta_1) g_k
$$

$$
v_k = \beta_2 v_{k-1} + (1 - \beta_2) g_k^2
$$

It removes the initialization bias with

$$
\hat m_k = \frac{m_k}{1 - \beta_1^k},
\qquad
\hat v_k = \frac{v_k}{1 - \beta_2^k}.
$$

A one-step AdamW update can be written as

$$
\theta_k = (1 - \eta\lambda)\theta_{k-1}
- \eta\frac{\hat m_k}{\sqrt{\hat v_k} + \epsilon}.
$$

Here <code>eta</code> is the learning rate and <code>lambda</code> is decoupled weight decay. The released GenMol configuration sets <code>lambda = 0</code>, so the decay multiplier is exactly 1. The class is still AdamW, but its decay term is inactive.

### Concrete scalar example

Use <code>theta0 = 2</code>, <code>g1 = 0.5</code>, <code>beta1 = 0.9</code>, and <code>beta2 = 0.999</code>:

    m1 = 0.1 * 0.5 = 0.05
    v1 = 0.001 * 0.5^2 = 0.00025
    m_hat1 = 0.05 / 0.1 = 0.5
    v_hat1 = 0.00025 / 0.001 = 0.25
    theta1 is approximately 1.9997

The next cell computes every quantity in float64 and checks it against one real <code>torch.optim.AdamW.step()</code>. This tiny oracle isolates optimizer algebra from BERT, diffusion sampling, padding, and GPU behavior.

### Comprehension checkpoint

1. Why are <code>m1</code> and <code>v1</code> smaller than the raw gradient quantities?
2. Why do the bias corrections recover 0.5 and 0.25 at the first step?
3. What happens to the decoupled decay term when <code>weight_decay = 0</code>?
4. Which call actually changes <code>theta</code>?


In [ ]:
stage6_theta0 = torch.tensor(2.0, dtype=torch.float64)
stage6_g1 = torch.tensor(0.5, dtype=torch.float64)
stage6_beta1, stage6_beta2 = stage6_optimizer_kwargs["betas"]
stage6_lr = stage6_optimizer_kwargs["lr"]
stage6_eps = stage6_optimizer_kwargs["eps"]
stage6_weight_decay = stage6_optimizer_kwargs["weight_decay"]

stage6_m1 = (1.0 - stage6_beta1) * stage6_g1
stage6_v1 = (1.0 - stage6_beta2) * stage6_g1.square()
stage6_m_hat1 = stage6_m1 / (1.0 - stage6_beta1)
stage6_v_hat1 = stage6_v1 / (1.0 - stage6_beta2)
stage6_expected_theta1 = (
    stage6_theta0 * (1.0 - stage6_lr * stage6_weight_decay)
    - stage6_lr * stage6_m_hat1 / (stage6_v_hat1.sqrt() + stage6_eps)
)

stage6_scalar_parameter = torch.nn.Parameter(stage6_theta0.clone())
stage6_scalar_optimizer = torch.optim.AdamW(
    [stage6_scalar_parameter],
    **stage6_optimizer_kwargs,
)
stage6_scalar_parameter.grad = stage6_g1.clone()
stage6_scalar_before_step = stage6_scalar_parameter.detach().clone()
stage6_scalar_optimizer.step()
stage6_scalar_after_step = stage6_scalar_parameter.detach().clone()
stage6_scalar_state = stage6_scalar_optimizer.state[stage6_scalar_parameter]

assert torch.equal(stage6_scalar_before_step, stage6_theta0)
torch.testing.assert_close(stage6_scalar_state["exp_avg"], stage6_m1, rtol=1e-12, atol=1e-12)
torch.testing.assert_close(stage6_scalar_state["exp_avg_sq"], stage6_v1, rtol=1e-12, atol=1e-12)
torch.testing.assert_close(stage6_scalar_after_step, stage6_expected_theta1, rtol=1e-12, atol=1e-12)
assert int(stage6_scalar_state["step"].item()) == 1
assert not torch.equal(stage6_scalar_before_step, stage6_scalar_after_step)

stage6_scalar_df = pd.DataFrame(
    [
        ("g1", float(stage6_g1)),
        ("m1", float(stage6_m1)),
        ("v1", float(stage6_v1)),
        ("m_hat1", float(stage6_m_hat1)),
        ("v_hat1", float(stage6_v_hat1)),
        ("theta before optimizer.step", float(stage6_scalar_before_step)),
        ("theta after optimizer.step", float(stage6_scalar_after_step)),
        ("optimizer state step", int(stage6_scalar_state["step"].item())),
    ],
    columns=["quantity", "value"],
)
stage6_scalar_optimizer.zero_grad(set_to_none=True)
assert stage6_scalar_parameter.grad is None

display(stage6_scalar_df)
print("Scalar hand calculation and torch.optim.AdamW agree.")

del stage6_scalar_state, stage6_scalar_optimizer, stage6_scalar_parameter


## Lesson 6.3 - One real BERT optimizer transaction

### Gradient clipping

After <code>backward()</code>, concatenate all parameter gradients conceptually into one vector <code>g</code>. With threshold <code>C = 1</code>, norm clipping rescales all gradients by the same factor:

$$
c = \min\left(1, \frac{C}{\lVert g \rVert_2 + 10^{-6}}\right),
\qquad
g \leftarrow c g.
$$

This preserves the gradient direction while limiting its length. Clipping belongs after <code>backward()</code> because gradients must exist, and before <code>optimizer.step()</code> because AdamW must consume the clipped gradients. The paper does not state clipping; threshold 1.0 is a released-repository setting.

### Why clear gradients twice?

PyTorch accumulates into <code>parameter.grad</code>. Without <code>zero_grad(set_to_none=True)</code>, a later backward pass adds to old gradients. We clear before the transaction to prevent accidental accumulation and after the transaction to prove the next step would start clean.

### GPU selection without assuming physical GPU 0

Stage 0 already exposed the requested count of dynamically selected GPUs and stored their physical IDs in <code>SELECTED_PHYSICAL_GPU_IDS</code>. The code below:

1. re-probes every selected physical GPU with the complete Stage 0 policy and exact UUID check;
2. retains only cards with at least 30,000 MiB free, utilization strictly below 10 percent, and non-prohibited compute mode; active process rows are recorded but do not disqualify a card;
3. chooses the eligible selected logical device with the least notebook allocation, breaking ties by logical index; and
4. prints the physical ID, derived logical device, and current eligibility telemetry.

This lesson needs one device for one optimizer transaction. If <code>NUM_GPUS</code> is greater than one, the other selected devices remain unused in this lesson. That is not multi-GPU training. The released implementation uses <code>torchrun</code> and DDP, where replicas synchronize gradients before a common update; DDP belongs in a later stage. Independent unsynchronized optimizers would train different models.

### What the code below proves

A disposable deep copy of the Stage 5 random model is updated so the original Stage 5 object stays unchanged. A bounded deterministic loop resamples the legitimate forward process until the teaching batch contains at least one supervised masked token; it never forces a token by hand. The cell then checks finite gradients, clips them, chooses a coordinate with the largest absolute clipped gradient, performs one AdamW step, verifies the parameter changed, verifies first-moment and second-moment state, clears gradients, and releases the disposable GPU model.

It does **not** run a second stochastic loss and does not claim that one update must lower loss. Stochastic training loss is not guaranteed to decrease monotonically after every individual step.

### Comprehension checkpoint

1. Why is clipping applied between <code>backward()</code> and <code>optimizer.step()</code>?
2. Does clipping change the gradient direction when all gradients share one scale factor?
3. Why would omitting <code>zero_grad()</code> mix two batches?
4. Why is the working model a deep copy of the Stage 5 model?
5. If two GPUs hold independent replicas, what must happen to their gradients before they make the same update?


In [ ]:
import copy
import math

stage6_eligible_candidates = []
stage6_ineligible_selected_gpus = {}
for stage6_logical_candidate, stage6_physical_candidate in enumerate(
    SELECTED_PHYSICAL_GPU_IDS
):
    stage6_reprobe = probe_physical_gpu(stage6_physical_candidate)
    stage6_expected_uuid = SELECTED_GPU_UUIDS[stage6_logical_candidate]
    stage6_rejection_reasons = list(stage6_reprobe["reasons"])
    if stage6_reprobe.get("uuid") != stage6_expected_uuid:
        stage6_rejection_reasons.append(
            "UUID changed from "
            f"{stage6_expected_uuid!r} to {stage6_reprobe.get('uuid')!r}"
        )
    if stage6_rejection_reasons:
        stage6_ineligible_selected_gpus[stage6_physical_candidate] = (
            stage6_rejection_reasons
        )
        continue
    stage6_eligible_candidates.append(
        {
            "logical_index": stage6_logical_candidate,
            "physical_index": stage6_physical_candidate,
            "uuid": stage6_reprobe["uuid"],
            "free_memory_mib": stage6_reprobe["free_memory_mib"],
            "utilization_percent": stage6_reprobe["utilization_percent"],
            "compute_process_count": stage6_reprobe["compute_process_count"],
            "compute_processes": stage6_reprobe["compute_processes"],
            "allocated_bytes": int(torch.cuda.memory_allocated(stage6_logical_candidate)),
        }
    )

if not stage6_eligible_candidates:
    raise RuntimeError(
        "No Stage 0-selected GPU still passes the utilization/free-memory/"
        "compute-mode and UUID policy: "
        f"{stage6_ineligible_selected_gpus}. Restart the kernel so Stage 0 can "
        "select again."
    )

stage6_gpu_choice = min(
    stage6_eligible_candidates,
    key=lambda item: (item["allocated_bytes"], item["logical_index"]),
)
stage6_logical_index = stage6_gpu_choice["logical_index"]
stage6_physical_gpu_id = stage6_gpu_choice["physical_index"]
stage6_device = torch.device(f"cuda:{stage6_logical_index}")

assert next(stage5_denoiser.parameters()).device.type == "cpu"
stage6_original_sentinel_before = (
    next(stage5_denoiser.parameters()).detach().reshape(-1)[0].clone()
)

stage6_valid_cpu = model_batch["attention_mask"].bool()
stage6_time_generator = torch.Generator(device="cpu").manual_seed(6601)
stage6_mask_generator = torch.Generator(device="cpu").manual_seed(6602)
for stage6_resampling_attempt in range(1, 33):
    stage6_times_cpu = sample_ordered_stratified_times(
        clean_token_ids.shape[0],
        generator=stage6_time_generator,
    )
    (
        stage6_xt_cpu,
        stage6_mask_events_cpu,
        stage6_mask_probability_cpu,
    ) = sample_forward_absorbing(
        clean_token_ids,
        stage6_times_cpu,
        generator=stage6_mask_generator,
    )
    stage6_supervised_mask_cpu = stage6_mask_events_cpu & stage6_valid_cpu
    if bool(stage6_supervised_mask_cpu.any()):
        break
else:
    raise RuntimeError("No supervised mask was sampled in 32 legitimate forward-process draws.")

stage6_working_model = copy.deepcopy(stage5_denoiser).to(stage6_device)
stage6_working_model.train()
stage6_optimizer = torch.optim.AdamW(
    stage6_working_model.parameters(),
    **stage6_optimizer_kwargs,
)
stage6_optimizer.zero_grad(set_to_none=True)

stage6_target = clean_token_ids.to(stage6_device)
stage6_time = stage6_times_cpu.to(stage6_device)
stage6_xt = stage6_xt_cpu.to(stage6_device)
stage6_attention = model_batch["attention_mask"].to(stage6_device)
stage6_valid = stage6_valid_cpu.to(stage6_device)

torch.cuda.reset_peak_memory_stats(stage6_device)
stage6_output = stage6_working_model(
    input_ids=stage6_xt,
    attention_mask=stage6_attention,
    return_dict=True,
)
stage6_logits = stage6_output.logits
stage6_loss_parts = mdlm_token_losses(
    stage6_logits,
    stage6_target,
    stage6_xt,
    stage6_time,
    mask_id=tokenizer.mask_token_id,
    valid_positions=stage6_valid,
)
stage6_loss = reduce_mdlm_loss(
    stage6_loss_parts["token_losses"],
    stage6_valid,
    mode="repo_global_token_mean",
)
assert stage6_loss.ndim == 0
assert bool(torch.isfinite(stage6_loss))
stage6_loss.backward()

stage6_parameters_with_gradient = 0
for stage6_parameter in stage6_working_model.parameters():
    if stage6_parameter.grad is None:
        continue
    if not bool(torch.isfinite(stage6_parameter.grad).all()):
        raise FloatingPointError("A Stage 6 BERT gradient is not finite.")
    stage6_parameters_with_gradient += 1
assert stage6_parameters_with_gradient > 0

stage6_raw_gradient_norm_tensor = torch.nn.utils.clip_grad_norm_(
    stage6_working_model.parameters(),
    max_norm=stage6_gradient_clip_norm,
)
stage6_raw_gradient_norm = float(stage6_raw_gradient_norm_tensor)
if not math.isfinite(stage6_raw_gradient_norm):
    raise FloatingPointError("The global pre-clipping gradient norm is not finite.")

stage6_post_gradient_square_sum = torch.zeros((), device=stage6_device)
for stage6_parameter in stage6_working_model.parameters():
    if stage6_parameter.grad is not None:
        stage6_post_gradient_square_sum += (
            stage6_parameter.grad.detach().float().square().sum()
        )
stage6_post_gradient_norm = float(stage6_post_gradient_square_sum.sqrt())
assert stage6_post_gradient_norm <= stage6_gradient_clip_norm + 1e-5

stage6_sentinel_name = None
stage6_sentinel_parameter = None
stage6_sentinel_flat_index = None
stage6_largest_clipped_gradient = -1.0
for stage6_candidate_name, stage6_candidate_parameter in stage6_working_model.named_parameters():
    if stage6_candidate_parameter.grad is None:
        continue
    stage6_candidate_abs = stage6_candidate_parameter.grad.detach().abs().reshape(-1)
    stage6_candidate_value, stage6_candidate_index = stage6_candidate_abs.max(dim=0)
    stage6_candidate_value_float = float(stage6_candidate_value)
    if stage6_candidate_value_float > stage6_largest_clipped_gradient:
        stage6_largest_clipped_gradient = stage6_candidate_value_float
        stage6_sentinel_name = stage6_candidate_name
        stage6_sentinel_parameter = stage6_candidate_parameter
        stage6_sentinel_flat_index = int(stage6_candidate_index)

if stage6_sentinel_parameter is None or stage6_largest_clipped_gradient <= 0.0:
    raise RuntimeError("Could not find a nonzero clipped gradient coordinate.")

stage6_sentinel_gradient = (
    stage6_sentinel_parameter.grad.detach().reshape(-1)[stage6_sentinel_flat_index].clone()
)
stage6_parameter_before_step = (
    stage6_sentinel_parameter.detach().reshape(-1)[stage6_sentinel_flat_index].clone()
)
stage6_optimizer.step()
stage6_parameter_after_step = (
    stage6_sentinel_parameter.detach().reshape(-1)[stage6_sentinel_flat_index].clone()
)

stage6_optimizer_state = stage6_optimizer.state[stage6_sentinel_parameter]
stage6_state_step = int(stage6_optimizer_state["step"].item())
stage6_expected_first_moment = (
    (1.0 - stage6_optimizer_kwargs["betas"][0]) * stage6_sentinel_gradient
)
stage6_expected_second_moment = (
    (1.0 - stage6_optimizer_kwargs["betas"][1]) * stage6_sentinel_gradient.square()
)
stage6_actual_first_moment = (
    stage6_optimizer_state["exp_avg"].reshape(-1)[stage6_sentinel_flat_index]
)
stage6_actual_second_moment = (
    stage6_optimizer_state["exp_avg_sq"].reshape(-1)[stage6_sentinel_flat_index]
)

assert stage6_state_step == 1
assert not torch.equal(stage6_parameter_before_step, stage6_parameter_after_step)
torch.testing.assert_close(
    stage6_actual_first_moment,
    stage6_expected_first_moment,
    rtol=1e-5,
    atol=1e-12,
)
torch.testing.assert_close(
    stage6_actual_second_moment,
    stage6_expected_second_moment,
    rtol=1e-5,
    atol=1e-12,
)

stage6_original_sentinel_after = (
    next(stage5_denoiser.parameters()).detach().reshape(-1)[0].clone()
)
assert torch.equal(stage6_original_sentinel_before, stage6_original_sentinel_after)

stage6_optimizer.zero_grad(set_to_none=True)
stage6_gradients_cleared = all(
    stage6_parameter.grad is None for stage6_parameter in stage6_working_model.parameters()
)
assert stage6_gradients_cleared

torch.cuda.synchronize(stage6_device)
stage6_peak_memory_mib = torch.cuda.max_memory_allocated(stage6_device) / (1024**2)
stage6_parameter_delta = float(stage6_parameter_after_step - stage6_parameter_before_step)
stage6_clip_scale = min(
    1.0,
    stage6_gradient_clip_norm / (stage6_raw_gradient_norm + 1e-6),
)
stage6_results = {
    "user_selected_GPU_count": NUM_GPUS,
    "physical_GPU_used": stage6_physical_gpu_id,
    "derived_logical_device": str(stage6_device),
    "other_selected_GPUs_unused_in_this_lesson": NUM_GPUS - 1,
    "reprobe_free_memory_MiB": stage6_gpu_choice["free_memory_mib"],
    "reprobe_utilization_percent": stage6_gpu_choice["utilization_percent"],
    "reprobe_compute_process_count": stage6_gpu_choice["compute_process_count"],
    "reprobe_compute_processes": stage6_gpu_choice["compute_processes"],
    "legitimate_corruption_draws_needed": stage6_resampling_attempt,
    "supervised_masked_tokens": int(stage6_supervised_mask_cpu.sum()),
    "loss_before_update": float(stage6_loss.detach()),
    "parameters_with_gradient": stage6_parameters_with_gradient,
    "raw_global_gradient_L2_norm": stage6_raw_gradient_norm,
    "clip_threshold": stage6_gradient_clip_norm,
    "common_clip_scale": stage6_clip_scale,
    "post_clip_global_gradient_L2_norm": stage6_post_gradient_norm,
    "sentinel_parameter": stage6_sentinel_name,
    "sentinel_flat_index": stage6_sentinel_flat_index,
    "sentinel_parameter_delta": stage6_parameter_delta,
    "optimizer_state_step": stage6_state_step,
    "first_and_second_moments_match": True,
    "gradients_cleared_after_step": stage6_gradients_cleared,
    "original_stage5_model_unchanged": True,
    "fraction_of_reported_steps_percent": 100.0 / stage6_paper_training_steps,
    "loss_decrease_claimed": False,
    "peak_allocated_GPU_memory_MiB": stage6_peak_memory_mib,
}
print(stage6_results)

stage6_working_model.zero_grad(set_to_none=True)
del stage6_optimizer_state, stage6_optimizer, stage6_sentinel_parameter
del stage6_output, stage6_logits, stage6_loss_parts, stage6_loss
del stage6_target, stage6_time, stage6_xt, stage6_attention, stage6_valid
del stage6_working_model, stage6_parameter, stage6_candidate_parameter
del stage6_raw_gradient_norm_tensor, stage6_post_gradient_square_sum
del stage6_sentinel_gradient, stage6_parameter_before_step, stage6_parameter_after_step
del stage6_expected_first_moment, stage6_expected_second_moment
del stage6_actual_first_moment, stage6_actual_second_moment
torch.cuda.empty_cache()
print(
    "Stage 6 disposable model released; original Stage 5 model remains on",
    next(stage5_denoiser.parameters()).device,
)


## Stage 6 completion gate

The optimization boundary is now explicit:

    backward():       computes and accumulates parameter.grad
    clip_grad_norm_(): rescales the gradient vector when its norm is too large
    optimizer.step(): reads gradients, updates Adam moments, and mutates parameters
    zero_grad():      removes gradients before the next batch

### What the executed evidence establishes

- The paper values are learning rate <code>3e-4</code> and AdamW betas <code>(0.9, 0.999)</code>.
- The released repository adds epsilon <code>1e-8</code>, zero weight decay, and global gradient clipping at 1.0.
- The float64 scalar oracle matches PyTorch's first AdamW step, including <code>exp_avg</code>, <code>exp_avg_sq</code>, and state step 1.
- A fresh, legitimate diffusion corruption produces a finite Equation (3) loss and finite BERT gradients.
- The global gradient norm is measured before clipping and bounded after clipping.
- One nonzero-gradient BERT coordinate changes only after <code>optimizer.step()</code>.
- AdamW moment state is created from the clipped gradient.
- <code>zero_grad(set_to_none=True)</code> clears every working gradient.
- The disposable Stage 6 model changes, while the original Stage 5 random model remains unchanged.
- The physical GPU is selected by Stage 0 and rechecked; no code assumes physical GPU 0.

### Still intentionally absent

No scheduler or 2,500-step warmup, EMA, bf16 experiment, data loader, global batch construction, gradient accumulation, multi-step loop, DDP synchronization, checkpoint, validation, generation, confidence sampling, remasking, or molecular guidance has been implemented. In particular, selecting several GPUs in Stage 0 does not by itself make this single-device lesson distributed. Those are later dependencies.

One stochastic update is not expected to make the model chemically meaningful, and its loss is not required to decrease monotonically.

### Comprehension checkpoint

1. Why can <code>backward()</code> leave every parameter unchanged even when gradients are nonzero?
2. For <code>theta0 = 2</code> and <code>g1 = 0.5</code>, what are <code>m1</code>, <code>v1</code>, <code>m_hat1</code>, and <code>v_hat1</code>?
3. Why is AdamW's decoupled decay term inactive in the released GenMol configuration?
4. What is the exact order of clearing, forward/loss, backward, clipping, stepping, and clearing again?
5. Why is the optimizer state built from the clipped gradient rather than the raw gradient?
6. Why does choosing two GPUs not permit two independent unsynchronized optimizer steps?
7. Why would it be incorrect to claim that this one update reproduced paper training or must lower the next stochastic loss?

**Stop here. Do not continue to Stage 7 until every checkpoint answer is clear.**


# Stage 7 - Data loading, padding, and antithetic training times

## What enters one training step

The model never trains directly on a Python string. A dataset returns a clean SAFE string, the tokenizer converts it to token IDs, and a collator builds a rectangular batch. Shorter rows are padded and attention_mask marks which positions are real. This notebook's molecule-view examples ignore PAD, BOS, and EOS. The released training path ignores PAD but can corrupt BOS/EOS; Stage 2 labels that deliberate policy difference.

For B sequences, independent uniform times can accidentally cluster. Antithetic sampling first draws one offset u and then uses evenly spaced values

t_i = (u + i / B) modulo 1,

with a small epsilon clamp away from exactly 0 and 1. Each individual time is uniform, but the batch covers the interval more evenly.

The effective global batch is

global batch = per-device batch times number of processes times accumulation steps.

This notebook solves the integer equation for the user-selected GPU count. It never assumes physical GPU 0.

## Concrete example

With four examples and offset 0.10, antithetic times are approximately 0.10, 0.35, 0.60, and 0.85. Independent draws could all land near 0.8. If the target global batch is 2,048, four processes with per-process batch 64 require eight accumulation steps.

## Paper, released GitHub code, and notebook

- The paper specifies global batch 2,048 but does not prescribe a particular per-GPU split.
- The repository dataset class has an indexing typo: its get-item method refers to undefined i instead of the requested index. The local dataset below fixes that boundary.
- The repository uses antithetic uniform time sampling with epsilon 0.001. This notebook reproduces and tests that behavior.
- The executable cell reuses Stage 2's `sample_ordered_stratified_times`; the repository calls this construction antithetic sampling.
- The paper-scale SAFE corpus is not present, so the cell uses deterministic local examples to validate mechanics only.

## What the next code proves

It tests dataset indexing, tokenizer round-trip, dynamic padding, attention masks, excluded special tokens, antithetic coverage, and exact global-batch arithmetic for arbitrary NUM_GPUS.

## Comprehension checkpoint

1. Why must padding be excluded from both diffusion and loss?
2. What marginal distribution does each antithetic time have?
3. Why can antithetic times reduce batch-to-batch variance?
4. If NUM_GPUS changes, which factor must be recomputed to retain global batch 2,048?


In [ ]:
from dataclasses import dataclass
from torch.utils.data import Dataset
import math

@dataclass(frozen=True)
class BatchPlan:
    global_batch_size: int
    world_size: int
    micro_batch_per_rank: int | None
    accumulation_steps: int | None
    effective_global_batch: int | None
    exact: bool

def choose_exact_batch_plan(global_batch_size, world_size, preferred_micro_batch=16):
    if not isinstance(global_batch_size, int) or global_batch_size < 1:
        raise ValueError("global_batch_size must be a positive integer")
    if not isinstance(world_size, int) or world_size < 1:
        raise ValueError("world_size must be a positive integer")
    upper = min(preferred_micro_batch, global_batch_size)
    candidates = [
        micro for micro in range(upper, 0, -1)
        if global_batch_size % (world_size * micro) == 0
    ]
    if not candidates:
        return BatchPlan(global_batch_size, world_size, None, None, None, False)
    micro = candidates[0]
    accumulation = global_batch_size // (world_size * micro)
    return BatchPlan(
        global_batch_size,
        world_size,
        micro,
        accumulation,
        world_size * micro * accumulation,
        True,
    )

def released_repository_batch_plan(global_batch_size, world_size):
    micro = math.ceil(global_batch_size / world_size)
    accumulation = math.ceil(global_batch_size / (world_size * micro))
    effective = world_size * micro * accumulation
    return {
        "micro_batch_per_rank": micro,
        "accumulation_steps": accumulation,
        "effective_global_batch": effective,
        "exact": effective == global_batch_size,
    }

class LocalSafeDataset(Dataset):
    def __init__(self, safe_strings):
        self.safe_strings = tuple(str(value).strip() for value in safe_strings)
        if not self.safe_strings or any(not value for value in self.safe_strings):
            raise ValueError("Every local SAFE example must be a non-empty string")

    def __len__(self):
        return len(self.safe_strings)

    def __getitem__(self, index):
        if isinstance(index, torch.Tensor):
            if index.numel() != 1:
                raise TypeError("LocalSafeDataset expects scalar indices")
            index = int(index.item())
        if not isinstance(index, int):
            raise TypeError("LocalSafeDataset expects an integer index")
        return {"input": self.safe_strings[index]}

class SafeBatchCollator:
    def __init__(self, tokenizer, max_length=256):
        self.tokenizer = tokenizer
        self.max_length = int(max_length)

    def __call__(self, examples):
        safe_strings = [example["input"] for example in examples]
        batch = self.tokenizer(
            safe_strings,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=self.max_length,
        )
        batch.pop("token_type_ids", None)
        return batch


stage7_plan = choose_exact_batch_plan(2048, NUM_GPUS, preferred_micro_batch=16)
stage7_repository_plan = released_repository_batch_plan(2048, NUM_GPUS)

stage7_safe_strings = tokenizer.batch_decode(
    clean_token_ids[: min(4, clean_token_ids.shape[0])].detach().cpu(),
    skip_special_tokens=True,
)
stage7_dataset = LocalSafeDataset(stage7_safe_strings)
stage7_collator = SafeBatchCollator(tokenizer, max_length=256)
stage7_batch = stage7_collator([stage7_dataset[i] for i in range(len(stage7_dataset))])

assert len(stage7_dataset) == len(stage7_safe_strings)
assert set(stage7_batch) == {"input_ids", "attention_mask"}
assert stage7_batch["input_ids"].shape == stage7_batch["attention_mask"].shape
assert int(stage7_batch["input_ids"].min()) >= 0
assert int(stage7_batch["input_ids"].max()) < 1880
assert torch.all((stage7_batch["attention_mask"] == 0) | (stage7_batch["attention_mask"] == 1))
stage7_padding_loss_mask = stage7_batch["attention_mask"].bool()
assert not stage7_padding_loss_mask[stage7_batch["attention_mask"] == 0].any()

stage7_generator = torch.Generator().manual_seed(7007)
stage7_times = sample_ordered_stratified_times(8, sampling_eps=1e-3, generator=stage7_generator)
stage7_unit_times = (stage7_times - 1e-3) / (1.0 - 1e-3)
stage7_strata = torch.floor(stage7_unit_times * 8).to(torch.long)
assert torch.equal(stage7_strata, torch.arange(8))
assert torch.all((stage7_times >= 1e-3) & (stage7_times < 1.0))

if stage7_plan.exact:
    assert stage7_plan.effective_global_batch == 2048

stage7_summary = pd.DataFrame(
    [
        ["paper global batch", 2048],
        ["user-selected GPU count", NUM_GPUS],
        ["exact plan available", stage7_plan.exact],
        ["our microbatch per rank", stage7_plan.micro_batch_per_rank],
        ["our accumulation steps", stage7_plan.accumulation_steps],
        ["GitHub effective batch", stage7_repository_plan["effective_global_batch"]],
        ["GitHub plan exact", stage7_repository_plan["exact"]],
        ["smoke batch shape", tuple(stage7_batch["input_ids"].shape)],
        ["time strata", stage7_strata.tolist()],
    ],
    columns=["quantity", "value"],
)
display(stage7_summary)
print("Stage 7 checks passed: local data, collation, padding, time strata, and batch math.")

# Stage 8 - A scalable training state machine

## One optimizer step is a transaction

A correct training update has a strict order:

1. make several microbatches and divide each loss by the accumulation count;
2. backpropagate every microbatch;
3. let distributed data parallel synchronize gradients across processes;
4. clip the global gradient norm to 1;
5. apply AdamW;
6. advance the learning-rate schedule;
7. update exponential-moving-average weights;
8. clear gradients;
9. periodically save model, optimizer, scheduler, EMA, step, and provenance.

Gradient accumulation changes memory use, not the intended global batch. Distributed processes must each receive different shuffled samples, while all ranks agree on optimizer-step boundaries.

The repository warmup factor is min((step + 1) / 2500, 1). Its EMA decay after update n is

decay_n = min(0.9999, (1 + n) / (10 + n)).

The early ramp prevents an almost-frozen average before useful parameters exist.

## Concrete example

Suppose two GPU processes each handle 32 molecules and accumulate four microbatches. One optimizer step represents 2 times 32 times 4 = 256 molecules. The cell uses a tiny CPU model to show that accumulated gradients change parameters once, EMA tracks the new parameters, and a checkpoint restores every state exactly.

## Paper, released GitHub code, and notebook

- The paper specifies AdamW with beta1 0.9, beta2 0.999, learning rate 0.0003, 50,000 optimizer steps, global batch 2,048, and eight A100 GPUs.
- The repository adds epsilon 1e-8, weight decay 0, gradient clipping 1, a 2,500-step linear warmup followed by constant learning rate, EMA target 0.9999, Lightning DDP, and checkpoints every 5,000 steps.
- The configuration requests bfloat16 training, while the model wrapper explicitly opens float32 autocast; this notebook records that discrepancy rather than claiming one is mandated by the paper.
- Full training remains behind RUN_PAPER_SCALE_TRAINING=False. The launch command is built from the dynamically selected physical IDs and NUM_GPUS.

## What the next code proves

It validates warmup endpoints, adaptive EMA arithmetic, accumulated optimizer order, gradient clipping, zeroed gradients, checkpoint round-trip, and a multi-process launch command that never hard-codes a physical GPU.

### Two distributed details that are easy to miss

A global token mean is total loss numerator over all valid tokens divided by the total valid-token count across every rank. It is not the mean of per-sequence means or per-rank means. Because DDP averages gradients, each rank backpropagates its local numerator times world_size divided by the all-reduced global count.

Generation and evaluation must use the EMA shadow parameters, matching the repository's model-loading path. The context manager below swaps EMA weights in temporarily and restores live training weights afterward; the smoke test checks both directions.

## Comprehension checkpoint

1. Why is each microbatch loss divided by the accumulation count?
2. Which step must happen before clipping in distributed training?
3. Why save optimizer, scheduler, and EMA state in addition to model weights?
4. Which training details come from the paper, and which are repository additions?


In [ ]:
import copy
import io
import os
import random
import numpy as np
from torch import nn

STAGE8_BASE_LR = 3e-4
STAGE8_WARMUP_STEPS = 2500
STAGE8_EMA_DECAY = 0.9999

def warmup_multiplier(step, warmup_steps=STAGE8_WARMUP_STEPS):
    if step < 0 or warmup_steps < 1:
        raise ValueError("step must be nonnegative and warmup_steps positive")
    return min(float(step) / float(warmup_steps), 1.0)

def warmup_learning_rate(step, base_lr=STAGE8_BASE_LR):
    return base_lr * warmup_multiplier(step)

class AdaptiveEMA:
    def __init__(self, parameters, decay=STAGE8_EMA_DECAY):
        parameters = list(parameters)
        self.decay = float(decay)
        self.num_updates = 0
        self.shadow = [parameter.detach().clone() for parameter in parameters]

    def effective_decay(self):
        return min(self.decay, (1.0 + self.num_updates) / (10.0 + self.num_updates))

    @torch.no_grad()
    def update(self, parameters):
        parameters = list(parameters)
        if len(parameters) != len(self.shadow):
            raise ValueError("EMA parameter count changed")
        self.num_updates += 1
        decay = self.effective_decay()
        for shadow, parameter in zip(self.shadow, parameters):
            shadow.mul_(decay).add_(parameter.detach(), alpha=1.0 - decay)
        return decay

    def state_dict(self):
        return {
            "decay": self.decay,
            "num_updates": self.num_updates,
            "shadow": [tensor.clone() for tensor in self.shadow],
        }

    def load_state_dict(self, state):
        self.decay = float(state["decay"])
        self.num_updates = int(state["num_updates"])
        self.shadow = [tensor.clone() for tensor in state["shadow"]]

from contextlib import contextmanager


def ddp_global_token_mean_for_backward(local_loss_sum, local_valid_count):
    if local_loss_sum.ndim != 0:
        raise ValueError("local_loss_sum must be a scalar tensor.")
    local_count = torch.as_tensor(
        float(local_valid_count),
        device=local_loss_sum.device,
        dtype=local_loss_sum.dtype,
    )
    if local_count.item() <= 0:
        raise ValueError("Each participating rank needs at least one valid token.")

    global_count = local_count.clone()
    world_size = 1
    if torch.distributed.is_available() and torch.distributed.is_initialized():
        torch.distributed.all_reduce(
            global_count,
            op=torch.distributed.ReduceOp.SUM,
        )
        world_size = torch.distributed.get_world_size()

    # DDP averages gradients across ranks. Multiplying the local numerator by
    # world_size / global_count yields grad(sum_r numerator_r / sum_r count_r).
    return local_loss_sum * (float(world_size) / global_count)


@contextmanager
def use_ema_parameters(model, ema):
    parameters = list(model.parameters())
    if len(parameters) != len(ema.shadow):
        raise ValueError("EMA shadow and model parameter counts differ.")
    original = [parameter.detach().clone() for parameter in parameters]
    try:
        with torch.no_grad():
            for parameter, shadow in zip(parameters, ema.shadow):
                parameter.copy_(shadow.to(device=parameter.device, dtype=parameter.dtype))
        yield model
    finally:
        with torch.no_grad():
            for parameter, saved in zip(parameters, original):
                parameter.copy_(saved)


def accumulated_optimizer_transaction(
    model,
    optimizer,
    scheduler,
    ema,
    loss_closures,
    clip_norm=1.0,
):
    if not loss_closures:
        raise ValueError("At least one microbatch loss is required")
    optimizer.zero_grad(set_to_none=True)
    micro_losses = []
    for closure in loss_closures:
        loss = closure()
        if loss.ndim != 0 or not torch.isfinite(loss):
            raise ValueError("Each microbatch closure must return one finite scalar")
        (loss / len(loss_closures)).backward()
        micro_losses.append(float(loss.detach()))
    raw_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)
    if not torch.isfinite(raw_norm):
        raise FloatingPointError("Non-finite global gradient norm")
    optimizer.step()
    scheduler.step()
    ema_decay = ema.update(model.parameters())
    optimizer.zero_grad(set_to_none=True)
    assert all(parameter.grad is None for parameter in model.parameters())
    return {
        "mean_micro_loss": sum(micro_losses) / len(micro_losses),
        "raw_gradient_norm": float(raw_norm),
        "ema_decay": float(ema_decay),
        "learning_rate_after_scheduler": optimizer.param_groups[0]["lr"],
    }

stage8_model = nn.Linear(2, 1, bias=False, dtype=torch.float64)
with torch.no_grad():
    stage8_model.weight.copy_(torch.tensor([[0.25, -0.50]], dtype=torch.float64))

stage8_optimizer = torch.optim.AdamW(
    stage8_model.parameters(),
    lr=STAGE8_BASE_LR,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.0,
)
stage8_scheduler = torch.optim.lr_scheduler.LambdaLR(
    stage8_optimizer,
    lr_lambda=lambda step: warmup_multiplier(step + 1),
)
stage8_ema = AdaptiveEMA(stage8_model.parameters(), decay=STAGE8_EMA_DECAY)

stage8_x1 = torch.tensor([[1.0, 2.0], [2.0, -1.0]], dtype=torch.float64)
stage8_y1 = torch.tensor([[0.5], [1.0]], dtype=torch.float64)
stage8_x2 = torch.tensor([[-1.0, 1.0], [0.5, 0.5]], dtype=torch.float64)
stage8_y2 = torch.tensor([[-0.25], [0.0]], dtype=torch.float64)
stage8_before = copy.deepcopy(stage8_model.state_dict())

stage8_transaction = accumulated_optimizer_transaction(
    stage8_model,
    stage8_optimizer,
    stage8_scheduler,
    stage8_ema,
    [
        lambda: torch.nn.functional.mse_loss(stage8_model(stage8_x1), stage8_y1),
        lambda: torch.nn.functional.mse_loss(stage8_model(stage8_x2), stage8_y2),
    ],
    clip_norm=1.0,
)

assert warmup_learning_rate(0) == 0.0
assert math.isclose(warmup_learning_rate(1), 1.2e-7, rel_tol=0.0, abs_tol=1e-15)
assert math.isclose(warmup_learning_rate(2500), 3e-4, rel_tol=0.0, abs_tol=1e-15)
assert stage8_transaction["ema_decay"] == 2.0 / 11.0
assert any(
    not torch.equal(stage8_before[name], stage8_model.state_dict()[name])
    for name in stage8_before
)

stage8_local_mean = ddp_global_token_mean_for_backward(
    torch.tensor(6.0),
    local_valid_count=3,
)
assert torch.isclose(stage8_local_mean, torch.tensor(2.0))

stage8_before_ema_swap = [
    parameter.detach().clone() for parameter in stage8_model.parameters()
]
with use_ema_parameters(stage8_model, stage8_ema):
    assert all(
        torch.equal(parameter.detach(), shadow.to(parameter))
        for parameter, shadow in zip(stage8_model.parameters(), stage8_ema.shadow)
    )
assert all(
    torch.equal(parameter.detach(), saved)
    for parameter, saved in zip(stage8_model.parameters(), stage8_before_ema_swap)
)


stage8_checkpoint = {
    "model": copy.deepcopy(stage8_model.state_dict()),
    "optimizer": copy.deepcopy(stage8_optimizer.state_dict()),
    "scheduler": copy.deepcopy(stage8_scheduler.state_dict()),
    "ema": stage8_ema.state_dict(),
    "global_step": 1,
    "python_rng": random.getstate(),
    "numpy_rng": np.random.get_state(),
    "torch_rng": torch.get_rng_state(),
}
stage8_buffer = io.BytesIO()
torch.save(stage8_checkpoint, stage8_buffer)
stage8_buffer.seek(0)
try:
    stage8_loaded = torch.load(stage8_buffer, map_location="cpu", weights_only=False)
except TypeError:
    stage8_buffer.seek(0)
    stage8_loaded = torch.load(stage8_buffer, map_location="cpu")

stage8_restored_model = nn.Linear(2, 1, bias=False, dtype=torch.float64)
stage8_restored_optimizer = torch.optim.AdamW(
    stage8_restored_model.parameters(),
    lr=STAGE8_BASE_LR,
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.0,
)
stage8_restored_scheduler = torch.optim.lr_scheduler.LambdaLR(
    stage8_restored_optimizer,
    lr_lambda=lambda step: warmup_multiplier(step),
)
stage8_restored_ema = AdaptiveEMA(stage8_restored_model.parameters())
stage8_restored_model.load_state_dict(stage8_loaded["model"])
stage8_restored_optimizer.load_state_dict(stage8_loaded["optimizer"])
stage8_restored_scheduler.load_state_dict(stage8_loaded["scheduler"])
stage8_restored_ema.load_state_dict(stage8_loaded["ema"])

for name, value in stage8_model.state_dict().items():
    torch.testing.assert_close(value, stage8_restored_model.state_dict()[name])
assert stage8_restored_scheduler.state_dict() == stage8_scheduler.state_dict()
assert stage8_restored_ema.num_updates == stage8_ema.num_updates

stage8_visible_ids = ",".join(str(index) for index in SELECTED_PHYSICAL_GPU_IDS)
stage8_launch_command = (
    f"CUDA_VISIBLE_DEVICES={stage8_visible_ids} "
    f"torchrun --nproc_per_node={NUM_GPUS} scripts/train.py "
    "hydra.run.dir=outputs/paper_v1"
)
assert f"--nproc_per_node={NUM_GPUS}" in stage8_launch_command
assert stage8_visible_ids == ",".join(str(index) for index in SELECTED_PHYSICAL_GPU_IDS)

RUN_PAPER_SCALE_TRAINING = False
stage8_report = {
    **stage8_transaction,
    "lr_step_0": warmup_learning_rate(0),
    "lr_step_1": warmup_learning_rate(1),
    "lr_step_2500": warmup_learning_rate(2500),
    "checkpoint_round_trip": True,
    "paper_scale_training": "SKIPPED" if not RUN_PAPER_SCALE_TRAINING else "OPTED IN",
    "selected_gpu_count": NUM_GPUS,
    "launch_command": stage8_launch_command,
    "precision_claim": "not inferred from contradictory repository settings",
}
print(stage8_report)

del stage8_restored_model, stage8_restored_optimizer, stage8_restored_scheduler
del stage8_model, stage8_optimizer, stage8_scheduler
print("Stage 8 checks passed: transaction order, scheduler, EMA, and resume state.")

# Stage 9 - Equation 2 standard reverse-transition baseline

## Reverse an absorbing-mask step

Let alpha_t be the probability that a clean token is still visible at time t. Reverse sampling moves from a noisier t to an earlier, cleaner s, so alpha_s is greater than or equal to alpha_t.

If the current token is visible, absorbing diffusion proves it was never masked, so the earlier token is exactly the same token.

If the current token is MASK, the denoiser supplies p_theta(x | z_t, t). The exact posterior is

P(z_s = x | z_t = MASK) =
((alpha_s - alpha_t) / (1 - alpha_t)) times p_theta(x | z_t, t)

for each clean token x, and

P(z_s = MASK | z_t = MASK) =
(1 - alpha_s) / (1 - alpha_t).

These probabilities sum to one. At s=t the state cannot change. At the clean endpoint alpha_s=1, no mask probability remains.

## Concrete example from our discussion

Take alpha_s=0.90, alpha_t=0.75, p_theta(A)=0.60, and p_theta(B)=0.40. Then

P(A)=((0.90-0.75)/(1-0.75)) times 0.60 = 0.36,
P(B)=((0.90-0.75)/(1-0.75)) times 0.40 = 0.24,
P(MASK)=(1-0.90)/(1-0.75) = 0.40.

The total is 1.00. Your reasoning was exactly right.

## Paper, released GitHub code, and notebook

- Equation (2) in the paper defines this standard reverse kernel.
- The released sampler delegates the step implementation to BioNeMo-MoCo instead of spelling out the probabilities locally.
- Stage 3 already implemented the kernel term by term. This stage reuses that tested primitive and checks it as the standard ancestral baseline instead of maintaining a second copy.
- Confidence sampling in Stage 10 is a different reveal policy; this stage remains the Equation 2 standard-diffusion baseline used by the paper's ablation.

## What the next code proves

It checks probability normalization, visible-token invariance, s=t identity, the clean endpoint, a grid of valid time pairs, and the numerical example above.

## Comprehension checkpoint

1. Why can a visible token not change during the reverse process?
2. Where does the remaining MASK probability come from?
3. What happens when s=t?
4. Can you recompute 0.36, 0.24, and 0.40 without looking above?


In [ ]:
stage9_clean = torch.tensor([[[0.6, 0.4, 0.0], [0.2, 0.8, 0.0]]], dtype=torch.float64)
stage9_current = torch.tensor([[2, 0]], dtype=torch.long)
stage9_probs = reverse_posterior_from_alphas(stage9_current, stage9_clean, alpha_s=1.0 - 0.1, alpha_t=1.0 - 0.25, mask_id=2)[0]
torch.testing.assert_close(stage9_probs[0, 0], torch.tensor([0.36, 0.24, 0.4], dtype=torch.float64), rtol=1e-12, atol=1e-12)
torch.testing.assert_close(stage9_probs[0, 1], torch.tensor([1.0, 0.0, 0.0], dtype=torch.float64), rtol=0.0, atol=0.0)
stage9_endpoint = reverse_posterior_from_alphas(torch.tensor([[2]]), stage9_clean[:, :1], alpha_s=1.0 - 0.0, alpha_t=1.0 - 0.25, mask_id=2)[0]
assert float(stage9_endpoint[0, 0, 2]) == 0.0
stage9_grid_checks = []
for stage9_t in (0.2, 0.5, 0.9, 1.0):
    for stage9_s in (0.0, stage9_t / 2.0):
        checked = reverse_posterior_from_alphas(torch.tensor([[2]]), stage9_clean[:, :1], alpha_s=1.0 - stage9_s, alpha_t=1.0 - stage9_t, mask_id=2)[0]
        stage9_grid_checks.append({'s': stage9_s, 't': stage9_t, 'reveal_mass': float(checked[0, 0, :2].sum()), 'mask_mass': float(checked[0, 0, 2])})
stage9_draw = sample_reverse_from_alphas(stage9_current, stage9_clean, 0.9, 0.75, mask_id=2, generator=torch.Generator().manual_seed(9009))[0]
assert stage9_draw[0, 1].item() == 0
display(pd.DataFrame(stage9_grid_checks))
print({'numerical_example': stage9_probs[0, 0].tolist(), 'visible_token_distribution': stage9_probs[0, 1].tolist(), 'sampled_tokens': stage9_draw.tolist()})
print('Stage 9 checks passed: the Equation 2 transition kernel is normalized and absorbing.')


# Stage 10 - Confidence sampling (paper Appendix B)

## Why this sampler exists

The exact reverse kernel from Stage 9 treats masked positions independently. GenMol instead predicts a candidate at every masked position, keeps only the most trustworthy predictions, and asks the denoiser to reconsider the rest. A token that becomes visible is never changed again.

For vocabulary category i at position l, Equation (4) applies temperature tau:

p(i) = exp(logit_i / tau) / sum_j exp(logit_j / tau).

After sampling candidate i*, Equation (8) gives its confidence:

c_t^l = log p(i*) + r * t * epsilon, where epsilon is Gumbel(0,1).

The top N currently masked positions are confirmed. The others remain masks. Because reverse time decreases from 1 to 0, the random term r*t is strongest early and fades late. Thus tau changes which token is sampled, r changes reveal-order randomness, and N changes speed versus quality.

## Concrete example

With five masks and N=2, the mask-count trace is 5 -> 3 -> 1 -> 0. The last step safely confirms one token; the paper does not specify this edge-case convention, so we state and test it.

## Paper vs released GitHub code

- The paper defines the probability and confidence equations and warns that N>1 improves speed but reduces quality.
- The repository delegates confidence sampling to BioNeMo-MoCo. Its public generate wrapper does not forward N, so the external sampler's default effectively matters.
- This notebook implements the equations locally, samples independently per row, forbids accidental special-token proposals, and makes the decreasing time grid explicit.
- This is a sampler unit test with an oracle denoiser. It does not demonstrate that an untrained network generates molecules.

## What the next code proves

The cell checks reproducibility under a fixed generator, monotone mask removal, preservation of visible context, exact top-N progress, and safe handling of fewer than N remaining masks.

## Comprehension checkpoint

1. Why is log probability used in Equation (8), rather than the raw model logit?
2. Which knob changes candidate-token probabilities: tau, r, or N?
3. Why does the Gumbel perturbation naturally weaken near t=0?
4. If N is doubled, what speed-quality trade-off does the paper report?


In [ ]:
import math
import torch.nn.functional as F


def _gumbel_noise(shape, *, device, dtype, generator=None):
    uniform = torch.rand(shape, device=device, dtype=dtype, generator=generator)
    uniform = uniform.clamp(min=1e-6, max=1.0 - 1e-6)
    return -torch.log(-torch.log(uniform))


def confidence_sampling_step(
    logits,
    current_ids,
    *,
    mask_token_id,
    tokens_per_step,
    temperature,
    randomness,
    time_value,
    generator=None,
    forbidden_token_ids=(),
):
    if logits.ndim != 3 or current_ids.shape != logits.shape[:2]:
        raise ValueError("Expected logits [batch, length, vocab] and matching token ids.")
    if tokens_per_step < 1 or temperature <= 0:
        raise ValueError("tokens_per_step and temperature must be positive.")

    adjusted = logits.clone()
    for token_id in forbidden_token_ids:
        adjusted[..., int(token_id)] = -torch.inf

    log_probabilities = F.log_softmax(adjusted / float(temperature), dim=-1)
    probabilities = log_probabilities.exp()
    flat_samples = torch.multinomial(
        probabilities.reshape(-1, probabilities.shape[-1]),
        num_samples=1,
        replacement=True,
        generator=generator,
    )
    sampled_ids = flat_samples.reshape_as(current_ids)
    sampled_log_probability = log_probabilities.gather(
        dim=-1, index=sampled_ids.unsqueeze(-1)
    ).squeeze(-1)

    confidence = sampled_log_probability
    if randomness:
        confidence = confidence + float(randomness) * float(time_value) * _gumbel_noise(
            confidence.shape,
            device=confidence.device,
            dtype=confidence.dtype,
            generator=generator,
        )

    masked = current_ids.eq(int(mask_token_id))
    confidence = confidence.masked_fill(~masked, -torch.inf)
    updated = current_ids.clone()

    for row in range(current_ids.shape[0]):
        remaining = int(masked[row].sum().item())
        number_to_confirm = min(int(tokens_per_step), remaining)
        if number_to_confirm:
            positions = confidence[row].topk(number_to_confirm).indices
            updated[row, positions] = sampled_ids[row, positions]

    return updated


def confidence_sample(
    denoise_logits,
    initial_ids,
    *,
    mask_token_id,
    tokens_per_step=1,
    temperature=0.5,
    randomness=0.5,
    generator=None,
    forbidden_token_ids=(),
):
    state = initial_ids.clone()
    initial_masks = int(state.eq(mask_token_id).sum(dim=1).max().item())
    maximum_steps = max(1, math.ceil(initial_masks / int(tokens_per_step)))
    mask_trace = [int(state.eq(mask_token_id).sum().item())]

    for step in range(maximum_steps):
        if not state.eq(mask_token_id).any():
            break
        time_value = 1.0 - step / maximum_steps
        times = torch.full(
            (state.shape[0],), time_value, device=state.device, dtype=torch.float32
        )
        logits = denoise_logits(state, times)
        state = confidence_sampling_step(
            logits,
            state,
            mask_token_id=mask_token_id,
            tokens_per_step=tokens_per_step,
            temperature=temperature,
            randomness=randomness,
            time_value=time_value,
            generator=generator,
            forbidden_token_ids=forbidden_token_ids,
        )
        mask_trace.append(int(state.eq(mask_token_id).sum().item()))

    if state.eq(mask_token_id).any():
        raise RuntimeError("The confidence schedule ended with unresolved masks.")
    return state, mask_trace


stage10_targets = torch.tensor([[5, 6, 7, 8, 9]], dtype=torch.long)
stage10_vocab_size = max(len(tokenizer), int(stage10_targets.max().item()) + 1)
stage10_initial = torch.full_like(stage10_targets, tokenizer.mask_token_id)


def stage10_scripted_logits(token_ids, times):
    oracle = torch.full(
        (*token_ids.shape, stage10_vocab_size),
        -30.0,
        dtype=torch.float32,
        device=token_ids.device,
    )
    targets = stage10_targets.to(token_ids.device).expand_as(token_ids)
    oracle.scatter_(-1, targets.unsqueeze(-1), 30.0)
    return oracle


def run_stage10(seed):
    return confidence_sample(
        stage10_scripted_logits,
        stage10_initial,
        mask_token_id=tokenizer.mask_token_id,
        tokens_per_step=2,
        temperature=0.5,
        randomness=0.5,
        generator=torch.Generator().manual_seed(seed),
        forbidden_token_ids=(tokenizer.mask_token_id,),
    )


stage10_result_a, stage10_trace_a = run_stage10(1010)
stage10_result_b, _ = run_stage10(1010)
assert torch.equal(stage10_result_a, stage10_result_b)
assert torch.equal(stage10_result_a, stage10_targets)
assert stage10_trace_a == [5, 3, 1, 0]
assert all(later <= earlier for earlier, later in zip(stage10_trace_a, stage10_trace_a[1:]))

stage10_visible = stage10_initial.clone()
stage10_visible[0, 0] = stage10_targets[0, 0]
stage10_visible_result, _ = confidence_sample(
    stage10_scripted_logits,
    stage10_visible,
    mask_token_id=tokenizer.mask_token_id,
    tokens_per_step=3,
    temperature=0.5,
    randomness=0.0,
    generator=torch.Generator().manual_seed(12),
    forbidden_token_ids=(tokenizer.mask_token_id,),
)
assert stage10_visible_result[0, 0].item() == stage10_visible[0, 0].item()

display(pd.DataFrame([{
    "test": "oracle confidence sampling",
    "mask trace": str(stage10_trace_a),
    "matches target": bool(torch.equal(stage10_result_a, stage10_targets)),
    "visible token preserved": True,
    "scientific claim": "algorithmic unit test only",
}]))


# Stage 11 - De novo generation: choose a length, denoise, decode

## From an empty canvas to a molecule

De novo generation starts from BOS, a sampled number L of mask tokens, and EOS. GenMol samples L from the empirical ZINC250k SAFE-token length distribution rather than fixing every molecule to the same size. Confidence sampling then replaces masks, and the tokenizer plus SAFE decoder maps tokens back to a molecular graph.

This cell deliberately separates two decode policies:

- strict decode reports exactly what the model produced;
- repair is an optional operational policy and must never be silently counted as raw validity.

Silently taking the largest connected component or repairing invalid text would bias evaluation upward.

## Concrete example

If L=4, the model input is BOS MASK MASK MASK MASK EOS. The code also constructs an oracle exercise from a real clean SAFE sequence already used earlier: it hides all interior tokens, reconstructs them, decodes the exact SAFE string, and checks the resulting SMILES with RDKit.

## Paper vs released GitHub code

- The paper samples generation lengths from ZINC250k and reports N=1, tau=0.5, r=0.5 as its main high-quality setting.
- The released data/len.pk file supplies the empirical distribution. Its public sampling utility also performs postprocessing; this notebook keeps strict validity separate from any repair.
- The repository checkpoint directory is empty here, so random Stage-5 weights cannot reproduce Table 1. The real model call is wired but paper-scale sampling stays behind an explicit false flag.

## What the next code proves

It validates the saved length distribution, constructs padding and attention masks correctly, exercises the complete token-to-SAFE-to-SMILES route with a deterministic oracle, and exposes a guarded entry point for a trained denoiser.

## Comprehension checkpoint

1. Why should molecule length be sampled from data rather than chosen arbitrarily?
2. Which tokens are fixed throughout generation?
3. Why must strict decoding and repaired decoding be reported separately?
4. What does the oracle reconstruction prove, and what does it not prove?


In [ ]:
import pickle
from pathlib import Path
from rdkit import Chem
from safe import decode as safe_decode


def _first_available_token_id(tokenizer_object, names):
    for name in names:
        value = getattr(tokenizer_object, name, None)
        if value is not None:
            return int(value)
    raise ValueError(f"Tokenizer provides none of these token ids: {names}")


STAGE11_BOS_ID = _first_available_token_id(tokenizer, ("bos_token_id", "cls_token_id"))
STAGE11_EOS_ID = _first_available_token_id(tokenizer, ("eos_token_id", "sep_token_id"))
STAGE11_PAD_ID = _first_available_token_id(tokenizer, ("pad_token_id",))

stage11_length_path = Path(PROJECT_ROOT) / "data" / "len.pk"
with stage11_length_path.open("rb") as handle:
    stage11_lengths = [int(value) for value in pickle.load(handle)]
assert stage11_lengths and min(stage11_lengths) > 0


def build_masked_safe_batch(lengths, *, device="cpu"):
    lengths = [int(length) for length in lengths]
    if not lengths or min(lengths) < 1:
        raise ValueError("Every requested interior length must be positive.")
    width = max(lengths) + 2
    input_ids = torch.full(
        (len(lengths), width), STAGE11_PAD_ID, dtype=torch.long, device=device
    )
    attention_mask = torch.zeros_like(input_ids)
    for row, length in enumerate(lengths):
        input_ids[row, 0] = STAGE11_BOS_ID
        input_ids[row, 1 : 1 + length] = tokenizer.mask_token_id
        input_ids[row, 1 + length] = STAGE11_EOS_ID
        attention_mask[row, : length + 2] = 1
    return {"input_ids": input_ids, "attention_mask": attention_mask}


def token_ids_to_safe(token_ids):
    text = tokenizer.decode(
        token_ids.tolist(),
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )
    return text.replace(" ", "")


def strict_safe_to_smiles(safe_text):
    smiles = safe_decode(safe_text)
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise ValueError("SAFE decoded to an invalid SMILES.")
    return Chem.MolToSmiles(molecule)


stage11_preview_lengths = [
    min(stage11_lengths),
    int(round(float(torch.tensor(stage11_lengths, dtype=torch.float32).median().item()))),
    max(stage11_lengths),
]
stage11_preview_batch = build_masked_safe_batch(stage11_preview_lengths)
for row, length in enumerate(stage11_preview_lengths):
    assert int(stage11_preview_batch["attention_mask"][row].sum()) == length + 2
    assert int(stage11_preview_batch["input_ids"][row].eq(tokenizer.mask_token_id).sum()) == length

stage11_target = clean_token_ids[:1].detach().cpu().clone()
stage11_fixed = (
    stage11_target.eq(STAGE11_BOS_ID)
    | stage11_target.eq(STAGE11_EOS_ID)
    | stage11_target.eq(STAGE11_PAD_ID)
)
stage11_masked = stage11_target.clone()
stage11_masked[~stage11_fixed] = tokenizer.mask_token_id
stage11_vocab_size = len(tokenizer)


def stage11_scripted_logits(token_ids, times):
    logits = torch.full(
        (*token_ids.shape, stage11_vocab_size),
        -30.0,
        dtype=torch.float32,
        device=token_ids.device,
    )
    target = stage11_target.to(token_ids.device).expand_as(token_ids)
    logits.scatter_(-1, target.unsqueeze(-1), 30.0)
    return logits


stage11_reconstructed, stage11_trace = confidence_sample(
    stage11_scripted_logits,
    stage11_masked,
    mask_token_id=tokenizer.mask_token_id,
    tokens_per_step=8,
    temperature=0.5,
    randomness=0.0,
    generator=torch.Generator().manual_seed(111),
    forbidden_token_ids=(
        tokenizer.mask_token_id,
        STAGE11_PAD_ID,
        STAGE11_BOS_ID,
        STAGE11_EOS_ID,
    ),
)
assert torch.equal(stage11_reconstructed, stage11_target)
stage11_safe = token_ids_to_safe(stage11_reconstructed[0])
stage11_smiles = strict_safe_to_smiles(stage11_safe)
assert Chem.MolFromSmiles(stage11_smiles) is not None

RUN_PAPER_SCALE_DE_NOVO = False
display(pd.DataFrame([{
    "len.pk count": len(stage11_lengths),
    "minimum": min(stage11_lengths),
    "median": float(torch.tensor(stage11_lengths, dtype=torch.float32).median()),
    "maximum": max(stage11_lengths),
    "oracle SAFE": stage11_safe,
    "strict SMILES": stage11_smiles,
    "mask trace": str(stage11_trace),
    "paper-scale run": RUN_PAPER_SCALE_DE_NOVO,
}]))


# Stage 12 - Molecular Context Guidance (MCG)

## Guidance without task-specific fine-tuning

MCG asks the same denoiser two questions. D_good is its prediction from the current partially generated sequence. D_poor is its prediction after additionally masking gamma times 100 percent of the currently visible, editable tokens. Equation (7) combines logits as

guided = w * D_good + (1 - w) * D_poor
       = D_good + (w - 1) * (D_good - D_poor).

For w greater than 1, features that rely on coherent molecular context are amplified. This is autoguidance, not property conditioning: no QED, docking score, or oracle label enters the network.

## Concrete example

If one token has good logits [2, 0] and poor logits [1, 1], then w=2 gives [3, -1]. The contrast between contextual and corrupted predictions is strengthened.

The corruption must be sampled independently for every batch row. Reusing row 0's selected positions for all rows can mask padding or the wrong chemical tokens.

## Paper vs released GitHub code

- The paper defines Equation (7), masks gamma percent of tokens for D_poor, and uses w=2 in the reported guided experiments.
- The GitHub implementation chooses corruptible positions from the first row and reuses them across the batch.
- This notebook samples each row separately and explicitly protects BOS, EOS, PAD, and existing MASK tokens.
- Gamma is task-specific and tuned in the paper; public experiment defaults of gamma=0 are not evidence that zero was the reported optimum.

## What the next code proves

It verifies the algebra, the w=1 identity, the gamma=0 identity, protected-token invariance, truly row-local corruption, and the reusable two-pass denoiser wrapper.

### Why mixing raw logits is valid

Equation (7) is written using log probabilities, while an implementation normally mixes raw logits. At one position, log-softmax differs from logits only by a category-independent normalization constant. The weighted constants also remain category-independent and cancel in the next softmax, so both forms produce the same categorical distribution.

## Comprehension checkpoint

1. Why can w>1 sharpen context-dependent predictions?
2. Does MCG tell the denoiser that a molecule has a high docking score?
3. Why must poor-context masks be sampled separately per row?
4. What should happen exactly when w=1 or gamma=0?


In [ ]:
def make_poor_context(
    input_ids,
    *,
    gamma,
    mask_token_id,
    protected_token_ids,
    generator=None,
):
    if not 0.0 <= gamma <= 1.0:
        raise ValueError("gamma must lie in [0, 1].")
    poor = input_ids.clone()
    protected = torch.zeros_like(input_ids, dtype=torch.bool)
    for token_id in protected_token_ids:
        protected |= input_ids.eq(int(token_id))
    eligible = ~protected

    for row in range(input_ids.shape[0]):
        positions = eligible[row].nonzero(as_tuple=False).flatten()
        count = int(round(float(gamma) * positions.numel()))
        count = min(count, positions.numel())
        if count:
            order = torch.randperm(
                positions.numel(), generator=generator, device=positions.device
            )
            poor[row, positions[order[:count]]] = int(mask_token_id)
    return poor


def combine_mcg_logits(good_logits, poor_logits, *, guidance_weight):
    if good_logits.shape != poor_logits.shape:
        raise ValueError("Good and poor logits must have identical shapes.")
    weight = float(guidance_weight)
    return weight * good_logits + (1.0 - weight) * poor_logits


def molecular_context_guided_logits(
    denoiser,
    input_ids,
    times,
    *,
    gamma,
    guidance_weight,
    mask_token_id,
    protected_token_ids,
    generator=None,
):
    good_logits = denoiser(input_ids, times)
    if gamma == 0.0:
        return good_logits, input_ids.clone()
    poor_ids = make_poor_context(
        input_ids,
        gamma=gamma,
        mask_token_id=mask_token_id,
        protected_token_ids=protected_token_ids,
        generator=generator,
    )
    poor_logits = denoiser(poor_ids, times)
    return combine_mcg_logits(
        good_logits, poor_logits, guidance_weight=guidance_weight
    ), poor_ids


stage12_good = torch.tensor([[[2.0, 0.0]]])
stage12_poor = torch.tensor([[[1.0, 1.0]]])
assert torch.equal(
    combine_mcg_logits(stage12_good, stage12_poor, guidance_weight=2.0),
    torch.tensor([[[3.0, -1.0]]]),
)
assert torch.equal(
    combine_mcg_logits(stage12_good, stage12_poor, guidance_weight=1.0),
    stage12_good,
)

stage12_rows = torch.tensor([
    [STAGE11_BOS_ID, 11, 12, STAGE11_EOS_ID, STAGE11_PAD_ID],
    [STAGE11_BOS_ID, 21, 22, STAGE11_EOS_ID, STAGE11_PAD_ID],
])
stage12_protected = (
    STAGE11_BOS_ID,
    STAGE11_EOS_ID,
    STAGE11_PAD_ID,
    tokenizer.mask_token_id,
)
stage12_poor_rows = make_poor_context(
    stage12_rows,
    gamma=0.5,
    mask_token_id=tokenizer.mask_token_id,
    protected_token_ids=stage12_protected,
    generator=torch.Generator().manual_seed(1212),
)
assert torch.all(stage12_poor_rows[:, 0] == STAGE11_BOS_ID)
assert torch.all(stage12_poor_rows[:, 3] == STAGE11_EOS_ID)
assert torch.all(stage12_poor_rows[:, 4] == STAGE11_PAD_ID)
assert torch.all(stage12_poor_rows.eq(tokenizer.mask_token_id).sum(dim=1) == 1)

stage12_zero = make_poor_context(
    stage12_rows,
    gamma=0.0,
    mask_token_id=tokenizer.mask_token_id,
    protected_token_ids=stage12_protected,
    generator=torch.Generator().manual_seed(1),
)
assert torch.equal(stage12_zero, stage12_rows)

stage12_guided_logits, stage12_guided_poor_rows = molecular_context_guided_logits(
    lambda ids, _times: torch.nn.functional.one_hot(
        ids, num_classes=len(tokenizer)
    ).to(torch.float32),
    stage12_rows,
    torch.tensor([0.25, 0.75], dtype=torch.float32),
    gamma=0.5,
    guidance_weight=2.0,
    mask_token_id=tokenizer.mask_token_id,
    protected_token_ids=stage12_protected,
    generator=torch.Generator().manual_seed(2),
)
assert stage12_guided_logits.shape == (*stage12_rows.shape, len(tokenizer))
assert torch.all(
    (stage12_guided_poor_rows == tokenizer.mask_token_id).sum(dim=1) >= 1
)
print("Two-pass MCG wrapper output shape:", tuple(stage12_guided_logits.shape))

display(pd.DataFrame({
    "row": [0, 1],
    "original": [str(row.tolist()) for row in stage12_rows],
    "poor context": [str(row.tolist()) for row in stage12_poor_rows],
    "new masks": stage12_poor_rows.eq(tokenizer.mask_token_id).sum(dim=1).tolist(),
}))


# Stage 13 - Fragment-constrained generation

## One denoiser, several molecular-design tasks

SAFE makes fixed molecular fragments contiguous, so a constraint can be represented by visible token blocks while unknown regions are masks. The same sampler then supports linker design, scaffold morphing, motif extension, scaffold decoration, and superstructure generation. The task changes the template, not the network weights.

Two primitives are enough to expose the idea:

1. insert a mask block before EOS to extend a known sequence;
2. replace an interior token interval by masks while leaving the surrounding context visible.

A chemical post-check is still required: token preservation alone does not prove that the decoded graph contains the intended substructure.

## Concrete example

For a clean sequence BOS A B C D EOS, masking B C gives BOS A MASK MASK D EOS. An oracle denoiser reconstructs B C while A and D must stay byte-for-byte unchanged. Separately, RDKit checks that the query CO is a substructure of CCO.

## Paper vs released GitHub code

- The paper uses N=1 and tau=1.2. Randomness r is 3 for linker design and scaffold morphing, 1.2 for motif extension, and 2 for scaffold decoration and superstructure generation.
- The released fragment-completion helper ignores its min_add_len argument, and its addmask path forwards an unused mask_len.
- This notebook makes the requested mask length an explicit, tested input and keeps task settings in one auditable table.
- The oracle smoke test validates constraint plumbing, not the paper's benchmark numbers.

## What the next code proves

It checks exact mask insertion, exact interval replacement, immutable visible context during sampling, the published task hyperparameters, and graph-level substructure containment.

## Comprehension checkpoint

1. Why can one unconditional denoiser support several fragment tasks?
2. Why is preserving visible tokens necessary but not sufficient?
3. Which task parameters change between linker design and motif extension?
4. What repository argument-wiring issue is avoided here?


In [ ]:
FRAGMENT_TASK_SETTINGS = {
    "linker_design": {"N": 1, "tau": 1.2, "r": 3.0},
    "scaffold_morphing": {"N": 1, "tau": 1.2, "r": 3.0},
    "motif_extension": {"N": 1, "tau": 1.2, "r": 1.2},
    "scaffold_decoration": {"N": 1, "tau": 1.2, "r": 2.0},
    "superstructure_generation": {"N": 1, "tau": 1.2, "r": 2.0},
}


def _trim_after_eos(token_ids, *, eos_token_id, pad_token_id):
    values = [int(value) for value in token_ids.tolist()]
    if eos_token_id not in values:
        raise ValueError("EOS token is required.")
    return values[: values.index(eos_token_id) + 1]


def insert_masks_before_eos(
    token_ids,
    *,
    number_of_masks,
    eos_token_id,
    pad_token_id,
    mask_token_id,
):
    if number_of_masks < 1:
        raise ValueError("number_of_masks must be positive.")
    values = _trim_after_eos(
        token_ids, eos_token_id=eos_token_id, pad_token_id=pad_token_id
    )
    eos_position = values.index(eos_token_id)
    expanded = (
        values[:eos_position]
        + [int(mask_token_id)] * int(number_of_masks)
        + values[eos_position:]
    )
    return torch.tensor(expanded, dtype=torch.long)


def mask_known_sequence_interval(
    token_ids,
    *,
    start,
    stop,
    mask_token_id,
    protected_token_ids,
):
    if not 0 <= start < stop <= token_ids.numel():
        raise ValueError("Expected a non-empty half-open interval inside the sequence.")
    result = token_ids.clone()
    protected = torch.zeros_like(result, dtype=torch.bool)
    for token_id in protected_token_ids:
        protected |= result.eq(int(token_id))
    interval = torch.zeros_like(result, dtype=torch.bool)
    interval[start:stop] = True
    editable = interval & ~protected
    if not editable.any():
        raise ValueError("The requested interval contains no editable tokens.")
    result[editable] = int(mask_token_id)
    return result, editable


def contains_substructure(candidate_smiles, query_smiles):
    candidate = Chem.MolFromSmiles(candidate_smiles)
    query = Chem.MolFromSmiles(query_smiles)
    if candidate is None or query is None:
        return False
    return candidate.HasSubstructMatch(query)


stage13_clean = stage11_target[0].clone()
stage13_editable_positions = (
    ~stage13_clean.eq(STAGE11_BOS_ID)
    & ~stage13_clean.eq(STAGE11_EOS_ID)
    & ~stage13_clean.eq(STAGE11_PAD_ID)
).nonzero(as_tuple=False).flatten()
assert stage13_editable_positions.numel() >= 2
stage13_start = int(stage13_editable_positions[0])
stage13_stop = int(stage13_editable_positions[min(2, stage13_editable_positions.numel() - 1)]) + 1
stage13_template, stage13_was_masked = mask_known_sequence_interval(
    stage13_clean,
    start=stage13_start,
    stop=stage13_stop,
    mask_token_id=tokenizer.mask_token_id,
    protected_token_ids=(STAGE11_BOS_ID, STAGE11_EOS_ID, STAGE11_PAD_ID),
)
stage13_extended = insert_masks_before_eos(
    stage13_clean,
    number_of_masks=3,
    eos_token_id=STAGE11_EOS_ID,
    pad_token_id=STAGE11_PAD_ID,
    mask_token_id=tokenizer.mask_token_id,
)
assert int(stage13_extended.eq(tokenizer.mask_token_id).sum()) == 3

stage13_target = stage13_clean.unsqueeze(0)
def stage13_scripted_logits(token_ids, times):
    logits = torch.full(
        (*token_ids.shape, len(tokenizer)),
        -30.0,
        dtype=torch.float32,
        device=token_ids.device,
    )
    target = stage13_target.to(token_ids.device).expand_as(token_ids)
    logits.scatter_(-1, target.unsqueeze(-1), 30.0)
    return logits

stage13_result, stage13_trace = confidence_sample(
    stage13_scripted_logits,
    stage13_template.unsqueeze(0),
    mask_token_id=tokenizer.mask_token_id,
    tokens_per_step=1,
    temperature=1.2,
    randomness=0.0,
    generator=torch.Generator().manual_seed(1313),
    forbidden_token_ids=(
        tokenizer.mask_token_id,
        STAGE11_PAD_ID,
        STAGE11_BOS_ID,
        STAGE11_EOS_ID,
    ),
)
assert torch.equal(stage13_result, stage13_target)
assert torch.equal(
    stage13_result[0][~stage13_was_masked],
    stage13_template[~stage13_was_masked],
)
assert contains_substructure("CCO", "CO")
assert not contains_substructure("CC", "CO")

display(pd.DataFrame([
    {"check": "task settings", "result": str(FRAGMENT_TASK_SETTINGS)},
    {"check": "masked interval tokens", "result": int(stage13_was_masked.sum())},
    {"check": "oracle mask trace", "result": str(stage13_trace)},
    {"check": "CCO contains CO", "result": contains_substructure("CCO", "CO")},
]))


# Stage 14 - Two decomposition rules and fragment scoring

## Training fragments and remasking fragments are intentionally different

The paper uses two representations of the same molecular graph:

- R_vocab makes three independent augmentations, each cutting one randomly chosen single, non-ring bond. These larger fragments populate and score the optimization vocabulary.
- R_remask cuts every single, non-ring bond. This produces more dots and smaller fragments so one local region can be replaced precisely.

The underlying molecule is unchanged: SAFE attachment labels retain how the pieces reconnect. As you observed earlier, R_remask usually has more fragments, but it is still the same graph after decoding.

For fragment f, Equation (5) uses the mean property of all molecules containing f:

score(f) = sum over containing molecules of score(molecule) / count of containing molecules.

A fragment is counted once per molecule even if it occurs twice in that molecule.

## Concrete example

If fragment A occurs in molecules scoring 0.2 and 0.8, its score is 0.5. The cell also cuts CCOC(=O)N1CCCCC1 with both rules and proves every SAFE variant decodes to the same canonical SMILES.

## Paper vs released GitHub code

- The paper specifies three independent one-bond R_vocab cuts and the all-bond R_remask cut.
- The repository implements the same conceptual distinction for vocabulary construction and remasking.
- During online PMO updates, the released optimizer initializes a newly seen fragment with the current parent score; that shortcut is not yet the exact running mean in Equation (5).
- This notebook implements exact sum/count statistics for fragments emitted by the sampled R_vocab decompositions.

## What the next code proves

It verifies bond eligibility, graph identity after both SAFE decompositions, the expected increase in fragment count for R_remask, and a hand-calculated mean over emitted-fragment observations.

### A scoring-estimator nuance

Equation (5) can be read literally as averaging over every molecule whose graph contains fragment f. Exhaustive literal evaluation would require subgraph checks against all molecules. The released code, and the table below, instead update f when R_vocab emits it in a sampled decomposition. This is a decomposition-occurrence estimator. The running sum/count is exact for those emitted observations, but it is not an exhaustive subgraph census.

## Comprehension checkpoint

1. Why does R_remask deliberately create smaller fragments than R_vocab?
2. Do extra dots mean a different molecule?
3. Why are ring bonds excluded?
4. Why should a repeated fragment be counted only once for one molecule?


In [ ]:
from collections import defaultdict
import numpy as np
from safe import encode as safe_encode


def canonical_smiles(smiles):
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise ValueError(f"Invalid SMILES: {smiles}")
    return Chem.MolToSmiles(molecule)


def eligible_non_ring_single_bonds(molecule):
    return [
        (bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())
        for bond in molecule.GetBonds()
        if bond.GetBondType() == Chem.BondType.SINGLE and not bond.IsInRing()
    ]


def random_vocab_cut_sets(molecule, *, number_of_augmentations=3, seed=0):
    eligible = eligible_non_ring_single_bonds(molecule)
    if not eligible:
        raise ValueError("The molecule has no eligible non-ring single bond.")
    rng = np.random.default_rng(seed)
    return [[eligible[int(rng.integers(len(eligible)))]] for _ in range(number_of_augmentations)]


def all_remask_cut_set(molecule):
    return eligible_non_ring_single_bonds(molecule)


def encode_with_atom_pair_cuts(smiles, atom_pair_cuts):
    selected = [tuple(map(int, pair)) for pair in atom_pair_cuts]
    return safe_encode(
        smiles,
        canonical=True,
        slicer=lambda molecule: selected,
    )


def decode_to_canonical_smiles(safe_text):
    return canonical_smiles(safe_decode(safe_text))


class FragmentScoreTable:
    def __init__(self):
        self._sum = defaultdict(float)
        self._molecule_count = defaultdict(int)

    def update(self, safe_text, molecule_score):
        for fragment in set(safe_text.split(".")):
            self._sum[fragment] += float(molecule_score)
            self._molecule_count[fragment] += 1

    def score(self, fragment):
        count = self._molecule_count[fragment]
        if not count:
            raise KeyError(fragment)
        return self._sum[fragment] / count

    def rows(self):
        return [
            {
                "fragment": fragment,
                "mean score": self.score(fragment),
                "molecule count": self._molecule_count[fragment],
            }
            for fragment in sorted(self._molecule_count)
        ]


stage14_smiles = canonical_smiles("CCOC(=O)N1CCCCC1")
stage14_molecule = Chem.MolFromSmiles(stage14_smiles)
stage14_vocab_cut_sets = random_vocab_cut_sets(
    stage14_molecule, number_of_augmentations=3, seed=1414
)
stage14_vocab_safe = [
    encode_with_atom_pair_cuts(stage14_smiles, cuts)
    for cuts in stage14_vocab_cut_sets
]
stage14_all_cuts = all_remask_cut_set(stage14_molecule)
stage14_remask_safe = encode_with_atom_pair_cuts(stage14_smiles, stage14_all_cuts)

for safe_text in stage14_vocab_safe + [stage14_remask_safe]:
    assert decode_to_canonical_smiles(safe_text) == stage14_smiles
assert len(stage14_all_cuts) >= len(stage14_vocab_cut_sets[0])
assert stage14_remask_safe.count(".") >= stage14_vocab_safe[0].count(".")

stage14_scores = FragmentScoreTable()
stage14_scores.update("A.B", 0.2)
stage14_scores.update("A.A.C", 0.8)
assert abs(stage14_scores.score("A") - 0.5) < 1e-12
assert stage14_scores._molecule_count["A"] == 2

display(pd.DataFrame([
    {
        "representation": "R_vocab augmentation 1",
        "cuts": len(stage14_vocab_cut_sets[0]),
        "fragments": stage14_vocab_safe[0].count(".") + 1,
        "SAFE": stage14_vocab_safe[0],
    },
    {
        "representation": "R_remask",
        "cuts": len(stage14_all_cuts),
        "fragments": stage14_remask_safe.count(".") + 1,
        "SAFE": stage14_remask_safe,
    },
]))
display(pd.DataFrame(stage14_scores.rows()))


# Stage 15 - Attach promising fragments, then remask one fragment

## The optimization proposal operator

Algorithm 1 alternates a graph operation and a sequence operation:

1. choose high-scoring vocabulary fragments;
2. attach a pair through matching dummy-atom labels;
3. convert the molecule to R_remask;
4. choose exactly one dot-delimited fragment;
5. replace that fragment by a mask block whose length is sampled from empirical p_len;
6. let GenMol complete the masks.

Attachment preserves valence only if the resulting graph sanitizes, so proposals need bounded retries and explicit failure. Unbounded retry loops can hang an experiment and hide a broken vocabulary.

## Concrete example

The fragments [1*]CC and [1*]O share attachment label 1. Removing the two dummy atoms and bonding their neighbors yields CCO. For sequence remasking, all non-selected SAFE fragments and dot separators remain unchanged while one selected fragment becomes a run of MASK tokens.

## Paper vs released GitHub code

- The paper samples replacement length from empirical p_len.
- The released remask path samples a hard-coded integer from 5 through 14, which is a practical divergence from the paper.
- This notebook accepts the empirical fragment-length list explicitly, replaces exactly one fragment, and bounds attachment attempts.
- The following smoke test uses known fragments and an oracle; it is not an optimization result.

## What the next code proves

It validates labeled-dummy attachment, RDKit sanitization, exact one-fragment replacement, empirical length sampling, unchanged surrounding context, and successful oracle completion.

## Comprehension checkpoint

1. Why are attachment labels needed?
2. Why must a proposed molecule be sanitized after graph surgery?
3. What remains fixed when one R_remask fragment is replaced?
4. How does the paper's p_len differ from the released hard-coded range?


In [ ]:
def attach_labeled_fragments(fragment_a, fragment_b, *, max_attempts=3):
    if max_attempts < 1:
        raise ValueError("max_attempts must be positive.")
    last_reason = "no attempt"
    for _ in range(max_attempts):
        molecule_a = Chem.MolFromSmiles(fragment_a)
        molecule_b = Chem.MolFromSmiles(fragment_b)
        if molecule_a is None or molecule_b is None:
            last_reason = "invalid fragment SMILES"
            continue

        combined = Chem.CombineMols(molecule_a, molecule_b)
        dummy_atoms = [
            atom for atom in combined.GetAtoms()
            if atom.GetAtomicNum() == 0 and atom.GetIsotope() != 0
        ]
        labels = sorted({atom.GetIsotope() for atom in dummy_atoms})
        shared = [
            label for label in labels
            if sum(atom.GetIsotope() == label for atom in dummy_atoms) == 2
        ]
        if not shared:
            last_reason = "no attachment label appearing exactly twice"
            continue

        label = shared[0]
        selected = [atom for atom in dummy_atoms if atom.GetIsotope() == label]
        if any(atom.GetDegree() != 1 for atom in selected):
            last_reason = "dummy attachment atom must have degree one"
            continue

        neighbors = [atom.GetNeighbors()[0].GetIdx() for atom in selected]
        dummy_indices = [atom.GetIdx() for atom in selected]
        editable = Chem.RWMol(combined)
        editable.AddBond(neighbors[0], neighbors[1], Chem.BondType.SINGLE)
        for atom_index in sorted(dummy_indices, reverse=True):
            editable.RemoveAtom(atom_index)
        proposal = editable.GetMol()
        try:
            Chem.SanitizeMol(proposal)
        except Exception as error:
            last_reason = str(error)
            continue
        return Chem.MolToSmiles(proposal)

    raise ValueError(f"Attachment failed after {max_attempts} attempts: {last_reason}")


def replace_one_fragment_with_masks(
    token_ids,
    *,
    fragment_index,
    replacement_length,
    dot_token_id,
    mask_token_id,
    eos_token_id,
    pad_token_id,
):
    values = _trim_after_eos(
        token_ids, eos_token_id=eos_token_id, pad_token_id=pad_token_id
    )
    if values[-1] != eos_token_id:
        raise ValueError("Trimmed sequence must end in EOS.")
    interior = values[1:-1]
    boundaries = [-1] + [
        index for index, value in enumerate(interior) if value == int(dot_token_id)
    ] + [len(interior)]
    fragment_count = len(boundaries) - 1
    if not 0 <= fragment_index < fragment_count:
        raise IndexError("fragment_index is outside the dot-delimited SAFE sequence.")
    if replacement_length < 1:
        raise ValueError("replacement_length must be positive.")

    start = boundaries[fragment_index] + 1
    stop = boundaries[fragment_index + 1]
    replaced = (
        interior[:start]
        + [int(mask_token_id)] * int(replacement_length)
        + interior[stop:]
    )
    return torch.tensor([values[0]] + replaced + [values[-1]], dtype=torch.long), {
        "fragment_count": fragment_count,
        "start": start + 1,
        "stop": start + 1 + int(replacement_length),
        "original_length": stop - start,
    }


def sample_empirical_length(lengths, *, generator=None):
    choices = torch.as_tensor(list(lengths), dtype=torch.long)
    if choices.numel() == 0 or int(choices.min()) < 1:
        raise ValueError("Empirical lengths must be a non-empty positive collection.")
    index = torch.randint(choices.numel(), (1,), generator=generator)
    return int(choices[index].item())


stage15_attached = attach_labeled_fragments("[1*]CC", "[1*]O")
assert stage15_attached == canonical_smiles("CCO")

stage15_encoded = tokenizer(
    stage14_remask_safe,
    add_special_tokens=True,
    return_tensors="pt",
)["input_ids"][0]
stage15_dot_id = int(tokenizer.convert_tokens_to_ids("."))
assert stage15_dot_id != tokenizer.unk_token_id
stage15_parts = stage14_remask_safe.split(".")
stage15_empirical_fragment_lengths = [
    len(tokenizer(part, add_special_tokens=False)["input_ids"])
    for part in stage15_parts
]
stage15_replacement_length = sample_empirical_length(
    stage15_empirical_fragment_lengths,
    generator=torch.Generator().manual_seed(1515),
)
stage15_fragment_index = min(1, len(stage15_parts) - 1)
stage15_template, stage15_meta = replace_one_fragment_with_masks(
    stage15_encoded,
    fragment_index=stage15_fragment_index,
    replacement_length=stage15_replacement_length,
    dot_token_id=stage15_dot_id,
    mask_token_id=tokenizer.mask_token_id,
    eos_token_id=STAGE11_EOS_ID,
    pad_token_id=STAGE11_PAD_ID,
)
assert int(stage15_template.eq(tokenizer.mask_token_id).sum()) == stage15_replacement_length
assert stage15_template.tolist().count(stage15_dot_id) == stage15_encoded.tolist().count(stage15_dot_id)

stage15_scripted_template, _ = replace_one_fragment_with_masks(
    stage15_encoded,
    fragment_index=stage15_fragment_index,
    replacement_length=stage15_meta["original_length"],
    dot_token_id=stage15_dot_id,
    mask_token_id=tokenizer.mask_token_id,
    eos_token_id=STAGE11_EOS_ID,
    pad_token_id=STAGE11_PAD_ID,
)
stage15_scripted_target = torch.tensor(
    _trim_after_eos(
        stage15_encoded,
        eos_token_id=STAGE11_EOS_ID,
        pad_token_id=STAGE11_PAD_ID,
    ),
    dtype=torch.long,
).unsqueeze(0)


def stage15_scripted_logits(token_ids, times):
    logits = torch.full(
        (*token_ids.shape, len(tokenizer)),
        -30.0,
        dtype=torch.float32,
        device=token_ids.device,
    )
    target = stage15_scripted_target.to(token_ids.device).expand_as(token_ids)
    logits.scatter_(-1, target.unsqueeze(-1), 30.0)
    return logits


stage15_scripted_result, stage15_scripted_trace = confidence_sample(
    stage15_scripted_logits,
    stage15_scripted_template.unsqueeze(0),
    mask_token_id=tokenizer.mask_token_id,
    tokens_per_step=4,
    temperature=1.2,
    randomness=0.0,
    generator=torch.Generator().manual_seed(1516),
    forbidden_token_ids=(
        tokenizer.mask_token_id,
        STAGE11_PAD_ID,
        STAGE11_BOS_ID,
        STAGE11_EOS_ID,
    ),
)
assert torch.equal(stage15_scripted_result, stage15_scripted_target)
stage15_context_mask = ~stage15_scripted_template.eq(tokenizer.mask_token_id)
assert torch.equal(
    stage15_scripted_result[0][stage15_context_mask],
    stage15_scripted_template[stage15_context_mask],
)

display(pd.DataFrame([
    {"check": "attachment", "result": f"[1*]CC + [1*]O -> {stage15_attached}"},
    {"check": "R_remask fragments", "result": len(stage15_parts)},
    {"check": "sampled empirical mask length", "result": stage15_replacement_length},
    {"check": "replaced fragment index", "result": stage15_fragment_index},
    {"check": "surrounding context unchanged", "result": True},
]))


# Stage 16 - Goal-directed hit generation (PMO)

## Close the optimization loop

Algorithm 1 is an online search loop. It ranks fragments by the running mean from Stage 14, attaches two promising fragments, optionally remasks one fine fragment after warmup, generates a completion, evaluates it with a black-box oracle, and updates the vocabulary.

The paper limits the vocabulary to V=100. For the first 1,000 generated molecules it performs attachment only; zero-based generation 1000 is therefore the first that may remask. It uses N=1, tau=1.2, r=2, w=2, and a task-specific gamma.

PMO performance is the area under the average-top-10-score curve versus unique oracle calls, with at most 10,000 calls on each of 23 tasks and three runs. The paper does not specify every bookkeeping convention, so this notebook explicitly canonicalizes molecules, caches duplicates, rejects invalid SMILES without charging the oracle, and then reproduces the released repository's precise AUC convention.

## Concrete example

Calling an oracle on CCO and then OCC should consume one call because both canonicalize to CCO. Under the repository convention, 100 calls whose top-10 average is always 0.5 yield AUC 0.4975: the first 100-call trapezoid rises from zero, then the value is extended through call 10,000. Generation indices 0 through 999 are warmup; index 1000 enables remasking.

## Paper vs released GitHub code

- The paper defines AUC of average top-10 score; the repository makes it executable by measuring every 100 calls, starting from zero, extending early termination to 10,000, integrating trapezoids, and dividing by 10,000.
- The paper says first 1,000 generations are attachment-only.
- The released condition uses iter > warmup, producing an off-by-one boundary under its indexing.
- The paper's Table 8 gives tuned gamma values; the public command-line default is gamma=0.
- The released online vocabulary shortcut initializes unseen fragments from the current parent. This notebook's FragmentScoreTable keeps the exact running mean over emitted R_vocab observations; Stage 14 distinguishes that estimator from exhaustive subgraph containment.

## What the next code proves

It checks all 23 gamma values, canonical oracle caching, budget enforcement, the warmup boundary, AUC arithmetic, toy bounded-loop control-flow state updates, and vocabulary ranking with a deterministic fake oracle. The 10,000-call benchmark remains disabled.

## Comprehension checkpoint

1. Why must duplicate canonical molecules not consume two oracle calls?
2. What exactly changes after the first 1,000 generations?
3. Why is top-10 AUC more informative than only the final best score?
4. Which PMO conventions are notebook choices because the paper leaves them unspecified?


In [ ]:
from dataclasses import dataclass


PMO_GAMMA = {
    "albuterol_similarity": 0.2,
    "amlodipine_mpo": 0.3,
    "celecoxib_rediscovery": 0.0,
    "deco_hop": 0.2,
    "drd2": 0.0,
    "fexofenadine_mpo": 0.0,
    "gsk3b": 0.0,
    "isomers_c7h8n2o2": 0.5,
    "isomers_c9h10n2o2pf2cl": 0.0,
    "jnk3": 0.5,
    "median1": 0.2,
    "median2": 0.2,
    "mestranol_similarity": 0.0,
    "osimertinib_mpo": 0.0,
    "perindopril_mpo": 0.4,
    "qed": 0.0,
    "ranolazine_mpo": 0.0,
    "scaffold_hop": 0.0,
    "sitagliptin_mpo": 0.2,
    "thiothixene_rediscovery": 0.3,
    "troglitazone_rediscovery": 0.0,
    "valsartan_smarts": 0.4,
    "zaleplon_mpo": 0.4,
}
assert len(PMO_GAMMA) == 23


class CachedBudgetedOracle:
    def __init__(self, score_function, *, budget):
        if budget < 1:
            raise ValueError("budget must be positive.")
        self.score_function = score_function
        self.budget = int(budget)
        self.cache = {}
        self.call_order = []

    @property
    def calls(self):
        return len(self.call_order)

    def __call__(self, smiles):
        molecule = Chem.MolFromSmiles(smiles)
        if molecule is None:
            raise ValueError("Invalid SMILES are rejected before the oracle.")
        canonical = Chem.MolToSmiles(molecule)
        if canonical in self.cache:
            return self.cache[canonical]
        if self.calls >= self.budget:
            raise RuntimeError("Oracle-call budget exhausted.")
        score = float(self.score_function(canonical))
        self.cache[canonical] = score
        self.call_order.append(canonical)
        return score




def repository_top_k_auc(
    scores,
    *,
    k=10,
    oracle_budget=10_000,
    reporting_frequency=100,
):
    values = [float(score) for score in scores[:oracle_budget]]
    if not values:
        raise ValueError("At least one score is required.")
    if oracle_budget < reporting_frequency or oracle_budget % reporting_frequency:
        raise ValueError("The repository convention requires an evenly spaced budget.")

    x_values = [0.0]
    y_values = [0.0]
    for call_count in range(
        reporting_frequency,
        len(values) + 1,
        reporting_frequency,
    ):
        top = sorted(values[:call_count], reverse=True)[:k]
        x_values.append(float(call_count))
        y_values.append(float(np.mean(top)))

    if x_values[-1] < oracle_budget:
        x_values.append(float(oracle_budget))
        y_values.append(y_values[-1])
    return float(np.trapz(y_values, x_values) / oracle_budget)


def remasking_enabled(generation_index, *, warmup_generations=1000):
    if generation_index < 0:
        raise ValueError("generation_index must be non-negative.")
    return int(generation_index) >= int(warmup_generations)


def run_bounded_hit_loop(
    proposal_batches,
    *,
    oracle,
    fragment_decomposer,
    vocabulary_size=100,
    warmup_generations=1000,
):
    fragment_statistics = FragmentScoreTable()
    records = []
    for generation_index, candidates in enumerate(proposal_batches):
        mode = "attach+remask" if remasking_enabled(
            generation_index, warmup_generations=warmup_generations
        ) else "attachment-only"
        for candidate in candidates:
            score = oracle(candidate)
            fragments = list(dict.fromkeys(fragment_decomposer(candidate)))
            if not fragments:
                raise ValueError("Every valid candidate must yield at least one fragment.")
            fragment_statistics.update(".".join(fragments), score)
            records.append({
                "generation": generation_index,
                "mode": mode,
                "canonical_smiles": canonical_smiles(candidate),
                "score": score,
            })

    ranked = sorted(
        fragment_statistics.rows(),
        key=lambda row: (-row["mean score"], row["fragment"]),
    )[: int(vocabulary_size)]
    return records, ranked


assert not remasking_enabled(999)
assert remasking_enabled(1000)
stage16_constant_curve_auc = repository_top_k_auc([0.5] * 100, k=10)
assert abs(stage16_constant_curve_auc - 0.4975) < 1e-12

stage16_toy_oracle = CachedBudgetedOracle(
    lambda smiles: Chem.MolFromSmiles(smiles).GetNumHeavyAtoms() / 10.0,
    budget=10,
)
assert stage16_toy_oracle("CCO") == stage16_toy_oracle("OCC")
assert stage16_toy_oracle.calls == 1
stage16_records, stage16_ranked = run_bounded_hit_loop(
    [["CCO", "OCC"], ["CCCO", "c1ccccc1"]],
    oracle=stage16_toy_oracle,
    fragment_decomposer=lambda smiles: [canonical_smiles(smiles)],
    vocabulary_size=3,
    warmup_generations=1000,
)
assert stage16_toy_oracle.calls == 3
assert all(record["mode"] == "attachment-only" for record in stage16_records)

RUN_PMO_BENCHMARK = False
display(pd.DataFrame(stage16_records))
display(pd.DataFrame([{
    "PMO tasks": len(PMO_GAMMA),
    "paper budget per task": 10_000,
    "paper runs": 3,
    "paper vocabulary size": 100,
    "smoke unique oracle calls": stage16_toy_oracle.calls,
    "repository constant-curve AUC check": stage16_constant_curve_auc,
    "benchmark executed": RUN_PMO_BENCHMARK,
}]))


# Stage 17 - Lead-optimization qualification gates

## Optimize while preserving the lead

Lead optimization uses the same fragment loop but starts from a known active molecule and keeps the vocabulary unbounded. A generated candidate succeeds only if all four gates hold:

docking(candidate) < docking(seed),
QED(candidate) >= 0.6,
SA(candidate) <= 4,
Tanimoto(candidate, seed) >= delta.

Lower docking score means stronger predicted binding. The paper evaluates delta in {0.4, 0.6}, runs 10 iterations with 100 generations each, uses three seeds for each of five targets, and reports the best docking score among qualifying molecules.

## Concrete example

For a seed docking score of -7.5, a candidate at -8.0 passes the affinity gate, but it still fails if QED is 0.59, SA is 4.01, or similarity is below delta. Equality at the QED, SA, and similarity thresholds is accepted; docking must improve strictly.

## Paper vs released GitHub code

- The paper uses V=infinity, N=1, tau=1.2, r=2, w=2 and Table 9's gamma per target and seed.
- It reports success on 26 of 30 target/seed/delta tasks.
- The paper does not fully specify the QuickVina protocol, fingerprint radius/size, tie handling, or code-level invalid-molecule policy. This notebook records Morgan radius 2 and 2,048 bits as explicit evaluation choices.
- The released public lead command defaults gamma to zero; reproducing the table requires the tuned values below.

## What the next code proves

It encodes every Table 9 gamma, computes modern RDKit Morgan similarity without deprecated calls, tests every inequality boundary, and selects the lowest-docking qualifying candidate. Actual QuickVina docking is guarded and not fabricated.

## Comprehension checkpoint

1. Why is a lower docking score better here?
2. Why is similarity a constraint rather than the optimization objective?
3. Which threshold comparisons allow equality?
4. Why can the smoke test verify gate logic but not reproduce a docking result?


In [ ]:
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator


LEAD_GAMMA_ROWS = [
    {"target": "parp1", "seed_docking": -7.3, "gamma": 0.2},
    {"target": "parp1", "seed_docking": -7.8, "gamma": 0.2},
    {"target": "parp1", "seed_docking": -8.2, "gamma": 0.2},
    {"target": "fa7", "seed_docking": -6.4, "gamma": 0.3},
    {"target": "fa7", "seed_docking": -6.7, "gamma": 0.4},
    {"target": "fa7", "seed_docking": -8.5, "gamma": 0.0},
    {"target": "5ht1b", "seed_docking": -4.5, "gamma": 0.3},
    {"target": "5ht1b", "seed_docking": -7.6, "gamma": 0.0},
    {"target": "5ht1b", "seed_docking": -9.8, "gamma": 0.4},
    {"target": "braf", "seed_docking": -9.3, "gamma": 0.2},
    {"target": "braf", "seed_docking": -9.4, "gamma": 0.1},
    {"target": "braf", "seed_docking": -9.8, "gamma": 0.5},
    {"target": "jak2", "seed_docking": -7.7, "gamma": 0.5},
    {"target": "jak2", "seed_docking": -8.0, "gamma": 0.0},
    {"target": "jak2", "seed_docking": -8.6, "gamma": 0.1},
]
assert len(LEAD_GAMMA_ROWS) == 15
assert {row["target"] for row in LEAD_GAMMA_ROWS} == {
    "parp1", "fa7", "5ht1b", "braf", "jak2"
}

stage17_morgan = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


def morgan_tanimoto(smiles_a, smiles_b):
    molecule_a = Chem.MolFromSmiles(smiles_a)
    molecule_b = Chem.MolFromSmiles(smiles_b)
    if molecule_a is None or molecule_b is None:
        raise ValueError("Morgan similarity requires two valid molecules.")
    fingerprint_a = stage17_morgan.GetFingerprint(molecule_a)
    fingerprint_b = stage17_morgan.GetFingerprint(molecule_b)
    return float(DataStructs.TanimotoSimilarity(fingerprint_a, fingerprint_b))


@dataclass(frozen=True)
class LeadMeasurements:
    smiles: str
    docking: float
    qed: float
    sa: float
    similarity: float


def is_qualifying_lead(measurement, *, seed_docking, delta):
    return (
        measurement.docking < float(seed_docking)
        and measurement.qed >= 0.6
        and measurement.sa <= 4.0
        and measurement.similarity >= float(delta)
    )


def select_best_qualifying_lead(measurements, *, seed_docking, delta):
    qualifying = [
        measurement for measurement in measurements
        if is_qualifying_lead(
            measurement, seed_docking=seed_docking, delta=delta
        )
    ]
    return min(qualifying, key=lambda item: item.docking) if qualifying else None


stage17_boundary_pass = LeadMeasurements("CCO", -8.0, 0.6, 4.0, 0.4)
stage17_equal_docking_fail = LeadMeasurements("CCO", -7.5, 0.9, 1.0, 1.0)
stage17_low_qed_fail = LeadMeasurements("CCO", -9.0, 0.59, 1.0, 1.0)
assert is_qualifying_lead(
    stage17_boundary_pass, seed_docking=-7.5, delta=0.4
)
assert not is_qualifying_lead(
    stage17_equal_docking_fail, seed_docking=-7.5, delta=0.4
)
assert not is_qualifying_lead(
    stage17_low_qed_fail, seed_docking=-7.5, delta=0.4
)
stage17_best = select_best_qualifying_lead(
    [
        stage17_boundary_pass,
        LeadMeasurements("CCCO", -8.4, 0.7, 3.0, 0.5),
        LeadMeasurements("CCCCO", -9.0, 0.7, 3.0, 0.3),
    ],
    seed_docking=-7.5,
    delta=0.4,
)
assert stage17_best.smiles == "CCCO"
stage17_similarity_example = morgan_tanimoto("CCO", "CCCO")
assert 0.0 <= stage17_similarity_example <= 1.0

RUN_QUICKVINA_DOCKING = False
display(pd.DataFrame(LEAD_GAMMA_ROWS))
display(pd.DataFrame([{
    "gate boundary accepted": True,
    "strict docking improvement": True,
    "best qualifying smoke lead": stage17_best.smiles,
    "CCO/CCCO Morgan similarity": stage17_similarity_example,
    "QuickVina executed": RUN_QUICKVINA_DOCKING,
}]))


# Stage 18 - Evaluation metrics and reproduction ledger

## Measure claims without mixing denominators

This final V1 stage computes:

- validity: valid RDKit molecules divided by all generated strings;
- uniqueness: unique canonical molecules divided by valid molecules;
- diversity: mean pairwise one-minus-Tanimoto over unique valid molecules;
- quality: generated molecules satisfying QED >= 0.6 and SA <= 4, with invalid strings counted as failures.

Canonicalization makes CCO and OCC duplicates. Diversity is undefined for fewer than two unique valid molecules; this notebook returns 0 and records that convention.

The reproduction ledger separates three levels of evidence: implemented, deterministic smoke-tested, and paper-scale reproduced. A green unit test cannot be promoted into a paper result.

## Concrete example

For [CCO, OCC, benzene, invalid], three of four strings are valid but only two canonical molecules are unique. Thus validity is 0.75 and uniqueness is 2/3.

## Paper vs released GitHub code

- The paper uses RDKit Morgan fingerprints and TDC for diversity, QED, and SA, reporting three-run means and standard deviations.
- The notebook uses RDKit's contributed SA scorer and records its Morgan settings. Exact library versions and denominator conventions matter when comparing numbers.
- The paper's 945M-example training corpus, trained checkpoint, PMO oracle suite, and docking assets are not present. Those rows correctly remain not reproduced.
- Every algorithmic component is now executable and tested without pretending that smoke outputs equal Tables 1 through 5.

## What the next code proves

It hand-checks validity and uniqueness, evaluates diversity and quality, confirms expensive-run guards are false, and prints a stage-by-stage evidence matrix.

## Comprehension checkpoint

1. Why canonicalize before measuring uniqueness?
2. Should an invalid string count as a high-quality molecule?
3. Why are implemented, smoke-tested, and paper-scale-reproduced separate columns?
4. What additional assets and compute are required before comparing against paper tables?


In [ ]:
from itertools import combinations
from rdkit.Chem import QED
from importlib import import_module

sascorer = import_module("rdkit.Contrib.SA_Score.sascorer")
from rdkit import RDLogger


def evaluate_generation(smiles_values):
    if not smiles_values:
        raise ValueError("At least one generated string is required.")

    RDLogger.DisableLog("rdApp.error")
    parsed = []
    try:
        for text in smiles_values:
            molecule = Chem.MolFromSmiles(text)
            if molecule is not None:
                parsed.append((Chem.MolToSmiles(molecule), molecule))
    finally:
        RDLogger.EnableLog("rdApp.error")

    canonical_valid = [canonical for canonical, _ in parsed]
    unique_canonical = sorted(set(canonical_valid))
    unique_molecules = [Chem.MolFromSmiles(text) for text in unique_canonical]
    fingerprints = [
        stage17_morgan.GetFingerprint(molecule) for molecule in unique_molecules
    ]
    pair_distances = [
        1.0 - float(DataStructs.TanimotoSimilarity(left, right))
        for left, right in combinations(fingerprints, 2)
    ]

    quality_count = sum(
        float(QED.qed(molecule)) >= 0.6
        and float(sascorer.calculateScore(molecule)) <= 4.0
        for _, molecule in parsed
    )
    return {
        "validity": len(parsed) / len(smiles_values),
        "uniqueness": len(unique_canonical) / len(parsed) if parsed else 0.0,
        "diversity": float(np.mean(pair_distances)) if pair_distances else 0.0,
        "quality": quality_count / len(smiles_values),
        "valid_count": len(parsed),
        "unique_count": len(unique_canonical),
    }


stage18_examples = ["CCO", "OCC", "c1ccccc1", "not-a-smiles"]
stage18_metrics = evaluate_generation(stage18_examples)
assert abs(stage18_metrics["validity"] - 0.75) < 1e-12
assert abs(stage18_metrics["uniqueness"] - 2.0 / 3.0) < 1e-12
assert 0.0 <= stage18_metrics["diversity"] <= 1.0
assert 0.0 <= stage18_metrics["quality"] <= 1.0
assert not RUN_PAPER_SCALE_TRAINING
assert not RUN_PAPER_SCALE_DE_NOVO
assert not RUN_PMO_BENCHMARK
assert not RUN_QUICKVINA_DOCKING

REPRODUCTION_LEDGER = [
    {"stage": "0 runtime/GPU selection", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "1 SAFE representation", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "2 forward diffusion", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "3 Equation 2 reverse posterior", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "4 weighted MDLM loss", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "5 BERT denoiser", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "6 optimizer transaction", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "7-8 data/DDP/EMA/checkpoint", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "9-12 reverse/confidence/MCG", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "13-15 fragment pipeline", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "16 PMO hit loop", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "17 lead gates", "implemented": True, "smoke tested": True, "paper scale": False},
    {"stage": "18 metrics", "implemented": True, "smoke tested": True, "paper scale": False},
]
assert all(row["implemented"] and row["smoke tested"] for row in REPRODUCTION_LEDGER)
NOTEBOOK_V1_SMOKE_TESTS_PASSED = True

display(pd.DataFrame([stage18_metrics]))
display(pd.DataFrame(REPRODUCTION_LEDGER))
print("GenMol V1 implementation smoke tests passed:", NOTEBOOK_V1_SMOKE_TESTS_PASSED)
print("Paper-scale claims reproduced:", False)


# Appendix - Extended SAFE V2 is post-paper

## Keep the scientific target clean

The linked repository's current main branch includes Extended SAFE V2. It adds angle-bracket tokens and bracket-SAFE conversion utilities, and its V2 configuration uses batch size 1,024. Those changes postdate the GenMol V1 paper, whose Appendix D specifies vocabulary size 1,880 and global batch size 2,048.

Mixing V2 into the V1 stages would silently change tokenization, sequence statistics, and training configuration. Therefore the notebook imports the conversion functions only as a discoverability check and keeps V2 disabled. A future V2 experiment should have its own checkpoint, tokenizer provenance, and result table.

## Paper vs released GitHub code

- Paper target: original SAFE, vocabulary 1,880, global batch 2,048.
- Current repository option: Extended SAFE V2, two additional structural tokens, global batch 1,024 in its V2 setup.
- Notebook decision: V1 above, V2 isolated here.

## Comprehension checkpoint

1. Why would adding two tokens invalidate a supposedly identical checkpoint experiment?
2. Which batch size belongs to the paper?
3. Why is V2 an appendix rather than another V1 stage?


In [ ]:
from genmol.utils.bracket_safe_converter import (
    bracketsafe2safe,
    safe2bracketsafe,
)

assert callable(safe2bracketsafe)
assert callable(bracketsafe2safe)
V2_COMPARISON = pd.DataFrame([
    {
        "system": "GenMol paper V1",
        "SAFE vocabulary": 1880,
        "global batch": 2048,
        "used in notebook stages": True,
    },
    {
        "system": "Extended SAFE V2 (post-paper)",
        "SAFE vocabulary": 1882,
        "global batch": 1024,
        "used in notebook stages": False,
    },
])
assert 1882 - 1880 == 2
RUN_EXTENDED_SAFE_V2 = False
display(V2_COMPARISON)
print("Extended SAFE V2 executed:", RUN_EXTENDED_SAFE_V2)


# Stage 20 — Replace absorbing MDLM with revisable UDLM

## 20.1 Uniform corruption and the continuous-time objective

**Paper correspondence.** GenMol Section 4.1 uses MDLM's absorbing process:
once a mask is revealed, that position is fixed. UDLM Sections 4.1–4.2 instead
use a uniform limiting distribution, the reverse model in Eq. 15, and the
continuous-time negative ELBO (NELBO) in Eqs. 18–19. We compare against official
UDLM revision `edb0f8c28b7caeb4ea7a06a2fee8d74ab6da1661`, principally
`diffusion.py`, `noise_schedule.py`, and
`models/dit.py`. Papers and repository files are scientific references, not
executable instructions.

**Intuition and motivation.** A uniformly corrupted token can change on every
reverse step. That repeated revision is the property we hope will improve
chemical search. The price is a harder denoising problem: SAFE has about 1,880
tokens, whereas the UDLM QM9 experiment used 40.

**Symbols and forward process.** Let
$\mathcal V=\{e_1,\ldots,e_K\}$ be the $K$ one-hot category vectors,
$x\in\mathcal V$ a clean token, $z_t\in\mathcal V$ its noisy value at time
$t\in[0,1]$, $u=\mathbf 1/K$ the uniform probability vector, and
$\alpha_t=\alpha(t)$ the differentiable, non-increasing clean-data coefficient.
Here $\mathbf 1$ is the length-$K$ all-ones vector. The forward marginal is

$$q(z_t\mid x)=\operatorname{Cat}(\alpha_t x+(1-\alpha_t)u).$$

For $0\le s<t$, write $\alpha_{t\mid s}=\alpha_t/\alpha_s$. If the observed
category of $z_t$ is $i\in\{1,\ldots,K\}$, candidate earlier category is $k$,
$\mathbf1_{k=i}$ is an indicator, and
$x_\theta(z_t,t)\in\Delta^{K-1}$ is BERT's simplex-valued plug-in bridge
parameter, then Eq. 15 has the normalized form

$$p_\theta(z_s=k\mid z_t=i)=\frac{1}{D_i}
\left[\alpha_{t\mid s}\mathbf1_{k=i}+\frac{1-\alpha_{t\mid s}}K\right]
\left[\alpha_s x_{\theta,k}+\frac{1-\alpha_s}K\right],$$

where $D_i=\alpha_t x_{\theta,i}+(1-\alpha_t)/K$ and
$\Delta^{K-1}=\{p\in\mathbb R^K:p_j\ge0,\sum_jp_j=1\}$.

The 2024 UDLM paper describes $x_\theta$ as a clean-token prediction. A later
[analysis of uniform diffusion](https://arxiv.org/abs/2605.22765) proves a
sharper interpretation: under this standard bridge plug-in ELBO, the optimum is
the leave-one-out (LOO) posterior, which predicts the clean token at position
$\ell$ from $z_t^{-\ell}$ without its own noisy observation, not the ordinary
denoising posterior conditioned on all of $z_t$. Let $r_j$ be that LOO
probability, $d_j$ the ordinary denoising probability, and $i$ the observed
category. For any full-support rank-one prior $\pi$ they convert exactly as

$$d_j\propto r_j[\alpha_t\mathbf1\{j=i\}+(1-\alpha_t)\pi_i],\qquad
r_j\propto\frac{d_j}{\alpha_t\mathbf1\{j=i\}+(1-\alpha_t)\pi_i}.$$

Our raw logits parameterize $r$ inside the bridge. The BERT architecture does
not enforce exact invariance to its own noisy token, so “LOO predictor” names
the objective's optimal target rather than asserting architectural invariance.
This later clarification does not change the faithful 2024 implementation or
the frozen screens, but temperature/top-p should act on the raw LOO logits;
quantities interpreted as a true denoiser require the conversion above.

**Full variational objective.** On a finite reverse grid
$1=t_T>t_{T-1}>\cdots>t_0>0$, write $s_m=t_{m-1}$ and let $q$ denote the
joint forward path. Before taking a continuous-time limit, the non-negative
variational upper bound is

$$\mathcal L_T=\mathbb E_q\!\left[
-\log p_\theta(x\mid z_{t_0})
+\sum_{m=1}^{T}D_{\rm KL}\!\left(
q(z_{s_m}\mid z_{t_m},x)\,\|\,p_\theta(z_{s_m}\mid z_{t_m})\right)
+D_{\rm KL}\!\left(q(z_{t_T}\mid x)\,\|\,p(z_{t_T})\right)
\right].$$

Here $T$ is the number of finite transitions and $D_{\rm KL}$ is categorical
Kullback--Leibler divergence. Under the ideal endpoint conditions
$\alpha(0)=1$ and $\alpha(1)=0$, a copying reconstruction parameterization and
$p(z_1)=u$ make the first and last terms vanish as $T\to\infty$. The middle
sum then becomes the Eq. 18 integral below.

For the loss, define $\bar x=K\alpha_t x+(1-\alpha_t)\mathbf1$,
$\bar x_\theta=K\alpha_t x_\theta+(1-\alpha_t)\mathbf1$,
$r_j=\bar x_j/\bar x_i$, $\hat r_j=\bar x_{\theta,j}/\bar x_{\theta,i}$,
and $v_j=\log\hat r_j-\log r_j$ for $j\in\{1,\ldots,K\}$. For
$t\in(0,1)$, with $\alpha'_t=d\alpha(t)/dt\le0$, the per-token integrand is

$$\ell_\theta(x,z_t,t)=\frac{-\alpha'_t}{K\alpha_t}
\sum_{j\ne i}r_j\,[\exp(v_j)-1-v_j].$$

Since $\exp(v)-1-v\ge0$, this form is non-negative and avoids cancellation.
For a length-$L$ sequence $x^{(1:L)}$, let $p_{\rm data}$ be the clean-sequence
distribution, let $q(z_t^{(1:L)}\mid x^{(1:L)})$ be the token-factorized forward
law, and let $C(x)\subseteq\{1,\ldots,L\}$ contain exactly the positions that
are valid under the attention mask **and are not PAD, BOS, or EOS**. Thus $C(x)$
is the molecular-content support used by `GenMol.diffusion_token_mask`, not just
"all non-padding positions." Let $\theta$ denote all model parameters. The
prediction for position $\ell$ is $x_\theta^{(\ell)}(z_t^{(1:L)},t)$, so
$\ell_\theta^{(\ell)}(x,z_t,t)$ means the scalar integrand above evaluated at
that position while conditioning on the whole noisy sequence. The full
sequence objective is

$$\mathcal L^\infty=
\mathbb E_{x^{(1:L)}\sim p_{\rm data}}\int_0^1
\mathbb E_{z_t^{(1:L)}\sim q(\cdot\mid x^{(1:L)})}
\left[\sum_{\ell\in C(x)}
\ell_\theta^{(\ell)}(x,z_t,t)\right]dt.$$

Training estimates both expectations with sampled data, one sampled time per
sequence, and one sampled noisy sequence. Within one process-local microbatch,
the configured `global_mean_loss=True` reduction for $B$ sequences is

$$\widehat{\mathcal L}_{\rm token}=
\frac{\sum_{b=1}^B\sum_{\ell\in C(x_b)}
\ell_\theta^{(\ell)}(x_b,z_{t_b},t_b)}
{\sum_{b=1}^B|C(x_b)|}.$$

For one fixed microbatch this is exactly $B/\sum_b|C(x_b)|$ times the mean of
the $B$ sequence sums, so it does **not** change their relative contributions
inside that fixed group; longer sequences contain more summands in both forms.
The scale factor does vary with the length composition of successive
microbatches, so their stochastic contributions can differ. There is also an
important distributed-training qualification. Let $R$ be the number of DDP
ranks and $A$ the number of gradient-accumulation microbatches in a complete
optimizer step. Ordinary DDP and Lightning accumulation produce an equal mean
of the $RA$ local ratios,

$$\frac1{RA}\sum_{r=1}^{R}\sum_{a=1}^{A}
\frac{\sum_{b,\ell\in C(x_{r,a,b})}\ell_\theta^{(\ell)}}
{\sum_b |C(x_{r,a,b})|},$$

not one numerator divided by the token count pooled across all ranks and
microbatches. The two coincide when $R=A=1$ or all local denominators match.
This follows NVIDIA's released process-local `global_mean` behavior for a
faithful baseline; it is a labeled weighting limitation, not an exact global
token mean. With `global_mean_loss=False`, the code first divides each sequence
sum by $|C(x_b)|$ and then averages sequences locally before the same DDP and
accumulation averaging, giving each sequence equal weight inside its
microbatch. These normalizations are not a single constant rescaling unless the
relevant content lengths and group denominators are equal. An exact
cross-rank/accumulation token reduction must be tested as a separate repair.

The definition of $C(x)$ is a **GenMol-specific training-support adaptation**.
The current faithful MDLM path uses the full attention mask and therefore may
corrupt and supervise BOS/EOS, whereas this UDLM path clamps and removes them
from the loss. This support change is distinct from the separate prior-alphabet
ablation that optionally removes UNK/CLS/SEP/PAD/MASK from uniform refreshes.
Consequently a final method-causality claim requires an MDLM control with the
same content-only framing; without it, the support mismatch must remain an
explicit limitation even if the operational UDLM system wins.

**Concrete example.** With $K=3$, clean token A, and $\alpha_t=0.2$, the noisy
probabilities are $(0.2+0.8/3,\,0.8/3,\,0.8/3)=(0.4667,0.2667,0.2667)$.
Unlike masking, B or C can later return to A—or change again.

**Code below.** Notebook-native functions implement the forward probabilities,
Eq. 15 posterior, and stable Eq. 18 integrand directly. They are independent
oracles for `ContinuousUniformDiffusion`. Token IDs are `(B,L)`, logits and
probabilities are `(B,L,K)`, and times are `(B,)`. We also compare empirical
forward frequencies with the analytical three-token example.

**Released code and schedule qualification.** The release evaluates Eq. 18 by
subtracting two potentially large terms; production GenMol uses the equivalent
`expm1(v)-v` identity. The release contains a compatibility mismatch, which we
preserve for the faithful control:
corruption and reverse sampling use
$\alpha_{\rm rel}(t)=1-(1-\epsilon)t$ with $\epsilon=10^{-3}$, while the loss
hard-codes $\alpha_{\rm ideal}(t)=1-t$ and $\alpha'_{\rm ideal}=-1$. Therefore
the stable loss is exactly equivalent to the *released idealized loss*, but is
not the exact NELBO of the residual-clean forward schedule. In particular,
$\alpha_{\rm rel}(1)=10^{-3}$ means $q(z_1\mid x)$ is not exactly uniform, so
the paper's analytically zero prior-loss condition does not hold exactly even
though sampling starts from an exact uniform prior. A schedule-consistent repair
is a later, separately labeled ablation.

**Comprehension checkpoint.** Why can UDLM repair an early mistake while MDLM
cannot? Expected reasoning: uniform reverse transitions resample all editable
positions, whereas the absorbing posterior copies every already-visible MDLM
token. Why is the compatibility loss not the exact NELBO of its forward process?
Expected reasoning: the forward process uses $\alpha_{\rm rel}$ with a nonzero
endpoint, while the loss and zero-prior derivation use $\alpha_{\rm ideal}$.
Why is large $K$ risky? Expected reasoning: corruption can replace a token with
any of many rare or incompatible alternatives, so clean prediction is harder.
Why is the raw plug-in output not automatically the ordinary denoising
posterior? Expected reasoning: for uniform diffusion the bridge is nonlinear in
the clean proxy, and the expected plug-in ELBO is optimized by a LOO posterior;
the ordinary denoiser includes an extra factor from the observed token's forward
likelihood.


In [ ]:
import torch
from torch.nn import functional as F
from genmol.diffusion import ContinuousUniformDiffusion


def reference_uniform_forward_probs(clean_ids, alpha, num_classes):
    '''Implement q(z_t | x) directly; return shape (B, L, K).'''
    if clean_ids.ndim != 2 or alpha.shape != (clean_ids.shape[0],):
        raise ValueError("expected clean_ids (B,L) and alpha (B,)")
    clean_one_hot = F.one_hot(clean_ids, num_classes=num_classes).to(torch.float64)
    alpha = alpha.to(torch.float64)[:, None, None]
    return alpha * clean_one_hot + (1.0 - alpha) / num_classes


def reference_uniform_reverse_probs(clean_logits, noisy_ids, t, s, noise_eps):
    '''Implement the normalized Eq. 15 product directly.'''
    num_classes = clean_logits.shape[-1]
    clean_probs = clean_logits.to(torch.float64).softmax(-1)
    noisy_one_hot = F.one_hot(noisy_ids, num_classes).to(torch.float64)
    alpha_t = (1.0 - (1.0 - noise_eps) * t.to(torch.float64))[:, None, None]
    alpha_s = (1.0 - (1.0 - noise_eps) * s.to(torch.float64))[:, None, None]
    alpha_t_given_s = alpha_t / alpha_s
    transition_to_observation = (
        alpha_t_given_s * noisy_one_hot
        + (1.0 - alpha_t_given_s) / num_classes
    )
    predicted_marginal_s = alpha_s * clean_probs + (1.0 - alpha_s) / num_classes
    observed_clean_probability = torch.gather(
        clean_probs, -1, noisy_ids[..., None]
    )
    normalizer = alpha_t * observed_clean_probability + (1.0 - alpha_t) / num_classes
    return transition_to_observation * predicted_marginal_s / normalizer


def reference_stable_udlm_integrand(clean_logits, clean_ids, noisy_ids, t):
    '''Implement the release-compatible idealized Eq. 18 integrand.'''
    num_classes = clean_logits.shape[-1]
    clean_probs = clean_logits.to(torch.float64).softmax(-1)
    alpha = (1.0 - t.to(torch.float64))[:, None, None]
    clean_one_hot = F.one_hot(clean_ids, num_classes).to(torch.float64)
    noisy_one_hot = F.one_hot(noisy_ids, num_classes).to(torch.bool)
    x_bar = num_classes * alpha * clean_one_hot + (1.0 - alpha)
    x_bar_theta = num_classes * alpha * clean_probs + (1.0 - alpha)
    gather_index = noisy_ids[..., None]
    log_r = x_bar.log() - torch.gather(x_bar.log(), -1, gather_index)
    log_r_hat = x_bar_theta.log() - torch.gather(
        x_bar_theta.log(), -1, gather_index
    )
    v = log_r_hat - log_r
    phi = torch.expm1(v) - v
    phi = torch.where(noisy_one_hot, torch.zeros_like(phi), phi)
    return (log_r.exp() * phi).sum(-1) / (num_classes * alpha.squeeze(-1))


def reference_released_literal_udlm_integrand(clean_logits, clean_ids, noisy_ids, t):
    '''Implement the released Eq. 18 subtraction before stable rearrangement.'''
    num_classes = clean_logits.shape[-1]
    clean_probs = clean_logits.to(torch.float64).log_softmax(-1).exp()
    alpha = (1.0 - t.to(torch.float64))[:, None, None]
    clean_one_hot = F.one_hot(clean_ids, num_classes).to(torch.float64)
    x_bar = num_classes * alpha * clean_one_hot + (1.0 - alpha)
    x_bar_theta = num_classes * alpha * clean_probs + (1.0 - alpha)
    gather_index = noisy_ids[..., None]
    x_bar_i = torch.gather(x_bar, -1, gather_index)
    x_bar_theta_i = torch.gather(x_bar_theta, -1, gather_index)
    coefficient = -1.0 / (num_classes * alpha)
    term_1 = num_classes / x_bar_i - num_classes / x_bar_theta_i
    term_2 = (
        (x_bar / x_bar_i)
        * (
            x_bar_theta_i.log()
            - x_bar_theta.log()
            + x_bar.log()
            - x_bar_i.log()
        )
    ).sum(-1, keepdim=True)
    return (coefficient * (term_1 - term_2)).squeeze(-1)


toy_udlm = ContinuousUniformDiffusion(
    num_classes=3,
    noise_eps=1e-3,
    antithetic_sampling=False,
)
x0 = torch.tensor([[0, 1, 2]])                           # (B=1, L=3)
xt = torch.tensor([[2, 1, 0]])                           # (B=1, L=3)
t = torch.tensor([0.8], dtype=torch.float64)             # (B=1,)
s = torch.tensor([0.5], dtype=torch.float64)             # (B=1,)

forward_probs = reference_uniform_forward_probs(x0, toy_udlm.alpha(t), 3)
expected_first_forward = torch.tensor(
    [0.2008 + 0.7992 / 3, 0.7992 / 3, 0.7992 / 3],
    dtype=torch.float64,
)
assert torch.allclose(forward_probs[0, 0], expected_first_forward, atol=1e-12)

draw_count = 30_000
draw_clean = torch.zeros((draw_count, 1), dtype=torch.long)
draw_times = torch.full((draw_count,), 0.8)
draws = toy_udlm.forward_process(
    draw_clean,
    draw_times,
    generator=torch.Generator().manual_seed(20),
)
empirical_forward = torch.bincount(draws[:, 0], minlength=3) / draw_count
assert torch.allclose(empirical_forward, expected_first_forward.float(), atol=0.01)

imperfect_logits = torch.zeros(1, 3, 3, dtype=torch.float64)  # (B, L, K)
posterior_reference = reference_uniform_reverse_probs(
    imperfect_logits, xt, t, s, noise_eps=1e-3
)
posterior = toy_udlm.posterior_probs(imperfect_logits, xt, t, s)
assert posterior.shape == (1, 3, 3)
assert torch.all(posterior >= 0)
assert torch.allclose(posterior.sum(-1), torch.ones(1, 3, dtype=torch.float64))
assert torch.allclose(posterior, posterior_reference, atol=1e-12, rtol=1e-12)

perfect_logits = torch.full((1, 3, 3), -100.0, dtype=torch.float64)
perfect_logits.scatter_(-1, x0[..., None], 100.0)
perfect_loss = toy_udlm.loss_per_token(
    perfect_logits, x0, xt, t
)
imperfect_loss = toy_udlm.loss_per_token(imperfect_logits, x0, xt, t)
reference_imperfect_loss = reference_stable_udlm_integrand(
    imperfect_logits, x0, xt, t
)
assert torch.allclose(perfect_loss, torch.zeros_like(perfect_loss), atol=1e-12)
assert torch.isfinite(imperfect_loss).all() and torch.all(imperfect_loss > 0)
assert torch.allclose(imperfect_loss, reference_imperfect_loss, atol=1e-12, rtol=1e-12)

batched_x0 = torch.tensor([[0, 1], [2, 0]])
batched_xt = torch.tensor([[1, 1], [0, 2]])
batched_t = torch.tensor([0.35, 0.9], dtype=torch.float64)
batched_s = torch.tensor([0.1, 0.4], dtype=torch.float64)
batched_logits = torch.randn(
    2, 2, 3, dtype=torch.float64, generator=torch.Generator().manual_seed(21)
)
batched_posterior_reference = reference_uniform_reverse_probs(
    batched_logits, batched_xt, batched_t, batched_s, noise_eps=1e-3
)
batched_posterior = toy_udlm.posterior_probs(
    batched_logits, batched_xt, batched_t, batched_s
)
batched_loss_reference = reference_stable_udlm_integrand(
    batched_logits, batched_x0, batched_xt, batched_t
)
batched_loss = toy_udlm.loss_per_token(
    batched_logits, batched_x0, batched_xt, batched_t
)
assert batched_posterior.shape == (2, 2, 3)
assert batched_loss.shape == (2, 2)
assert torch.allclose(
    batched_posterior, batched_posterior_reference, atol=1e-12, rtol=1e-12
)
assert torch.allclose(
    batched_loss, batched_loss_reference, atol=1e-12, rtol=1e-12
)

gradient_logits = batched_logits.detach().clone().requires_grad_()
production_for_gradient = toy_udlm.loss_per_token(
    gradient_logits, batched_x0, batched_xt, batched_t
)
released_literal = reference_released_literal_udlm_integrand(
    gradient_logits, batched_x0, batched_xt, batched_t
)
production_gradient = torch.autograd.grad(
    production_for_gradient.sum(), gradient_logits, retain_graph=True
)[0]
released_literal_gradient = torch.autograd.grad(
    released_literal.sum(), gradient_logits
)[0]
assert torch.allclose(
    production_for_gradient, released_literal, atol=2e-12, rtol=2e-12
)
assert torch.allclose(
    production_gradient, released_literal_gradient, atol=2e-11, rtol=2e-11
)

stage20_reference_checks = {
    "forward_shape": tuple(forward_probs.shape),
    "analytical_forward_A": forward_probs[0, 0].tolist(),
    "empirical_forward_A": empirical_forward.tolist(),
    "posterior_shape": tuple(posterior.shape),
    "posterior_matches_reference": True,
    "stable_loss_matches_reference": True,
    "batched_loss_shape": tuple(batched_loss.shape),
    "batched_cross_checks_passed": True,
    "released_literal_value_and_gradient_match": True,
    "perfect_loss_zero": True,
}

print(stage20_reference_checks)


## 20.2 Time-condition GenMol's BERT

**Paper correspondence and motivation.** UDLM predicts the plug-in bridge
parameter $x_\theta(z_t,t)$, so the LOO predictor must know the noise level. The official DiT
maps the log-linear total noise $\sigma(t)=-\log[1-(1-\epsilon)t]$ through
sinusoidal features and adaptive layer normalization (AdaLN). Here
$\epsilon=10^{-3}$ is the residual clean coefficient at $t=1$. GenMol's
released BERT has no time input.

**Mathematics and intuition.** For an even feature dimension $d$, let
$m\in\{0,\ldots,d/2-1\}$ and
$\omega_m=\exp[-\log(10{,}000)m/(d/2)]$. Concatenating
$(\cos(\sigma\omega_m))_m$ and $(\sin(\sigma\omega_m))_m$ gives a vector in
$\mathbb R^d$; an odd $d$ receives one trailing zero. An MLP maps it to
$c\in\mathbb R^H$, where $H$ is BERT's hidden size, and our adaptation adds $c$
to every word embedding. Low $\sigma$ means nearly clean input; high $\sigma$
means the network should rely more on global context.

**Concrete example and code below.** Times `(0.1, 0.9)` become two scalar noise
levels and then sinusoidal tensors `(2,16)`. The reference features must match
the production embedder exactly. A tiny BERT receives two copies of token IDs
with shape `(2,4)` and produces logits `(2,4,13)`. The adapter output projection
starts at zero, so the two outputs agree before training; its nonzero gradient
shows that time conditioning can begin learning on the first update.

**Difference from released code.** Additive conditioning preserves GenMol's
BERT and absolute positional embeddings. It is not the official DiT, which
applies an outer SiLU to the timestep MLP, uses rotary positions, and injects
zero-initialized shift/scale/gate vectors through every AdaLN block and the
output layer. Zero-initializing our adapter output is a GenMol warm-start
adaptation, not a released-UDLM detail. An MDLM checkpoint is therefore only a
BERT/MLM-head initialization, never a resumed UDLM run: optimizer, scheduler,
step counter, adapter, and EMA restart.

**Comprehension checkpoint.** Why not omit time because MDLM did? Expected
reasoning: MDLM's SUBS parameterization can be time-independent, but UDLM must
distinguish the reliability of the same visible token pattern at different
noise levels. Why zero-initialize only the adapter output? Expected reasoning:
it preserves imported BERT behavior while still giving that output layer a
gradient on the first update.


In [ ]:
from transformers.models.bert.configuration_bert import BertConfig
from genmol.backbone import TimestepEmbedder, TimeConditionedBertForMaskedLM


def reference_sinusoidal_noise_embedding(values, dimension, max_period=10_000):
    '''Notebook-native version of the official timestep features.'''
    half = dimension // 2
    frequencies = torch.exp(
        -torch.log(values.new_tensor(float(max_period), dtype=torch.float32))
        * torch.arange(half, device=values.device, dtype=torch.float32)
        / half
    )
    arguments = values.float()[:, None] * frequencies[None]
    features = torch.cat((torch.cos(arguments), torch.sin(arguments)), dim=-1)
    if dimension % 2:
        features = torch.cat((features, torch.zeros_like(features[:, :1])), dim=-1)
    return features


time_values = torch.tensor([0.1, 0.9])                   # (B=2,)
noise = -torch.log1p(-(1.0 - 1e-3) * time_values)        # sigma(t), (B=2,)
reference_features = reference_sinusoidal_noise_embedding(noise, 16)
production_features = TimestepEmbedder.sinusoidal_embedding(noise, 16)
assert reference_features.shape == (2, 16)
assert torch.allclose(reference_features, production_features)

tiny_bert = TimeConditionedBertForMaskedLM(
    BertConfig(
        vocab_size=13,
        hidden_size=24,
        num_hidden_layers=2,
        num_attention_heads=4,
        intermediate_size=48,
        max_position_embeddings=16,
        pad_token_id=3,
    ),
    time_embedding_size=16,
)
token_ids = torch.tensor([[1, 5, 8, 2], [1, 5, 8, 2]])  # same input, (B=2,L=4)
attention = torch.ones_like(token_ids)                    # (B=2, L=4)
conditioner_at_initialization = tiny_bert.time_conditioner(noise)
assert torch.count_nonzero(conditioner_at_initialization) == 0

tiny_bert.eval()
with torch.no_grad():
    initial_logits = tiny_bert(token_ids, attention, noise_level=noise).logits
assert initial_logits.shape == (2, 4, 13)
assert torch.allclose(initial_logits[0], initial_logits[1])

tiny_bert.train()
tiny_bert.zero_grad(set_to_none=True)
training_logits = tiny_bert(token_ids, attention, noise_level=noise).logits
training_logits.square().mean().backward()
adapter_gradient = tiny_bert.time_conditioner.mlp[-1].weight.grad
assert adapter_gradient is not None
assert torch.isfinite(adapter_gradient).all()
assert torch.count_nonzero(adapter_gradient) > 0

stage20_time_checks = {
    "time_shape": tuple(time_values.shape),
    "noise_levels": noise.tolist(),
    "feature_shape": tuple(reference_features.shape),
    "logit_shape": tuple(initial_logits.shape),
    "zero_init_preserves_same_input": True,
    "adapter_output_gradient_norm": float(adapter_gradient.norm()),
}
print(stage20_time_checks)


## 20.3 Reverse sampling, molecular invariants, and evidence gates

**Paper correspondence.** UDLM sampling begins with iid uniform tokens at
$t=1$ and walks a fixed grid $1=t_M>\cdots>t_0\approx0$. At each step BERT
emits the bridge proxy $x_\theta$, the posterior from 20.1 is formed, and every editable
position is sampled again. Here $M$ is the number of reverse steps and $t_m$ is
one grid time. There is no MDLM confidence ranking or monotone unmasking. The
official UDLM configurations disable cache reuse because the predictor is time
conditioned, although the generic released sampler contains a cache path.

**Molecular adaptation and intuition.** GenMol represents a requested edit with
`[MASK]` placeholders. The UDLM sampler first records their boolean locations,
replaces only those locations with the uniform prior, and clamps BOS, EOS,
padding, and supplied fragment context after every reverse step. For example,
`[BOS] context [MASK] [MASK] [EOS]` may revise the two editable tokens many
times, but `context` cannot drift.

**Shapes and invariants in the code below.** `state` is `(1,4)`, clean-token
logits are `(1,4,K)`, and `editable` is Boolean `(1,4)`. A seeded two-step toy
trajectory first draws the editable positions from the uniform prior and then
uses two different clean predictions. Both editable tokens change on both
steps, while positions 0 and 3 remain exactly unchanged throughout. Production
experiments will separately compare 32/64 reverse steps and retain the official
128-step setting as the faithful control.

**Differences and hypotheses.** Clamping is absent from the released QM9 code
and is labeled here as required molecular inpainting behavior. Excluding five
tokenizer control symbols from the uniform prior is optional and must be
compared against the faithful full-vocabulary prior. GenMol's raw-logit MCG is
disabled: UDLM D-CFG must combine conditional and unconditional *reverse
posterior* log probabilities.

**Concrete example.** Starting from `[BOS, 4, 3, EOS]`, the seeded toy path below
visits `[BOS, 2, 0, EOS]` and then `[BOS, 4, 4, EOS]`. Revision means the first
generated value need not be permanent; clamping means BOS and EOS are permanent.

**Comprehension checkpoint.** Why is cache reuse invalid even if a sampled state
does not change? Expected reasoning: the next predictor call receives a different
time/noise level. Why is a 100-sample win insufficient? Expected reasoning:
generation is stochastic and the paper reports three 1,000-sample runs, so a
small pilot has wide uncertainty and serves only as a progression gate.


In [ ]:
sampling_udlm = ContinuousUniformDiffusion(5, noise_eps=1e-3)
template = torch.tensor([[1, 3, 4, 2]])
editable = torch.tensor([[False, True, True, False]])
initial_context = template[~editable].clone()
sampling_generator = torch.Generator().manual_seed(0)
prior_draw = sampling_udlm.sample_prior(template.shape, generator=sampling_generator)
state = torch.where(editable, prior_draw, template)
trajectory = [state.clone()]

for target_id, time_t, time_s in ((0, 1.0, 0.5), (4, 0.5, 0.0)):
    clean_logits = torch.full((1, 4, 5), -80.0)
    clean_logits[..., target_id] = 80.0
    state = sampling_udlm.step(
        clean_logits,
        state,
        t=torch.tensor([time_t]),
        s=torch.tensor([time_s]),
        mutable_mask=editable,
        generator=sampling_generator,
    )
    assert torch.equal(state[~editable], initial_context)
    trajectory.append(state.clone())

changed_on_both_steps = (
    (trajectory[0][editable] != trajectory[1][editable])
    & (trajectory[1][editable] != trajectory[2][editable])
)
assert torch.all(changed_on_both_steps)
assert all(torch.equal(item[~editable], initial_context) for item in trajectory)

stage20_sampling_trace = [
    {"step": index, "time": time_value, "token_ids": item.tolist()[0]}
    for index, (time_value, item) in enumerate(zip((1.0, 0.5, 0.0), trajectory))
]
print(stage20_sampling_trace)


## 20.4 Validate bounded evidence and define the advancement gate

**Why this stage exists.** Equation checks establish correctness, not molecular
quality. The committed MDLM manifest in
[`experiments/udlm/baselines/`](experiments/udlm/baselines/) freezes the exact
local comparator, and the recorded CPU overfit artifacts in
[`experiments/udlm/cpu_smoke/`](experiments/udlm/cpu_smoke/) report git SHA
`02595d1ecf994bbd432ec67ef21b0d56ed97918d`, seed 1, 100 updates on the same
16-molecule toy set, and a 32-step chain. Their byte hashes freeze the recorded
results, but the generating runner revision, source-tree cleanliness, tokenizer
and model-config hashes, and exact command were not retained; they are bounded
integration evidence, not exactly reproducible benchmark evidence. The next
cell reads those JSON files; it does not train, sample, call an oracle, use a
GPU, or access a network.

**Concrete evidence.** Both full-vocabulary and special-token-excluded runs must
have finite lower last-five-step loss, improved fixed-$t=0.5$ loss, and at least
one strictly decoded molecule. These are integration gates only. Comparing 5/16
with 3/16 does not rank the priors because the sample is tiny and stochastic.

**Immutable launch evidence.** The user first chooses a count $W\in\{1,2\}$;
$W$ is a world size, not a physical device ID. Before any GPU query, the
launcher must exclusively acquire the repository-global single-training-job
lease. It then scans every NVIDIA device, chooses an ordered UUID tuple
$U=(u_1,\ldots,u_W)$, and re-probes exactly those UUIDs immediately before
launch. Each final record $f_i$ must have utilization strictly below 10%, at
least 30,000 MiB free, and non-prohibited compute mode. Active compute
processes are recorded as evidence but do not disqualify an otherwise eligible
device under the user's utilization-based idle definition; the launcher never
interrupts or kills them. Thus
$|U|=W$, the UUID order in the final telemetry is exactly $U$, and only one
reviewed pilot may hold the lease.

The raw bytes of `launch_manifest.json` are frozen by repository-relative path,
SHA-256, and schema 2. Training receives that digest out of band so the manifest
need not hash itself. Runtime-config schema 2, training-summary schema 5, and
successful-exit-receipt schema 5 must each repeat the same stable manifest
snapshot and exact $U$. The receipt also validates the still-held lease before
publication; after tmux handoff, only its writer may then unlink that exact
unchanged lease. Before handoff, the launcher may release only its own exact
lease if launch fails. Unexplained or stale leases fail closed for manual
review. A candidate lock that lacks any link in
`launch -> runtime -> summary -> receipt -> checkpoint` is inadmissible.
Summary schema 5 requires every model, EMA, optimizer, and other non-sentinel
floating checkpoint tensor to be finite. It separately verifies and records
the sole non-finite framework exception: Lightning 2.5.1's scalar float32
`ModelCheckpoint.kth_value=+inf` bookkeeping sentinel under the exact
unmonitored, minimum-mode callback state. This is not a general `Inf`
allowance; a wrong callback identity, value, shape, path, or second non-finite
tensor fails.
It also closes the checkpoint's top-level and EMA key schemas, binds the saved
Hydra configuration and Lightning loop progress to the live model and trainer,
and binds the live anomaly, clipping, precision, optimizer, scheduler, sampler,
and checkpoint callback state. For example, an opaque extra object hidden under
`ema` is rejected even if its tensor visitor cannot inspect that object's
internals. The independent optimization-screen verifier rechecks closed summary
and receipt shapes and their manifest/runtime/config joins instead of trusting
the producer's `completed` label.

Protocol v3 records why training-artifact schema 5 was needed. The first
10-update R health process completed its updates but failed closed before a
valid summary because Lightning's exact `kth_value=+inf` sentinel met the older
blanket finiteness rule. Its namespace and artifact hashes remain immutable and
scientifically ineligible. Protocol v4 now binds the independently validated
terminal 1,000-update R/S/E panel and freezes the generation framework before
any registered candidate generation. V4 changes no claim, operating point,
selection firewall, point or uncertainty threshold, candidate-lock rule, final
decision, or claim boundary from v3. Earlier ineligible CPU generation smokes,
the failed health namespace, and the audited MDLM baseline rescoring remain
disclosed rather than being erased by this scoped amendment.
Candidate-lock schema 2 additionally binds one terminal E successful receipt.
The final gate reconstructs that receipt's transitive R→S→E chain, requires
both the E receipt and selected training receipt to predate the lock, and
requires its matched-panel digest to equal the selected checkpoint's panel
digest. The selected receipt must be the exact R/S/E member of that chain, not
a separate run sharing its panel digest. The pilot ledger's winning checkpoint
must also equal the lock's training checkpoint. Thus R or S may still win pilot
selection, but the superiority claim cannot proceed from an incomplete or
cherry-picked panel.

**Progressive experiment gate and motivation.** First run the warm-start R/S/E
panel for only 10 optimizer updates. The only supported entry point is
`scripts/udlm/launch_health_panel.py`; it exposes only the user-selected world
size $W\in\{1,2\}$ and a CPU-only dry-run, not scientific or safety overrides.
The fixed launcher variants are `udlm` (R, released schedule and uniform prior),
`schedule_uniform` (S, schedule-consistent uniform control), and
`udlm_categorical` (E, schedule-consistent empirical-frequency prior). Every arm
uses seed 1, one loader worker, the full 1,880-token vocabulary, the audited
empirical-mixture field 0.0002 (active only for E), and the verified MDLM EMA
warm start from `outputs/paper_v1/checkpoints/50000.ckpt`, whose size is
1,396,998,679 bytes and SHA-256 is
`8d00aa47b02f64bf39ff6b0b2e786f213587366fc2c3d29712a00f3f84108dd6`.
For per-process microbatch $m=2$, accumulation is $a_1=8$ or $a_2=4$, so
$B_{\mathrm{eff}}=Wma_W=16$ for either supported world size. The wrapper fixes
utilization strictly below 10%, at least 30,000 MiB free, and non-prohibited
compute mode. It records active compute processes without using their presence
as a rejection criterion and never interrupts or kills them.

Let $H$ be the full clean pushed 40-character source revision. The deterministic
run names are `health-w{W}-r-{H}`, `health-w{W}-s-{H}`, and
`health-w{W}-e-{H}`. Each wrapper invocation launches only the first missing
member after a successful prefix and rejects incomplete, failed, malformed, or
out-of-order directories. A failed receipt closes that H namespace: preserve it
in place, repair and push a descendant, and restart R under the descendant's
distinct full-revision names. R requires explicit genesis. S and E cannot reach
a GPU probe without a validated successful receipt
from the immediately preceding arm; the exact receipt, manifest, and summary
snapshots are bound into the successor manifest and revalidated before its
successful exit receipt. The predecessor receipt must predate the successor's
lease acquisition and first GPU inventory query. The global lease separately
machine-enforces one-job concurrency.

The pass artifact is the schema-5 terminal receipt at
`output/udlm/health-w{W}-e-{H}/pilot_exit_status.json`, accepted by the CPU-only
`scripts/udlm/validate_health_panel.py`. That validator checks the complete
R→S→E chain and narrows it to this exact contract, including verified EMA
loading, finite training/checkpoint state, and checkpoint save/reload. Its
normalized result permits only screen authorization: generation, ranking,
superiority, and candidate-lock eligibility are false. The health panel cannot
rank the variants: the current constant schedule has 2,500 linear-warmup
updates, so at peak learning rate $3\times10^{-4}$ its value by update 10 is
only approximately $(10/2500)(3\times10^{-4})=1.2\times10^{-6}$.

**Completed registered screens, not a molecular experiment.** The exact W=1
health chain and both optimization screens have now completed under the
registry-aware launcher. Their fixed denoising panel selected E-L1 and then
E-A1; Stage 20.9 verifies the four committed evidence/selection envelopes and
teaches the next matched scale-up. No screen generated or scored molecules,
used a final seed, or established UDLM-over-GenMol superiority.

**Completed health-to-selection firewall.** The immutable publication chain is
$H=$`34856c275049cd329320f6c01171f0d2d34cd814`, R0=`95bb397`,
R1=`2d33e56`, R2=`d32df6`, and R3=`b49e900`. H produced the successful
ten-update W=1 R/S/E health chain. R0 added exactly the six W=1 resolved screen
configs; R1 added only the frozen registry and produced the scheduler runs; R2
added only scheduler evidence/selection and authorized two fresh conditioning
arms; R3 added only conditioning evidence/selection. The independent verifier
reconstructs this chronology and returns no winner for missing or unmatched
evidence. These screen decisions alone do not authorize the later 1,000-update
R/S/E panel, which needs its own selection-bound configuration and registry
firewall.

**Paper correspondence and motivation.** UDLM's plug-in bridge predictor emits
$x_\theta(z_t,t)$, so it needs the noise time (implemented here through total
noise $\sigma(t)$). The UDLM paper and official code motivate strong per-block
time conditioning; they do not establish that the following BERT adapter or
learning-rate path improves molecules. At the pinned official revision, the
QM9 recipe uses 25,000 optimizer steps, global batch 2,048, peak learning rate
$3\times10^{-4}$, 1,000 warmup steps, and cosine decay to $3\times10^{-6}$.
L0 instead preserves this project's inherited GenMol-style constant schedule
with 2,500-step warmup; it is not the released UDLM QM9 schedule. L1 scales the
warmup and horizon for a 100-update screen, so it is also a pilot hypothesis,
not an exact replay of the official recipe. The small screens separately ask
whether the existing empirical-prior arm E is starved by the inherited long
warmup and whether per-layer time modulation is more expressive than one
additive vector.

**L0/L1 mathematics and confound boundary.** Let optimizer-update index
$k\in\{0,\ldots,99\}$, peak learning rate $\eta=3\times10^{-4}$, L0 warmup
$w_0=2500$, L1 warmup $w_1=50$, L1 horizon $h=1000$, and L1 floor
$\eta_{\min}=3\times10^{-6}$. L0 uses
$\eta_{L0}(k)=\eta k/w_0$. L1 uses $\eta k/w_1$ for $k<w_1$ and

$$
\eta_{L1}(k)=\eta_{\min}+(\eta-\eta_{\min})
\frac{1+\cos\!\left(\pi(k-w_1)/(h-w_1)\right)}{2}
$$

for $w_1\le k\le h$, then clamps at $\eta_{\min}$. Here $k$ counts optimizer
updates, not microbatches. Across the 100 used indices,
$\sum_k\eta_{L0}(k)=5.94\times10^{-4}$ whereas
$\sum_k\eta_{L1}(k)=0.022317219370972547$; L1 therefore supplies
$37.571076\ldots$ times the cumulative learning-rate exposure. This sum is not
an equivalent AdamW step count or parameter-distance bound. E-L1 must be called
an **optimizer-schedule bundle**—shorter warmup, early exposure, later
half-cosine shape, and floor—not an isolated cosine-curvature ablation.

Both E-L0 and E-L1 use training seed 17, additive A0 conditioning, and 100
updates. Let $S_{a,j}$ be summed content-token loss and $N_{a,j}$ its integer
token denominator for arm $a$ at noise-time bin
$t_j\in\{0.1,0.5,0.9\}$. Define $\ell_{a,j}=S_{a,j}/N_{a,j}$ and pooled
$L_a=(\sum_jS_{a,j})/(\sum_jN_{a,j})$. Select L1 only if
$L_{L1}\le0.98L_{L0}$, at least two of three bin losses strictly decrease,
and every $\ell_{L1,j}\le1.02\ell_{L0,j}$. The verifier uses exact integer
cross-products rather than rounded ratios. For an equal-denominator example,
$N_{a,j}=100$, L0 sums $(1000,600,200)$ and L1 sums $(970,570,202)$ produce
means $(10,6,2)$ and $(9.7,5.7,2.02)$: pooled loss improves about 3.2%, two bins
improve, and the last worsens only 1%. A complete valid screen that misses a
threshold retains L0; missing, malformed, or unmatched evidence yields no
winner rather than a silent fallback.

**A0/A1 mathematics, shapes, and invariants.** With the selected scheduler,
both 500-update E arms start independently from the same verified MDLM-EMA
checkpoint; neither continues a scheduler-screen checkpoint. Let batch size be
$B$, sequence length $S$, hidden width $H$, BERT layer index
$l\in\{1,\ldots,L\}$, per-example total noise
$\sigma\in\mathbb R^B$, and $y_l\in\mathbb R^{B\times S\times H}$ be layer
$l$'s ordinary post-LayerNorm output. A0 keeps the zero-output additive
timestep adapter. A1 normally initializes the timestep MLP
$g:\mathbb R\to\mathbb R^H$, applies the official outer activation
$c=\operatorname{SiLU}(g(\sigma))\in\mathbb R^{B\times H}$, and computes one
zero-initialized projection per layer:

$$
[\beta_l,\gamma_l]=W_lc+b_l\in\mathbb R^{B\times2H},\qquad
h_l=(1+\gamma_l[:,\mathrm{None},:])\odot y_l+
\beta_l[:,\mathrm{None},:].
$$

$\beta_l,\gamma_l\in\mathbb R^{B\times H}$ are a shift and residual scale,
$W_l\in\mathbb R^{2H\times H}$ and $b_l\in\mathbb R^{2H}$ are trainable
parameters, and $\odot$ is elementwise multiplication. An explicit encoder loop
passes $h_l$ to stock BERT layer $l+1$ and sends final $h_L$ to the classifier;
it uses no hooks or mutable forward state. At initialization $W_l=b_l=0$, so
every $h_l=y_l$ exactly for every $\sigma$, the Jacobian with respect to $y_l$
is the identity, and MDLM warm-start logits and base gradients are unchanged.

**Concrete tensor example and checkpoint behavior.** For $B=2$, $S=4$, $H=24$,
and $L=2$, $c$ has shape `[2, 24]`; each projection emits `[2, 48]`, splits into
two `[2, 24]` tensors, broadcasts them as `[2, 1, 24]`, and preserves hidden
shape `[2, 4, 24]`. Production uses $H=768$ and $L=12$: each FiLM weight is
`[1536, 768]`, each bias `[1536]`, and all 12 projections contain 14,174,208
parameters. With the 787,968-parameter timestep MLP, A1 has 14,962,176
conditioning parameters. The explicit warm-start loader still maps the same 202
MDLM EMA base tensors, excludes four timestep plus 24 FiLM parameter tensors,
and creates 230 fresh EMA shadows. A0 retains its legacy state-key set; A1 saves
an exact conditioning manifest, and cross-topology checkpoint loads fail closed.

**Gradient staging and fair RNG initialization.** Zeroing both the timestep MLP
output and the FiLM projections would give $c=0$ and permanently zero FiLM
weight gradients, so A1 rejects that dead combination. With normally initialized
$g$, each FiLM parameter can receive a finite nonzero gradient at optimizer-
gradient observation one (after accumulation). Because $W_l=0$ then, the chain
rule gives zero gradient into $g$ at that observation. Both registered schedules
use learning-rate index zero on optimizer update one, so that zero-rate step
leaves $W_l=0$ and observation two also gives zero gradient to $g$. Optimizer
update two has positive learning rate and changes $W_l$; observation three can
then give every parameter in $g$ a finite nonzero gradient. The gate says
"after the first nonzero-learning-rate FiLM update" rather than assuming the
first optimizer call changes weights. Both A0 and A1 reseed all training RNG
streams to seed 17 *after* construction and MDLM-EMA loading. Otherwise A1's
extra parameter initialization would consume random draws and shift later
corruption/dropout randomness even under the same initial seed.

The exact production observation topology is frozen in
`experiments/udlm/protocols/film_gradient_contract_v1.json` with raw SHA-256
`b2a666a23351eb0882a179f7ae5d09fafd2188fee924313cdf60ee94888e7ac5` and
canonical SHA-256
`ff45961276df75f445221fd1aa4629262d21fdb852bd9b226ad56fe2559315d5`.
It binds all 24 FiLM and four timestep-MLP tensor names and shapes. Summary
schema 5 and receipt schema 5 require an explicit null for non-A1 arms or a contract-bound
A1 gradient certificate.

**Exact initialization-state attestation.** Immediately after each verified
MDLM-EMA warm start, and before RNG reseeding, dataloader/trainer creation, or
optimizer construction, the training entry point snapshots
`backbone.state_dict()`. Let $S$ be its name-sorted tensor sequence and let
$C\subset S$ exclude every timestep-MLP and FiLM tensor. A domain-separated
SHA-256 frames each tensor name, dtype, shape, and exact raw bytes. The ten-field
`screen_initialization_state_audit` records $|S|$, $H(S)$, $|C|$, and $H(C)$
alongside the checkpoint, resolved config, seed, phase, and conditioning
variant. L0/L1 must match in both $H(S)$ and $H(C)$; A0/A1 must match in
$H(C)$ even though their conditioning topology makes $H(S)$ differ. Summary
and receipt schema 5 carry the exact same object, preventing evidence assembly
from inventing initial-state hashes.

For a two-tensor toy state containing one shared BERT weight and one timestep
weight, changing only the timestep tensor changes $H(S)$ but not $H(C)$;
changing the shared tensor changes both. A separately pinned literal CPU probe
checks the functional invariant: A0 and zero-FiLM A1 must produce byte-exact
float32 logits before training. Released GenMol has no analogous attestation
because it does not compare these two time-conditioning topologies.

The full-size pre-registry CPU diagnostic exercised that invariant against the
actual 50,000-step MDLM EMA checkpoint. Both A0 and A1 emitted shape
`[2, 4, 1880]`, or 60,160 raw little-endian float32 bytes, with identical
SHA-256
`3e6ef7368f9a11d061640948ac5955fba81c2acac6546a12adc4efc5e22e15b8`.
The sequential check took 33.55 seconds and about 3,578,044 KiB peak RSS. It is
a topology diagnostic rather than registered screen evidence. The later frozen
W=1 conditioning screen independently reproduced the same logit shape and digest
before training and used that registered audit in the A0/A1 decision.

Select A1 only if exact initialization equality, the staged gradient checks,
pooled content-token fixed-panel loss improves by at least 2%, no time-bin
regression exceeds 2%, and clean-token accuracy is nondecreasing under an exact
integer cross-product. A complete valid screen that misses a condition retains
A0; missing, malformed, or unmatched evidence yields no winner.

**What the code below verifies.** It evaluates both learning-rate formulas on
the exact update indices 0 through 99, asserts their cumulative sums and ratio,
records the concrete A1 tensor shapes, derives production FiLM and conditioner
parameter counts, checks the 202 + 4 + 24 = 230 warm-start/EMA accounting, and
records the completed health/screen flags while keeping scale-up authorization
false until its separate registry is frozen. These are executable arithmetic
and protocol invariants; the screen results remain denoising evidence, not
molecular-generation evidence.

**Difference from released implementations.** Released GenMol's BERT has no
time input. Official UDLM uses a rotary, pre-LayerNorm DiT: its normally
initialized timestep MLP has an outer SiLU, every block produces two
shift/scale/gate triplets, and the output layer has another shift/scale pair.
A1 retains GenMol's absolute-position, post-LayerNorm BERT and applies only one
shift/scale pair after each stock layer, without residual gates. It is our
warm-start-compatible architecture hypothesis, not released UDLM and not a
paper result.

With the selected scheduler and architecture, train matched R/S/E controls for
1,000 updates each. Only after each 1,000-update training receipt validates, a
32-request generation at seed 1100 is an ineligible post-scale-up decode
diagnostic. It cannot authorize either earlier optimization screen or rank
candidates. A candidate is eligible for selection only after exactly
256 requests at 128 NFE for each reserved generation seed 1000 and 1001. If
$q_{c,s}$ and $d_{c,s}$ are released-branch quality and diversity for candidate
attempt $c$ and generation seed $s$, the frozen scores are
$\bar q_c=\tfrac12(q_{c,1000}+q_{c,1001})$ and
$\bar d_c=\tfrac12(d_{c,1000}+d_{c,1001})$. The ledger selects maximum
$\bar q_c$, then maximum $\bar d_c$, then the lexicographically smallest
attempt ID. For example, attempts A and B with scores $(0.86,0.82)$ and
$(0.86,0.825)$ select B; even a seed-1100 diagnostic with quality 1.0 is
disclosed but cannot enter this ordering.

Schema-2 pilot envelopes contain references, not trusted scores. For every
completed seed, the collector and final gate validate the schema-7 summary,
successful schema-5 training receipt, and raw CSV, then use a fresh CPU process
to re-decode `raw_model_text` and recompute QED, SA, and both diversity
branches. For a failed seed, the envelope instead binds the launcher's
schema-1 no-clobber failure receipt, exact command and source revision, log,
checkpoint/config identity, pilot mode, requested sample count, and any partial
artifacts. The same CPU-only writer publishes completed envelopes with
`--outcome completed` and failed envelopes with `--outcome failed`; envelopes
are never hand-authored. A failed sibling makes the attempt ineligible without
deleting a seed that completed. Every outcome must predate the candidate lock
and its producer revision must be an ancestor of the final benchmark revision.
Because `output/` is ignored, all referenced summaries, raw CSVs, receipts,
logs, and partials must be force-added and verified as Git-tracked before the
ledger is committed; the revision-time gate rejects any missing Git blob.
This authenticates every disclosed outcome but cannot independently establish
ledger completeness: without a host-wide append-only launch registry, the
absence of an omitted or deleted pilot remains a cooperative operator/launcher
assumption that the final decision records explicitly.

Before final generation, one checkpoint and sampling configuration must be
frozen in a committed, pushed candidate lock using only training, the fixed
validation panel, and registered pilot seeds. The three final seeds are then
evaluated once, with 1,000 requests each, under the same repaired definitions
as the audited MDLM control.
The final gate independently repeats raw-text rescoring for all three candidate
seeds before it can publish a decision.

The candidate lock's terminal-E receipt is the later 1,000-update matched-panel
receipt, not the 10-update health-terminal receipt. The latter authorizes only
screen preparation and is explicitly ineligible for the candidate lock; the
later receipt must share the locked candidate's matched-panel digest.

**Baseline recomputation evidence.** The immutable rescore attestation does not
regenerate molecules. Three fresh CPU interpreters re-decode and re-score the
1,000 historical raw rows for each MDLM seed, compare all 21 CSV fields, and
reproduce both released and strict aggregates: $3\times1000\times21=63{,}000$
matching cells. It binds the old row/summary hashes, current benchmark/report
schemas, pinned SA input, loaded SAFE/RDKit module bytes, and the clean pushed
source revision. Offline environment variables and four Python TCP/name-
resolution APIs were guarded during computation; this is explicitly not
OS-level or process-level network isolation. The code below reads the frozen
protocol, baseline manifest, and attestation from their exact retained bytes
and checks the cross-links before displaying any criterion.

The exact point-estimate gates use fractions, not rounded display labels:

- repaired validity must be at least $1.0$ (therefore all 3,000 requests);
- repaired uniqueness must be at least $0.9986666666666667$;
- repaired quality must exceed $0.858$; and
- repaired diversity must be at least
  $0.8230213192558725-0.005=0.8180213192558725$.

Uncertainty is metric-specific. Validity is a request-level binary outcome, so
its pooled UDLM-minus-MDLM lower bound uses the one-sided 95% Newcombe--Wilson
hybrid score interval; this remains non-degenerate when both observed rates are
100%. Uniqueness and quality are nonlinear per-run set functionals because the
released metric deduplicates first, and diversity is a per-run pairwise
functional. For those three metrics, the primary lower bound is the one-sided
95% Welch interval over the three independent seed-level UDLM estimates versus
the three MDLM estimates. No row bootstrap may re-deduplicate resampled rows:
that would manufacture duplicate molecules and invalidate uniqueness. The
quality lower bound must exceed 0; validity, uniqueness, and diversity lower
bounds must exceed -0.005. Seed means and sample SDs remain the paper-compatible
summaries. With only three runs per method, the Welch normality assumption and
low power are explicit limitations.

Report strict unrepaired diagnostics, checkpoint and source hashes, training
and generation seeds, initialization, added optimizer updates and requested
example exposure, parameter counts, checkpoint-selection rule, immutable
launch-manifest path/hash/schema, exact UUID and final-idle telemetry bindings,
global-lease evidence, 128-NFE UDLM inference cost, wall time, and exact
denominators. The receipt does not claim a content-token exposure count. The
exact 10-update health panel granted only permission to prepare the screens.
Passing the v4 two-seed, 256-request eligible stage may select one checkpoint
and sampling configuration for a decision, deterministic ledger, and lock; it
permits no extra candidate training or final-seed tuning. Neither result is
itself a superiority claim. If the
locked candidate is an MDLM-EMA warm start
with extra UDLM training, passing the gate supports only an operational
continuation-system claim on molecular metrics, not a from-scratch causal claim
for the diffusion method and not a speed win. A method-only claim requires a
from-scratch UDLM comparison or an equal-extra-update MDLM continuation control.

**Difference from released code.** The full-vocabulary artifact is the faithful
UDLM prior control. Excluding UNK/CLS/SEP/PAD/MASK is a GenMol-specific ablation;
the committed semantic pilot ledger, immutable MDLM rescore, launch-evidence
chain, global lease, machine-enforced R/S/E predecessor protocol, and intersection-union
publication gate are local reproducibility controls, not features of released
GenMol or UDLM. The per-layer FiLM/AdaLN-style plumbing is a local BERT
hypothesis: official UDLM uses richer per-block modulation in a DiT. The frozen
W=1 screen selected E-A1 on its registered denoising gates, but it produced no
molecular benchmark and is not a paper-scale or superiority result. Neither CPU
smoke result is paper-scale.

**Comprehension checkpoint.** Why does falling toy loss not show that UDLM beats
GenMol? Expected reasoning: it checks optimization mechanics on 16 memorized
molecules, not the matched de-novo distribution. Why retain strict diagnostics
when the headline comparison is repaired? Expected reasoning: repair can hide
invalid raw SAFE generations, so both views are needed. Why may the post-scale-up
32-request diagnostic not win even if its observed quality is highest? Expected
reasoning: its variance and operating point differ from the frozen selector, so
allowing it would reintroduce post-hoc selection. What does 63,000 matching row fields show?
Expected reasoning: current decoding/scoring exactly reproduces the frozen raw
MDLM evidence within the declared numeric tolerance; it does not repair the
historical training or GPU-provenance limitations. Why must the same manifest
hash and UUID tuple appear through runtime, summary, and receipt? Expected
reasoning: a launch-time idle snapshot is useful only if the process and
completed artifacts are cryptographically joined to exactly that launch. Why
is the 10-update R/S/E panel health-only? Expected reasoning: at roughly
$1.2\times10^{-6}$ by update 10 under the 2,500-step warmup, meaningful learning
has barely started. Why are both a global lease and predecessor receipts needed?
Expected reasoning: the lease prevents concurrent reviewed jobs, while each
successor's immutable predecessor binding proves R-to-S-to-E order and successful
advancement. What does the health-terminal E receipt authorize? Expected
reasoning: config materialization and registry freezing only; it permits no
generation, ranking, superiority claim, or candidate lock, and the later
1,000-update panel needs its own terminal E receipt. Why is L1 an
optimizer-schedule bundle rather than a clean cosine
ablation? Expected reasoning: shortening warmup changes its first-100-update
cumulative learning-rate exposure by about 37.57 times, so early exposure and
curve shape cannot be separated. Why must both 500-update arms freshly reload
the same MDLM EMA and reseed after construction? Expected reasoning: otherwise
one arm could inherit extra scheduler-screen exposure, and A1's extra parameter
initialization would shift later stochastic training draws. Why are A1's FiLM
gradients nonzero at optimizer-gradient observation one while its timestep-MLP
gradients need not be nonzero until observation three? Expected reasoning:
nonzero $c$ enters zero projection weights directly, but the gradient returning
to $c$ is multiplied by those zero weights; optimizer update one has zero
learning rate, and update two is the first that makes them nonzero. Why is A1 retained only after its
fixed-panel, clean-accuracy, exact-initialization, and staged-gradient gates all
pass? Expected reasoning: stronger conditioning is useful only if the adapter
is an exact warm-start no-op, becomes active as designed, learns across noise
levels, and does not damage clean predictions.


In [ ]:
import hashlib as stage20_hashlib
import json as stage20_json
import math as stage20_math
import pandas as stage20_pd

stage20_superiority_protocol_path = (
    PROJECT_ROOT
    / "experiments"
    / "udlm"
    / "protocols"
    / "de_novo_superiority_v4.json"
)
stage20_superiority_protocol_bytes = stage20_superiority_protocol_path.read_bytes()
assert stage20_hashlib.sha256(stage20_superiority_protocol_bytes).hexdigest() == (
    "9432360dad30a01de7ededf62db77470af9a0b8297fc78f06d330dbc73e826b7"
)
stage20_superiority_protocol = stage20_json.loads(stage20_superiority_protocol_bytes)
stage20_superiority_protocol_canonical_sha256 = stage20_hashlib.sha256(
    stage20_json.dumps(
        stage20_superiority_protocol,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")
).hexdigest()
assert stage20_superiority_protocol_canonical_sha256 == (
    "4edb0d193fcedc76913905220f3431fed8f0dd416f900c8da5f433a6071d4bfa"
)
assert stage20_superiority_protocol["schema_version"] == 4
assert stage20_superiority_protocol["protocol_id"] == (
    "genmol_udlm_de_novo_superiority_v4"
)
assert stage20_superiority_protocol["status"] == (
    "frozen_after_terminal_scale_up_validation_before_registered_candidate_generation"
)
stage20_protocol_amendment = stage20_superiority_protocol["amends"]
assert stage20_protocol_amendment["relative_path"] == (
    "experiments/udlm/protocols/de_novo_superiority_v3.json"
)
assert not stage20_protocol_amendment["scientific_settings_or_thresholds_changed"]
assert not stage20_protocol_amendment[
    "registered_candidate_generation_executed_before_amendment"
]
assert not stage20_protocol_amendment[
    "candidate_checkpoint_or_sampling_configuration_ranked_before_amendment"
]
assert not stage20_protocol_amendment[
    "candidate_final_evaluation_executed_before_amendment"
]

stage20_previous_protocol_path = stage20_superiority_protocol_path.with_name(
    "de_novo_superiority_v3.json"
)
stage20_previous_protocol_bytes = stage20_previous_protocol_path.read_bytes()
assert stage20_hashlib.sha256(stage20_previous_protocol_bytes).hexdigest() == (
    stage20_protocol_amendment["raw_sha256"]
)
stage20_previous_protocol = stage20_json.loads(stage20_previous_protocol_bytes)
assert stage20_hashlib.sha256(
    stage20_json.dumps(
        stage20_previous_protocol,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")
).hexdigest() == stage20_protocol_amendment["canonical_sha256"]
stage20_unchanged_scientific_fields = (
    "primary_claim",
    "baseline",
    "final_operating_point",
    "selection_firewall",
    "point_estimate_gates",
    "uncertainty_gates",
    "candidate_lock_requirements",
    "decision",
    "claim_boundaries",
)
assert stage20_protocol_amendment["scientific_decision_subtrees_unchanged"] == list(
    stage20_unchanged_scientific_fields
)
for stage20_unchanged_scientific_field in stage20_unchanged_scientific_fields:
    assert (
        stage20_superiority_protocol[stage20_unchanged_scientific_field]
        == stage20_previous_protocol[stage20_unchanged_scientific_field]
    )
stage20_candidate_lock_requirements = stage20_superiority_protocol[
    "candidate_lock_requirements"
]
assert stage20_candidate_lock_requirements["candidate_lock_schema_version"] == 2
assert stage20_candidate_lock_requirements["candidate_ledger_schema_version"] == 2
assert stage20_candidate_lock_requirements["pilot_evidence_schema_version"] == 2
assert stage20_candidate_lock_requirements[
    "pilot_failure_receipt_schema_version"
] == 1
assert stage20_candidate_lock_requirements[
    "accepted_training_artifact_schema_versions"
] == {
    "launch_manifest": 2,
    "runtime_config": 2,
    "training_summary": 5,
    "successful_exit_receipt": 5,
}
for stage20_required_launch_binding in (
    "immutable_launch_manifest_relative_path_raw_hash_and_schema_required",
    "exact_selected_gpu_uuids_and_final_idle_telemetry_bound_through_launch_runtime_summary_and_receipt_required",
    "global_single_training_job_lease_acquired_before_gpu_probe_and_validated_before_receipt_publication_required",
    "matched_r_s_e_registered_order_and_receipt_gated_advancement_machine_enforced",
    "predecessor_receipt_chain_is_machine_enforced_by_each_per_run_launch_manifest",
    "predecessor_artifacts_revalidated_unchanged_before_successful_exit_receipt",
    "predecessor_receipt_must_predate_successor_lock_and_gpu_probe",
    "terminal_e_successful_exit_receipt_hash_and_full_chain_required",
    "terminal_e_receipt_must_predate_candidate_lock",
    "selected_candidate_receipt_must_predate_candidate_lock",
    "selected_candidate_receipt_must_be_exact_member_of_terminal_r_s_e_chain",
    "terminal_e_and_selected_candidate_must_share_matched_panel",
    "candidate_ledger_winner_checkpoint_must_equal_locked_training_checkpoint",
    "per_seed_success_failure_outcomes_and_partial_failure_retention_required",
    "completed_pilot_summary_raw_receipt_hashes_required",
    "completed_pilot_training_receipt_full_validation_required",
    "pilot_quality_diversity_independently_recomputed_from_raw_model_text",
    "producer_authored_failure_receipt_command_source_log_and_partial_hashes_required",
    "pilot_failure_receipt_mode_and_requested_samples_required",
    "all_pilot_outcomes_must_predate_lock_and_source_revisions_be_ancestors",
    "selected_pilot_checkpoint_config_sampling_ema_source_metric_and_receipt_identity_must_equal_lock",
    "final_candidate_qed_sa_diversity_independently_recomputed_from_raw_model_text",
    "independent_rescore_source_and_dependency_hashes_required",
    "gate_report_rescore_launcher_writer_source_hashes_plus_scipy_version_required",
    "training_summary_checkpoint_audit_excludes_only_exact_live_bound_lightning_sentinel_required",
    "training_summary_checkpoint_exact_top_level_schema_and_nonfinite_python_numpy_rejection_required",
    "training_summary_checkpoint_hyperparameters_loop_progress_and_live_trainer_configuration_match_required",
    "training_summary_optimizer_scheduler_sampler_and_model_checkpoint_live_state_matches_required",
    "independent_optimization_screen_exact_closed_schema_and_cross_artifact_bindings_required",
    "failed_health_namespaces_are_immutable_and_never_eligible_for_candidate_selection",
):
    assert stage20_candidate_lock_requirements[stage20_required_launch_binding] is True
stage20_protocol_prior_floor = stage20_candidate_lock_requirements[
    "audited_empirical_prior_floor"
]
assert stage20_protocol_prior_floor["empirical_uniform_mix"] == 0.0002
assert stage20_protocol_prior_floor["selection_scope"] == (
    "retrospective_training_only_engineering_selection"
)
stage20_selection_firewall = stage20_superiority_protocol["selection_firewall"]
assert stage20_selection_firewall["eligible_pilot_generation_seeds"] == [1000, 1001]
assert stage20_selection_firewall["eligible_requested_samples_per_seed"] == 256
assert stage20_selection_firewall["eligible_nfe"] == 128
assert stage20_selection_firewall["eligible_metric_branch"] == "released_comparable"
stage20_terminal_scale_up = stage20_superiority_protocol[
    "terminal_scale_up_authority"
]
assert stage20_terminal_scale_up["status"] == "validated"
assert stage20_terminal_scale_up["source_revision"] == (
    "83c92963690aa0c41fa4d86dcc69fa0f692f656a"
)
assert stage20_terminal_scale_up["arm_order"] == ["R", "S", "E"]
assert stage20_terminal_scale_up["gpu_count_per_member"] == 1
assert stage20_terminal_scale_up["optimizer_updates_per_member"] == 1000
assert [member["candidate_id"] for member in stage20_terminal_scale_up["members"]] == [
    "r-w1-1000u-dcb271453411",
    "s-w1-1000u-dcb271453411",
    "e-w1-1000u-dcb271453411",
]
assert [
    member["checkpoint"]["sha256"] for member in stage20_terminal_scale_up["members"]
] == [
    "d0310d2e2402043ab9ab6a99268263581a35d60fb2262ddd988ea38cd6f6395c",
    "100f467b94766f2c87cc734398c8590bd14a1c8e0722ce31dbca7b446d986e72",
    "dce870e8d63453f73c428b9115f33556f67a21d45788d3112c2777ea82623005",
]

stage20_future_pilot_plan = {
    "status": "completed_terminal_scale_up_bound_by_protocol_v4",
    "execution_authority": {
        "exact_health_launcher_implemented": True,
        "exact_health_validator_implemented": True,
        "two_phase_registry_preparer_implemented": True,
        "registry_aware_launcher_implemented": True,
        "evidence_collector_implemented": True,
        "independent_selection_verifier_implemented": True,
        "failed_health_r_executed": True,
        "successful_health_panel_completed": True,
        "exact_arm_registry_frozen": True,
        "reviewed_launcher_authorized_screen_arms": True,
        "gpu_screens_completed": True,
        "scheduler_selection_committed": True,
        "conditioning_selection_committed": True,
        "screen_selections_alone_authorize_scale_up": False,
        "scale_up_registry_required": False,
        "terminal_scale_up_validated": True,
        "v4_generation_framework_pending_publication": True,
    },
    "registry_git_firewall": {
        "H": "34856c275049cd329320f6c01171f0d2d34cd814",
        "R0": "95bb3971354b4712135a30e6e58326a954917e1f",
        "R1": "2d33e565d19f585f75f0a1c1d849c4311ce9714d",
        "R2": "d32df6abe4b46589f4259b04e67cc4040f25aaa8",
        "R3": "b49e9006fe3d65f2a1e92f1f100c9adcfe596a58",
        "terminal_e_health_receipt_required_before_materialize": True,
        "same_terminal_e_health_receipt_revalidated_before_freeze": True,
        "H_to_R0_exact_selected_config_only_transition_required": True,
        "unselected_gpu_count_config_family_absent_in_H_and_R0": True,
        "resolved_config_count": 6,
        "effective_global_batch_size": 16,
        "micro_batch_size_per_process": 2,
        "gpu_count_materialized_before_R0": True,
    },
    "pre_registry_full_size_initialization_diagnostic": {
        "device": "cpu",
        "checkpoint_sha256": (
            "8d00aa47b02f64bf39ff6b0b2e786f213587366fc2c3d29712a00f3f84108dd6"
        ),
        "arms_constructed_sequentially": ["E-A0", "E-A1"],
        "logits_shape_each": [2, 4, 1880],
        "raw_float32_bytes_each": 60160,
        "raw_logits_sha256_each": (
            "3e6ef7368f9a11d061640948ac5955fba81c2acac6546a12adc4efc5e22e15b8"
        ),
        "byte_exact_equal": True,
        "wall_seconds": 33.55,
        "peak_rss_kib": 3578044,
        "registered_selection_evidence": False,
    },
    "health_panel": {
        "launcher": "scripts/udlm/launch_health_panel.py",
        "validator": "scripts/udlm/validate_health_panel.py",
        "launcher_choices": ["gpu_count_1_or_2", "cpu_only_dry_run"],
        "source_revision_symbol": "H",
        "source_revision_requirement": "full_clean_pushed_40_character_revision",
        "run_name_templates": [
            "health-w{W}-r-{H}",
            "health-w{W}-s-{H}",
            "health-w{W}-e-{H}",
        ],
        "terminal_receipt_template": (
            "output/udlm/health-w{W}-e-{H}/pilot_exit_status.json"
        ),
        "variant_order": ["R_release_uniform", "S_schedule_uniform", "E_empirical_frequency"],
        "launcher_variant_order": ["udlm", "schedule_uniform", "udlm_categorical"],
        "supported_world_sizes": [1, 2],
        "num_nodes": 1,
        "optimizer_updates_each": 10,
        "training_seed": 1,
        "loader_workers": 1,
        "micro_batch_size_per_process": 2,
        "gradient_accumulation_by_world_size": {1: 8, 2: 4},
        "effective_global_batch_size": 16,
        "vocabulary_size": 1880,
        "exclude_special_tokens": False,
        "scratch_mode": False,
        "empirical_uniform_mix": 0.0002,
        "warm_start": {
            "weights": "ema",
            "checkpoint_project_relative_path": (
                "outputs/paper_v1/checkpoints/50000.ckpt"
            ),
            "checkpoint_size_bytes": 1396998679,
            "checkpoint_sha256": (
                "8d00aa47b02f64bf39ff6b0b2e786f213587366fc2c3d29712a00f3f84108dd6"
            ),
            "byte_identity_verified_before_and_after_load": True,
        },
        "gpu_safety_policy": {
            "max_utilization_percent": 10,
            "utilization_comparison": "strictly_less_than",
            "min_free_memory_mib": 30000,
            "active_compute_processes_allowed": True,
            "compute_mode_prohibited_allowed": False,
        },
        "purpose": "health_and_provenance_only",
        "normalized_evidence_schema_version": 1,
        "successful_exit_receipt_schema_version": 5,
        "terminal_e_receipt_required_before": [
            "screen_config_materialization",
            "screen_registry_freeze",
        ],
        "eligibility": {
            "screen_authorization": True,
            "generation": False,
            "ranking": False,
            "superiority": False,
            "candidate_lock": False,
        },
        "distinct_from_later_1000_update_terminal_e_receipt": True,
        "single_job_concurrency_machine_enforced": True,
        "order_and_predecessor_receipt_gate_machine_enforced": True,
    },
    "scheduler_screen": {
        "variant": "E_only",
        "training_seed": 17,
        "optimizer_updates": 100,
        "fresh_verified_mdlm_ema_start_each_arm": True,
        "post_initialization_reseed_each_arm": False,
        "constructor_rng_path_is_identical_between_arms": True,
        "full_and_common_backbone_state_hashes_must_match": True,
        "state_audit_phase": "after_warm_start_before_reseed_and_optimizer",
        "fixed_noise_times": [0.1, 0.5, 0.9],
        "E_L0": {
            "conditioning": "additive",
            "schedule": "constant_with_linear_warmup",
            "warmup_updates": 2500,
        },
        "E_L1": {
            "interpretation": "optimizer_schedule_bundle_not_isolated_cosine_shape",
            "schedule": "half_cosine_with_linear_warmup_and_floor",
            "horizon_updates": 1000,
            "warmup_updates": 50,
            "peak_learning_rate": 3e-4,
            "minimum_learning_rate": 3e-6,
            "first_100_cumulative_lr_exposure_ratio_vs_E_L0": 37.571076382108664,
        },
        "select_E_L1_only_if": {
            "pooled_content_token_loss_relative_improvement_min": 0.02,
            "improved_time_bins_min": 2,
            "per_time_bin_relative_regression_max": 0.02,
            "comparison_arithmetic": "exact_sum_denominator_cross_products",
        },
        "complete_valid_threshold_miss": "E_L0",
        "missing_malformed_or_unmatched_evidence": "incomplete_no_winner",
    },
    "conditioning_screen": {
        "variant": "E_only",
        "training_seed": 17,
        "optimizer_updates": 500,
        "scheduler": "winner_of_scheduler_screen",
        "fresh_verified_mdlm_ema_start_each_arm": True,
        "scheduler_screen_checkpoint_continuation": False,
        "post_initialization_reseed_each_arm": True,
        "common_backbone_state_hash_must_match": True,
        "full_state_hash_may_differ_by_conditioning_topology": True,
        "summary_and_receipt_echo_exact_state_audit": True,
        "E_A0": {
            "topology": "zero_output_additive_time_conditioner",
            "legacy_state_keys_preserved": True,
        },
        "E_A1": {
            "topology": "post_mlp_outer_silu_plus_per_layer_post_bert_film",
            "timestep_mlp_output_initialization": "normal_nonzero",
            "per_layer_projection": "hidden_to_two_hidden_shift_then_scale",
            "per_layer_projection_initialization": "zero_weight_and_bias",
            "official_udlm_exact_architecture": False,
        },
        "select_E_A1_only_if": {
            "exact_warm_start_output_preserved_at_initialization": True,
            "pooled_content_token_loss_relative_improvement_min": 0.02,
            "per_time_bin_relative_regression_max": 0.02,
            "clean_token_accuracy_nondecreasing_by_integer_cross_product": True,
            "each_film_parameter_gradient_finite_nonzero_at_optimizer_observation_one": True,
            "timestep_mlp_gradient_finite_nonzero_after_first_nonzero_lr_film_update": True,
            "required_timestep_mlp_nonzero_gradient_optimizer_observation": 3,
        },
        "complete_valid_threshold_miss": "E_A0",
        "missing_malformed_or_unmatched_evidence": "incomplete_no_winner",
    },
    "matched_scale_up": {
        "variant_order": ["R_release_uniform", "S_schedule_uniform", "E_empirical_frequency"],
        "optimizer_updates_each": 1000,
        "post_training_decode_diagnostic": {
            "after_optimizer_updates_each": 1000,
            "seed": 1100,
            "requested": 32,
            "eligible": False,
            "can_authorize_optimization_screen": False,
            "can_rank_candidates": False,
        },
        "registered_selection_generation": {
            "seeds": [1000, 1001],
            "requested_per_seed": 256,
            "nfe": 128,
            "eligible": True,
        },
    },
}
stage20_screen_authority = stage20_future_pilot_plan["execution_authority"]
for stage20_implemented_authority in (
    "exact_health_launcher_implemented",
    "exact_health_validator_implemented",
    "two_phase_registry_preparer_implemented",
    "registry_aware_launcher_implemented",
    "evidence_collector_implemented",
    "independent_selection_verifier_implemented",
):
    assert stage20_screen_authority[stage20_implemented_authority] is True
assert stage20_screen_authority["exact_arm_registry_frozen"] is True
assert stage20_screen_authority["scheduler_selection_committed"] is True
assert stage20_screen_authority["conditioning_selection_committed"] is True
assert stage20_screen_authority["screen_selections_alone_authorize_scale_up"] is False
assert stage20_screen_authority["scale_up_registry_required"] is False
assert stage20_screen_authority["terminal_scale_up_validated"] is True
assert stage20_screen_authority["v4_generation_framework_pending_publication"] is True
assert stage20_future_pilot_plan["registry_git_firewall"]["resolved_config_count"] == 6
assert (
    stage20_future_pilot_plan["registry_git_firewall"][
        "effective_global_batch_size"
    ]
    == 16
)
stage20_health_contract = stage20_future_pilot_plan["health_panel"]
assert stage20_health_contract["launcher"] == (
    "scripts/udlm/launch_health_panel.py"
)
assert stage20_health_contract["validator"] == (
    "scripts/udlm/validate_health_panel.py"
)
assert stage20_health_contract["run_name_templates"] == [
    "health-w{W}-r-{H}",
    "health-w{W}-s-{H}",
    "health-w{W}-e-{H}",
]
assert stage20_health_contract["terminal_e_receipt_required_before"] == [
    "screen_config_materialization",
    "screen_registry_freeze",
]
assert stage20_health_contract["eligibility"] == {
    "screen_authorization": True,
    "generation": False,
    "ranking": False,
    "superiority": False,
    "candidate_lock": False,
}
for stage20_health_world_size in stage20_health_contract["supported_world_sizes"]:
    stage20_health_effective_batch = (
        stage20_health_world_size
        * stage20_health_contract["micro_batch_size_per_process"]
        * stage20_health_contract["gradient_accumulation_by_world_size"][
            stage20_health_world_size
        ]
    )
    assert stage20_health_effective_batch == (
        stage20_health_contract["effective_global_batch_size"]
    )
stage20_post_training_decode = stage20_future_pilot_plan["matched_scale_up"][
    "post_training_decode_diagnostic"
]
assert stage20_post_training_decode["after_optimizer_updates_each"] == 1000
assert stage20_post_training_decode["seed"] == 1100
assert stage20_post_training_decode["requested"] == 32
assert stage20_post_training_decode["eligible"] is False
assert stage20_post_training_decode["can_authorize_optimization_screen"] is False
assert stage20_post_training_decode["can_rank_candidates"] is False
stage20_initialization_diagnostic = stage20_future_pilot_plan[
    "pre_registry_full_size_initialization_diagnostic"
]
assert stage20_initialization_diagnostic["logits_shape_each"] == [2, 4, 1880]
assert stage20_initialization_diagnostic["raw_float32_bytes_each"] == (
    2 * 4 * 1880 * 4
)
assert stage20_initialization_diagnostic["byte_exact_equal"] is True
assert stage20_initialization_diagnostic["registered_selection_evidence"] is False
stage20_lr_at_health_update_10 = 3e-4 * 10 / 2500
assert stage20_math.isclose(stage20_lr_at_health_update_10, 1.2e-6)
stage20_screen_update_indices = range(100)
stage20_peak_lr = 3e-4
stage20_floor_lr = 3e-6


def stage20_l0_lr(optimizer_update_index):
    return stage20_peak_lr * optimizer_update_index / 2500


def stage20_l1_lr(optimizer_update_index):
    if optimizer_update_index < 50:
        return stage20_peak_lr * optimizer_update_index / 50
    progress = (optimizer_update_index - 50) / (1000 - 50)
    cosine = 0.5 * (1.0 + stage20_math.cos(stage20_math.pi * progress))
    return stage20_floor_lr + (stage20_peak_lr - stage20_floor_lr) * cosine


stage20_l0_cumulative_lr = stage20_math.fsum(
    stage20_l0_lr(index) for index in stage20_screen_update_indices
)
stage20_l1_cumulative_lr = stage20_math.fsum(
    stage20_l1_lr(index) for index in stage20_screen_update_indices
)
stage20_l1_to_l0_cumulative_lr_ratio = (
    stage20_l1_cumulative_lr / stage20_l0_cumulative_lr
)
assert stage20_math.isclose(stage20_l0_cumulative_lr, 5.94e-4, rel_tol=1e-15)
assert stage20_math.isclose(
    stage20_l1_cumulative_lr, 0.022317219370972547, rel_tol=1e-15
)
assert stage20_math.isclose(
    stage20_l1_to_l0_cumulative_lr_ratio,
    37.571076382108664,
    rel_tol=1e-15,
)

stage20_a1_example_shapes = {
    "batch": 2,
    "sequence": 4,
    "hidden": 24,
    "layers": 2,
    "conditioning": [2, 24],
    "projection_output": [2, 48],
    "shift": [2, 24],
    "scale": [2, 24],
    "broadcast_shift_and_scale": [2, 1, 24],
    "modulated_hidden": [2, 4, 24],
}
stage20_a1_hidden = 768
stage20_a1_layers = 12
stage20_a1_film_parameters_per_layer = (
    (2 * stage20_a1_hidden) * stage20_a1_hidden + 2 * stage20_a1_hidden
)
stage20_a1_film_parameters = (
    stage20_a1_layers * stage20_a1_film_parameters_per_layer
)
stage20_a1_time_mlp_parameters = 787_968
stage20_a1_conditioning_parameters = (
    stage20_a1_film_parameters + stage20_a1_time_mlp_parameters
)
assert stage20_a1_film_parameters_per_layer == 1_181_184
assert stage20_a1_film_parameters == 14_174_208
assert stage20_a1_conditioning_parameters == 14_962_176
stage20_a1_warm_start_tensor_counts = {
    "loaded_mdlm_ema_base": 202,
    "new_timestep": 4,
    "new_film": 24,
    "fresh_ema_shadows": 230,
}
assert (
    stage20_a1_warm_start_tensor_counts["loaded_mdlm_ema_base"]
    + stage20_a1_warm_start_tensor_counts["new_timestep"]
    + stage20_a1_warm_start_tensor_counts["new_film"]
    == stage20_a1_warm_start_tensor_counts["fresh_ema_shadows"]
)
stage20_scheduler_example_l0_sums = [1000, 600, 200]
stage20_scheduler_example_l1_sums = [970, 570, 202]
stage20_scheduler_example_denominators = [100, 100, 100]
stage20_scheduler_example_l0_total = sum(stage20_scheduler_example_l0_sums)
stage20_scheduler_example_l1_total = sum(stage20_scheduler_example_l1_sums)
stage20_scheduler_example_denominator = sum(
    stage20_scheduler_example_denominators
)
stage20_scheduler_example_pooled_improvement = 1.0 - (
    stage20_scheduler_example_l1_total / stage20_scheduler_example_l0_total
)
# Exact 2% pooled threshold: L1 <= 98/100 * L0.
assert (
    100
    * stage20_scheduler_example_l1_total
    * stage20_scheduler_example_denominator
    <= 98
    * stage20_scheduler_example_l0_total
    * stage20_scheduler_example_denominator
)
stage20_scheduler_example_strictly_better_bins = 0
for l0_sum, l1_sum, denominator in zip(
    stage20_scheduler_example_l0_sums,
    stage20_scheduler_example_l1_sums,
    stage20_scheduler_example_denominators,
    strict=True,
):
    if l1_sum * denominator < l0_sum * denominator:
        stage20_scheduler_example_strictly_better_bins += 1
    # Exact +2% per-bin ceiling: L1 <= 102/100 * L0.
    assert 100 * l1_sum * denominator <= 102 * l0_sum * denominator
assert stage20_scheduler_example_pooled_improvement >= 0.02
assert stage20_scheduler_example_strictly_better_bins >= 2

stage20_rescore_path = (
    PROJECT_ROOT
    / "experiments"
    / "udlm"
    / "baselines"
    / "mdlm_50000_rescore_attestation.json"
)
stage20_rescore_bytes = stage20_rescore_path.read_bytes()
stage20_rescore_sha256 = stage20_hashlib.sha256(stage20_rescore_bytes).hexdigest()
assert stage20_rescore_sha256 == (
    "6326b63c38c7052d0b47282d611618f77637496da2785779af69097fc1441323"
)
stage20_mdlm_rescore = stage20_json.loads(stage20_rescore_bytes)
assert stage20_mdlm_rescore["schema_version"] == 1
assert stage20_mdlm_rescore["status"] == "completed_exact_match"
assert stage20_mdlm_rescore["source"]["revision"] == (
    "74482c2742ab5ad15def122c809a6b4e403e94cf"
)
assert all(stage20_mdlm_rescore["source"]["clean_pushed_checks"].values())
assert stage20_mdlm_rescore["implementation"]["benchmark_schema_version"] == 7
assert stage20_mdlm_rescore["implementation"]["report_schema_version"] == 6
assert stage20_mdlm_rescore["protocol"]["network_controls"][
    "os_or_process_network_isolation"
] is False
stage20_rescore_seed_rows = stage20_mdlm_rescore["seed_results"]
assert [row["seed"] for row in stage20_rescore_seed_rows] == [0, 1, 2]
assert all(
    row["status"] == "exact_match"
    and row["row_comparison"]["all_match"] is True
    and row["row_comparison"]["row_count"] == 1000
    and row["row_comparison"]["field_count"] == 21
    and row["row_comparison"]["cell_count"] == 21000
    for row in stage20_rescore_seed_rows
)
assert sum(row["row_comparison"]["cell_count"] for row in stage20_rescore_seed_rows) == 63000

stage20_baseline_path = (
    PROJECT_ROOT / "experiments" / "udlm" / "baselines" / "mdlm_50000.json"
)
stage20_baseline_bytes = stage20_baseline_path.read_bytes()
stage20_baseline_manifest_sha256 = stage20_hashlib.sha256(
    stage20_baseline_bytes
).hexdigest()
assert (
    stage20_baseline_manifest_sha256
    == "6da46fc615dedbcca436da087a2c1e9145f5d110036e0c15bb431ded3c2e5539"
)
stage20_mdlm_baseline = stage20_json.loads(stage20_baseline_bytes)
assert stage20_mdlm_baseline["schema_version"] == 1
assert stage20_mdlm_baseline["purpose"] == (
    "frozen audited local MDLM comparator for UDLM; not an exact paper reproduction"
)
assert (
    isinstance(stage20_mdlm_baseline["historical_run_caveats"], list)
    and len(stage20_mdlm_baseline["historical_run_caveats"]) == 5
    and all(
        isinstance(caveat, str) and caveat
        for caveat in stage20_mdlm_baseline["historical_run_caveats"]
    )
)
assert stage20_mdlm_baseline["protocol"] == {
    "seeds": [0, 1, 2],
    "samples_per_seed": 1000,
    "total_requested_samples": 3000,
    "single_generation_batch_per_seed": True,
    "softmax_temperature": 0.5,
    "randomness": 0.5,
    "minimum_added_length": 40,
    "safe_version": "V1",
    "use_bracket_safe": False,
}
assert stage20_mdlm_baseline["checkpoint"]["sha256"] == (
    "8d00aa47b02f64bf39ff6b0b2e786f213587366fc2c3d29712a00f3f84108dd6"
)
assert stage20_mdlm_baseline["source_aggregate"]["sha256"] == (
    "b474efc593b665489359425dbe1ed0873f8ae1d44b77478b871aff6d6b555904"
)
stage20_registered_baseline = stage20_superiority_protocol["baseline"]
assert stage20_registered_baseline["manifest_sha256"] == (
    stage20_baseline_manifest_sha256
)
assert stage20_registered_baseline["rescore_attestation_sha256"] == (
    stage20_rescore_sha256
)
assert stage20_registered_baseline["rescore_source_revision"] == (
    stage20_mdlm_rescore["source"]["revision"]
)
assert stage20_mdlm_rescore["manifest_comparison"] == {
    "all_seed_rows_metrics_failures_hashes_and_aggregates_match": True,
    "manifest_sha256": stage20_baseline_manifest_sha256,
}
stage20_exact_mdlm_means = stage20_mdlm_baseline["released_comparable"]["mean"]
assert stage20_exact_mdlm_means == {
    "validity": 1.0,
    "uniqueness": 0.9986666666666667,
    "quality": 0.858,
    "diversity": 0.8230213192558725,
}
for stage20_metric_name, stage20_metric_value in stage20_exact_mdlm_means.items():
    assert stage20_mdlm_rescore["aggregate_metrics"]["released_comparable"][
        stage20_metric_name
    ]["mean"] == stage20_metric_value

stage20_expected_smoke_commit = "02595d1ecf994bbd432ec67ef21b0d56ed97918d"
assert stage20_expected_smoke_commit == UDLM_BASE_COMMIT
stage20_evidence_paths = (
    PROJECT_ROOT / "experiments" / "udlm" / "cpu_smoke" / "full_vocab_s100.json",
    PROJECT_ROOT / "experiments" / "udlm" / "cpu_smoke" / "no_special_s100.json",
)
stage20_expected_artifact_sha256 = {
    "full_vocab_s100.json": "2a9dd7078887a7c679fc4c673981b52791c0a03c8b46e7743b061209a61ec06a",
    "no_special_s100.json": "7fe113dd227d144ea46c222fb1c0a3b579d634a4490f4f9ed570ae2a5ee16df7",
}
stage20_cpu_rows = []
for evidence_path in stage20_evidence_paths:
    raw_bytes = evidence_path.read_bytes()
    artifact_sha256 = stage20_hashlib.sha256(raw_bytes).hexdigest()
    assert artifact_sha256 == stage20_expected_artifact_sha256[evidence_path.name]
    evidence = stage20_json.loads(raw_bytes)
    required_keys = {
        "device", "exclude_special_tokens", "fixed_diagnostics_before",
        "fixed_diagnostics_after", "generated_smiles", "git_sha",
        "loss_first_five_mean", "loss_last_five_mean", "purpose",
        "runtime_seconds", "sample_count_requested", "sampling_steps",
        "seed", "steps", "strict_valid_samples",
    }
    assert required_keys <= evidence.keys()
    assert evidence["git_sha"] == stage20_expected_smoke_commit
    assert evidence["purpose"] == "integration smoke test; not benchmark evidence"
    assert evidence["device"] == "cpu"
    assert evidence["seed"] == 1 and evidence["steps"] == 100
    assert evidence["sample_count_requested"] == 16
    assert evidence["sampling_steps"] == 32
    assert isinstance(evidence["exclude_special_tokens"], bool)
    assert isinstance(evidence["generated_smiles"], list)
    assert stage20_math.isfinite(evidence["runtime_seconds"])
    assert evidence["runtime_seconds"] > 0
    assert stage20_math.isfinite(evidence["loss_first_five_mean"])
    assert stage20_math.isfinite(evidence["loss_last_five_mean"])
    assert evidence["loss_first_five_mean"] >= 0
    assert evidence["loss_last_five_mean"] >= 0
    assert evidence["loss_last_five_mean"] < evidence["loss_first_five_mean"]
    assert stage20_math.isfinite(
        evidence["fixed_diagnostics_before"]["0.5"]["loss"]
    )
    assert stage20_math.isfinite(
        evidence["fixed_diagnostics_after"]["0.5"]["loss"]
    )
    assert (
        evidence["fixed_diagnostics_after"]["0.5"]["loss"]
        < evidence["fixed_diagnostics_before"]["0.5"]["loss"]
    )
    assert evidence["strict_valid_samples"] == len(evidence["generated_smiles"])
    assert 0 < evidence["strict_valid_samples"] <= evidence["sample_count_requested"]
    stage20_cpu_rows.append(
        {
            "artifact": str(evidence_path.relative_to(PROJECT_ROOT)),
            "sha256": artifact_sha256,
            "exclude_special_tokens": evidence["exclude_special_tokens"],
            "seed": evidence["seed"],
            "updates": evidence["steps"],
            "first_5_loss": evidence["loss_first_five_mean"],
            "last_5_loss": evidence["loss_last_five_mean"],
            "strict_valid": evidence["strict_valid_samples"],
            "requested": evidence["sample_count_requested"],
        }
    )

assert {row["exclude_special_tokens"] for row in stage20_cpu_rows} == {False, True}
stage20_cpu_evidence = stage20_pd.DataFrame(stage20_cpu_rows)
stage20_full_vocab_row = next(
    row for row in stage20_cpu_rows if not row["exclude_special_tokens"]
)
stage20_excluded_vocab_row = next(
    row for row in stage20_cpu_rows if row["exclude_special_tokens"]
)
stage20_prior_ablation_summary = (
    f"full vocabulary: {stage20_full_vocab_row['strict_valid']}/"
    f"{stage20_full_vocab_row['requested']} strict valid; control-token excluded: "
    f"{stage20_excluded_vocab_row['strict_valid']}/"
    f"{stage20_excluded_vocab_row['requested']}; artifact-byte hashes validated; "
    "generating runner revision, source cleanliness, config hashes, and command "
    "were not retained"
)
stage20_prior_ablation_sample_scope = (
    "2 configurations x 16 requested; same seed 1; paired engineering smoke, "
    "not independent replicates"
)
stage20_success_criteria = {
    "metric_family": "released-compatible repaired de-novo metrics",
    "baseline_manifest": {
        "path": str(stage20_baseline_path.relative_to(PROJECT_ROOT)),
        "sha256": stage20_baseline_manifest_sha256,
        "rescore_attestation_path": str(stage20_rescore_path.relative_to(PROJECT_ROOT)),
        "rescore_attestation_sha256": stage20_rescore_sha256,
        "rescore_source_revision": stage20_mdlm_rescore["source"]["revision"],
        "rescored_row_cells_all_match": 63000,
        "source_aggregate_sha256": stage20_mdlm_baseline["source_aggregate"]["sha256"],
        "checkpoint_sha256": stage20_mdlm_baseline["checkpoint"]["sha256"],
        "metric_definition_schema": stage20_mdlm_baseline["source_aggregate"]["schema_version"],
        "metric_runner_sha256": stage20_mdlm_baseline["source_aggregate"]["runner_sha256"],
    },
    "protocol": {
        "final_generation_seeds": [0, 1, 2],
        "requested_samples_per_seed": 1000,
        "final_seeds_evaluated_once_after_lock": True,
        "candidate_manifest_committed_and_pushed_before_final_seeds": True,
        "candidate_selected_without_final_seed_results": True,
        "eligible_pilot_generation_seeds": [1000, 1001],
        "eligible_pilot_samples_per_seed": 256,
        "eligible_pilot_nfe": 128,
        "eligible_pilot_metric_branch": "released_comparable",
        "selection_rule": stage20_selection_firewall["selection_rule"],
        "faithful_udlm_sampling_nfe": 128,
        "training_artifact_schemas": stage20_candidate_lock_requirements[
            "accepted_training_artifact_schema_versions"
        ],
        "launch_evidence_requirements": [
            "immutable launch_manifest.json repository-relative path, raw SHA-256, and schema",
            "exact ordered selected UUIDs and final idle telemetry cross-bound through runtime, summary, and receipt",
            "global single-job lease acquired before GPU probing and validated before receipt publication",
            "machine-enforced R/S/E genesis/predecessor chain with exact predecessor artifacts",
            "schema-2 candidate lock binds a terminal E receipt proving the full matched panel before lock",
            "selected receipt is an exact member of that chain and the ledger winner checkpoint equals the training lock",
            "schema-2 per-seed success/failure envelopes retain partial failures and bind producer receipts",
            "completed pilot and final candidate metrics are independently recomputed from raw_model_text",
        ],
        "required_matching": [
            "training data and tokenizer",
            "BERT width and depth",
            "generation length distribution",
            "released-compatible and strict metric definitions",
        ],
        "training_support_fairness_requirement": (
            "Run an MDLM-matched content-only framing control or retain the "
            "BOS/EOS training-support mismatch as a method-causality limitation."
        ),
        "required_candidate_provenance": [
            "initialization checkpoint and raw-or-EMA choice",
            "optimizer updates and requested example exposure (content-token exposure is not claimed)",
            "base and adapter parameter counts",
            "checkpoint-selection rule",
            "sampling temperature, NFE, seeds, source and checkpoint hashes",
            "summary/raw/receipt hashes plus launcher/rescore/evidence-writer source hashes",
            "launch-manifest path/hash/schema and exact selected UUID/final-idle telemetry chain",
            "global lease evidence, machine-enforced R/S/E predecessor chain, and wall time",
        ],
    },
    "local_mdlm_baseline_exact_fractions": stage20_exact_mdlm_means,
    "point_estimate_gate": {
        "validity_fraction_min": 1.0,
        "uniqueness_fraction_min": 0.9986666666666667,
        "quality_fraction_strictly_above": 0.858,
        "diversity_min": 0.8180213192558725,
    },
    "one_sided_95pct_lower_bound_gate": {
        "validity": {
            "method": "Newcombe-Wilson hybrid score difference on pooled request counts",
            "udlm_minus_mdlm_strictly_above": -0.005,
            "boundary_safe": True,
        },
        "uniqueness": {
            "method": "Welch interval over three independent seed-level estimates per method",
            "udlm_minus_mdlm_strictly_above": -0.005,
        },
        "quality": {
            "method": "Welch interval over three independent seed-level estimates per method",
            "udlm_minus_mdlm_strictly_above": 0.0,
        },
        "diversity": {
            "method": "Welch interval over three independent seed-level estimates per method",
            "udlm_minus_mdlm_strictly_above": -0.005,
        },
    },
    "forbidden_inference": "row bootstrap that re-deduplicates resampled molecule identities",
    "welch_small_sample_limitation_reported": True,
    "strict_metrics_reported_separately": True,
    "claim_scope": (
        "If MDLM-EMA warm-started with extra UDLM updates, passing supports a locked "
        "continuation-system molecular-metric claim only; it does not isolate diffusion "
        "method causality and does not claim an inference-speed win."
    ),
}
stage20_interval_gates = stage20_success_criteria[
    "one_sided_95pct_lower_bound_gate"
]
stage20_point_gate = stage20_success_criteria["point_estimate_gate"]
stage20_decision_gate_report_rows = [
    (
        "Comparator status",
        f"{stage20_mdlm_baseline['purpose']}; manifest SHA-256 "
        f"{stage20_baseline_manifest_sha256}; checkpoint SHA-256 "
        f"{stage20_mdlm_baseline['checkpoint']['sha256']}",
    ),
    (
        "Comparator exact means",
        "validity=1; uniqueness=0.9986666666666667; quality=0.858; "
        "diversity=0.8230213192558725",
    ),
    (
        "Comparator implementation provenance",
        f"source aggregate SHA-256 "
        f"{stage20_mdlm_baseline['source_aggregate']['sha256']}; metric runner "
        f"SHA-256 {stage20_mdlm_baseline['source_aggregate']['runner_sha256']}; "
        f"metric schema {stage20_mdlm_baseline['source_aggregate']['schema_version']}",
    ),
    (
        "Comparator current-code rescore",
        f"{stage20_success_criteria['baseline_manifest']['rescored_row_cells_all_match']}/"
        "63000 row fields matched; exact released and strict aggregates; historical "
        "molecules were not regenerated.",
    ),
    *[
        (f"Comparator caveat {index}", caveat)
        for index, caveat in enumerate(
            stage20_mdlm_baseline["historical_run_caveats"], start=1
        )
    ],
    (
        "Point-estimate gate",
        f"validity >= {stage20_point_gate['validity_fraction_min']}; "
        f"uniqueness >= {stage20_point_gate['uniqueness_fraction_min']}; "
        f"quality > {stage20_point_gate['quality_fraction_strictly_above']}; "
        f"diversity >= {stage20_point_gate['diversity_min']}",
    ),
    (
        "Validity uncertainty gate",
        f"{stage20_interval_gates['validity']['method']}; lower bound of "
        f"UDLM-MDLM > {stage20_interval_gates['validity']['udlm_minus_mdlm_strictly_above']}",
    ),
    *[
        (
            f"{metric.title()} uncertainty gate",
            f"{stage20_interval_gates[metric]['method']}; lower bound of "
            f"UDLM-MDLM > {stage20_interval_gates[metric]['udlm_minus_mdlm_strictly_above']}",
        )
        for metric in ("uniqueness", "quality", "diversity")
    ],
    (
        "Final-candidate lock",
        "Commit and push the candidate manifest before seeds 0,1,2; bind launch "
        "candidate-lock schema 2, terminal-E full-chain receipt, launch manifest schema 2, "
        "runtime schema 2, summary schema 5, receipt schema 5, exact UUIDs, "
        "final-idle telemetry, global-lease evidence, checkpoint, schema-2 pilot "
        "envelopes, and schema-1 failure receipts; evaluate and independently "
        "re-score each final seed once; do not select or tune from final-seed results.",
    ),
    (
        "Registered pilot selector",
        "Completed W=1 health and denoising screens selected E-L1 then E-A1; "
        "the corrected selection-bound R/S/E chain then completed 1000 updates "
        "per arm. V4 runs structural D at seed 1100 x 32, staged A/B/C at "
        "engineering seeds 1101/1102/1103 x 32/64/96, then eligible survivors "
        "at seeds 1000,1001 x 256 and 128 NFE. Ranking is released-compatible "
        "quality, diversity, config ID, then attempt ID using current-stage raw "
        "scores only. Schema-8 outcomes are independently rescored; failures "
        "remain disclosed and unrankable, with no retry or substitution.",
    ),
    (
        "Final evaluation protocol",
        "3 seeds x 1000 requests = exact denominator 3000; seeds 0,1,2; each "
        "evaluated once after lock; faithful UDLM sampling cost = 128 NFE.",
    ),
    (
        "Matching constraints",
        "; ".join(stage20_success_criteria["protocol"]["required_matching"]),
    ),
    (
        "Required candidate provenance",
        "; ".join(
            stage20_success_criteria["protocol"]["required_candidate_provenance"]
        ),
    ),
    (
        "Training-support fairness",
        stage20_success_criteria["protocol"][
            "training_support_fairness_requirement"
        ],
    ),
    (
        "Forbidden inference",
        stage20_success_criteria["forbidden_inference"],
    ),
    (
        "Small-sample limitation",
        "Welch intervals use only three independent seed-level estimates per method; "
        "normality is weakly checkable at n=3.",
    ),
    (
        "Strict metrics",
        "Report strict metrics separately from released-compatible repaired metrics.",
    ),
    ("Claim scope", stage20_success_criteria["claim_scope"]),
]
assert len(stage20_decision_gate_report_rows) == 24
assert stage20_decision_gate_report_rows[-1][0] == "Claim scope"
STAGE20_UDLM_SMOKE_TESTS_PASSED = True
stage20_summary = {
    "status": "bounded equation and integration evidence only",
    "official_udlm_revision": "edb0f8c28b7caeb4ea7a06a2fee8d74ab6da1661",
    "implementation_base_commit": stage20_expected_smoke_commit,
    "reference_checks": stage20_reference_checks,
    "time_checks": stage20_time_checks,
    "sampling_trace": stage20_sampling_trace,
    "cpu_artifact_count": len(stage20_cpu_evidence),
    "mdlm_baseline_manifest_sha256": stage20_baseline_manifest_sha256,
    "mdlm_rescore_attestation_sha256": stage20_rescore_sha256,
    "superiority_protocol_sha256": stage20_hashlib.sha256(
        stage20_superiority_protocol_bytes
    ).hexdigest(),
    "future_pilot_plan": stage20_future_pilot_plan,
    "screen_and_scale_up_status": {
        "status": stage20_future_pilot_plan["status"],
        "l1_interpretation": "optimizer_schedule_bundle_not_isolated_cosine_shape",
        "l1_to_l0_cumulative_lr_exposure_ratio_100_updates": (
            stage20_l1_to_l0_cumulative_lr_ratio
        ),
        "a1_example_shapes": stage20_a1_example_shapes,
        "a1_conditioning_parameters": stage20_a1_conditioning_parameters,
        "a1_warm_start_tensor_counts": stage20_a1_warm_start_tensor_counts,
        "fresh_mdlm_ema_start_each_500_update_arm": True,
        "post_initialization_reseed_each_500_update_arm": True,
        "exact_health_launcher_implemented": True,
        "exact_health_validator_implemented": True,
        "health_terminal_e_required_before_materialize_and_freeze": True,
        "health_evidence_screen_authorization_only": True,
        "two_phase_registry_preparer_implemented": True,
        "registry_aware_launcher_implemented": True,
        "evidence_collector_implemented": True,
        "independent_selection_verifier_implemented": True,
        "screen_registry_frozen": True,
        "scheduler_selection_committed": True,
        "conditioning_selection_committed": True,
        "screen_selections_alone_authorize_scale_up": False,
        "scale_up_registry_required": False,
        "terminal_scale_up_validated": True,
    },
    "training_artifact_schemas": stage20_candidate_lock_requirements[
        "accepted_training_artifact_schema_versions"
    ],
    "r_s_e_order_evidence": (
        "single-job concurrency and exact predecessor-receipt chaining are "
        "independently machine-enforced"
    ),
    "registered_selection_operating_point": {
        "seeds": [1000, 1001],
        "samples_per_seed": 256,
        "nfe": 128,
        "metric_branch": "released_comparable",
    },
    "paper_scale_superiority_claim": False,
}

REPRODUCTION_LEDGER = [
    row for row in REPRODUCTION_LEDGER if row["stage"] != "20 UDLM extension"
]
stage20_ledger_record = {
    "stage": "20 UDLM extension",
    "implemented": True,
    "smoke tested": True,
    "paper scale": False,
}
REPRODUCTION_LEDGER.append(stage20_ledger_record)

display(stage20_cpu_evidence)
print(stage20_summary)


## 20.5 A schedule-matched categorical prior experiment

**Paper correspondence.** UDLM derives its main result for the uniform
stationary distribution. The construction below generalizes the same
rank-one transition family to a strictly positive categorical stationary
distribution $\pi$. This arbitrary-$\pi$ process and the molecular frequency
prior are **our experimental extension**, not a result claimed by the UDLM or
GenMol papers. The faithful released control remains `release_uniform`.
`schedule_uniform` uses the generalized implementation with uniform $\pi$ and
the exact residual-clean schedule; it is the required control for
`empirical_frequency`, because those two variants differ only in $\pi$.

**Intuition and motivation.** Uniform refreshes spend probability on many SAFE
tokens that are rare in molecules. A training-frequency prior may create more
chemically plausible noisy states and make denoising easier. It may also copy
dataset imbalance, suppress useful rare chemistry, or reduce exploration. We
therefore treat it as a falsifiable hypothesis, not an automatic improvement.

**Symbols and mathematics.** Let $\mathcal A\subseteq\{1,\ldots,K\}$ be the
configured active diffusion alphabet and let $A=|\mathcal A|$. The primary
three-way comparison keeps all $K=1880$ tokenizer entries active so that
`release_uniform`, `schedule_uniform`, and `empirical_frequency` have identical
support. Removing the five tokenizer-control IDs gives $A=1875$ and is a
separate, explicitly labeled support ablation. Let
$\pi=(\pi_j)_{j\in\mathcal A}$ satisfy $\pi_j>0$ and
$\sum_{j\in\mathcal A}\pi_j=1$. For clean token $x$, candidate token $j$,
time $t\in[0,1]$, and
$\alpha_t=1-(1-\epsilon)t$ with residual clean mass
$\epsilon=10^{-3}$, the forward marginal is

$$q(z_t=j\mid x)=\alpha_t\mathbf 1_{j=x}+(1-\alpha_t)\pi_j.$$

For $0\le s<t\le1$, put $a_{t\mid s}=\alpha_t/\alpha_s$. If the observed
current token is $i$, the transition likelihood for an earlier candidate $j$
is

$$q(z_t=i\mid z_s=j)=a_{t\mid s}\mathbf1_{j=i}
 +(1-a_{t\mid s})\pi_i.$$

Notice that the refresh factor is $\pi_i$, the probability of the *observed*
token, not $\pi_j$. If $p_{\theta,j}$ is the raw plug-in/LOO probability,
define
$m_{\theta,s}(j)=\alpha_s p_{\theta,j}+(1-\alpha_s)\pi_j$. The exact reverse
posterior is

$$p_\theta(z_s=j\mid z_t=i)=
\frac{[a_{t\mid s}\mathbf1_{j=i}+(1-a_{t\mid s})\pi_i]
m_{\theta,s}(j)}
{\sum_{k\in\mathcal A}[a_{t\mid s}\mathbf1_{k=i}
+(1-a_{t\mid s})\pi_i]m_{\theta,s}(k)}.$$

For the continuous loss, define
$m_x(j)=\alpha_t\mathbf1_{j=x}+(1-\alpha_t)\pi_j$,
$\bar m_x(j)=m_x(j)/\pi_j$, and
$\bar m_\theta(j)=[\alpha_t p_{\theta,j}+(1-\alpha_t)\pi_j]/\pi_j$.
With $R_j=\bar m_x(j)/\bar m_x(i)$,
$S_j=\bar m_\theta(j)/\bar m_\theta(i)$,
$\phi(u)=e^u-1-u$, and the exact jump rate
$\beta(t)=-\alpha'_t/\alpha_t=(1-\epsilon)/\alpha_t$, our model-dependent
integrand is

$$\ell_\theta=\beta(t)\sum_{j\ne i}\pi_jR_j
\phi\!\left(\log S_j-\log R_j\right).$$

Because $\alpha_1=\epsilon>0$, the full finite-endpoint NELBO also contains
the parameter-independent term

$$D_{\rm KL}(q(z_1\mid x)\|\pi)
=\sum_{j\in\mathcal A}q(z_1=j\mid x)
\log\frac{q(z_1=j\mid x)}{\pi_j}.$$

The code exposes this term separately: omitting it does not change gradients,
but it must not be silently called zero in a reported full NELBO.

For frequency counts $c_j$ from $N=10{,}000$ fixed training examples, write
$f_j=c_j/\sum_{k\in\mathcal A}c_k$. The historical/manual configuration and
the immutable CPU smoke artifacts use

$$\pi_j=(1-\lambda)f_j+\lambda/A,\qquad \lambda=0.01.$$

Stage 20.7b later audits a smaller $\lambda=0.0002$ on disjoint, ordered
training blocks and applies it only to reviewed pilot launches. That
retrospective engineering choice does not rewrite the historical artifacts or
constitute molecular-quality evidence.

The uniform component gives every active category positive mass, including
categories unseen in the prefix and, in the primary full-support comparison,
the five zero-count control tokens. The dataset revision, ordered-text digest,
tokenizer revision, counts, and artifact bytes are pinned; the prefix is a
prior-design diagnostic, not an estimate with a benchmark claim.

**Concrete example.** For three categories with counts $(6,3,1)$ and
$\lambda=0.1$, $f=(0.6,0.3,0.1)$ and
$\pi\approx(0.5733,0.3033,0.1233)$. If $x$ is category B and
$\alpha_t=0.2$, then
$q(z_t\mid x)\approx(0.4587,0.4427,0.0987)$. At high noise, common category A
can be more likely than the clean token; the time-conditioned LOO predictor must
undo that structured corruption.

**Code below.** Notebook-native formulas cross-check production forward,
posterior, loss, and endpoint-KL values on a three-category example. Tensors
`x0` and `xt` have shape `(B,L)`, logits have shape `(B,L,A)`, and posterior
probabilities have shape `(B,L,A)`. A second part validates the exact pinned
10,000-row frequency artifact and constructs both the primary 1,880-entry
prior and the separate 1,875-entry control-token-excluded ablation without
network or GPU access.

**Difference from released implementations.** Official UDLM uses uniform
$\pi$ and an idealized loss schedule in its released compatibility path.
`schedule_uniform` and `empirical_frequency` instead share the exact
residual-clean rate above. NVIDIA GenMol has no categorical-prior selector.
Checkpoint and benchmark metadata bind the variant, active alphabet, schedule,
stationary-probability digest, and—only for the empirical treatment—the source
artifact and smoothing weight.

**Comprehension checkpoint.** Why is `release_uniform` not a clean control for
the empirical prior? Expected reasoning: both the prior and loss schedule
change, so their effects are confounded. Why must $\lambda$ be positive?
Expected reasoning: zero-count active tokens would otherwise have zero
stationary mass and make density ratios undefined. Why report endpoint KL
separately? Expected reasoning: it belongs to the full NELBO but is constant in
$\theta$, so it matters for objective accounting but not optimization
gradients.


In [ ]:
from genmol.diffusion import ContinuousCategoricalDiffusion


def reference_categorical_forward(clean_ids, alpha, stationary_probs):
    '''Return q(z_t | x) with shape (B,L,A).'''
    num_active = stationary_probs.numel()
    one_hot = F.one_hot(clean_ids, num_active).to(torch.float64)
    alpha = alpha.to(torch.float64)[:, None, None]
    return alpha * one_hot + (1.0 - alpha) * stationary_probs


def reference_categorical_posterior(
    clean_logits, noisy_ids, t, s, stationary_probs, noise_eps
):
    '''Direct Bayes-rule oracle for p_theta(z_s | z_t).'''
    clean_probs = clean_logits.to(torch.float64).softmax(-1)
    alpha_t = (1.0 - (1.0 - noise_eps) * t.to(torch.float64))[:, None, None]
    alpha_s = (1.0 - (1.0 - noise_eps) * s.to(torch.float64))[:, None, None]
    conditional_alpha = alpha_t / alpha_s
    observed_prior = stationary_probs[noisy_ids][..., None]
    likelihood = (1.0 - conditional_alpha) * observed_prior
    likelihood = likelihood.expand_as(clean_probs).clone()
    likelihood.scatter_add_(
        -1, noisy_ids[..., None], conditional_alpha.expand_as(observed_prior)
    )
    marginal_s = alpha_s * clean_probs + (1.0 - alpha_s) * stationary_probs
    unnormalized = likelihood * marginal_s
    return unnormalized / unnormalized.sum(-1, keepdim=True)


def reference_categorical_loss(
    clean_logits, clean_ids, noisy_ids, t, stationary_probs, noise_eps
):
    '''Direct density-ratio expression for the model-dependent CT integrand.'''
    clean_probs = clean_logits.to(torch.float64).softmax(-1)
    alpha = (1.0 - (1.0 - noise_eps) * t.to(torch.float64))[:, None, None]
    one_hot = F.one_hot(clean_ids, stationary_probs.numel()).to(torch.float64)
    marginal_x = alpha * one_hot + (1.0 - alpha) * stationary_probs
    marginal_theta = alpha * clean_probs + (1.0 - alpha) * stationary_probs
    density_x = marginal_x / stationary_probs
    density_theta = marginal_theta / stationary_probs
    gather_index = noisy_ids[..., None]
    ratio_x = density_x / torch.gather(density_x, -1, gather_index)
    ratio_theta = density_theta / torch.gather(
        density_theta, -1, gather_index
    )
    log_ratio_error = ratio_theta.log() - ratio_x.log()
    phi = torch.expm1(log_ratio_error) - log_ratio_error
    phi.scatter_(-1, gather_index, 0.0)
    beta = ((1.0 - noise_eps) / alpha).squeeze(-1)
    return beta * (stationary_probs * ratio_x * phi).sum(-1)


toy_pi = torch.tensor([0.55, 0.30, 0.15], dtype=torch.float64)
categorical_udlm = ContinuousCategoricalDiffusion(
    num_classes=3,
    stationary_probs=toy_pi,
    noise_eps=1e-3,
    antithetic_sampling=False,
)
categorical_x0 = torch.tensor([[1, 0, 2], [2, 1, 0]])       # (B=2,L=3)
categorical_xt = torch.tensor([[0, 2, 2], [1, 0, 2]])       # (B=2,L=3)
categorical_t = torch.tensor([0.75, 0.40], dtype=torch.float64)
categorical_s = torch.tensor([0.30, 0.10], dtype=torch.float64)
categorical_logits = torch.tensor(
    [
        [[0.2, 1.1, -0.4], [1.3, -0.2, 0.1], [-0.7, 0.3, 1.4]],
        [[-0.1, 0.2, 1.2], [0.4, 1.0, -0.5], [1.1, 0.1, -0.2]],
    ],
    dtype=torch.float64,
)                                                               # (B,L,A)

categorical_alpha_reference = (
    1.0 - (1.0 - 1e-3) * categorical_t
)
categorical_forward_reference = reference_categorical_forward(
    categorical_x0, categorical_alpha_reference, toy_pi
)
categorical_posterior_reference = reference_categorical_posterior(
    categorical_logits,
    categorical_xt,
    categorical_t,
    categorical_s,
    toy_pi,
    noise_eps=1e-3,
)
categorical_posterior_production = categorical_udlm.posterior_probs(
    categorical_logits, categorical_xt, categorical_t, categorical_s
)
assert categorical_forward_reference.shape == (2, 3, 3)
categorical_draw_count = 50_000
categorical_draw_clean = torch.zeros(
    (categorical_draw_count, 1), dtype=torch.long
)
categorical_draw_times = torch.full((categorical_draw_count,), 0.75)
categorical_draws = categorical_udlm.forward_process(
    categorical_draw_clean,
    categorical_draw_times,
    generator=torch.Generator().manual_seed(205),
)
categorical_forward_empirical = (
    torch.bincount(categorical_draws[:, 0], minlength=3).to(torch.float64)
    / categorical_draw_count
)
categorical_forward_expected = reference_categorical_forward(
    torch.zeros((1, 1), dtype=torch.long),
    torch.tensor([1.0 - (1.0 - 1e-3) * 0.75], dtype=torch.float64),
    toy_pi,
)[0, 0]
assert torch.allclose(
    categorical_forward_empirical,
    categorical_forward_expected,
    atol=0.006,
    rtol=0,
)
assert categorical_posterior_production.shape == (2, 3, 3)
assert torch.allclose(
    categorical_posterior_production,
    categorical_posterior_reference,
    atol=2e-12,
    rtol=2e-12,
)
assert torch.allclose(
    categorical_posterior_production.sum(-1),
    torch.ones(2, 3, dtype=torch.float64),
)

categorical_loss_reference = reference_categorical_loss(
    categorical_logits,
    categorical_x0,
    categorical_xt,
    categorical_t,
    toy_pi,
    noise_eps=1e-3,
)
categorical_loss_production = categorical_udlm.loss_per_token(
    categorical_logits, categorical_x0, categorical_xt, categorical_t
)
assert torch.allclose(
    categorical_loss_production,
    categorical_loss_reference,
    atol=2e-12,
    rtol=2e-12,
)
assert torch.all(categorical_loss_production >= 0)

endpoint_q = (
    1e-3 * F.one_hot(categorical_x0, 3).to(torch.float64)
    + (1.0 - 1e-3) * toy_pi
)
endpoint_kl_reference = (
    endpoint_q * (endpoint_q.log() - toy_pi.log())
).sum(-1)
endpoint_kl_production = categorical_udlm.endpoint_prior_kl(categorical_x0)
assert torch.allclose(
    endpoint_kl_production, endpoint_kl_reference, atol=2e-12, rtol=2e-12
)
assert torch.all(endpoint_kl_production > 0)

stage20_frequency_path = (
    PROJECT_ROOT
    / "experiments"
    / "udlm"
    / "token_frequency"
    / "train_first_10000.json"
)
stage20_frequency_bytes = stage20_frequency_path.read_bytes()
assert stage20_hashlib.sha256(stage20_frequency_bytes).hexdigest() == (
    "088c78e75611f3cc42c4011e1da6f65a377e673b9cba07a28b126b0fc62f06ed"
)
stage20_frequency = stage20_json.loads(stage20_frequency_bytes)
assert stage20_frequency["schema_version"] == 1
assert stage20_frequency["example_count"] == 10_000
assert stage20_frequency["content_token_count"] == 517_090
assert stage20_frequency["dataset"] == {
    "ordered_safe_text_sha256": (
        "53aee8e5592fc96159788e86519abbbcc9f1ab7c6348a1cb59a939bd57051d8f"
    ),
    "repo_id": "datamol-io/safe-gpt",
    "revision": "b83175cd7394e7a4027478a35b2f9d1dda3ac62f",
    "selection": "first 10000 streaming rows",
    "split": "train",
}
stage20_special_ids = tuple(
    stage20_frequency["tokenizer"]["special_token_ids"]
)
stage20_counts = torch.tensor(
    stage20_frequency["counts_by_token_id"], dtype=torch.float64
)
assert stage20_counts.shape == (1880,)
assert stage20_special_ids == (0, 1, 2, 3, 4)
assert torch.count_nonzero(stage20_counts[list(stage20_special_ids)]) == 0
stage20_uniform_mix = 0.01
stage20_primary_active_ids = torch.arange(stage20_counts.numel())
stage20_primary_counts = stage20_counts[stage20_primary_active_ids]
stage20_primary_empirical = stage20_primary_counts / stage20_primary_counts.sum()
stage20_primary_pi = (
    (1.0 - stage20_uniform_mix) * stage20_primary_empirical
    + stage20_uniform_mix / stage20_primary_active_ids.numel()
)
assert stage20_primary_pi.shape == (1880,)
assert torch.all(stage20_primary_pi > 0)
assert torch.isclose(
    stage20_primary_pi.sum(), torch.tensor(1.0, dtype=torch.float64)
)
assert not torch.allclose(
    stage20_primary_pi,
    torch.full_like(
        stage20_primary_pi, 1.0 / stage20_primary_active_ids.numel()
    ),
)

stage20_excluded_active_ids = torch.tensor(
    [
        token_id
        for token_id in range(stage20_counts.numel())
        if token_id not in stage20_special_ids
    ]
)
stage20_excluded_counts = stage20_counts[stage20_excluded_active_ids]
stage20_excluded_empirical = (
    stage20_excluded_counts / stage20_excluded_counts.sum()
)
stage20_excluded_pi = (
    (1.0 - stage20_uniform_mix) * stage20_excluded_empirical
    + stage20_uniform_mix / stage20_excluded_active_ids.numel()
)
assert stage20_excluded_pi.shape == (1875,)
assert torch.all(stage20_excluded_pi > 0)
assert torch.isclose(
    stage20_excluded_pi.sum(), torch.tensor(1.0, dtype=torch.float64)
)

stage20_categorical_checks = {
    "toy_forward_shape": tuple(categorical_forward_reference.shape),
    "forward_sampler_matches_analytical_frequencies": True,
    "toy_posterior_shape": tuple(categorical_posterior_production.shape),
    "posterior_matches_bayes_oracle": True,
    "loss_matches_density_ratio_oracle": True,
    "endpoint_kl_matches_direct_sum": True,
    "frequency_artifact_sha256": stage20_hashlib.sha256(
        stage20_frequency_bytes
    ).hexdigest(),
    "frequency_examples": stage20_frequency["example_count"],
    "frequency_content_tokens": int(stage20_primary_counts.sum()),
    "primary_active_vocab_size": stage20_primary_active_ids.numel(),
    "primary_zero_count_active_tokens": int(
        (stage20_primary_counts == 0).sum()
    ),
    "excluded_ablation_active_vocab_size": stage20_excluded_active_ids.numel(),
    "excluded_ablation_zero_count_active_tokens": int(
        (stage20_excluded_counts == 0).sum()
    ),
    "uniform_mixture_weight": stage20_uniform_mix,
    "primary_stationary_prior_min": float(stage20_primary_pi.min()),
    "primary_stationary_prior_max": float(stage20_primary_pi.max()),
}
print(stage20_categorical_checks)


## 20.6 Read the matched categorical CPU smoke panel

**Paper correspondence.** UDLM Sections 4.1–4.2 derive a uniform stationary
diffusion and its continuous-time objective; GenMol Section 4.1 instead uses
the absorbing MDLM process. The empirical categorical prior below is our
molecular hypothesis, not a result claimed in either paper. This stage asks
only whether all three implementations can take finite optimization steps and
execute a reverse chain under one exactly matched toy setup. It does not test
the GenMol paper's molecular benchmarks.

**Intuition and motivation.** Before spending GPU time, we want two clean
controls around the empirical-prior idea. A release-compatible control tells us
whether the official UDLM compatibility path runs. A schedule-matched uniform
control then separates repairing the schedule from changing the prior. Only
after that control is present can the empirical-frequency row be interpreted as
a prior intervention rather than a mixture of two interventions.

**Mathematics and controlled contrasts.** Let $V\in\{R,S,E\}$ name the
`release_uniform`, `schedule_uniform`, and `empirical_frequency` variants. Let
$K=1880$ be the full tokenizer vocabulary size, $t\in[0,1]$ diffusion time,
$\epsilon=10^{-3}$ the residual clean mass,
$\alpha_{\rm rel}(t)=1-(1-\epsilon)t$ the residual-clean schedule, and
$\alpha_{\rm ideal}(t)=1-t$ the ideal endpoint schedule. Let
$\pi_U=(1/K,\ldots,1/K)$ be the uniform stationary distribution. Let
$n_{\rm ex}=10{,}000$ denote the number of training examples used for the prior
artifact. For token index $j\in\{1,\ldots,K\}$ with content-token count $c_j$,
define $C_{\rm tok}=\sum_{j=1}^K c_j=517{,}090$ and empirical frequency
$f_j=c_j/C_{\rm tok}$. Finally, let $\lambda=0.01$ be the smoothing weight and
$\pi_{E,j}=(1-\lambda)f_j+\lambda/K$. Thus every entry of $\pi_E$ is
positive and $\sum_j\pi_{E,j}=1$.

**Terminology qualification for Section 20.5.** The clean token $x$, observed
token $i$, and candidate token $j$ all belong to the active alphabet
$\mathcal A$. The quantity $p_{\theta,j}$ means
$P_\theta(x=j\mid z_t^{(1:L)},t)$, conditioned on the whole noisy sequence and
time. The displayed Bayes-normalized expression is therefore a model reverse
kernel; it equals the true data posterior only when the clean predictor is
exact. A full active alphabet that includes five tokenizer-control IDs does not
make BOS, EOS, or PAD *positions* editable—the content mask still clamps and
excludes them. The endpoint KL is per selected token and independent of
$\theta$ because the stationary $\pi$ is fixed. Also, the Section 20.5 toy
probabilities were rounded component by component; unigram prior plausibility
alone does not establish grammatical or chemical sequence plausibility.

| Variant $V$ | Stationary prior | Forward/reverse schedule | Training integrand | Role |
|---|---|---|---|---|
| $R$: `release_uniform` | $\pi_U$ | $\alpha_{\rm rel}$ | released $\alpha_{\rm ideal}$ compatibility loss | release-compatible process/loss control in our GenMol-BERT runner |
| $S$: `schedule_uniform` | $\pi_U$ | $\alpha_{\rm rel}$ | exact residual-schedule rate | schedule-repair control |
| $E$: `empirical_frequency` | $\pi_E$ | $\alpha_{\rm rel}$ | exact residual-schedule rate | empirical-prior treatment |

For a future matched benchmark metric $M(V)$, where $M$ is a fully specified
molecular metric evaluated for variant $V$, the contrast
$\Delta_{\rm schedule}=M(S)-M(R)$ changes the objective schedule at fixed prior,
while $\Delta_{\rm prior}=M(E)-M(S)$ changes the prior at fixed schedule. The
direct contrast $M(E)-M(R)$ is confounded. The CPU panel below does **not**
estimate either $\Delta$: it has one seed, 20 toy updates, and no benchmark
metric.

**Small concrete example.** With categories (A, B, C), $K=3$, counts
$(6,3,1)$, and
$\lambda=0.1$, $\pi_U=(1/3,1/3,1/3)$ while
$\pi_E\approx(0.5733,0.3033,0.1233)$. At $t=1$ and
$\epsilon=10^{-3}$, a clean B token is therefore almost uniform under $S$ but
is refreshed mostly toward A under $E$. Comparing $E$ with $S$ isolates that
prior change; comparing $E$ directly with $R$ would also include the objective
schedule repair.

**Code and recorded invariants.** The next cell reads
`experiments/udlm/categorical_cpu_smoke/panel_seed1_steps20_n32_nfe16.json`,
whose SHA-256 is
`9c9cd3ce11157dbc5a053b03c28a3f87923a89bd660b41e3e8eac12b371b21e4`.
The panel binds clean, pushed source commit
`a9bb67c445da8cb3d4f7b6017c05f9b77896bf9b`; every embedded raw artifact and
each source blob is hash-checked. All rows use CPU, seed 1, 20 updates, 32
requested samples, 16 reverse steps, the full 1,880-token support, and the same
token-ID and attention-mask bytes. Here $B$ is minibatch size and $L$ is padded
token length. The recorded training IDs have shape `(B,L)=(16,19)`. Given
the configured vocabulary, the denoiser's categorical logits are expected to
have Stage 20 shape `(B,L,K)=(16,19,1880)`; the panel does not record a
logit-shape trace. The displayed evidence table has shape `(3,14)`.
Loading this cell performs no optimization, sampling, GPU use, or network
access.

All three rows recorded finite losses and gradient norms. Their first-five to
last-five mean losses were respectively $6.2195\to2.3224$,
$5.5463\to2.1544$, and $3.9215\to1.8306$ in $(R,S,E)$ order; no-repair
decoding returned 2/32, 4/32, and 5/32 strings. Here “no repair” disables the
sampler's `fix` path, but SAFE decoding still uses permissive error handling and
largest-component selection; the raw failed strings were not retained. These
counts are therefore not strict validity and cannot be independently re-audited.
The all-step finiteness booleans are producer-recorded summaries because the
panel does not retain complete loss or gradient traces. These numbers are
**engineering-only, non-ranking evidence**. In particular, loss magnitudes
across $R$ and the categorical variants use different integrands, and the
single-seed decode counts have no uncertainty estimate, chemical-quality
metric, or held-out evaluation. The common seed and clean-input hashes also do
not prove bitwise-identical initial weights or paired corruption draws: the
variant-specific diagnostic corruption digests differ and no weight/RNG trace
hash was recorded. Neither raw losses nor CPU runtimes may rank variants, and
the 16-step chain is an engineering shortcut rather than the faithful 128-NFE
UDLM evaluation setting.

**Difference from NVIDIA's released GenMol.** NVIDIA GenMol supplies the
absorbing MDLM path, not a time-conditioned revisable UDLM, a categorical-prior
selector, the residual-schedule repair, or this three-way provenance panel.
`release_uniform` preserves the official UDLM process/loss compatibility
behavior inside our GenMol-BERT adaptation; it is not an official-architecture
or benchmark reproduction. The other two rows are explicitly labeled controls/
extensions in our implementation. The requested smoothing value 0.01 is stored
in every run configuration but is inert for both uniform-prior rows.

### Comprehension checkpoint

1. Why compare `empirical_frequency` with `schedule_uniform`, not only with
   `release_uniform`? **Expected reasoning:** $E$ and $S$ share schedule and
   objective, so their intended difference is $\pi$; $E$ versus $R$ also changes
   the loss schedule.
2. What does a decreasing finite toy loss establish? **Expected reasoning:** the
   forward/loss/backward plumbing executes and can optimize this tiny batch; it
   does not establish generalization or molecular quality.
3. Does 5/32 versus 4/32 show that the empirical prior is better? **Expected
   reasoning:** no—one small stochastic seed, no confidence interval, and no
   benchmark metric cannot rank the methods.
4. Why retain both the panel hash and its source commit? **Expected reasoning:**
   the panel hash freezes the evidence bytes, while the commit identifies the
   exact pushed implementation whose tracked source blobs produced them.


In [ ]:
import base64 as stage20_base64

stage20_panel_path = (
    PROJECT_ROOT
    / "experiments"
    / "udlm"
    / "categorical_cpu_smoke"
    / "panel_seed1_steps20_n32_nfe16.json"
)
stage20_panel_bytes = stage20_panel_path.read_bytes()
stage20_panel_sha256 = stage20_hashlib.sha256(
    stage20_panel_bytes
).hexdigest()
assert stage20_panel_sha256 == (
    "9c9cd3ce11157dbc5a053b03c28a3f87923a89bd660b41e3e8eac12b371b21e4"
)
stage20_panel = stage20_json.loads(stage20_panel_bytes)
stage20_panel_source_commit = (
    "a9bb67c445da8cb3d4f7b6017c05f9b77896bf9b"
)
assert stage20_panel["schema_version"] == 1
assert stage20_panel["purpose"] == (
    "matched three-prior CPU integration panel"
)
assert stage20_panel["claim_scope"] == (
    "All rows are tiny same-seed optimization and reverse-chain checks. "
    "They cannot rank priors, estimate molecular quality, or support a "
    "UDLM-versus-GenMol superiority claim."
)
assert stage20_panel["git"] == {
    "commit": stage20_panel_source_commit,
    "dirty": False,
    "upstream": stage20_panel_source_commit,
}
stage20_panel_ancestry = subprocess.run(
    [
        "git", "merge-base", "--is-ancestor",
        stage20_panel_source_commit, actual_commit,
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=False,
)
assert stage20_panel_ancestry.returncode == 0, (
    stage20_panel_ancestry.stderr.strip()
    or "The panel source commit is not an ancestor of this notebook revision."
)

stage20_expected_matched_fields = {
    "attention_mask_sha256": (
        "6200c30399829918ab3428b4e680e40eddd179b41255a2f1ec220719d4794038"
    ),
    "batch_shape": [16, 19],
    "clean_input_ids_sha256": (
        "0913908bde0dacfe190496f8f181a7565fbf743289d275bab96afbb1ebf7dae5"
    ),
    "exclude_special_tokens": False,
    "sample_count_requested": 32,
    "sampling_steps": 16,
    "seed": 1,
    "steps": 20,
    "toy_smiles": [
        "CCO", "CCN", "CCS", "CCCl", "CCC", "CCCO",
        "CCCN", "CC(C)O", "CC(C)N", "CC(=O)O",
        "CCOC(=O)C", "c1ccccc1", "c1ccncc1",
        "CC1CCCCC1", "O=C(O)c1ccccc1", "CCOc1ccccc1",
    ],
}
assert stage20_panel["matched_fields"] == stage20_expected_matched_fields

stage20_variant_contracts = {
    "release_uniform": {
        "role": "faithful_release_control",
        "display_role": "release-compatible process/loss control",
        "schedule": "released_ideal_loss_residual_forward",
        "prior_source": "uniform",
        "source_path": (
            "output/udlm/categorical_cpu_smoke/release_uniform.json"
        ),
        "no_repair_decodable": 2,
    },
    "schedule_uniform": {
        "role": "schedule_repair_uniform_control",
        "display_role": "schedule-repair uniform control",
        "schedule": "schedule_consistent_residual_forward_and_loss",
        "prior_source": "uniform",
        "source_path": (
            "output/udlm/categorical_cpu_smoke/schedule_uniform.json"
        ),
        "no_repair_decodable": 4,
    },
    "empirical_frequency": {
        "role": "empirical_prior_treatment",
        "display_role": "empirical-prior treatment",
        "schedule": "schedule_consistent_residual_forward_and_loss",
        "prior_source": "pinned_frequency_artifact_uniform_mixture",
        "source_path": (
            "output/udlm/categorical_cpu_smoke/empirical_frequency.json"
        ),
        "no_repair_decodable": 5,
    },
}
assert set(stage20_panel["variants"]) == set(stage20_variant_contracts)

stage20_panel_rows = []
stage20_common_source_hashes = None
stage20_normalized_effective_configs = set()
for variant, contract in stage20_variant_contracts.items():
    entry = stage20_panel["variants"][variant]
    assert entry["source_path"] == contract["source_path"]
    embedded_bytes = stage20_base64.b64decode(
        entry["source_artifact_base64"], validate=True
    )
    assert stage20_hashlib.sha256(embedded_bytes).hexdigest() == (
        entry["source_artifact_sha256"]
    )
    result = stage20_json.loads(embedded_bytes)
    assert result == entry["result"]
    assert result["schema_version"] == 2
    for field, expected_value in stage20_expected_matched_fields.items():
        assert result[field] == expected_value
    assert result["purpose"] == (
        "bounded CPU integration smoke; not benchmark or superiority evidence"
    )
    assert result["claim_scope"] == (
        "Checks finite optimization, fixed-grid denoising diagnostics, and an "
        "executable reverse chain. Loss magnitudes across release_uniform and "
        "categorical variants are not directly comparable because their schedules "
        "and objectives differ. Sample validity is descriptive at this tiny size."
    )
    assert result["prior_variant"] == variant
    assert result["device"] == "cpu"
    assert result["seed"] == stage20_expected_matched_fields["seed"]
    assert result["steps"] == stage20_expected_matched_fields["steps"]
    assert result["sampling_steps"] == (
        stage20_expected_matched_fields["sampling_steps"]
    )
    assert result["sample_count_requested"] == (
        stage20_expected_matched_fields["sample_count_requested"]
    )
    assert result["batch_shape"] == (
        stage20_expected_matched_fields["batch_shape"]
    )
    assert result["clean_input_ids_sha256"] == (
        stage20_expected_matched_fields["clean_input_ids_sha256"]
    )
    assert result["attention_mask_sha256"] == (
        stage20_expected_matched_fields["attention_mask_sha256"]
    )
    assert result["exclude_special_tokens"] is False
    assert result["empirical_uniform_mix_requested"] == 0.01
    assert result["effective_config"]["model"]["vocab_size"] == 1880
    assert (
        result["effective_config"]["training"]["udlm"]["prior_variant"]
        == variant
    )
    normalized_config = stage20_json.loads(
        stage20_json.dumps(result["effective_config"])
    )
    normalized_config["training"]["udlm"]["prior_variant"] = "<controlled>"
    stage20_normalized_effective_configs.add(
        stage20_json.dumps(normalized_config, sort_keys=True)
    )
    assert result["git"]["commit"] == stage20_panel_source_commit
    assert result["git"]["upstream"] == stage20_panel_source_commit
    assert result["git"]["dirty"] is False
    assert result["git"]["status_porcelain"] == []
    assert result["all_losses_finite"] is True
    assert result["all_gradient_norms_finite"] is True
    assert stage20_math.isfinite(result["loss_first_five_mean"])
    assert stage20_math.isfinite(result["loss_last_five_mean"])
    assert result["loss_last_five_mean"] < result["loss_first_five_mean"]
    assert stage20_math.isfinite(result["runtime_seconds"])
    assert result["runtime_seconds"] > 0
    assert result["no_repair_decodable_samples"] == (
        contract["no_repair_decodable"]
    )
    generated_smiles = result["generated_smiles"]
    assert len(generated_smiles) == result["no_repair_decodable_samples"]
    assert len(generated_smiles) <= result["sample_count_requested"]
    assert all(
        isinstance(smiles, str) and smiles.strip()
        for smiles in generated_smiles
    )

    diagnostic_times = {"0.1", "0.5", "0.9"}
    diagnostics_before = result["fixed_diagnostics_before"]
    diagnostics_after = result["fixed_diagnostics_after"]
    assert set(diagnostics_before) == diagnostic_times
    assert set(diagnostics_after) == diagnostic_times
    for time_key in sorted(diagnostic_times):
        before = diagnostics_before[time_key]
        after = diagnostics_after[time_key]
        assert before["corrupted_token_ids_sha256"] == (
            after["corrupted_token_ids_sha256"]
        )
        for diagnostic in (before, after):
            assert diagnostic["content_token_count"] == 114
            assert 0 <= diagnostic["changed_content_tokens"] <= 114
            assert stage20_math.isfinite(diagnostic["loss"])
            assert diagnostic["loss"] >= 0
            assert stage20_math.isfinite(diagnostic["clean_token_accuracy"])
            assert 0 <= diagnostic["clean_token_accuracy"] <= 1
    assert diagnostics_after["0.5"]["loss"] < diagnostics_before["0.5"]["loss"]

    prior = result["prior_metadata"]
    assert prior["comparison_role"] == contract["role"]
    assert prior["schedule_variant"] == contract["schedule"]
    assert prior["prior_source"] == contract["prior_source"]
    assert prior["active_vocab_size"] == 1880
    assert prior["excluded_token_ids"] == []

    common_source_hashes = {
        name: result["source_inputs"][name]["sha256"]
        for name in ("diffusion", "model", "sampler", "smoke_runner")
    }
    if stage20_common_source_hashes is None:
        stage20_common_source_hashes = common_source_hashes
    else:
        assert common_source_hashes == stage20_common_source_hashes

    for source in result["source_inputs"].values():
        source_blob = subprocess.run(
            [
                "git", "show",
                f"{stage20_panel_source_commit}:{source['path']}",
            ],
            cwd=PROJECT_ROOT,
            capture_output=True,
            check=True,
        ).stdout
        assert len(source_blob) == source["size_bytes"]
        assert stage20_hashlib.sha256(source_blob).hexdigest() == source["sha256"]

    stage20_panel_rows.append(
        {
            "variant": variant,
            "role": contract["display_role"],
            "schedule": contract["schedule"],
            "prior_source": contract["prior_source"],
            "seed": result["seed"],
            "updates": result["steps"],
            "NFE": result["sampling_steps"],
            "requested": result["sample_count_requested"],
            "first_5_loss": result["loss_first_five_mean"],
            "last_5_loss": result["loss_last_five_mean"],
            "finite_losses": result["all_losses_finite"],
            "finite_gradients": result["all_gradient_norms_finite"],
            "no_repair_decodable": result["no_repair_decodable_samples"],
            "runtime_seconds": result["runtime_seconds"],
        }
    )

assert len(stage20_normalized_effective_configs) == 1
stage20_release_prior = stage20_panel["variants"]["release_uniform"][
    "result"
]["prior_metadata"]
stage20_schedule_prior = stage20_panel["variants"]["schedule_uniform"][
    "result"
]["prior_metadata"]
stage20_empirical_prior = stage20_panel["variants"]["empirical_frequency"][
    "result"
]["prior_metadata"]


def stage20_canonical_sequence_sha256(values):
    canonical = [
        value.hex() if isinstance(value, float) else value
        for value in values
    ]
    encoded = stage20_json.dumps(
        canonical, separators=(",", ":")
    ).encode("ascii")
    return stage20_hashlib.sha256(encoded).hexdigest()


stage20_active_ids_digest = stage20_canonical_sequence_sha256(
    [int(value) for value in stage20_primary_active_ids.tolist()]
)
stage20_empirical_canonical = torch.tensor(
    stage20_primary_pi.tolist(), dtype=torch.float64
)
stage20_empirical_canonical /= stage20_empirical_canonical.sum()
stage20_empirical_digest = stage20_canonical_sequence_sha256(
    [float(value) for value in stage20_empirical_canonical.tolist()]
)
stage20_uniform_canonical = torch.full(
    (stage20_primary_active_ids.numel(),),
    1.0 / stage20_primary_active_ids.numel(),
    dtype=torch.float64,
)
stage20_uniform_canonical /= stage20_uniform_canonical.sum()
stage20_uniform_digest = stage20_canonical_sequence_sha256(
    [float(value) for value in stage20_uniform_canonical.tolist()]
)
assert stage20_active_ids_digest == stage20_release_prior[
    "active_token_ids_sha256"
]
assert (
    stage20_release_prior["active_token_ids_sha256"]
    == stage20_schedule_prior["active_token_ids_sha256"]
    == stage20_empirical_prior["active_token_ids_sha256"]
)
assert stage20_uniform_digest == stage20_release_prior[
    "stationary_probs_sha256"
]
assert stage20_release_prior["stationary_probs_sha256"] == (
    stage20_schedule_prior["stationary_probs_sha256"]
)
assert stage20_empirical_digest == (
    stage20_empirical_prior["stationary_probs_sha256"]
)
stage20_categorical_single_factor_fields = (
    "schema_version",
    "process_family",
    "schedule_variant",
    "objective_scope",
    "active_token_ids_sha256",
    "active_vocab_size",
    "full_vocab_size",
    "excluded_token_ids",
    "noise_eps",
    "sampling_eps",
    "antithetic_sampling",
    "tokenizer_json_sha256",
    "tokenizer_repo_id",
    "tokenizer_revision",
)
for field in stage20_categorical_single_factor_fields:
    assert stage20_schedule_prior[field] == stage20_empirical_prior[field]
assert stage20_release_prior["stationary_probs_sha256"] == (
    stage20_schedule_prior["stationary_probs_sha256"]
)
assert stage20_empirical_prior["stationary_probs_sha256"] != (
    stage20_schedule_prior["stationary_probs_sha256"]
)
stage20_frequency_metadata_fields = tuple(
    field
    for field in stage20_release_prior
    if field.startswith("frequency_")
)
assert stage20_frequency_metadata_fields
for uniform_prior in (stage20_release_prior, stage20_schedule_prior):
    assert uniform_prior["uniform_mixture_weight"] is None
    assert all(
        uniform_prior[field] is None
        for field in stage20_frequency_metadata_fields
    )
assert stage20_empirical_prior["uniform_mixture_weight"] == 0.01
assert all(
    stage20_empirical_prior[field] is not None
    for field in stage20_frequency_metadata_fields
)
assert stage20_empirical_prior["frequency_artifact_sha256"] == (
    "088c78e75611f3cc42c4011e1da6f65a377e673b9cba07a28b126b0fc62f06ed"
)

stage20_categorical_cpu_evidence = stage20_pd.DataFrame(stage20_panel_rows)
assert stage20_categorical_cpu_evidence.shape == (3, 14)
stage20_summary["categorical_cpu_panel"] = {
    "path": str(stage20_panel_path.relative_to(PROJECT_ROOT)),
    "sha256": stage20_panel_sha256,
    "source_commit": stage20_panel_source_commit,
    "variant_count": len(stage20_panel_rows),
    "claim_scope": "engineering-only, non-ranking evidence",
}
stage20_summary["paper_scale_superiority_claim"] = False
display(stage20_categorical_cpu_evidence)
print(stage20_summary["categorical_cpu_panel"])


## 20.7 Audit the empirical stationary-prior geometry before GPU training

**Paper correspondence.** The UDLM paper derives its continuous-time process
for a *uniform* stationary distribution. GenMol instead uses MDLM's absorbing
mask state as its released reference process. Replacing UDLM's uniform law by
a SAFE-token frequency law is therefore our empirical-prior deviation, not a
claim from either paper. This CPU-only stage audits that deviation's unigram
geometry before the matched $R/S/E$ GPU health gate; it does not train or sample
a model.

**Intuition and motivation.** A concentrated molecular-token prior can put noisy
states near common SAFE syntax, which may ease denoising. A small uniform
component keeps training-unseen active tokens reachable, but increasing it also
spreads probability away from frequent tokens. Held-out unigram likelihood and
unseen-token mass expose that tradeoff cheaply. They do not measure sequence
grammar, molecular validity, chemical quality, or generator performance.

**Symbols and mathematics.** Let $\mathcal A$ be the active token-ID set and
$K=|\mathcal A|$ its size. For each $j\in\mathcal A$, let $c_j\geq0$ be its
count in the pinned training prefix and let $C=\sum_{k\in\mathcal A}c_k>0$.
For a uniform-mixture weight $w\in(0,1)$, the stationary probability is

$$\pi_{w,j}=(1-w)\frac{c_j}{C}+\frac{w}{K},\qquad j\in\mathcal A.$$

Thus $\pi_{w,j}>0$ and $\sum_{j\in\mathcal A}\pi_{w,j}=1$. For ordered
validation content-token IDs $y_1,\ldots,y_n\in\mathcal A$, where $n$ is the
number of validation content tokens, define the unigram negative log likelihood

$$\operatorname{NLL}(w)=-\frac1n\sum_{m=1}^{n}\log\pi_{w,y_m}.$$

Define stationary entropy $H(\pi_w)=-\sum_{j\in\mathcal A}
\pi_{w,j}\log\pi_{w,j}$ and effective vocabulary
$V_{\rm eff}(w)=\exp(H(\pi_w))$. Finally let
$\mathcal U=\{j\in\mathcal A:c_j=0\}$ be the training-unseen active IDs;
their stationary mass is $M_{\rm unseen}(w)=\sum_{j\in\mathcal U}
\pi_{w,j}$. All logarithms are natural, so NLL and entropy are in nats.

**Small concrete example.** Take $K=4$, counts $(6,3,1,0)$, $C=10$, and
$w=0.01$. Then
$\pi_w=(0.5965,0.2995,0.1015,0.0025)$. The fourth token is unseen, so
$M_{\rm unseen}=0.0025$. The entropy is about $0.91647$ nats and
$V_{\rm eff}\approx2.50044$. If a four-token validation example contains
each token once, its unigram NLL is about $2.50037$ nats. The rare fourth token
dominates that NLL even though the mixture makes its probability nonzero.

**Recorded result and its limit.** The configured process keeps all $K=1880$
tokenizer IDs active (`exclude_special_tokens=false`). Its 10,000-example
prefix contains 517,090 content tokens, 184 observed token types, and 1,696
unseen active types. At $w=0.01$, validation NLL is 2.759559 nats, perplexity
15.792879, entropy 2.860807 nats, effective vocabulary 17.475627, and
training-unseen mass 0.0090213. The fixed 256-example validation panel contains
13,627 content tokens but **zero tokens unseen in that training prefix**. It
therefore cannot test the proposed unseen-token-support benefit. The descriptive
grid minimum at $w=0.0001$ (NLL 2.750209) was found on this same panel: it is
exploratory and non-selective, and any prior choice would require a fresh
confirmatory panel. There is no quality, ranking, or UDLM-over-GenMol
superiority result here.

**Code, shapes, and invariants.** The next cell strictly reads the 14,582-byte
artifact, pins its SHA-256 and clean source commit, rejects duplicate/non-finite
JSON values, and verifies every tracked source blob recorded by the audit. The
training-count vector and independently reconstructed $\pi_w$ both have shape
`(K,)=(1880,)`; the grid table has shape `(10,6)`. The cell checks positivity,
unit mass, the configured stationary-probability digest, and the producer's live
process-versus-formula maximum difference $\leq2\times10^{-15}$. It only reads
CPU files and Git objects: no network, optimization, sampling, or GPU operation
occurs.

**Differences from released implementations.** Official UDLM uses uniform
$\pi$; NVIDIA's GenMol supplies an absorbing MDLM, neither this empirical
stationary prior nor this geometry auditor. Our `schedule_uniform` and
`empirical_frequency` variants keep the same rank-one process family and noise
schedule while varying only $\pi$, so only $E-S$ isolates the prior. The
proposed $w\in\{0.001,0.01,0.05\}$ follow-up is a hypothesis to test after the
matched $R/S/E$ health gate, not a parameter selected by this panel.

### Comprehension checkpoint

1. Why is $w>0$ useful? **Expected reasoning:** it gives every active token
   positive stationary mass, including zero-count tokens, so density ratios stay
   defined.
2. Why can the minimum validation NLL not select $w$? **Expected reasoning:** the
   same fixed panel was used to inspect the grid, and a fresh confirmatory panel
   is required after exploratory choice.
3. What does $V_{\rm eff}\ll K$ mean? **Expected reasoning:** probability mass is
   concentrated on far fewer tokens than a uniform $K$-category distribution.
4. Why does zero unseen validation tokens matter? **Expected reasoning:** this
   panel supplies no observation of whether uniform smoothing helps predict
   training-unseen token types.
5. Does lower unigram NLL imply better molecules? **Expected reasoning:** no; it
   ignores order, decoding, validity, chemistry, sampling uncertainty, and the
   learned denoiser.

### 20.7b Select a pilot-only floor from later training blocks

**Paper correspondence and deviation.** Official UDLM uses a uniform stationary
law; neither the UDLM paper nor NVIDIA GenMol proposes this empirical smoothing
choice. The historical/manual categorical configuration and the immutable CPU
artifacts above therefore remain at $w=0.01$. For reviewed pilot launches only,
we disclose a new training-data engineering choice rather than rewriting that
history.

**Intuition and motivation.** The 1% floor assigns about 0.00902 total mass to
the 1,696 token IDs absent from the first 10,000 rows, even though later training
blocks use such IDs only about 0.00015--0.00018 of the time. Too much floor mass
can waste corruption probability; too little makes genuinely new token types
extremely unlikely. Two disjoint later *training* blocks give a cheap scale check
without consuming validation molecules, final seeds, or generation metrics.

**Mathematics.** Keep the first-prefix counts $c_j$ and
$\pi_{w,j}=(1-w)c_j/C+w/K$ defined above. For block $b$, let $d_{b,j}$ be
the number of held-out content tokens of type $j$ and
$D_b=\sum_j d_{b,j}$. Its unigram loss is

$$L_b(w)=-\frac{1}{D_b}\sum_j d_{b,j}\log\pi_{w,j}.$$

The one-dimensional maximum-likelihood weight $w_b^*$ minimizes $L_b(w)$;
the audit finds it by deterministic bisection of the log-likelihood derivative.
Rows 10,001--20,000 give $D_1=512{,}587$, 78 tokens from 41 first-prefix-
unseen types, and $w_1^*=0.0001659380$. Rows 20,001--30,000 give
$D_2=513{,}326$, 91 tokens from 62 such types, and $w_2^*=0.0001933411$.
The rounded candidate $w=0.0002$ lowers NLL relative to $0.01$ by
0.00860567 and 0.00848514 nats/token, respectively.

**Small concrete example.** In the first later block, only
$78/512{,}587\approx1.52\times10^{-4}$ tokens come from types unseen in the
prefix. A floor near $2\times10^{-4}$ preserves full support on all 1,880 IDs
while matching that observed scale far more closely than a 1% mixture. This is
a unigram argument, not evidence that the denoiser will learn syntax or chemistry.

**Code, shapes, and invariants.** The appended code strictly reads the 73,953-byte
floor-selection artifact, pins raw SHA-256 and source commit, verifies each source
blob at that commit, and checks that the replayed first-prefix vector of shape
`(1880,)` exactly equals the original frozen counts. Each later block has a
10-row mixture grid, so the displayed table has shape `(20,5)`. It verifies the
two optima, NLL improvements, unseen counts, and selected value without network,
training, sampling, or GPU access.

**Scientific status and released-code difference.** The selection rule was
written after exploratory inspection of these same training blocks, so the second
block is a retrospective replication, not a preregistered confirmation. The
result justifies $w=0.0002$ as a disclosed pilot hyperparameter; it cannot rank
generators or establish a UDLM-over-GenMol win. Official UDLM still uses uniform
$\pi$, and released GenMol still uses absorbing MDLM. Only a later matched
$E-S$ comparison can estimate the empirical-prior effect.

**Comprehension checkpoint.** Why not call the second block confirmatory?
**Expected reasoning:** its data were inspected before this rule was frozen. Why
apply the chosen field to R and S launch configs too, even though they ignore it?
**Expected reasoning:** keeping the nuisance field identical preserves the matched
config contract while only E consumes it. Does the lower unigram NLL mean
UDLM beats MDLM? **Expected reasoning:** no; no model was trained or molecule
generated, and all registered quality/diversity gates remain ahead.


In [ ]:
import hashlib as stage207_hashlib
import json as stage207_json
import math as stage207_math
import subprocess as stage207_subprocess

import pandas as stage207_pd
import torch as stage207_torch


def stage207_unique_object(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError(f"duplicate JSON key: {key!r}")
        result[key] = value
    return result


def stage207_reject_constant(value):
    raise ValueError(f"non-finite JSON constant: {value}")


def stage207_assert_finite_json(value, path="$"):
    if isinstance(value, float):
        assert stage207_math.isfinite(value), f"non-finite number at {path}"
    elif isinstance(value, list):
        for index, item in enumerate(value):
            stage207_assert_finite_json(item, f"{path}[{index}]")
    elif isinstance(value, dict):
        for key, item in value.items():
            stage207_assert_finite_json(item, f"{path}.{key}")
    else:
        assert value is None or type(value) in (bool, int, str)


def stage207_load_strict_json(raw_bytes):
    value = stage207_json.loads(
        raw_bytes,
        object_pairs_hook=stage207_unique_object,
        parse_constant=stage207_reject_constant,
    )
    stage207_assert_finite_json(value)
    return value


def stage207_sequence_sha256(values):
    canonical = [
        value.hex() if isinstance(value, float) else value
        for value in values
    ]
    encoded = stage207_json.dumps(
        canonical, separators=(",", ":")
    ).encode("ascii")
    return stage207_hashlib.sha256(encoded).hexdigest()


stage20_prior_geometry_path = (
    PROJECT_ROOT
    / "experiments"
    / "udlm"
    / "prior_geometry"
    / "validation_grid.json"
)
stage20_prior_geometry_bytes = stage20_prior_geometry_path.read_bytes()
assert len(stage20_prior_geometry_bytes) == 14_582
stage20_prior_geometry_sha256 = stage207_hashlib.sha256(
    stage20_prior_geometry_bytes
).hexdigest()
assert stage20_prior_geometry_sha256 == (
    "b818e145cdde1a29532c64a351f1aff2eecf36b174828038c14902d9b515d560"
)
stage20_prior_geometry = stage207_load_strict_json(
    stage20_prior_geometry_bytes
)
assert set(stage20_prior_geometry) == {
    "claim_scope", "configured_process_contract", "created_at_utc",
    "definitions", "geometry", "git", "inputs", "purpose",
    "schema_version",
}
assert stage20_prior_geometry["schema_version"] == 1
assert stage20_prior_geometry["purpose"] == (
    "CPU-only empirical-UDLM stationary-prior geometry audit"
)
assert stage20_prior_geometry["claim_scope"] == (
    "This held-out unigram/concentration analysis performs no model training "
    "or molecular generation. It cannot rank generators, estimate chemical "
    "quality, or establish that UDLM beats GenMol."
)
stage20_prior_geometry_source_commit = (
    "6b312750bcc8861d8ff423f959e44764d121c3b1"
)
assert stage20_prior_geometry["git"] == {
    "commit": stage20_prior_geometry_source_commit,
    "dirty": False,
    "upstream": stage20_prior_geometry_source_commit,
}
stage20_prior_geometry_ancestry = stage207_subprocess.run(
    [
        "git", "merge-base", "--is-ancestor",
        stage20_prior_geometry_source_commit, actual_commit,
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=False,
)
assert stage20_prior_geometry_ancestry.returncode == 0, (
    stage20_prior_geometry_ancestry.stderr.strip()
    or "Prior-geometry source is not an ancestor of this notebook revision."
)

for source_path, source_record in stage20_prior_geometry["inputs"][
    "source_files"
].items():
    assert source_record["git_blob_verified"] is True
    source_bytes = stage207_subprocess.run(
        [
            "git", "show",
            f"{stage20_prior_geometry_source_commit}:{source_path}",
        ],
        cwd=PROJECT_ROOT,
        capture_output=True,
        check=True,
    ).stdout
    assert len(source_bytes) == source_record["size_bytes"]
    assert stage207_hashlib.sha256(source_bytes).hexdigest() == (
        source_record["sha256"]
    )

stage20_prior_contract = stage20_prior_geometry["configured_process_contract"]
assert stage20_prior_contract["exact_process_backend"] == (
    "genmol.diffusion.ContinuousCategoricalDiffusion"
)
assert stage20_prior_contract["active_token_count"] == 1880
assert stage20_prior_contract["configured_probability_sum"] == 1.0
assert stage20_prior_contract["resolved_relevant_config"] == {
    "model": {"vocab_size": 1880},
    "training": {
        "diffusion": "udlm",
        "sampling_eps": 0.001,
        "antithetic_sampling": True,
        "udlm": {
            "prior_variant": "empirical_frequency",
            "empirical_uniform_mix": 0.01,
            "exclude_special_tokens": False,
            "noise_eps": 0.001,
        },
    },
}
assert stage20_prior_contract["resolved_relevant_config_sha256"] == (
    "c030d4948365b1be7bc3b8e67a12c733dd3dfc6e32b1305ff92b567976a3708e"
)
stage20_prior_metadata = stage20_prior_contract["prior_metadata"]
assert stage20_prior_contract["prior_metadata_sha256"] == (
    "21cc62825086fa381e7918c6feb81ee6b4f8bcabd0857e26b3d32fc701540bbe"
)
assert stage20_prior_metadata == stage20_empirical_prior
assert stage20_prior_metadata["stationary_probs_sha256"] == (
    "51aa38acaf5cf4d5642c30dbdf14246e9540d4711917265cd1961e0df1902c97"
)
assert stage20_prior_metadata["active_vocab_size"] == 1880
assert stage20_prior_metadata["excluded_token_ids"] == []
assert stage20_prior_metadata["uniform_mixture_weight"] == 0.01

stage20_frequency_input = stage20_prior_geometry["inputs"][
    "frequency_artifact"
]
assert stage20_frequency_input == {
    "path": "experiments/udlm/token_frequency/train_first_10000.json",
    "sha256": (
        "088c78e75611f3cc42c4011e1da6f65a377e673b9cba07a28b126b0fc62f06ed"
    ),
}
stage20_frequency_bytes_207 = (
    PROJECT_ROOT / stage20_frequency_input["path"]
).read_bytes()
assert stage207_hashlib.sha256(stage20_frequency_bytes_207).hexdigest() == (
    stage20_frequency_input["sha256"]
)
stage20_frequency_207 = stage207_load_strict_json(stage20_frequency_bytes_207)
stage20_counts_207 = stage207_torch.tensor(
    stage20_frequency_207["counts_by_token_id"], dtype=stage207_torch.float64
)
stage20_active_ids_207 = stage207_torch.arange(1880, dtype=stage207_torch.long)
assert stage20_counts_207.shape == stage20_active_ids_207.shape == (1880,)
assert stage207_torch.all(stage20_counts_207 >= 0)
assert stage20_counts_207.sum().item() == 517_090
stage20_uniform_mix_207 = 0.01
stage20_empirical_207 = stage20_counts_207 / stage20_counts_207.sum()
stage20_pi_207 = (
    (1.0 - stage20_uniform_mix_207) * stage20_empirical_207
    + stage20_uniform_mix_207 / stage20_active_ids_207.numel()
)
stage20_pi_207 /= stage20_pi_207.sum()
assert stage20_pi_207.shape == (1880,)
assert stage207_torch.all(stage20_pi_207 > 0)
assert stage207_torch.equal(stage20_pi_207, stage20_primary_pi)
assert stage207_torch.isclose(
    stage20_pi_207.sum(), stage207_torch.tensor(1.0, dtype=stage207_torch.float64)
)
assert stage207_sequence_sha256(
    [float(value) for value in stage20_pi_207.tolist()]
) == stage20_prior_metadata["stationary_probs_sha256"]

stage20_geometry = stage20_prior_geometry["geometry"]
stage20_training_geometry = stage20_geometry["training"]
assert stage20_training_geometry == {
    "examples": 10_000,
    "full_vocabulary_size": 1880,
    "full_content_tokens": 517_090,
    "active_vocabulary_size": 1880,
    "active_content_tokens": 517_090,
    "active_observed_token_types": 184,
    "active_unobserved_token_types": 1696,
    "excluded_token_ids": [],
}
assert stage20_geometry["validation"] == {
    "examples": 256,
    "content_tokens": 13_627,
    "observed_token_types": 64,
    "tokens_unseen_in_training_prefix": 0,
}
stage20_configured_geometry = stage20_geometry["configured_0_01"]
stage20_configured_expected = {
    "uniform_mixture_weight": 0.01,
    "validation_content_token_nll_nats": 2.759559111037053,
    "validation_content_token_perplexity": 15.792878507273613,
    "stationary_entropy_nats": 2.860807184198026,
    "stationary_effective_vocabulary": 17.47562729522436,
    "training_unseen_token_mass": 0.009021276595744681,
    "maximum_token_probability": 0.18293198568667624,
    "minimum_token_probability": 5.319148936170213e-06,
    "top_10_token_mass": 0.790211384453836,
}
assert stage20_configured_geometry == stage20_configured_expected
stage20_process_agreement = stage20_geometry["configured_process_agreement"]
assert stage20_process_agreement["status"] == (
    "live_process_matches_audited_formula"
)
assert stage20_process_agreement["probability_count"] == 1880
assert stage20_process_agreement["probability_sum"] == 1.0
assert stage20_process_agreement["uniform_mixture_weight"] == 0.01
assert (
    stage20_process_agreement["maximum_absolute_formula_difference"]
    <= 2e-15
)

stage20_grid_rows = stage20_geometry["mixture_grid"]
assert [row["uniform_mixture_weight"] for row in stage20_grid_rows] == [
    0.0001, 0.001, 0.003, 0.01, 0.03, 0.05, 0.1, 0.2, 0.5, 0.75
]
stage20_prior_geometry_grid = stage207_pd.DataFrame(
    [
        {
            "w": row["uniform_mixture_weight"],
            "validation NLL": row["validation_content_token_nll_nats"],
            "perplexity": row["validation_content_token_perplexity"],
            "H(pi)": row["stationary_entropy_nats"],
            "effective vocab": row["stationary_effective_vocabulary"],
            "unseen mass": row["training_unseen_token_mass"],
        }
        for row in stage20_grid_rows
    ]
)
assert stage20_prior_geometry_grid.shape == (10, 6)
assert stage20_prior_geometry_grid.map(stage207_math.isfinite).all().all()
stage20_grid_minimum = stage20_geometry["grid_minimum_validation_nll"]
assert stage20_grid_minimum["uniform_mixture_weight"] == 0.0001
assert stage20_grid_minimum["validation_content_token_nll_nats"] == (
    2.7502090487521036
)
assert "not a generator selection rule" in stage20_grid_minimum[
    "qualification"
]
assert "no tokens unseen" in stage20_grid_minimum["qualification"]
assert stage20_geometry["proposed_followup_grid"][
    "uniform_mixture_weights"
] == [0.001, 0.01, 0.05]
assert stage20_geometry["proposed_followup_grid"]["status"] == (
    "hypothesis_to_test_after_the_matched_R_S_E_health_gate"
)

stage20_summary["prior_geometry"] = {
    "path": str(stage20_prior_geometry_path.relative_to(PROJECT_ROOT)),
    "sha256": stage20_prior_geometry_sha256,
    "source_commit": stage20_prior_geometry_source_commit,
    "configured_uniform_mixture_weight": 0.01,
    "validation_unseen_token_count": 0,
    "claim_scope": "CPU-only unigram geometry; non-ranking evidence",
}
stage20_summary["paper_scale_superiority_claim"] = False
display(stage20_prior_geometry_grid)
print(stage20_summary["prior_geometry"])

stage20_floor_audit_path = (
    PROJECT_ROOT
    / "experiments"
    / "udlm"
    / "prior_geometry"
    / "floor_selection_train_rows_10001_30000.json"
)
stage20_floor_audit_bytes = stage20_floor_audit_path.read_bytes()
assert len(stage20_floor_audit_bytes) == 73_953
stage20_floor_audit_sha256 = stage207_hashlib.sha256(
    stage20_floor_audit_bytes
).hexdigest()
assert stage20_floor_audit_sha256 == (
    "02908dafaf589ca9a49e560aa1eab470a18d6bfe616b781164784c489f54a9f1"
)
stage20_floor_audit = stage207_load_strict_json(stage20_floor_audit_bytes)
assert set(stage20_floor_audit) == {
    "block_analyses", "claim_scope", "created_at_utc", "data_use",
    "definitions", "git", "inputs", "purpose", "recommendation",
    "schema_version", "stream_checkpoints",
}
assert stage20_floor_audit["schema_version"] == 1
assert stage20_floor_audit["purpose"] == (
    "CPU-only empirical-UDLM uniform-floor training-data audit"
)
assert "does not train a denoiser" in stage20_floor_audit["claim_scope"]
stage20_floor_source_commit = "6424b323084358ea050ba22d7e13ef8d45962496"
assert stage20_floor_audit["git"] == {
    "commit": stage20_floor_source_commit,
    "dirty": False,
    "upstream": stage20_floor_source_commit,
}
stage20_floor_ancestry = stage207_subprocess.run(
    [
        "git", "merge-base", "--is-ancestor",
        stage20_floor_source_commit, actual_commit,
    ],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
    check=False,
)
assert stage20_floor_ancestry.returncode == 0, (
    stage20_floor_ancestry.stderr.strip()
    or "Prior-floor source is not an ancestor of this notebook revision."
)
for source_path, source_record in stage20_floor_audit["inputs"][
    "source_files"
].items():
    assert source_record["git_blob_verified"] is True
    source_bytes = stage207_subprocess.run(
        ["git", "show", f"{stage20_floor_source_commit}:{source_path}"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        check=True,
    ).stdout
    assert len(source_bytes) == source_record["size_bytes"]
    assert stage207_hashlib.sha256(source_bytes).hexdigest() == (
        source_record["sha256"]
    )

assert stage20_floor_audit["data_use"] == {
    "prior_estimation_rows": [1, 10_000],
    "exploratory_rows": [10_001, 20_000],
    "retrospective_replication_rows": [20_001, 30_000],
    "split": "training",
    "formal_preregistration_before_data_access": False,
    "final_generation_seeds_or_metrics_used": False,
}
stage20_floor_checkpoints = stage20_floor_audit["stream_checkpoints"]
assert stage20_floor_checkpoints["10000"]["counts_by_token_id"] == (
    stage20_frequency_207["counts_by_token_id"]
)
assert stage20_floor_checkpoints["10000"]["content_token_count"] == 517_090
assert stage20_floor_checkpoints["10000"][
    "ordered_safe_text_sha256"
] == stage20_frequency_207["dataset"]["ordered_safe_text_sha256"]
assert stage20_floor_checkpoints["20000"]["content_token_count"] == 1_029_677
assert stage20_floor_checkpoints["30000"]["content_token_count"] == 1_543_003
assert stage20_floor_checkpoints["30000"]["observed_token_types"] == 273

stage20_floor_blocks = stage20_floor_audit["block_analyses"]
stage20_floor_discovery = stage20_floor_blocks[
    "rows_10001_20000_exploratory"
]
stage20_floor_replication = stage20_floor_blocks[
    "rows_20001_30000_retrospective_replication"
]
assert stage20_floor_discovery["heldout_content_tokens"] == 512_587
assert stage20_floor_replication["heldout_content_tokens"] == 513_326
assert stage20_floor_discovery["heldout_tokens_from_training_unseen_types"] == 78
assert stage20_floor_replication["heldout_tokens_from_training_unseen_types"] == 91
assert stage20_floor_discovery[
    "continuous_maximum_likelihood_uniform_mixture_weight"
] == 0.00016593802382907556
assert stage20_floor_replication[
    "continuous_maximum_likelihood_uniform_mixture_weight"
] == 0.00019334112119092408
stage20_floor_grid = stage207_pd.DataFrame(
    [
        {"block": block_name, **row}
        for block_name, block in stage20_floor_blocks.items()
        for row in block["mixture_grid"]
    ]
)
assert stage20_floor_grid.shape == (20, 5)
assert all(
    stage207_math.isfinite(value)
    for block in stage20_floor_blocks.values()
    for row in block["mixture_grid"]
    for value in row.values()
)
stage20_floor_recommendation = stage20_floor_audit["recommendation"]
assert stage20_floor_recommendation["status"] == (
    "training_only_retrospective_engineering_recommendation"
)
assert stage20_floor_recommendation["recommended_uniform_mixture_weight"] == 0.0002
assert stage20_floor_recommendation[
    "candidate_nll_strictly_better_than_current_on_both_blocks"
] is True
assert stage20_floor_recommendation[
    "both_block_optima_within_0_0001_to_0_0003"
] is True
assert stage20_floor_recommendation["candidate_minus_current_nll_by_block"] == [
    -0.00860566722454914,
    -0.008485138280406979,
]
assert "not confirmatory" in stage20_floor_recommendation["qualification"]

stage20_summary["prior_floor_training_only"] = {
    "path": str(stage20_floor_audit_path.relative_to(PROJECT_ROOT)),
    "sha256": stage20_floor_audit_sha256,
    "source_commit": stage20_floor_source_commit,
    "historical_manual_weight": 0.01,
    "reviewed_pilot_weight": 0.0002,
    "claim_scope": "retrospective training-only prior fit; non-ranking",
}
stage20_summary["paper_scale_superiority_claim"] = False
display(stage20_floor_grid)
print(stage20_summary["prior_floor_training_only"])


## 20.8 Partition the pilot's hosted stream across DDP ranks

**Paper and released-code correspondence.** The GenMol and UDLM diffusion
equations act on examples; they do not specify how a hosted stream is assigned
to distributed workers. Correct assignment is therefore an engineering
precondition for the paper's nominal global batch, not a new diffusion method.
NVIDIA's released GenMol loader opens the SAFE Hugging Face dataset with
streaming enabled, while an iterable dataset cannot receive Lightning's usual
map-style DistributedSampler. Revision a4120fd adds an explicit, pilot-only
partition after Lightning has resolved each process's rank.

**Intuition and motivation.** If every rank replays the same stream, two ranks
can train on duplicate row identities while reporting twice the useful batch.
Modulo sharding makes the rank streams disjoint, so each synchronized update
combines different source rows. The pilot also validates one-node rank identity
and forces Lightning's self-spawning environment, preventing inherited scheduler
variables from silently changing who creates the DDP processes.

**Symbols and mathematics.** Let $D=(x_i)_{i\geq0}$ be the ordered hosted
stream, where $x_i$ is row $i$; let $W\in\mathbb N_{>0}$ be world size; and
let $r\in\{0,\ldots,W-1\}$ be a global rank. Rank $r$ receives
$D_r=(x_i:i\bmod W=r)$. For a finite prefix length
$N\in\mathbb N_{>0}$, define
$I_r^{(N)}=\{i\in\{0,\ldots,N-1\}:i\bmod W=r\}$. For distinct ranks
$r$ and $s$, $I_r^{(N)}\cap I_s^{(N)}=\varnothing$, while
$\bigcup_{r=0}^{W-1}I_r^{(N)}=\{0,\ldots,N-1\}$. Thus, for local batch
size $b\in\mathbb N_{>0}$,
the nominal $Wb$ row identities are distinct rather than only $b$ identities
repeated $W$ times.

**Concrete 12-row, two-rank example.** For rows $x_0,\ldots,x_{11}$ and
$W=2$, rank 0 receives IDs $(0,2,4,6,8,10)$ and rank 1 receives
$(1,3,5,7,9,11)$. Their intersection is empty and sorting their concatenation
reconstructs all 12 source IDs exactly once.

**Code and invariants.** The next cell imports the production
_shard_hosted_stream helper, builds a finite synthetic Hugging Face
IterableDataset, and asserts the exact even/odd split, disjointness, complete
reconstruction, and identity preservation when $W=1$. Each two-rank result is
a Python list of length 6; the reconstructed list has length 12. The cell uses
only a local generator and CPU iteration: it performs no download, network
request, CUDA query, GPU allocation, process spawn, training, or sampling.

**Difference from official code.** Neither the official UDLM release nor
NVIDIA's released SAFE loader supplies this reviewed pilot contract. Locally,
only the pilot resolves and passes $(r,W)$; single-rank and non-pilot/manual
calls preserve the original dataset object and released behavior. The pilot's
explicit LightningEnvironment is likewise local isolation; non-pilot training
retains Lightning's scheduler autodetection.

### Comprehension checkpoint

1. Why is a reported two-rank batch not necessarily twice as many useful rows
   without sharding? **Expected reasoning:** both ranks can replay the same row
   IDs, so synchronization averages duplicated evidence.
2. Why must the two index sets be both disjoint and reconstruct the source
   prefix? **Expected reasoning:** disjointness prevents duplication, while
   reconstruction detects dropped rows.
3. Why return the identical object for $W=1$? **Expected reasoning:** no split is
   needed, and identity preserves the released/manual single-rank path.
4. Why resolve $(r,W)$ after Trainer construction? **Expected reasoning:** the
   active Lightning process owns the authoritative rank; inherited environment
   text can be partial, stale, or scheduler-controlled.


In [ ]:
from datasets import IterableDataset as Stage208IterableDataset
from genmol.utils.utils_data import _shard_hosted_stream


def stage208_generate_rows():
    for row_id in range(12):
        yield {"row_id": row_id}


stage208_source = Stage208IterableDataset.from_generator(
    stage208_generate_rows
)
stage208_rank_rows = []
for stage208_rank in (0, 1):
    stage208_shard = _shard_hosted_stream(
        stage208_source,
        streaming_rank=stage208_rank,
        streaming_world_size=2,
    )
    stage208_rank_rows.append(
        [row["row_id"] for row in stage208_shard]
    )

stage208_rank0, stage208_rank1 = stage208_rank_rows
assert stage208_rank0 == list(range(0, 12, 2))
assert stage208_rank1 == list(range(1, 12, 2))
assert set(stage208_rank0).isdisjoint(stage208_rank1)
assert sorted(stage208_rank0 + stage208_rank1) == list(range(12))

stage208_single_rank_source = Stage208IterableDataset.from_generator(
    stage208_generate_rows
)
assert _shard_hosted_stream(
    stage208_single_rank_source,
    streaming_rank=0,
    streaming_world_size=1,
) is stage208_single_rank_source

stage208_stream_sharding = {
    "rank_0": stage208_rank0,
    "rank_1": stage208_rank1,
    "disjoint": True,
    "reconstructs_source": True,
    "single_rank_identity": True,
    "device": "cpu",
}
print(stage208_stream_sharding)


## 20.9 Freeze the selected bundle before a matched R/S/E scale-up

**Paper correspondence.** GenMol Section 4.1 uses an absorbing masked process,
whereas [UDLM Sections 4.1--4.2](https://arxiv.org/abs/2412.10193) derive the
uniform transition, reverse predictor in Eq. 15, and continuous-time objective
in Eqs. 18--19. [GenMol Section 4.1](https://arxiv.org/abs/2501.06158) supplies
the molecular BERT and SAFE setting. The completed screens below select local
engineering choices; neither paper claims that those choices improve GenMol.

**What the screens established.** The frozen one-GPU, seed-17 scheduler screen
compared 100-update E-L0 and E-L1 runs from byte-identical initialized states.
E-L1 reduced pooled fixed-panel production loss from
$68{,}513.0489201366/40{,}881$ to $44{,}204.2809926042/40{,}881$ (about
$35.48\%$), with all three time bins better. The subsequent two fresh
500-update warm starts compared additive E-A0 with FiLM E-A1 under L1. E-A1
reduced pooled loss from $114{,}483.5594098568/40{,}881$ to
$39{,}984.46089004135/40{,}881$ (about $65.07\%$), increased pooled clean-token
top-1 counts from $7{,}473$ to $24{,}382$, preserved the exact initial logits,
and passed the registered staged-gradient contract. These are denoising
results at different training lengths and must not be compared across screens.
No molecule was generated or scored, and final seeds were not used, so there
is still no molecular benchmark or superiority result.

**Causal three-arm design.** Let $a\in\{R,S,E\}$ identify a scale-up arm,
$x=i$ be a clean token, $z_t=j$ its category at noise time $t$, and
$\alpha_t^a\in[0,1]$ the clean-token coefficient for arm $a$. Let
$\mathbf1[j=i]$ be an indicator, $V=1880$ the tokenizer vocabulary size,
$u_j=1/V$ the uniform distribution, $f_j=c_j/\sum_k c_k$ the frequency of
token $j$ in the frozen 10,000-row training prefix, and $\lambda=0.0002$ the
training-only smoothing choice. Every arm follows

$$q_a(z_t=j\mid x=i)=\alpha_t^a\mathbf1[j=i]
 +(1-\alpha_t^a)\pi_j^a,$$

with $\pi^R=\pi^S=u$ and
$\pi_j^E=(1-\lambda)f_j+\lambda u_j$. Thus R versus S changes only the
registered release-compatible versus schedule-consistent process, isolating
the schedule repair. S versus E shares the repaired process and changes only
$\pi$, isolating the empirical-prior hypothesis. Comparing R directly with E
would confound both changes.

All three arms inherit the selected L1 learning-rate bundle. For optimizer
update $k$, peak $\eta_{\max}=3\times10^{-4}$, floor
$\eta_{\min}=3\times10^{-6}$, warmup $w=50$, and horizon $h=1000$,

$$\eta(k)=\begin{cases}
\eta_{\max}k/w,&0\le k<w,\\
\eta_{\min}+(\eta_{\max}-\eta_{\min})
\left[1+\cos\!\left(\pi(k-w)/(h-w)\right)\right]/2,&w\le k\le h.
\end{cases}$$

Each also inherits A1. With total noise $\sigma=-\log\alpha_t$, timestep MLP
$g$, conditioning vector $c=\operatorname{SiLU}(g(\sigma))\in\mathbb R^{B\times H}$,
ordinary BERT-layer output $y_l\in\mathbb R^{B\times L\times H}$, and one
zero-initialized projection $(W_l,b_l)$ for layer $l$,

$$[\beta_l,\gamma_l]=W_lc+b_l,\qquad
h_l=(1+\gamma_l[:,\mathrm{None},:])\odot y_l
 +\beta_l[:,\mathrm{None},:].$$

Here $B$ is batch size, $L$ sequence length, $H$ hidden width,
$\beta_l,\gamma_l\in\mathbb R^{B\times H}$ are shift and residual-scale
vectors, $h_l$ is the modulated hidden state, and $\odot$ is elementwise
multiplication. Zero $W_l,b_l$ make A1 an exact MDLM-logit identity before
training.

The `E-` prefix records where L1 and A1 were selected; it does not restrict
them to the empirical-prior arm at scale-up. Applying the same E-selected
bundle to R, S, and E preserves a matched causal contrast, but the result is
conditional on E-tuned optimization and conditioning. It is not a comparison
of three separately optimized methods. Matching BERT width and depth also does
not match parameter count: A1 adds 14,962,176 conditioning parameters, including
14,174,208 in the FiLM projections, so any later advantage must disclose this
capacity difference.

**Concrete example and intuition.** For a four-token toy alphabet with counts
$(6,3,1,0)$, $f=(0.6,0.3,0.1,0)$. Using a deliberately visible toy smoothing
$\lambda=0.2$ gives $\pi^E=(0.53,0.29,0.13,0.05)$. If the clean token is the
second category and $\alpha_t=0.4$, then
$q_E(z_t\mid x)=(0.318,0.574,0.078,0.030)$. The uniform S arm instead gives
$(0.15,0.55,0.15,0.15)$. R versus S asks whether correcting time/process
consistency helps at fixed uniform mass; S versus E asks whether moving refresh
mass toward observed SAFE tokens helps at that same corrected schedule.

**Code and tensor invariants.** The next cell opens only the four small,
Git-tracked screen envelopes, verifies their exact bytes, and recomputes their
pooled comparisons. It never opens the untracked checkpoints referenced by the
envelopes. During training, token IDs are integer `[B,L]`, $t$ and $\sigma$ are
`[B]`, logits are `[B,L,1880]`, the compact positive stationary prior is
float64 `[A]` and sums to one, reverse probabilities are finite `[B,L,A]` and
sum to one, and the Boolean content mask is `[B,L]` with supplied/special tokens
clamped. A1 uses `c`, `beta`, and `gamma` shaped `[B,H]`, broadcasts the latter
two as `[B,1,H]`, and preserves hidden shape `[B,L,H]`. A registered scale-up
must keep seed, data exposure, effective batch, initialization checkpoint, L1,
and A1 fixed; only the declared R/S/E process fields may differ. Every
1,000-update arm must reload the same verified MDLM EMA independently rather
than continue a screen checkpoint. With per-process microbatch $m=2$, the
preparer uses accumulation $(8,4,3,2)$ for world sizes $(1,2,3,4)$, giving
effective batches $(16,16,18,16)$; one registry freezes one world size, so all
three arms within that comparison still match exactly. The first scale-up rung
used one GPU: the selected full-size E-A1 topology had already completed 500
updates at W=1, making 1,000 updates a controlled twofold increase without also
introducing an untested multi-process topology. Protocol v4 binds the validated
terminal R/S/E chain; it does not infer molecule quality from training loss.

**Difference from released implementations and execution boundary.** NVIDIA
GenMol uses absorbing MDLM with BERT and no explicit time input. Official UDLM
uses a uniform prior with a DiT/AdaLN-style bridge predictor (called a denoiser
in the 2024 paper). The schedule-consistent S
arm, empirical $\pi^E$, scaled L1 recipe, and warm-start-compatible BERT FiLM
A1 are local repairs or hypotheses. The selection-bound framework provides a
registry preparer, verifier, launcher, and validator; those programs alone do
not authorize a run. A launch additionally requires GPU-count-specific configs
and a registry that were separately reviewed, committed, and pushed. Eligible physical GPUs must each have
utilization strictly below 10%, at least 30,000 MiB free, and non-prohibited
compute mode. The scale-up tooling can represent world sizes through four for
historical completeness, but the active user authorization permits at most
two GPUs without another request; no physical ID is hard-coded.

**Comprehension checkpoint.** Why is S required between R and E? Expected
reasoning: it holds the stationary prior uniform while repairing the schedule,
so S-to-E changes only the prior. Why does the selected screen bundle not prove
a molecule-level win? Expected reasoning: selection used a fixed denoising
panel, no generated molecules, and no final seeds. Why must each scale-up arm
restart from MDLM EMA? Expected reasoning: continuing a selected screen
checkpoint would give different exposure and initialization and destroy the
matched causal comparison. If `beta` is `[B,H]`, why insert a singleton axis?
Expected reasoning: `[B,1,H]` broadcasts the same per-example modulation across
all $L$ token positions without changing `[B,L,H]`.


In [ ]:
import hashlib as stage209_hashlib
import json as stage209_json
from fractions import Fraction as Stage209Fraction


stage209_screen_paths = {
    "scheduler_evidence": (
        PROJECT_ROOT / "experiments/udlm/screens/scheduler_evidence.json"
    ),
    "scheduler_selection": (
        PROJECT_ROOT / "experiments/udlm/screens/scheduler_selection.json"
    ),
    "conditioning_evidence": (
        PROJECT_ROOT / "experiments/udlm/screens/conditioning_evidence.json"
    ),
    "conditioning_selection": (
        PROJECT_ROOT / "experiments/udlm/screens/conditioning_selection.json"
    ),
}
stage209_expected_raw_sha256 = {
    "scheduler_evidence": "76a4ec9771dd0e875bf4532beb217da0ed54426427d39d92dbb218c50673c705",
    "scheduler_selection": "e93d1face65bab573b32898da1a0a7a209a95a1a21fde58c509bcb3db8bac272",
    "conditioning_evidence": "e63010c52f97e788bb25ea8f46b4f5ba9f2ae1e95bca484ee139a871ae57d612",
    "conditioning_selection": "ea1473ccfbff5b55e6f0e27dea4ea1c6ecca20de1a69935857da782b929193d0",
}
stage209_screen_records = {}
for stage209_name, stage209_path in stage209_screen_paths.items():
    stage209_bytes = stage209_path.read_bytes()
    assert stage209_hashlib.sha256(stage209_bytes).hexdigest() == (
        stage209_expected_raw_sha256[stage209_name]
    )
    stage209_screen_records[stage209_name] = stage209_json.loads(stage209_bytes)

stage209_scheduler_evidence = stage209_screen_records["scheduler_evidence"]
stage209_scheduler = stage209_screen_records["scheduler_selection"]
stage209_conditioning_evidence = stage209_screen_records["conditioning_evidence"]
stage209_conditioning = stage209_screen_records["conditioning_selection"]

assert stage209_scheduler["status"] == "completed"
assert stage209_scheduler["run_source_revision"] == (
    "2d33e565d19f585f75f0a1c1d849c4311ce9714d"
)
assert stage209_scheduler["selected_arm_id"] == "E-L1"
assert stage209_scheduler["gates"]["all_required_gates_pass"] is True
assert stage209_scheduler["gates"]["strictly_better_bin_count"] == 3
assert stage209_conditioning["status"] == "completed"
assert stage209_conditioning["run_source_revision"] == (
    "d32df6abe4b46589f4259b04e67cc4040f25aaa8"
)
assert stage209_conditioning["selected_arm_id"] == "E-A1"
assert all(stage209_conditioning["gates"].values())

for stage209_record in (
    stage209_scheduler_evidence,
    stage209_conditioning_evidence,
):
    assert stage209_record["status"] == "closed_after_registered_attempts"
    assert stage209_record["generation_metrics_included"] is False
    assert stage209_record["final_generation_seeds_included"] == []
for stage209_record in (stage209_scheduler, stage209_conditioning):
    assert stage209_record["generation_metrics_used"] is False
    assert stage209_record["final_generation_seeds_used"] == []


def stage209_pooled_loss(selection, arm_id):
    metrics = selection["metrics"][arm_id]
    return (
        Stage209Fraction(metrics["production_loss_sum_fraction"]),
        metrics["denominator_tokens"],
    )


stage209_l0_sum, stage209_scheduler_tokens = stage209_pooled_loss(
    stage209_scheduler, "E-L0"
)
stage209_l1_sum, stage209_scheduler_candidate_tokens = stage209_pooled_loss(
    stage209_scheduler, "E-L1"
)
assert stage209_scheduler_tokens == stage209_scheduler_candidate_tokens == 40_881
stage209_scheduler_reduction = 1 - stage209_l1_sum / stage209_l0_sum
assert stage209_scheduler_reduction > Stage209Fraction(35, 100)

stage209_a0_sum, stage209_conditioning_tokens = stage209_pooled_loss(
    stage209_conditioning, "E-A0"
)
stage209_a1_sum, stage209_conditioning_candidate_tokens = stage209_pooled_loss(
    stage209_conditioning, "E-A1"
)
assert stage209_conditioning_tokens == stage209_conditioning_candidate_tokens == 40_881
stage209_conditioning_reduction = 1 - stage209_a1_sum / stage209_a0_sum
assert stage209_conditioning_reduction > Stage209Fraction(65, 100)
assert stage209_conditioning["metrics"]["E-A1"]["clean_token_top1_correct"] == 24_382
assert stage209_conditioning["metrics"]["E-A0"]["clean_token_top1_correct"] == 7_473

stage209_init = stage209_conditioning_evidence["initialization_audit"]
assert stage209_init["exact_equal"] is True
assert stage209_init["logits_shape"] == [2, 4, 1880]
assert stage209_init["reference_logits"]["sha256"] == (
    "3e6ef7368f9a11d061640948ac5955fba81c2acac6546a12adc4efc5e22e15b8"
)
assert stage209_init["candidate_logits"]["sha256"] == (
    stage209_init["reference_logits"]["sha256"]
)
stage209_attempts = {
    attempt["arm_id"]: attempt
    for attempt in stage209_conditioning_evidence["attempts"]
}
stage209_gradient = stage209_attempts["E-A1"]["conditioning_gradient_audit"]
assert stage209_gradient["status"] == "completed"
assert stage209_gradient["optimizer_checks"][0]["film_groups"][0][
    "all_parameter_gradients_nonzero"
] is True
assert stage209_gradient["optimizer_checks"][2]["timestep_mlp_groups"][0][
    "all_parameter_gradients_nonzero"
] is True

stage209_selection_bundle = {
    "scheduler": "E-L1",
    "conditioning": "E-A1",
    "scheduler_pooled_loss_reduction": float(stage209_scheduler_reduction),
    "conditioning_pooled_loss_reduction": float(stage209_conditioning_reduction),
    "initial_logits_shape": stage209_init["logits_shape"],
    "initial_logits_sha256": stage209_init["reference_logits"]["sha256"],
    "generation_metrics_used": False,
    "final_generation_seeds_used": [],
    "molecular_superiority_claim": False,
}
stage209_future_scale_up_contract = {
    "variant_order": ["R_release_uniform", "S_schedule_uniform", "E_empirical_frequency"],
    "optimizer_updates_each": 1000,
    "fresh_verified_mdlm_ema_start_each": True,
    "reseed_after_model_initialization_each": True,
    "selected_scheduler": "E-L1",
    "selected_conditioning": "E-A1",
    "supported_gpu_counts": [1, 2, 3, 4],
    "maximum_user_authorized_gpu_count_without_additional_permission": 2,
    "completed_first_registered_gpu_count": 1,
    "terminal_panel_bound_by_protocol_v4": True,
    "gpu_utilization_must_be_strictly_below_percent": 10,
    "micro_batch_size_per_process": 2,
    "gradient_accumulation_by_gpu_count": {1: 8, 2: 4, 3: 3, 4: 2},
    "effective_global_batch_size_by_gpu_count": {1: 16, 2: 16, 3: 18, 4: 16},
    "screen_selection_artifacts_alone_authorize_scale_up": False,
    "valid_frozen_selection_bound_scale_up_registry_required": True,
    "configuration_and_registry_publication_are_separate_git_revisions": True,
}
print(stage209_selection_bundle)
print(stage209_future_scale_up_contract)


> Historical v4 teaching: D terminated on 2026-09-07 with a launcher configuration-identity failure. Its artifacts and protocol remain immutable; these examples do not resume it. Stage 21 describes the separate engineering-v5 experiment. Current GPU authorization is at most two cards.

## 20.10 Raw-LOO temperature and stable nucleus sampling

**Paper correspondence.** UDLM's reverse transition is the categorical bridge
in Eq. 15 of the [UDLM paper](https://arxiv.org/abs/2412.10193). The later
[LOO analysis](https://arxiv.org/abs/2605.22765) clarifies that the bridge's
plug-in distribution is optimally a leave-one-out (LOO) clean-token predictor,
not an ordinary denoising posterior. Neither paper prescribes this campaign's
top-$p$ search. Temperature and nucleus filtering are therefore labeled
inference hypotheses, applied to the raw LOO distribution before the exact
bridge rather than presented as a new training objective.

**Intuition and motivation.** Temperature controls how sharply the model ranks
plausible clean tokens, while nucleus filtering removes the long tail. Applying
either operation after the reverse bridge would answer a different question:
the bridge deliberately restores transition mass needed by the diffusion
process. We want to modify the model's raw prediction and then let UDLM perform
its mathematically specified transition.

**Mathematics and symbols.** Let $A=(a_1,\ldots,a_K)$ be the ordered active
model-token IDs, $h_j\in\mathbb R$ the raw LOO logit for active token $a_j$,
$\tau>0$ the softmax temperature, and $p\in(0,1]$ the nucleus threshold. First
form

$$r_j(\tau)=\frac{\exp(h_j/\tau)}{\sum_{k=1}^K\exp(h_k/\tau)}.$$

Sort indices by decreasing $r_j$, breaking an exact probability tie by smaller
active token ID. Retain the first sorted token whose cumulative mass becomes
strictly greater than $p$, and every token before it; zero only later tokens.
At least one token is retained. Renormalizing gives $\widetilde r$. If $i$ is
the observed noisy category, $0\le s<t\le1$, $\alpha_t$ is the clean
coefficient, $\alpha_{t\mid s}=\alpha_t/\alpha_s$, and $\pi$ is the stationary
prior, the unnormalized reverse probability is

$$w_k=\left[\alpha_{t\mid s}\mathbf1\{k=i\}
 +(1-\alpha_{t\mid s})\pi_i\right]
 \left[\alpha_s\widetilde r_k+(1-\alpha_s)\pi_k\right],$$

and $p_\theta(z_s=k\mid z_t=i)=w_k/\sum_jw_j$. The *reverse posterior itself*
is never nucleus-truncated. When `raw_loo_top_p=1.0`, production branches before
sorting, masking, or renormalizing and follows the literal pre-v4 path. The
required regression is stronger than numerical closeness: uniform and
categorical posterior tensors must satisfy `torch.equal`, and cloned RNG states
must produce exactly equal sampled IDs.

**Concrete example.** Suppose active token IDs `(7, 2, 9)` have raw LOO
probabilities `(0.5, 0.3, 0.2)` at $\tau=1$. For $p=0.7$, token 7 reaches
0.5 and token 2 crosses the threshold at 0.8, so both remain. The normalized
LOO vector is `(0.625, 0.375, 0)`. With a full-support stationary prior, the
bridge can still assign positive reverse probability to token 9 through its
noise term. Truncating *after* the bridge would incorrectly force that
probability to zero.

**Code below, shapes, and invariants.** The notebook oracle consumes one
length-$K$ logit vector and matching active-ID vector, checks finite
$\tau,p$, implements the stable ordering, and evaluates a three-category
uniform bridge. Production generalizes this independently at every editable
position of logits shaped `[B,L,K]`; token IDs and editable masks are `[B,L]`,
and each final posterior row is finite, nonnegative, and sums to one. The
training loss and forward process are unchanged.

**Difference from released implementations.** NVIDIA GenMol uses absorbing
MDLM: already revealed tokens are fixed, and there is no raw-LOO bridge input
on which to apply this rule. The official UDLM release supplies the bridge but
does not define this exact stable raw-LOO top-$p$ contract or its byte-exact
$p=1$ compatibility branch. Predictor-corrector sampling is intentionally
deferred because it is outside the frozen v4 campaign.

**Comprehension checkpoint.** Why retain the crossing token? Expected
reasoning: without it, retained mass can remain below the requested nucleus.
Why break ties by active token ID? Expected reasoning: a deterministic secondary
key prevents platform-dependent membership. Why filter $\widetilde r$ rather
than the final posterior? Expected reasoning: the bridge is part of the UDLM
transition and may legitimately restore full-support noise mass. Why is
`allclose` insufficient at $p=1$? Expected reasoning: the registered identity
claim includes exact posterior bytes and the exact random trajectory, so even a
tiny arithmetic-path change is disallowed.


In [ ]:
import math as stage210_math


stage210_contract = stage20_superiority_protocol["raw_loo_top_p_semantics"]
assert stage210_contract["schema_version"] == 1
assert stage210_contract["public_key"] == "raw_loo_top_p"
assert stage210_contract["omitted_value_normalizes_to"] == 1
assert stage210_contract["scope"] == "udlm_active_diffusion_alphabet_only"
assert stage210_contract["training_or_loss_changes"] is False
assert stage210_contract["mdlm_non_null_raw_loo_top_p_forbidden"] is True
assert stage210_contract["operation_order"] == [
    "select_raw_loo_logits_on_active_model_token_ids",
    "divide_raw_loo_logits_by_softmax_temperature",
    "softmax_to_raw_loo_probabilities",
    "apply_stable_nucleus_filter_to_raw_loo_probabilities",
    "renormalize_retained_raw_loo_probabilities",
    "apply_exact_uniform_or_categorical_reverse_bridge",
    "sample_from_the_untruncated_reverse_posterior",
]


def stage210_softmax(raw_logits, temperature):
    '''Return one raw-LOO probability vector; both inputs are one position.'''
    if (
        isinstance(temperature, bool)
        or not isinstance(temperature, (int, float))
        or not stage210_math.isfinite(temperature)
        or temperature <= 0
    ):
        raise ValueError("temperature must be a finite non-Boolean real > 0")
    scaled = [float(value) / temperature for value in raw_logits]
    maximum = max(scaled)
    weights = [stage210_math.exp(value - maximum) for value in scaled]
    total = sum(weights)
    return [value / total for value in weights]


def stage210_stable_nucleus(raw_probabilities, active_token_ids, top_p):
    '''Filter raw LOO probabilities, retaining the threshold-crossing token.'''
    if (
        isinstance(top_p, bool)
        or not isinstance(top_p, (int, float))
        or not stage210_math.isfinite(top_p)
        or not 0 < top_p <= 1
    ):
        raise ValueError("top_p must be a finite non-Boolean real in (0, 1]")
    if len(raw_probabilities) != len(active_token_ids) or not raw_probabilities:
        raise ValueError("probabilities and active IDs must have equal positive length")
    # The literal identity branch precedes sort, mask, and renormalization.
    if top_p == 1:
        return list(raw_probabilities), list(range(len(raw_probabilities)))
    order = sorted(
        range(len(raw_probabilities)),
        key=lambda index: (-raw_probabilities[index], active_token_ids[index]),
    )
    retained = []
    cumulative = 0.0
    for index in order:
        retained.append(index)
        cumulative += raw_probabilities[index]
        if cumulative > top_p:
            break
    retained_set = set(retained)
    retained_mass = sum(raw_probabilities[index] for index in retained)
    filtered = [
        probability / retained_mass if index in retained_set else 0.0
        for index, probability in enumerate(raw_probabilities)
    ]
    return filtered, retained


def stage210_uniform_bridge(raw_loo_probs, observed_index, alpha_t, alpha_s):
    '''Evaluate Eq. 15 after raw-LOO filtering for one K-category position.'''
    if not 0 <= alpha_t <= alpha_s <= 1 or alpha_s == 0:
        raise ValueError("expected 0 <= alpha_t <= alpha_s <= 1 and alpha_s > 0")
    num_classes = len(raw_loo_probs)
    alpha_t_given_s = alpha_t / alpha_s
    prior = 1.0 / num_classes
    weights = []
    for index, clean_probability in enumerate(raw_loo_probs):
        transition = (
            alpha_t_given_s * (index == observed_index)
            + (1.0 - alpha_t_given_s) * prior
        )
        earlier_marginal = alpha_s * clean_probability + (1.0 - alpha_s) * prior
        weights.append(transition * earlier_marginal)
    normalizer = sum(weights)
    return [weight / normalizer for weight in weights]


stage210_active_ids = [7, 2, 9]                          # [K=3]
stage210_logits = [stage210_math.log(x) for x in (0.5, 0.3, 0.2)]
stage210_raw = stage210_softmax(stage210_logits, 1.0)   # [K=3]
stage210_filtered, stage210_retained = stage210_stable_nucleus(
    stage210_raw, stage210_active_ids, 0.7
)
assert stage210_retained == [0, 1]
assert all(
    stage210_math.isclose(observed, expected, abs_tol=1e-15)
    for observed, expected in zip(stage210_filtered, (0.625, 0.375, 0.0))
)
stage210_reverse = stage210_uniform_bridge(
    stage210_filtered, observed_index=2, alpha_t=0.3, alpha_s=0.6
)
assert all(value > 0 for value in stage210_reverse)
assert stage210_math.isclose(sum(stage210_reverse), 1.0, abs_tol=1e-15)

stage210_identity, stage210_identity_order = stage210_stable_nucleus(
    stage210_raw, stage210_active_ids, 1.0
)
assert stage210_identity == stage210_raw
assert stage210_identity_order == [0, 1, 2]
assert stage210_contract["identity_branch"][
    "required_uniform_and_categorical_posterior_check"
] == "torch.equal"
assert stage210_contract["identity_branch"][
    "required_uniform_and_categorical_sample_check"
] == "exact_sampled_ids_under_cloned_rng_state"

stage210_example = {
    "active_token_ids": stage210_active_ids,
    "raw_loo_probabilities": stage210_raw,
    "retained_positions_at_p_0_7": stage210_retained,
    "filtered_raw_loo_probabilities": stage210_filtered,
    "untruncated_reverse_posterior": stage210_reverse,
    "p_1_literal_identity": stage210_identity == stage210_raw,
}
print(stage210_example)


> Historical v4 teaching: D terminated on 2026-09-07 with a launcher configuration-identity failure. Its artifacts and protocol remain immutable; these examples do not resume it. Stage 21 describes the separate engineering-v5 experiment. Current GPU authorization is at most two cards.

## 20.11 Schema-8 token-control evidence and completion-last publication

**Paper correspondence.** GenMol reports molecule-level validity, uniqueness,
quality, and diversity after decoding generated SAFE strings. UDLM defines the
categorical reverse process that produces token IDs. Neither paper specifies a
forensic record joining those two levels. Candidate benchmark schema 8 adds
that local evidence layer; it changes neither diffusion mathematics nor metric
definitions.

**Intuition and motivation.** A CSV of decoded strings cannot show whether the
sampler changed BOS/EOS/PAD context, left MASK tokens editable, or emitted a
control token later hidden by `skip_special_tokens=True`. Schema 8 stores the
complete sampler input, final sampled IDs, and editable mask inside the bounded
summary. A verifier can therefore reconstruct the exact token-to-text boundary
instead of trusting aggregate counts.

**Mathematics and symbols.** Let $B$ be the number of requested rows, $L$ the
padded token length, and $N=BL$. Flatten row-major token arrays
$x^{\mathrm{in}},x^{\mathrm{out}}\in\{0,\ldots,1879\}^N$. Each value is encoded
as an unsigned 16-bit little-endian integer. Let $m_n\in\{0,1\}$ mark editable
position $n$. With MSB-first bit packing, logical bit $n$ occupies byte
$\lfloor n/8\rfloor$ at bit $7-(n\bmod8)$. Unused tail bits must be zero. For
control token name $c\in\{\mathrm{unk,bos,eos,pad,mask}\}$ with ID $v_c$, the
audited count on support $S$ is

$$C_{c,S}=\sum_{n\in S}\mathbf1\{x_n=v_c\}.$$

Counts are recomputed for sampler input, all final positions, and editable
final positions. Immutable positions satisfy
$x^{\mathrm{out}}_n=x^{\mathrm{in}}_n$ whenever $m_n=0$. Decoding the final
`[B,L]` IDs with `skip_special_tokens=True` must equal every CSV
`raw_model_text` in row order.

**Concrete example.** The normative two-row, five-column input is
`[[BOS,MASK,MASK,EOS,PAD],[BOS,MASK,EOS,PAD,PAD]]`. Its editable positions are
flat indices 1, 2, and 6, so the first packed byte is binary `01100010`, or
hexadecimal `62`; six unused bits in the second byte are zero. The final IDs
replace those masks with `(7,8,9)` and leave every immutable value unchanged.
The next cell decodes the exact RFC 4648 base64 strings, checks their SHA-256
digests, and reconstructs both arrays and the mask.

**Code below, shapes, and invariants.** Schema 8 permits at most $B=1000$ and
$L=256$, and caps `summary.json` at 2 MiB. The model vocabulary is 1,880 while
the tokenizer reports an effective size of 1,882; every sampled model ID must
remain in the model range. The audit is a top-level summary member, never a
sidecar. Audit encoding time is excluded from generation timing and recorded
separately as `runtime_seconds.sampled_token_control_audit`.

Publication is equally strict. The controller reserves an output directory and
the child retains its descriptor. The benchmark publishes `raw_samples.csv`
first and links `summary.json` last with exclusive no-clobber operations. Thus a
reader may treat the completion member as a durable bundle boundary. Before
model import and again before publication, the child revalidates the directory
device/inode, repository-global generation lease, exact 24-element command,
source revision, selected GPU UUID, and `scripts/artifact_io.py` identity. If a
pre-completion operation fails, rollback may remove only members whose exact
owned identities still match; an ambiguous transition fails closed.

**Difference from released implementations.** NVIDIA GenMol and the official
UDLM release write ordinary generation outputs but do not provide this bounded
token audit, retained-directory authority, global generation lease, or
completion-last no-clobber bundle protocol. These are local reproducibility and
shared-host safety controls, not claims of a new molecular model.

**Comprehension checkpoint.** Why store final token IDs when raw text already
exists? Expected reasoning: decoding can hide special/control tokens and cannot
prove immutable context was preserved. Why is `summary.json` linked last?
Expected reasoning: its presence becomes an unambiguous completion signal after
all ordinary members exist. Why retain a directory descriptor? Expected
reasoning: a path string alone can be replaced or redirected between checks.
Why may rollback remove only exact owned identities? Expected reasoning: a
concurrent or externally replaced file must never be mistaken for the failed
writer's staging artifact.


In [ ]:
import base64 as stage211_base64
import hashlib as stage211_hashlib


stage211_contract = stage20_superiority_protocol["schema8_evidence_contract"]
assert stage211_contract["candidate_benchmark_schema_version"] == 8
assert stage211_contract["candidate_report_schema_version"] == 7
assert stage211_contract["audit_top_level_key"] == "sampled_token_control_audit"
assert stage211_contract["sidecar_audit_forbidden"] is True
assert stage211_contract["summary_maximum_size_bytes"] == 2 * 1024 * 1024
assert stage211_contract["maximum_rows"] == 1000
assert stage211_contract["maximum_columns"] == 256
assert stage211_contract["model_vocab_size"] == 1880
assert stage211_contract["tokenizer_effective_size"] == 1882


def stage211_decode_canonical_base64(text):
    raw = stage211_base64.b64decode(text, validate=True)
    assert stage211_base64.b64encode(raw).decode("ascii") == text
    return raw


def stage211_decode_uint16_little_endian(record):
    raw = stage211_decode_canonical_base64(record["data_base64"])
    assert len(raw) % 2 == 0
    assert stage211_hashlib.sha256(raw).hexdigest() == record["decoded_sha256"]
    return [
        int.from_bytes(raw[offset : offset + 2], "little", signed=False)
        for offset in range(0, len(raw), 2)
    ]


def stage211_decode_msb0_mask(record):
    raw = stage211_decode_canonical_base64(record["data_base64"])
    assert len(raw) == record["decoded_byte_count"]
    assert stage211_hashlib.sha256(raw).hexdigest() == record["decoded_sha256"]
    logical_count = record["logical_bit_count"]
    bits = [
        bool(raw[index // 8] & (1 << (7 - index % 8)))
        for index in range(logical_count)
    ]
    unused = 8 * len(raw) - logical_count
    assert unused == record["unused_tail_bit_count"]
    if unused:
        assert raw[-1] & ((1 << unused) - 1) == 0
    return bits


stage211_vector = stage211_contract["normative_vector"]
stage211_rows = stage211_vector["rows"]
stage211_columns = stage211_vector["columns"]
stage211_input = stage211_decode_uint16_little_endian(
    stage211_vector["sampler_input_ids"]
)
stage211_final = stage211_decode_uint16_little_endian(
    stage211_vector["final_sampled_ids"]
)
stage211_editable = stage211_decode_msb0_mask(stage211_vector["editable_mask"])
assert stage211_input == stage211_vector["sampler_input_ids"]["values"]
assert stage211_final == stage211_vector["final_sampled_ids"]["values"]
assert len(stage211_input) == len(stage211_final) == len(stage211_editable) == 10
assert stage211_rows * stage211_columns == len(stage211_input)
assert [index for index, value in enumerate(stage211_editable) if value] == [1, 2, 6]
assert all(
    before == after
    for before, after, editable in zip(
        stage211_input, stage211_final, stage211_editable
    )
    if not editable
)
assert all(0 <= value < stage211_contract["model_vocab_size"] for value in stage211_final)

stage211_control_ids = stage211_contract["control_token_ids"]


def stage211_control_counts(values, support):
    return {
        name: sum(values[index] == token_id for index in support)
        for name, token_id in stage211_control_ids.items()
    }


stage211_all_positions = range(len(stage211_final))
stage211_editable_positions = [
    index for index, value in enumerate(stage211_editable) if value
]
stage211_counts = {
    "sampler_input_all_positions": stage211_control_counts(
        stage211_input, stage211_all_positions
    ),
    "final_sampled_all_positions": stage211_control_counts(
        stage211_final, stage211_all_positions
    ),
    "final_sampled_editable_positions": stage211_control_counts(
        stage211_final, stage211_editable_positions
    ),
}
assert list(stage211_counts["sampler_input_all_positions"]) == [
    "unk", "bos", "eos", "pad", "mask"
]
assert stage211_counts["sampler_input_all_positions"] == {
    "unk": 0, "bos": 2, "eos": 2, "pad": 3, "mask": 3
}
assert stage211_counts["final_sampled_editable_positions"] == {
    "unk": 0, "bos": 0, "eos": 0, "pad": 0, "mask": 0
}

stage211_publication = stage211_contract["publication"]
assert stage211_publication["benchmark_bundle_members"] == [
    "raw_samples.csv", "summary.json"
]
assert stage211_publication["benchmark_completion_member"] == "summary.json"
assert stage211_publication["completion_last_no_clobber_owned_rollback"] is True
stage211_bundle = stage211_contract["benchmark_bundle_contract"]["exact_value"]
assert stage211_bundle["ordinary_members"] == ["raw_samples.csv"]
assert stage211_bundle["completion_member"] == "summary.json"
assert stage211_bundle["exclusive_no_clobber"] is True
assert stage211_bundle["completion_linked_last"] is True

stage211_audit_example = {
    "shape": [stage211_rows, stage211_columns],
    "sampler_input_ids": [
        stage211_input[offset : offset + stage211_columns]
        for offset in range(0, len(stage211_input), stage211_columns)
    ],
    "final_sampled_ids": [
        stage211_final[offset : offset + stage211_columns]
        for offset in range(0, len(stage211_final), stage211_columns)
    ],
    "editable_flat_indices": stage211_editable_positions,
    "control_token_counts": stage211_counts,
    "completion_member": stage211_bundle["completion_member"],
}
print(stage211_audit_example)


> Historical v4 teaching: D terminated on 2026-09-07 with a launcher configuration-identity failure. Its artifacts and protocol remain immutable; these examples do not resume it. Stage 21 describes the separate engineering-v5 experiment. Current GPU authorization is at most two cards.

## 20.12 Frozen 36-setting staged candidate campaign

**Paper correspondence.** The GenMol paper evaluates three 1,000-molecule
de-novo runs, while the UDLM paper motivates revisable uniform diffusion. The
papers do not choose a SAFE-specific raw-LOO temperature/top-$p$ operating
point. Protocol v4 therefore preregisters a small-first engineering campaign
and keeps the original three final seeds untouched until one candidate is
locked.

**Intuition and motivation.** The complete operating-point universe has three
trained checkpoints, four temperatures, and three top-$p$ values. Running the
largest evaluation for all 36 combinations would spend final-scale compute on
weak settings and encourage post-hoc choices. The campaign starts with tiny
diagnostics, narrows temperature, checks temperature/top-$p$ interactions,
confirms survivors on a new engineering seed, and only then uses the two
registered selection seeds.

**Mathematics and symbols.** Let arms $a\in\{R,S,E\}$ denote the three terminal
scale-up checkpoints, temperatures $\tau\in\{0.50,0.70,0.85,1.00\}$, and
nucleus thresholds $p\in\{1.00,0.98,0.95\}$. The universe size is
$3\times4\times3=36$. For a rankable attempt $c$, let $q_c$ and $d_c$ be raw,
unrounded released-compatible quality and diversity aggregated only across the
fixed seed tuple of its current stage. Ordering is

$$(-q_c,-d_c,\operatorname{ASCII}(\mathrm{config\_id}_c),
  \operatorname{ASCII}(\mathrm{attempt\_id}_c)).$$

No score is pooled across stages. Shared seeds are a blocking/common-random-
number control, not a claim that nonlinear set metrics support paired
inference. Every scheduled slot must become completed, failed, or undefined
before advancement. Failed and undefined outcomes remain in evidence but are
unrankable; retry and substitution are forbidden. If an arm lacks its required
rankable quota, the campaign is incomplete and no lock may be published.

**Concrete staged example.** D runs E at $(\tau,p)=(1,1)$ with seed 1100 and
32 requests; only schema-8 structural success advances. A runs all four
temperatures at $p=1$ for every arm with seed 1101 and 32 requests, retaining
two temperatures per arm. B crosses those six temperatures with three $p$
values at seed 1102 and 64 requests, retaining two configurations per arm. C
checks those six configurations at new seed 1103 and 96 requests, retaining
one per arm. The eligible stage evaluates those three survivors at both seeds
1000 and 1001 with 256 requests per child and chooses one global checkpoint
plus operating point. Only after a decision, deterministic ledger projection,
candidate lock, clean commits, and pushes may seeds 0, 1, and 2 request 1,000
molecules each.

The pre-final campaign has 40 distinct stage/config entries, 43 generation
children, and
$32+12(32)+18(64)+6(96)+6(256)=3680$ requested molecules (3,680 in display
notation). Final evaluation adds three children and 3,000 molecules, for 46
children and 6,680 requests.
At 128 network-function evaluations (NFE) per molecule, where one NFE is one
full backbone forward evaluation per reverse step, those totals are 471,040
and 855,040 molecule-NFE respectively.

**Code below, shapes, and invariants.** The next cell reads the already bound v4
protocol, reconstructs all 36 canonical IDs matching
`[rse]_t{050,070,085,100}_p{095,098,100}`, and verifies every stage count,
seed, sample size, and accounting identity. Candidate IDs name only immutable
training checkpoints; temperature and $p$ belong to config/attempt IDs. The
future registry has 36 config rows, but only configurations promoted into a
stage are executed there.

Generation uses one logical `cuda:0` per isolated child, mapped dynamically
from a qualifying physical UUID. Immediately before every launch, the
controller inspects the full inventory and re-probes the chosen UUID. A card is
idle only when utilization is *strictly* below 10%, at least 30,000 MiB is free,
and compute mode is not prohibited; exactly 10% is rejected. Active processes
may coexist only if those live checks pass, are recorded, and are never
interrupted. The current user-authorized maximum is two GPUs without another request,
physical GPU 0 is never assumed, and long jobs run in named detached `tmux`
sessions with logs under `output/logs/`.

**Publication firewall and released-code difference.** F contains framework,
tests, documentation, this notebook, and v4. Its clean pushed child C adds
exactly 33 generated YAMLs while reusing three historical identity YAMLs
byte-for-byte. G, the clean pushed child of C, adds only the registry; every GPU
child uses exact G. After all pre-final children terminate, tracked evidence is
followed by candidate-decision-only, deterministic-ledger-only, and candidate-
lock-only commits. NVIDIA GenMol and official UDLM provide neither this staged
search nor the generation lease, Git chronology, no-retry rule, and
decision/ledger/lock split. These controls limit researcher degrees of freedom;
they are not model improvements.

**Comprehension checkpoint.** Why is the universe 36 but Stage B runs only 18
children? Expected reasoning: A prospectively retains two of four temperatures
per arm before B crosses them with three $p$ values. Why must D ignore chemistry
metrics? Expected reasoning: it is an ineligible plumbing diagnostic, so using
its quality would leak tuning information. Why retain failures? Expected
reasoning: deleting them would hide the actual attempt set and bias selection.
Why can final seeds run only after the lock is clean and pushed? Expected
reasoning: otherwise their outcomes could influence checkpoint or sampling
selection. What limitation remains? Expected reasoning: greedy temperature
screening at $p=1$ can discard a temperature that would interact well with a
smaller $p$; this is a lean search, not exhaustive optimization.


In [ ]:
import itertools as stage212_itertools
import re as stage212_re


stage212_campaign = stage20_superiority_protocol["registered_generation_campaign"]
stage212_grid = stage212_campaign["grid"]
assert stage212_grid == {
    "arms": ["R", "S", "E"],
    "softmax_temperatures": [0.5, 0.7, 0.85, 1],
    "raw_loo_top_p_values": [1, 0.98, 0.95],
    "universe_config_count": 36,
    "historical_identity_config_count": 3,
    "new_config_count": 33,
}
stage212_temperature_codes = {0.5: "050", 0.7: "070", 0.85: "085", 1: "100"}
stage212_top_p_codes = {1: "100", 0.98: "098", 0.95: "095"}
stage212_config_ids = [
    f"{arm.lower()}_t{stage212_temperature_codes[temperature]}_p{stage212_top_p_codes[top_p]}"
    for arm, temperature, top_p in stage212_itertools.product(
        stage212_grid["arms"],
        stage212_grid["softmax_temperatures"],
        stage212_grid["raw_loo_top_p_values"],
    )
]
assert len(stage212_config_ids) == len(set(stage212_config_ids)) == 36
assert all(
    stage212_re.fullmatch(r"[rse]_t(?:050|070|085|100)_p(?:095|098|100)", config_id)
    for config_id in stage212_config_ids
)

stage212_stages = {
    stage["stage_id"]: stage for stage in stage212_campaign["stages"]
}
assert list(stage212_stages) == ["D", "A", "B", "C", "eligible", "final"]
stage212_expected = {
    "D": (1, 1, [1100], 32, 32),
    "A": (12, 12, [1101], 32, 384),
    "B": (18, 18, [1102], 64, 1152),
    "C": (6, 6, [1103], 96, 576),
    "eligible": (3, 6, [1000, 1001], 256, 1536),
    "final": (1, 3, [0, 1, 2], 1000, 3000),
}
for stage212_stage_id, stage212_values in stage212_expected.items():
    stage212_stage = stage212_stages[stage212_stage_id]
    (
        stage212_entry_count,
        stage212_child_count,
        stage212_seeds,
        stage212_requested_per_child,
        stage212_requested_total,
    ) = stage212_values
    assert stage212_stage["scheduled_entry_count"] == stage212_entry_count
    assert stage212_stage["generation_child_count"] == stage212_child_count
    assert stage212_stage["seeds"] == stage212_seeds
    assert stage212_stage["requested_samples_per_child"] == stage212_requested_per_child
    assert stage212_stage["requested_molecules"] == stage212_requested_total

stage212_prefinal_ids = ["D", "A", "B", "C", "eligible"]
stage212_prefinal_entries = sum(
    stage212_stages[stage_id]["scheduled_entry_count"]
    for stage_id in stage212_prefinal_ids
)
stage212_prefinal_children = sum(
    stage212_stages[stage_id]["generation_child_count"]
    for stage_id in stage212_prefinal_ids
)
stage212_prefinal_molecules = sum(
    stage212_stages[stage_id]["requested_molecules"]
    for stage_id in stage212_prefinal_ids
)
stage212_complete_entries = (
    stage212_prefinal_entries + stage212_stages["final"]["scheduled_entry_count"]
)
stage212_complete_children = (
    stage212_prefinal_children + stage212_stages["final"]["generation_child_count"]
)
stage212_complete_molecules = (
    stage212_prefinal_molecules + stage212_stages["final"]["requested_molecules"]
)
stage212_accounting = stage212_campaign["accounting"]
assert (
    stage212_prefinal_entries,
    stage212_prefinal_children,
    stage212_prefinal_molecules,
) == (40, 43, 3680)
assert (
    stage212_complete_entries,
    stage212_complete_children,
    stage212_complete_molecules,
) == (41, 46, 6680)
assert stage212_accounting["prefinal_distinct_stage_config_entries"] == 40
assert stage212_accounting["prefinal_generation_children"] == 43
assert stage212_accounting["prefinal_requested_molecules"] == 3680
assert stage212_accounting["prefinal_molecule_nfe"] == 3680 * 128
assert stage212_accounting["complete_distinct_stage_config_entries"] == 41
assert stage212_accounting["complete_generation_children"] == 46
assert stage212_accounting["complete_requested_molecules"] == 6680
assert stage212_accounting["complete_molecule_nfe"] == 6680 * 128

assert stage212_campaign["ranking"]["key"] == [
    "released_quality_descending",
    "released_diversity_descending",
    "config_id_ascii_ascending",
    "attempt_id_ascii_ascending",
]
assert stage212_campaign["failure_policy"] == {
    "all_scheduled_stage_slots_terminal_before_advancement": True,
    "failed_or_undefined_outcomes_retained": True,
    "failed_or_undefined_outcomes_rankable": False,
    "retry_forbidden": True,
    "substitution_forbidden": True,
    "insufficient_quota_result": "campaign_incomplete_and_candidate_lock_forbidden",
    "already_running_siblings_complete_after_failure": True,
}

stage212_resource = stage20_superiority_protocol["generation_resource_policy"]
assert stage212_resource["maximum_gpus_without_additional_user_permission"] == 3
assert stage212_resource["idle_definition"] == {
    "utilization_percent_strictly_less_than": 10,
    "exactly_10_percent_is_idle": False,
    "minimum_free_memory_mib": 30000,
    "prohibited_compute_mode_rejected": True,
    "active_processes_allowed_when_live_telemetry_qualifies": True,
    "active_processes_recorded_and_never_interrupted": True,
}
stage212_firewall = stage20_superiority_protocol["publication_firewall"]
assert [phase["phase"] for phase in stage212_firewall["ordered_phases"]] == [
    "F",
    "C",
    "G",
    "registered_prefinal_gpu_campaign",
    "tracked_envelopes_and_evidence",
    "candidate_decision_only",
    "candidate_ledger_only",
    "candidate_lock_only",
    "final_gpu_evaluation",
]
assert stage212_firewall["ordered_phases"][1]["content"] == (
    "exactly_33_new_candidate_yaml_files_only"
)
assert stage212_firewall["ordered_phases"][2]["content"] == (
    "candidate_config_registry_json_only"
)

stage212_campaign_summary = {
    "universe_config_count": len(stage212_config_ids),
    "prefinal": {
        "stage_config_entries": stage212_prefinal_entries,
        "generation_children": stage212_prefinal_children,
        "requested_molecules": stage212_prefinal_molecules,
        "molecule_nfe": stage212_prefinal_molecules * 128,
    },
    "including_final": {
        "stage_config_entries": stage212_complete_entries,
        "generation_children": stage212_complete_children,
        "requested_molecules": stage212_complete_molecules,
        "molecule_nfe": stage212_complete_molecules * 128,
    },
    "maximum_gpus_without_additional_permission": 3,
    "exactly_10_percent_idle": False,
}
print(stage212_campaign_summary)


> Historical v4 teaching: D terminated on 2026-09-07 with a launcher configuration-identity failure. Its artifacts and protocol remain immutable; these examples do not resume it. Stage 21 describes the separate engineering-v5 experiment. Current GPU authorization is at most two cards.

## 20.13 Exact prefix-resume and publication runbook

**Paper correspondence.** GenMol and UDLM describe model training, reverse
sampling, and molecular evaluation, but neither paper specifies how an
adaptive GPU search becomes immutable Git evidence without observing final
seeds early. This stage is operational provenance around those scientific
methods. It does not change the UDLM transition, SAFE decoding, or any metric.

**Intuition and motivation.** Each campaign invocation receives only a bounded
prefix authorization. First authorize D alone with `--through-stage D`; after
its durable decision, separately resume through A, then B, then C, then
`eligible`. This makes every adaptive boundary visible and prevents one early
command from silently authorizing the entire search. The controller recovers
the immutable contiguous decision prefix, so a resume advances only from the
next unfinished stage and never reruns a terminal stage.

The ignored campaign artifacts are not Git-recoverable. The controller,
evidence materializer, and authority builder must therefore stay on the
artifact-bearing host and worktree that contain the terminal scale checkpoints
and all 43 pre-final child outcomes. A fresh clone is insufficient unless those
ignored artifacts have been restored byte-for-byte and pass every independent
validation. Git history alone must never be substituted for live artifacts.

**Mathematics and symbols.** Let $G$ be the clean pushed registry revision,
$E_v$ the tracked evidence revision (`EVIDENCE`), $D_c$ the candidate-decision
revision, $L_c$ the deterministic ledger revision, and $K_c$ the candidate-lock
revision. The required strict history is

$$G\prec E_v\prec D_c\prec L_c\prec K_c\prec F_{\mathrm{final}},$$

where $x\prec y$ means that $x$ is the exact sole parent of $y$. $E_v$ is an
addition-only commit containing exactly the manifest-derived closure; the next
three commits add exactly one authority artifact each. If $T_X$ denotes a
stage-X decision-completion time, every non-D child start obeys
$t^{\mathrm{start}}_X>T_{\mathrm{pred}(X)}$, every child terminates before
$T_X$, and $T_{\mathrm{eligible}}<t(K_c)$. Before any stage ranks attempts,
its raw, unrounded quality and diversity must be recomputed by the fresh CPU
independent rescore rather than copied from producer summaries.

**Concrete runbook example.** Use exactly
`/home/aidar.alimbayev/Documents/genmolv2/.venv/bin/python`. Launch each prefix
in a separately named detached `tmux` controller with inherited
`CUDA_VISIBLE_DEVICES` removed and controller stdout/stderr redirected below
`output/logs/`. D had concurrency one. The archived v4 protocol encoded at most three
one-GPU children; current work is limited to two by the latest user instruction.
Every selected UUID must pass a fresh inventory and
re-probe with utilization *strictly* below 10% and at least 30,000 MiB free.
After the eligible decision, run the CPU evidence materializer at exact $G$,
then run the same materializer with `--stage-published-evidence`; that flag
force-adds and verifies exactly the manifest closure but never commits or
pushes. It belongs to `materialize_candidate_evidence.py`, not
`prepare_candidate_authority.py`.

Commit and push $E_v$. Run `prepare_candidate_authority.py` separately with
`--phase decision`, commit and push only `candidate_decision.json`; run
`--phase ledger`, commit and push only the deterministic ledger projection;
then run `--phase lock`, which deterministically builds schema-2
`candidate_lock.json`, and commit and push only that file. There is no draft or
hand-authored lock step. Each phase requires the exact clean pushed predecessor
revision. Final generation then invokes `launch_benchmark.py` with
`--candidate-lock experiments/udlm/candidates/candidate_lock.json`; checkpoint,
config, output root, seeds, sample count, and sampling fields must equal the
committed lock. Pilot flags and attempt/candidate IDs are forbidden on this
final command.

**Code below, shapes, and invariants.** The next cell is CPU-only and performs
no subprocess, Git, filesystem, tmux, or GPU action. It constructs the exact
argument-vector shapes for five campaign prefixes, evidence publication and
staging, three authority phases, and the final lock-bound launcher. Revision
and digest placeholders have length-one scalar positions; the final seed tuple
has shape `[3]`; every generation child itself still samples arrays shaped
`[B,L]`, with $B=1000$ for each final seed.

**Difference from released implementations.** NVIDIA GenMol's launcher and the
official UDLM research code do not impose this G→EVIDENCE→decision→ledger→lock
Git chronology, exact force-add closure, independent pre-advancement rescore,
prefix-limited campaign controller, or candidate-lock-bound final CLI. These
are local prospective-selection and publication controls, not architectural
advantages over either released model.

**Comprehension checkpoint.** Why not start with `--through-stage eligible`?
Expected reasoning: separate invocations expose each adaptive decision boundary
and limit authorization to what has been reviewed. Why can a fresh clone fail
even when Git is clean? Expected reasoning: ignored checkpoints, raw outputs,
and live envelopes are required inputs. Why stage evidence with the
materializer? Expected reasoning: it derives the exact force-added closure from
the completion-last manifest and verifies the index. Why is there no lock
draft? Expected reasoning: the lock is a deterministic projection of committed
decision and ledger authority. Why does the final command repeat checkpoint,
config, seeds, and output root? Expected reasoning: the launcher compares those
arguments to the committed lock before mutation or GPU work.


In [ ]:
stage213_python = "/home/aidar.alimbayev/Documents/genmolv2/.venv/bin/python"
stage213_registry = (
    "experiments/udlm/protocols/de_novo_candidate_config_registry_v1.json"
)
stage213_registry_raw = "<REGISTRY_RAW_SHA256>"
stage213_registry_canonical = "<REGISTRY_CANONICAL_SHA256>"
stage213_G = "<G_40_HEX_REVISION>"

stage213_campaign_base = [
    stage213_python,
    "scripts/udlm/launch_candidate_campaign.py",
    "--registry",
    stage213_registry,
    "--expected-registry-sha256",
    stage213_registry_raw,
    "--expected-registry-canonical-sha256",
    stage213_registry_canonical,
]
stage213_prefixes = ["D", "A", "B", "C", "eligible"]
stage213_campaign_commands = {
    prefix: [*stage213_campaign_base, "--through-stage", prefix]
    for prefix in stage213_prefixes
}
assert list(stage213_campaign_commands) == stage213_prefixes
assert stage213_campaign_commands["D"][-2:] == ["--through-stage", "D"]

stage213_evidence_base = [
    stage213_python,
    "scripts/udlm/materialize_candidate_evidence.py",
    "--expected-source-revision",
    stage213_G,
    "--expected-registry-sha256",
    stage213_registry_raw,
    "--expected-registry-canonical-sha256",
    stage213_registry_canonical,
]
stage213_materialize_command = list(stage213_evidence_base)
stage213_stage_evidence_command = [
    *stage213_evidence_base,
    "--stage-published-evidence",
]
assert "--stage-published-evidence" not in stage213_materialize_command
assert stage213_stage_evidence_command[-1] == "--stage-published-evidence"

stage213_authority_commands = {
    "decision": [
        stage213_python,
        "scripts/udlm/prepare_candidate_authority.py",
        "--phase",
        "decision",
        "--expected-source-revision",
        "<EVIDENCE_40_HEX_REVISION>",
        "--expected-registry-sha256",
        stage213_registry_raw,
        "--expected-registry-canonical-sha256",
        stage213_registry_canonical,
        "--registry-revision",
        stage213_G,
    ],
    "ledger": [
        stage213_python,
        "scripts/udlm/prepare_candidate_authority.py",
        "--phase",
        "ledger",
        "--expected-source-revision",
        "<DECISION_40_HEX_REVISION>",
    ],
    "lock": [
        stage213_python,
        "scripts/udlm/prepare_candidate_authority.py",
        "--phase",
        "lock",
        "--expected-source-revision",
        "<LEDGER_40_HEX_REVISION>",
    ],
}
assert list(stage213_authority_commands) == ["decision", "ledger", "lock"]
assert all(
    "--stage-published-evidence" not in command
    for command in stage213_authority_commands.values()
)

stage213_final_command = [
    stage213_python,
    "scripts/exps/denovo/launch_benchmark.py",
    "--checkpoint",
    "<EXACT_LOCK_CHECKPOINT_PATH>",
    "--config",
    "<EXACT_LOCK_CONFIG_PATH>",
    "--num-samples",
    "1000",
    "--seeds",
    "0",
    "1",
    "2",
    "--output-root",
    "<EXACT_LOCK_OUTPUT_ROOT>",
    "--candidate-lock",
    "experiments/udlm/candidates/candidate_lock.json",
    "--gpu-count",
    "<INTEGER_1_TO_3>",
    "--log-root",
    "output/logs",
]
assert stage213_final_command[stage213_final_command.index("--seeds") + 1 :][:3] == [
    "0",
    "1",
    "2",
]
assert not {"--pilot", "--selection-pilot", "--attempt-id", "--candidate-id"}.intersection(
    stage213_final_command
)

stage213_campaign = stage20_superiority_protocol["registered_generation_campaign"]
stage213_tracked = stage213_campaign["tracked_evidence_contract"]
assert stage213_tracked["materialization"][
    "all_43_children_are_independently_validated_and_rescored_before_any_target_publication"
] is True
assert stage213_tracked["evidence_commit"] == {
    "symbol": "EVIDENCE",
    "exact_sole_parent_is_G": True,
    "all_changes_are_additions": True,
    "exact_addition_set": "manifest_union_manifest_required_git_paths",
    "every_path_absent_at_G": True,
    "every_blob_matches_manifest_sha256_and_recorded_size_where_size_is_recorded": True,
    "decision_ledger_and_lock_absent_at_G_and_EVIDENCE": True,
    "verified_before_candidate_decision_publication": True,
}
assert stage213_campaign["ranking"][
    "ranked_stage_metrics_must_come_from_fresh_cpu_independent_rescore_before_advancement"
] is True
assert stage213_campaign["adaptive_stage_chronology"] == {
    "schema_version": 1,
    "completed_and_failed_child_intervals_recovered_from_committed_evidence": True,
    "every_non_d_child_start_strictly_after_predecessor_stage_decision_completion": True,
    "every_child_terminal_strictly_before_own_stage_decision_completion": True,
    "eligible_stage_decision_completion_strictly_before_candidate_lock": True,
}
stage213_resource = stage20_superiority_protocol["generation_resource_policy"]
assert stage213_resource["diagnostic_gpu_count"] == 1
assert stage213_resource["maximum_gpus_without_additional_user_permission"] == 3
assert stage213_resource["idle_definition"][
    "utilization_percent_strictly_less_than"
] == 10
assert stage213_resource["idle_definition"]["exactly_10_percent_is_idle"] is False
assert stage213_resource["idle_definition"]["minimum_free_memory_mib"] == 30000

stage213_runbook_summary = {
    "campaign_prefixes": stage213_prefixes,
    "evidence_symbol": stage213_tracked["evidence_commit"]["symbol"],
    "authority_phases": list(stage213_authority_commands),
    "final_seeds": [0, 1, 2],
    "diagnostic_concurrency": stage213_resource["diagnostic_gpu_count"],
    "maximum_gpu_concurrency": stage213_resource[
        "maximum_gpus_without_additional_user_permission"
    ],
}
print(stage213_runbook_summary)


# Stage 21 — Diagnose the failed pilot and preview temperature screening

**Paper correspondence and released-code differences.** GenMol combines a
SAFE representation, absorbing MDLM, and confidence-based token revelation.
Our R/S/E checkpoints instead update editable positions with UDLM reverse
bridges: R retains the released uniform process, S repairs schedule consistency,
and E adds an empirical stationary prior. These are warmed from GenMol EMA,
with a local BERT time conditioner, not fresh replicas of the official DiT
architecture. Each received 1,000 updates of batch 16, or 16,000 additional
example exposures. The [UDLM paper](https://arxiv.org/abs/2412.10193) motivates
continuous token editing; the [later LOO analysis](https://arxiv.org/abs/2605.22765)
clarifies the meaning of the plug-in predictor. Temperature screening is a
sampling hypothesis, not a change to either training objective.

**Intuition and motivation.** Archived v4 diagnostic D generated all 32 samples,
but completion validation failed because the producer inserted the default
`raw_loo_top_p=1.0` while the launcher expected a configuration without that
field. Its terminal failure is preserved, with no promotion or retry in v4.
Engineering v5 is a separately specified experiment using seeds 1200/1201.
It tests whether concentrating probability on plausible tokens improves SAFE
consistency before spending more training compute. Final seeds 0/1/2 remain
excluded from tuning.

The failed D artifacts contain observations, not a successful registered
benchmark: repaired validity was 29/32, strict validity 6/32, and repaired
quality 12/32. A direct syntax recount found odd ring-label occurrence counts
in 20/32 rows and unbalanced parentheses in 4/32. These diagnostics overlap and
are not an exhaustive chemistry validator. Repair recovered 23 strict failures
and largest-component selection affected 15 rows. No editable BOS, EOS, PAD,
MASK, or UNK survived. Thus this sample suggests a structural-consistency
problem; it does not support a superiority claim or identify its unique cause.

**Mathematics, with every symbol defined.** Let $B$ be the number of molecules,
$L$ the padded sequence length, and $K$ the active vocabulary size (1,880 here).
For example index $b$, position $\ell$, and token $j$, the network returns logit
$z_{b\ell j}$. Temperature $\tau>0$ gives raw leave-one-out probabilities

$$r^{(\tau)}_{b\ell j}=\frac{\exp(z_{b\ell j}/\tau)}
 {\sum_{k=1}^{K}\exp(z_{b\ell k}/\tau)}.$$

Here $k$ is the summation token index; $r$ predicts a clean token from its noisy
context. Let $t$ be current time, $s<t$ earlier time, $i$ the current token,
$\pi_j$ stationary noise mass, and $\alpha_u=1-(1-\epsilon)u$ the clean fraction
at time $u$, with residual-clean constant $\epsilon=0.001$. Set
$a=\alpha_t/\alpha_s$. The reverse probability of earlier token $j$ is

$$P(j\mid i)=\frac{[a\,\mathbf1\{j=i\}+(1-a)\pi_i]
 [\alpha_s r^{(\tau)}_j+(1-\alpha_s)\pi_j]}
 {\sum_k[a\,\mathbf1\{k=i\}+(1-a)\pi_i]
 [\alpha_s r^{(\tau)}_k+(1-\alpha_s)\pi_k]}.$$

$\mathbf1$ is one when its condition holds and zero otherwise. Notice the
likelihood uses $\pi_i$, the observed category's mass. Temperature acts on $r$
before the bridge; applying it to final $P$ changes the transition differently.
Top-$p$ would retain the smallest ranked prefix whose cumulative raw mass reaches
$p$, including the crossing token. V5 fixes $p=1$, so no category is truncated.

**Small concrete example.** For raw probabilities $(0.6,0.3,0.1)$, temperature
$\tau=0.5$ squares and normalizes them to $(0.782609,0.195652,0.021739)$.
With uniform $\pi=(1/3,1/3,1/3)$, $\alpha_t=0.2$, $\alpha_s=0.6$, and current
category $i=2$ (the second category), the following CPU calculation shows the
different reverse probabilities before and after sharpening. Decreasing
temperature concentrates model confidence; it does not guarantee chemical
validity and can reduce diversity.

**Code below, tensor shapes, and invariants.** This cell is self-contained from
a clean kernel in this worktree. It reads the small v5 protocol and pinned YAMLs,
hashes configuration bytes, and calculates the three-category example using
Python lists. It never imports PyTorch, opens a model checkpoint, queries a GPU,
launches a process, or writes an artifact. Real token IDs and editable masks
have shapes `[B,L]`; logits and reverse probabilities have `[B,L,K]`.
Probabilities must be finite, nonnegative, and normalized; immutable framing
must remain unchanged. The preview verifies 128 NFE, distinct engineering
seeds, equal requested counts, and at most two GPUs with utilization strictly
below 10% at launch. Historical protocol fields allowing more GPUs are not
current authorization. Actual jobs use dynamic UUID discovery in named tmux
sessions with logs in `output/logs/`, separately from this notebook.

**Comprehension checkpoint.** Why can identical sampling defaults still cause
a validation failure? Expected reasoning: literal effective dictionaries and
their hashes differed even though both meant $p=1$. Why not infer raw molecular
validity from 29/32 repaired successes? Expected reasoning: strict decoding
accepted only six, and repair or component removal changes the output. Why keep
the crossing token in top-$p$? Expected reasoning: omitting it can leave retained
mass below $p$. Why are 12 temperature configurations and two seeds still not a
final comparison? Expected reasoning: these outcomes select settings; an
independent locked evaluation is needed to estimate the selected model's
performance. What trade-off might sharpening create? Expected reasoning:
quality can rise while coverage and diversity fall, so both must be measured.


In [ ]:
import hashlib as stage21_hashlib
import json as stage21_json
import math as stage21_math
from pathlib import Path as Stage21Path

import yaml as stage21_yaml


stage21_relative_protocol = Stage21Path(
    "experiments/udlm/protocols/engineering_v5.json"
)
stage21_start = Stage21Path.cwd().resolve()
stage21_candidates = [
    stage21_start,
    stage21_start / "run_sources/udlm_genmol_worktree",
    *stage21_start.parents,
]
stage21_root = next(
    root for root in stage21_candidates
    if (root / stage21_relative_protocol).is_file()
)
stage21_protocol_bytes = (stage21_root / stage21_relative_protocol).read_bytes()
stage21_protocol = stage21_json.loads(stage21_protocol_bytes)
assert stage21_protocol["schema_version"] == 1
assert stage21_protocol["claim"] == "engineering_screen_only_no_superiority_claim"
assert stage21_protocol["gpu_policy"] == {
    "max_gpus": 2,
    "max_utilization_percent": 10,
    "min_free_memory_mib": 30000,
}
stage21_seeds = stage21_protocol["seeds"]
assert len(stage21_seeds) == len(set(stage21_seeds))
assert all(type(seed) is int and seed >= 1000 for seed in stage21_seeds)
assert not set(stage21_seeds).intersection({0, 1, 2})
assert 1 <= stage21_protocol["num_samples"] <= 100
assert stage21_protocol["nfe"] == 128
stage21_entries = stage21_protocol["entries"]
assert len({entry["attempt_id"] for entry in stage21_entries}) == len(stage21_entries)

stage21_preview_rows = []
for stage21_entry in stage21_entries:
    stage21_config_path = (stage21_root / stage21_entry["config"]).resolve()
    assert stage21_config_path.is_relative_to(stage21_root)
    stage21_config_bytes = stage21_config_path.read_bytes()
    assert stage21_hashlib.sha256(stage21_config_bytes).hexdigest() == (
        stage21_entry["config_sha256"]
    )
    stage21_config = stage21_yaml.safe_load(stage21_config_bytes)
    assert stage21_config["diffusion_type"] == "udlm"
    assert stage21_config["num_steps"] == stage21_protocol["nfe"]
    assert stage21_config.get("raw_loo_top_p", 1.0) == 1.0
    stage21_preview_rows.append({
        "attempt": stage21_entry["attempt_id"],
        "arm": stage21_entry["arm_id"],
        "temperature": stage21_config["softmax_temp"],
        "raw_loo_top_p": stage21_config.get("raw_loo_top_p", 1.0),
        "requests_per_seed": stage21_protocol["num_samples"],
        "checkpoint_sha256": stage21_entry["checkpoint_sha256"],
    })


def stage21_normalize(values):
    total = sum(values)
    probabilities = [value / total for value in values]
    assert all(stage21_math.isfinite(value) and value >= 0 for value in probabilities)
    assert stage21_math.isclose(sum(probabilities), 1.0, abs_tol=1e-12)
    return probabilities


def stage21_toy_bridge(raw_probabilities):
    alpha_t, alpha_s, current_index = 0.2, 0.6, 1
    stationary = [1 / 3] * 3
    ratio = alpha_t / alpha_s
    return stage21_normalize([
        (ratio * (j == current_index) + (1 - ratio) * stationary[current_index])
        * (alpha_s * raw_probabilities[j] + (1 - alpha_s) * stationary[j])
        for j in range(3)
    ])


stage21_raw = [0.6, 0.3, 0.1]
stage21_sharp = stage21_normalize([value ** (1 / 0.5) for value in stage21_raw])
assert stage21_math.isclose(stage21_sharp[0], 18 / 23)
stage21_toy_result = {
    "raw_tau_1": stage21_raw,
    "raw_tau_05": stage21_sharp,
    "bridge_tau_1": stage21_toy_bridge(stage21_raw),
    "bridge_tau_05": stage21_toy_bridge(stage21_sharp),
}
stage21_preview = {
    "protocol_sha256": stage21_hashlib.sha256(stage21_protocol_bytes).hexdigest(),
    "seeds": stage21_seeds,
    "entries": stage21_preview_rows,
    "total_requests": (
        len(stage21_entries) * len(stage21_seeds) * stage21_protocol["num_samples"]
    ),
    "gpu_queries": 0,
    "checkpoint_loads": 0,
    "process_launches": 0,
    "artifact_writes": 0,
    "superiority_claim": False,
}
print(stage21_json.dumps({"toy": stage21_toy_result, "preview": stage21_preview}, indent=2))


# Stage 22 — A Gibbs correction experiment at the same 128-NFE budget

**Paper correspondence.** The original [UDLM paper](https://arxiv.org/abs/2412.10193)
provides the reverse bridge used by the R/S/E predictors. Appendix E of
[Uniform Diffusion Models Revisited](https://arxiv.org/abs/2605.22765) derives a
conditional update from a leave-one-out (LOO) predictor, allowing correction at
fixed noise without a second trained model. The [authors' code](https://github.com/samsongourevitch/rev_udm)
also explores confidence-based and parallel variants. Here the proposed kernel
chooses one editable coordinate uniformly, which has the random-scan Gibbs
interpretation when its conditional is exact.

**Intuition and motivation.** The ancestral predictor advances to less noise;
the corrector can revise one coordinate at the resulting noise level. The
engineering question is whether spending model calls on revision helps SAFE
structure more than spending all calls on finer predictor steps. V6 compares
128 predictor calls against 64 predictor plus 64 fresh corrector calls, all
using the same checkpoint and requested count within each R/S/E pair.

Temperature 0.5 is an engineering choice informed by partial v5 observations
of E and S through temperature 0.85. It was fixed across arms to favor syntax,
not selected as a proven global quality winner. Thus this design is informed
by earlier exploration; its new seeds 1300/1301 do not make the overall search
confirmatory. The six settings request 64 molecules for each seed, or 768 total.
Final seeds 0/1/2 remain reserved. No v6 result is asserted by this preview.

**Mathematics, with every symbol defined.** Let $x_s=(x_s^1,\ldots,x_s^L)$ be a
noisy sequence of length $L$ at time $s$, and $x_s^{-\ell}$ all positions except
$\ell$. For clean token category $j$, define the exact LOO probability
$r_\ell(j)=P(X_0^\ell=j\mid X_s^{-\ell}=x_s^{-\ell})$, where $X_0$ is the
clean random sequence and $X_s$ the noisy random sequence. If $\pi_j$ is
stationary noise probability and $\alpha_s$ is the clean-data fraction, the
noisy one-coordinate conditional is

$$c_\ell(j\mid x_s^{-\ell})=\alpha_s r_\ell(j)+(1-\alpha_s)\pi_j.$$

Here $c_\ell$ sums to one over the active categories. Unlike the reverse bridge,
this formula has no observed-token likelihood multiplier: the current token
$x_s^\ell$ is being resampled given the other positions. Let $\mathcal M$ be a
fixed nonempty set of editable positions. Choose coordinate $I$ with
$P(I=\ell)=1/|\mathcal M|$ for $\ell\in\mathcal M$, where $|\mathcal M|$ is
its size, then draw its new token from $c_I$. All other positions are copied.
If $\mathcal M$ is empty, the row is unchanged. Exact compatible conditionals
with this state-independent scan preserve the noisy target distribution
conditional on the immutable context. One-coordinate resampling can retain the
old token, so at most one coordinate changes.

The network gives learned logits $z_{\ell j}$ rather than exact $r_\ell$.
With temperature $\tau$, it supplies
$\widehat r^{(\tau)}_{\ell j}=\exp(z_{\ell j}/\tau)/\sum_k\exp(z_{\ell k}/\tau)$,
where $k$ indexes active categories. V6 sets $\tau=0.5$ and top-$p=1$ (no
truncation). Prediction error, dependence on the coordinate's own noisy token,
and tempering make these approximate conditionals. They need not preserve the
original noisy target or correspond to a consistently tempered joint law.

Let $N$ be total network-function evaluations (NFE), $M=N/2$ predictor
transitions, $\delta$ the inference endpoint, and
$t_i=1-(1-\delta)i/M$ for grid index $i=0,\ldots,M$. Each predictor moves from
$t_i$ to $s=t_{i+1}$. Its output gets a **fresh** network evaluation at $s$ before
the Gibbs draw; the next predictor sees that corrected state. Hence
$N=M+M=64+64=128$. The last corrector is at
$\delta=10^{-5}$, below the usual training lower time $10^{-3}$, a disclosed
extrapolation. There is no additional terminal model call.

**Small concrete example.** With $\alpha_s=0.75$,
$r_\ell=(0.2,0.8)$ and $\pi=(0.7,0.3)$, the exact noisy conditional is
$(0.75\cdot0.2+0.25\cdot0.7,\;0.75\cdot0.8+0.25\cdot0.3)
=(0.325,0.675)$. These are already specified toy LOO probabilities; the example
does not apply the study temperature to them again. For template
`[BOS, 0, 1, EOS, PAD]` and editable mask `[False, True, True, False, False]`,
the code chooses coordinate 1 or 2 uniformly, samples category 0 or 1 with that
conditional, and checks every immutable position. A second all-immutable row
demonstrates the identity case. The local random generator does not alter any
training or sampling RNG state.

**Code below, shapes, and invariants.** This self-contained cell uses Python
lists and reads only the small v6 protocol and six pinned YAMLs. It verifies
configuration hashes, each matched checkpoint pair, the common temperature and
top-p, the 128-NFE allocations, and the total of 768 requests. It never imports
PyTorch, probes a GPU, opens a checkpoint, launches a job, or writes an artifact.
Real token IDs and Boolean masks have shape `[B,L]`, full model logits
`[B,L,K]`, and compact corrector probabilities `[B,L,A]`; $B$ is batch size,
$K$ full vocabulary size, and $A$ active vocabulary size. The mask comes from
the original template and stays fixed even if an editable token becomes MASK.
Probability rows are finite, nonnegative, and normalized; framing and supplied
fragments are invariant. Actual future jobs remain capped at two GPUs, each
strictly below 10% utilization immediately before dynamic UUID selection.

**Released-code differences and claim boundary.** This is an opt-in inference
experiment, with no retraining. The default sampler retains its ancestral RNG
sequence and IDs; MDLM rejects Gibbs mode. The option uses even integer NFE
budgets and changes the allocation of model calls, not their total. Production
CPU tests enumerate a correlated four-state target to establish exact oracle
stationarity and show that simultaneous updates from stale conditionals fail.
Those tests do not establish learned-model stationarity or improved molecular
quality. Fewer predictor transitions also increase their time intervals.

**Comprehension checkpoint.** Why must correction use the fresh predictor
sample at time $s$? Expected reasoning: both the context and noise level changed,
so previous logits describe a different conditional. Why choose a coordinate
uniformly from a fixed mask? Expected reasoning: the scan is independent of the
current token state, preserving the standard Gibbs argument and immutable
context. Does exactly one resampled coordinate mean one changed token? Expected
reasoning: the sampled category may equal the old one. Does temperature 0.5
preserve the original target? Expected reasoning: transformed approximate
conditionals generally lose that guarantee. Why is this still engineering work?
Expected reasoning: its temperature was informed by partial earlier outcomes,
and a locked held-out benchmark is still required to support superiority.


In [ ]:
import hashlib as stage22_hashlib
import json as stage22_json
import math as stage22_math
import random as stage22_random
from pathlib import Path as Stage22Path

import yaml as stage22_yaml


stage22_relative_protocol = Stage22Path("experiments/udlm/protocols/engineering_v6.json")
stage22_start = Stage22Path.cwd().resolve()
stage22_candidates = [
    stage22_start,
    stage22_start / "run_sources/udlm_corrector_worktree",
    *stage22_start.parents,
]
stage22_root = next(
    root for root in stage22_candidates
    if (root / stage22_relative_protocol).is_file()
)
stage22_protocol_bytes = (stage22_root / stage22_relative_protocol).read_bytes()
stage22_protocol = stage22_json.loads(stage22_protocol_bytes)
assert stage22_protocol["schema_version"] == 1
assert stage22_protocol["claim"] == "engineering_screen_only_no_superiority_claim"
assert stage22_protocol["gpu_policy"] == {
    "max_gpus": 2, "max_utilization_percent": 10, "min_free_memory_mib": 30000,
}
assert stage22_protocol["seeds"] == [1300, 1301]
assert stage22_protocol["num_samples"] == 64
assert stage22_protocol["nfe"] == 128
assert not set(stage22_protocol["seeds"]).intersection({0, 1, 2})
assert stage22_protocol["design"]["temperature"] == 0.5
assert stage22_protocol["design"]["top_p"] == 1.0
assert stage22_protocol["design"]["partial_v5_observed_before_design"]
stage22_entries = stage22_protocol["entries"]
assert len(stage22_entries) == 6
assert len({entry["attempt_id"] for entry in stage22_entries}) == 6
stage22_pairs = {}
stage22_preview_rows = []
for stage22_entry in stage22_entries:
    stage22_path = (stage22_root / stage22_entry["config"]).resolve()
    assert stage22_path.is_relative_to(stage22_root)
    stage22_config_bytes = stage22_path.read_bytes()
    assert stage22_hashlib.sha256(stage22_config_bytes).hexdigest() == (
        stage22_entry["config_sha256"]
    )
    stage22_config = stage22_yaml.safe_load(stage22_config_bytes)
    assert stage22_config["diffusion_type"] == "udlm"
    assert stage22_config["num_steps"] == 128
    assert stage22_config["softmax_temp"] == 0.5
    assert stage22_config.get("raw_loo_top_p", 1.0) == 1.0
    stage22_corrector = stage22_config.pop("gibbs_corrector", False)
    assert type(stage22_corrector) is bool
    stage22_arm_pair = stage22_pairs.setdefault(stage22_entry["arm_id"], {})
    assert stage22_corrector not in stage22_arm_pair
    stage22_arm_pair[stage22_corrector] = (
        stage22_config,
        stage22_entry["checkpoint"],
        stage22_entry["checkpoint_sha256"],
    )
    stage22_predictor_calls = 64 if stage22_corrector else 128
    stage22_corrector_calls = 64 if stage22_corrector else 0
    assert stage22_predictor_calls + stage22_corrector_calls == 128
    stage22_preview_rows.append({
        "attempt": stage22_entry["attempt_id"],
        "arm": stage22_entry["arm_id"],
        "gibbs_corrector": stage22_corrector,
        "predictor_calls": stage22_predictor_calls,
        "corrector_calls": stage22_corrector_calls,
        "total_nfe": 128,
    })
assert set(stage22_pairs) == {"R", "S", "E"}
for stage22_pair in stage22_pairs.values():
    assert set(stage22_pair) == {False, True}
    assert stage22_pair[False] == stage22_pair[True]

stage22_alpha, stage22_loo, stage22_prior = 0.75, [0.2, 0.8], [0.7, 0.3]
stage22_conditional = [
    stage22_alpha * raw + (1 - stage22_alpha) * prior
    for raw, prior in zip(stage22_loo, stage22_prior)
]
assert all(stage22_math.isfinite(p) and p >= 0 for p in stage22_conditional)
assert stage22_math.isclose(sum(stage22_conditional), 1.0)
assert all(
    stage22_math.isclose(actual, expected)
    for actual, expected in zip(stage22_conditional, [0.325, 0.675])
)
stage22_original = [
    ["BOS", 0, 1, "EOS", "PAD"],
    ["BOS", 1, 1, "EOS", "PAD"],
]
stage22_masks = [
    [False, True, True, False, False],
    [False, False, False, False, False],
]
stage22_rng = stage22_random.Random(1300)
stage22_corrected = [row.copy() for row in stage22_original]
stage22_selected_coordinates = []
for row, fixed_mask in zip(stage22_corrected, stage22_masks):
    positions = [index for index, editable in enumerate(fixed_mask) if editable]
    coordinate = stage22_rng.choice(positions) if positions else None
    stage22_selected_coordinates.append(coordinate)
    if coordinate is not None:
        row[coordinate] = stage22_rng.choices([0, 1], weights=stage22_conditional, k=1)[0]
for original, corrected, fixed_mask in zip(
    stage22_original, stage22_corrected, stage22_masks
):
    assert sum(a != b for a, b in zip(original, corrected)) <= 1
    assert all(a == b for a, b, editable in zip(original, corrected, fixed_mask) if not editable)
assert stage22_selected_coordinates[0] in {1, 2}
assert stage22_selected_coordinates[1] is None
stage22_preview = {
    "protocol_sha256": stage22_hashlib.sha256(stage22_protocol_bytes).hexdigest(),
    "entries": stage22_preview_rows,
    "total_requests": len(stage22_entries) * 2 * stage22_protocol["num_samples"],
    "toy_conditional": stage22_conditional,
    "toy_original": stage22_original,
    "toy_corrected": stage22_corrected,
    "toy_selected_coordinates": stage22_selected_coordinates,
    "gpu_queries": 0,
    "checkpoint_loads": 0,
    "process_launches": 0,
    "artifact_writes": 0,
    "superiority_claim": False,
}
assert stage22_preview["total_requests"] == 768
print(stage22_json.dumps(stage22_preview, indent=2))


# Stage 23 — Predicting clean tokens, then removing the local observation

**Paper correspondence.** [D3PM](https://arxiv.org/abs/2107.03006) gives the
forward posterior in Eq. 3 and clean-token prediction with auxiliary cross
entropy (CE) in Sections 3.3–3.4. Our proposed pure-CE adaptation is a new
objective choice, not a reproduction of D3PM's combined variational/CE loss.
The original UDLM bridge in earlier stages consumes clean leave-one-out (LOO)
probabilities. A clean denoiser conditions on an additional observation: the
noisy token at its own position. We must convert between these meanings.

**Intuition and motivation.** MDLM initialization already predicts clean tokens
from corrupted context. CE may provide a useful adaptation objective under
categorical replacement noise. This is an optimization hypothesis: no CE
molecular result is established. Simply passing CE predictions into the old
bridge would count the observed token's likelihood twice. Division removes
that local evidence before the bridge incorporates it once.

**Mathematics, with every symbol defined.** Let $j$ be a clean category and $i$
an earlier noisy category in the active alphabet of size $K$. At the current
position the observed category is $k$. Times satisfy $0<s<t$, with clean
retention fractions $0<\alpha_t<\alpha_s\leq1$. The stationary prior is
$\pi$, where $\pi_i>0$ and $\sum_i\pi_i=1$. Write $\delta_{ij}=1$ if $i=j$
and zero otherwise. Define the local observation likelihood and the two
forward factors by

$$L_j=\alpha_t\delta_{jk}+(1-\alpha_t)\pi_k,\quad
B_{ij}=\alpha_s\delta_{ij}+(1-\alpha_s)\pi_i,\quad
A_i=(\alpha_t/\alpha_s)\delta_{ik}+(1-\alpha_t/\alpha_s)\pi_k.$$

The denoiser probability $D_j=P(X_0^\ell=j\mid X_t)$ includes position
$\ell$ in its context. The desired LOO probability $R_j$ omits that position:

$$C=\sum_jD_j/L_j,\qquad R_j=D_j/(L_jC).$$

Here $X_0$ and $X_t$ denote clean and noisy random sequences. Bayes' rule gives
the LOO interpretation when $D$ is exact; an approximate network still yields
a normalized derived vector. The bridge gives unnormalized weights
$u_i=A_i\sum_jB_{ij}R_j$. Since $\sum_iA_iB_{ij}=L_j$, their sum is $1/C$,
so the normalized bridge is

$$p_i=\sum_jD_j\frac{A_iB_{ij}}{L_j}.$$

This is a mixture of normalized forward posteriors. It gives exact coordinate
marginals for an exact denoiser; independent draws at all positions generally
approximate the correlated joint reverse transition. D3PM Eq. 4 instead sums
joint kernels before normalization, so substituting $D$ directly there has
different mixture weights. In production, subtract $\log L_j$ from denoiser
logits before the existing raw-LOO temperature/top-p transformation. Never
evaluate the division at $t=0$ where a likelihood can vanish.

For a clean label $y$, CE is $-\log D_y$. With denoiser logits $h_j$ and
$D=\operatorname{softmax}(h)$, its logit gradient is
$D_j-\delta_{jy}$. The current CT objective targets $R$, so adding CE directly
to raw-LOO logits generally asks them to represent different distributions.

**Small concrete example.** Take $K=2$, $k=0$, $\pi=(1/2,1/2)$,
$\alpha_t=1/2$, $\alpha_s=4/5$ and $D=(3/4,1/4)$. Then
$L=(3/4,1/4)$, $C=2$, and $R=(1/2,1/2)$. Both the converted bridge and the
explicit posterior mixture give $(13/16,3/16)$. Passing $D$ to the old bridge
without conversion gives $(91/100,9/100)$, an absolute error of $39/400$.

**Code below, tensor shapes, and invariants.** The independent cell uses exact
fractions and two-element lists to check the example without a model, GPU,
checkpoint, or file write. Production logits have shape $[B,L,K]$ for batch
size $B$ and sequence length $L$; token IDs and editable masks have $[B,L]$.
CE normalizes over the active vocabulary only and excludes fixed controls and
padding from its token loss. Conversion uses the current IDs/time at every
predictor and fresh Gibbs call. A checkpoint must explicitly identify the new
parameterization; historical raw-LOO checkpoints keep their existing meaning.
The longer exact and production-kernel audit is in
`scripts/udlm/audit_denoiser_conversion.py`.

**Differences from released implementations.** NVIDIA GenMol uses absorbing
MDLM; this experiment retains categorical replacement and changes its training
target. The initial CE option uses schedule-consistent uniform or empirical
priors. It does not change the historical release-uniform arm. Temperature
and top-p act on the converted LOO distribution, so they generally break exact
posterior-mixture interpretation; Gibbs stationarity likewise requires exact,
compatible, untempered conditionals. A matched training and generation study
must assess usefulness, including repaired and strict molecular validity.

**Comprehension checkpoint.** Why does $D$ differ from $R$? Expected reasoning:
the current position contributes its own likelihood to $D$. Why divide by
$L_j$ rather than the earlier-time prior? Expected reasoning: remove precisely
the local observation at time $t$. Does the exact two-token calculation prove
better molecules? Expected reasoning: it checks algebra, while network error,
finite step size, factorized sampling and optimization remain empirical.
Why cannot an existing raw-LOO checkpoint be relabeled as CE? Expected
reasoning: its trained logits have a different probabilistic target.


In [ ]:
from fractions import Fraction as Stage23Fraction

stage23_pi = [Stage23Fraction(1, 2)] * 2
stage23_d = [Stage23Fraction(3, 4), Stage23Fraction(1, 4)]
stage23_alpha_t, stage23_alpha_s = Stage23Fraction(1, 2), Stage23Fraction(4, 5)
stage23_k = 0
stage23_likelihood = [
    stage23_alpha_t * (j == stage23_k) + (1 - stage23_alpha_t) * stage23_pi[stage23_k]
    for j in range(2)
]
stage23_c = sum(d / likelihood for d, likelihood in zip(stage23_d, stage23_likelihood))
stage23_loo = [
    d / likelihood / stage23_c
    for d, likelihood in zip(stage23_d, stage23_likelihood)
]
stage23_ratio = stage23_alpha_t / stage23_alpha_s
stage23_a = [
    stage23_ratio * (i == stage23_k) + (1 - stage23_ratio) * stage23_pi[stage23_k]
    for i in range(2)
]
stage23_b = [
    [stage23_alpha_s * (i == j) + (1 - stage23_alpha_s) * stage23_pi[i]
     for j in range(2)]
    for i in range(2)
]
stage23_weights = [
    stage23_a[i] * sum(stage23_b[i][j] * stage23_loo[j] for j in range(2))
    for i in range(2)
]
stage23_bridge = [value / sum(stage23_weights) for value in stage23_weights]
stage23_mixture = [
    sum(stage23_d[j] * stage23_a[i] * stage23_b[i][j] / stage23_likelihood[j]
        for j in range(2))
    for i in range(2)
]
stage23_wrong_weights = [
    stage23_a[i] * sum(stage23_b[i][j] * stage23_d[j] for j in range(2))
    for i in range(2)
]
stage23_wrong = [value / sum(stage23_wrong_weights) for value in stage23_wrong_weights]
assert stage23_loo == [Stage23Fraction(1, 2)] * 2
assert stage23_bridge == stage23_mixture == [Stage23Fraction(13, 16), Stage23Fraction(3, 16)]
assert sum(stage23_bridge) == 1 and all(value >= 0 for value in stage23_bridge)
assert stage23_wrong == [Stage23Fraction(91, 100), Stage23Fraction(9, 100)]
stage23_error = max(abs(a - b) for a, b in zip(stage23_wrong, stage23_bridge))
assert stage23_error == Stage23Fraction(39, 400)
print({"LOO": list(map(str, stage23_loo)), "reverse": list(map(str, stage23_bridge)),
       "unconverted_error": str(stage23_error), "molecular_superiority_established": False})


# Stage 24 — Comparing objectives with the same training examples and masks

**Paper correspondence.** Earlier stages implement the original UDLM CT
objective and the Stage 23 clean-denoiser CE hypothesis, motivated by the
[D3PM](https://arxiv.org/abs/2107.03006) clean-token parameterization. This
experiment compares two local adaptation procedures; it does not reproduce
either paper's original training architecture or establish their superiority.

**Intuition and motivation.** V5 and V6 tested sampling choices on small
1,000-update, batch-16 adaptations. Their best selected quality was 57.03125%,
below the local MDLM mean 85.8%. That observation motivates more adaptation
and a different objective. V8 gives CT and CE the same larger example budget,
starting both afresh from the same MDLM EMA. The throughput pilot's twenty
updates are not carried into either arm. Matching the tokenizer vocabulary
alone is insufficient: target masks must agree too. Both V8 arms explicitly
mask every tokenizer control, including UNK/MASK if present in clean input.

**Mathematics, with every symbol defined.** Let $b$ be examples per GPU per
microbatch, $W$ the number of GPUs, $G$ accumulated microbatches before one
optimizer update, $B$ the global batch, and $T$ the number of updates. Then
$B=bWG$ and the requested example exposure is $E=TB$. Here $b=16$, $B=128$,
$T=1000$ and $E=128000$ per arm. These are exposure counts, not a claim that all
molecular identities are unique. For a fixed $W$, both arms use the same
batch grouping, stream partition, initialization seed and learning-rate
schedule. CT logits target clean LOO probabilities; CE logits target clean
denoiser probabilities and require the Stage 23 inference conversion.

**Small concrete example.** One GPU uses $16\times1\times8=128$ examples per
update. Two GPUs use $16\times2\times4=128$. Each arm requests 128,000 exposures,
eight times the old 16,000 per-arm adaptation budget; together the arms request
256,000. Holding global
batch fixed across different GPU counts does not guarantee identical floating
point reductions or local antithetic time assignments. The actual comparison
therefore uses one common GPU count for both arms and records it.

**Code below, tensor shapes, and invariants.** The cell reads the prospective
protocol and asks its launcher to compose both one- and two-GPU plans on CPU.
It checks all resolved settings match after removing only the intended logit
parameterization and separate checkpoint output directories. IDs and common
target masks have shape $[b,L]$, logits $[b,L,K]$, with sequence length $L$ and
full vocabulary size $K$. Both masks exclude the same immutable controls; the
stationary prior and active vocabulary are identical. The preview checks
budgets, not completion. It does not inspect GPU inventory, load checkpoints,
create artifacts, or launch processes. Live runs must first verify the
throughput result and dynamically select at most two GPUs below 10% utilization.

**Differences from released implementations.** The common all-control target
mask is an explicit new CT setting; historical CT defaults retain their
BOS/EOS/PAD mask. CE retains its all-control invariant. Both new arms share
the same MDLM EMA, FiLM conditioner, empirical prior with uniform mixture
0.0002, seed 1500 and L1 scheduler: peak 0.0003, warmup 50 updates, horizon
1000, floor 0.000003. CE and CT differ in their statistical target and gradient
weighting, so equal updates/examples do not imply equal compute cost or equally
optimized hyperparameters. An observed difference is conditional on this
shared optimization bundle. Future generation must disclose both strict and
repaired metrics, raw rows, sample count, model evaluations and runtime. Final
evaluation seeds 0/1/2 remain reserved; no V8 molecular result is asserted here.

**Comprehension checkpoint.** Why start both arms from MDLM instead of resuming
the throughput pilot? Expected reasoning: otherwise one arm inherits extra
exposures and optimizer history. Why specify the target mask explicitly?
Expected reasoning: changed loss positions would confound the objective
comparison. Does 128,000 exposures mean 128,000 distinct molecules? Expected
reasoning: duplicate data and representation choices can repeat identities.
What does a better training loss prove across CT and CE? Expected reasoning:
their scales and statistical targets differ, so molecular generation must be
evaluated directly under a separately specified protocol.


In [ ]:
import copy as stage24_copy
import json as stage24_json
from pathlib import Path as Stage24Path
import sys as stage24_sys

stage24_relative = Stage24Path("experiments/udlm/protocols/engineering_v8_objectives.json")
stage24_start = Stage24Path.cwd().resolve()
stage24_candidates = [stage24_start, *stage24_start.parents]
stage24_candidates += [p / "run_sources/udlm_genmol_worktree" for p in stage24_candidates.copy()]
stage24_root = next(p for p in stage24_candidates if (p / stage24_relative).is_file())
if str(stage24_root) not in stage24_sys.path:
    stage24_sys.path.insert(0, str(stage24_root))
from scripts.udlm.launch_objective_training import build_plans as stage24_build_plans

stage24_protocol = stage24_json.loads((stage24_root / stage24_relative).read_text())
assert stage24_protocol["seed"] == 1500
assert stage24_protocol["optimizer_updates"] == 1000
stage24_preview = []
for stage24_w in (1, 2):
    stage24_plans = stage24_build_plans(stage24_w, root=stage24_root)
    assert len(stage24_plans) == 2
    stage24_comparable = []
    for stage24_plan in stage24_plans:
        stage24_cfg = stage24_copy.deepcopy(stage24_plan["config"])
        stage24_udlm = stage24_cfg["training"]["udlm"]
        assert stage24_udlm["mask_all_special_tokens"] is True
        assert stage24_udlm["exclude_special_tokens"] is False
        stage24_parameterization = stage24_udlm.pop("parameterization")
        assert stage24_parameterization in {"raw_loo", "x0_denoiser"}
        stage24_g = stage24_cfg["trainer"]["accumulate_grad_batches"]
        assert stage24_cfg["loader"]["batch_size"] * stage24_w * stage24_g == 128
        assert stage24_cfg["trainer"]["max_steps"] * 128 == 128000
        stage24_cfg["callback"].pop("dirpath")
        stage24_comparable.append(stage24_cfg)
        stage24_preview.append({"gpus": stage24_w, "accumulation": stage24_g,
                                "parameterization": stage24_parameterization,
                                "requested_exposures": 128000})
    assert stage24_comparable[0] == stage24_comparable[1]
print(stage24_json.dumps({"plans": stage24_preview, "molecular_result": None}, indent=2))


# Stage 25 — Evaluating the objective on generated molecules

**Paper correspondence.** GenMol's de novo evaluation measures validity,
uniqueness, diversity and quality. Our released-comparable quality counts
unique valid molecules with QED $\geq0.6$ and SA $\leq4$, divided by all
requested samples. Stage 23's CE-to-LOO identity supplies a valid inference
parameterization; it does not prove molecular quality. We therefore compare
generated samples directly, with strict decoding and the released
repair/largest-component path reported separately.
QED measures drug-likeness; SA is a synthetic-accessibility score, with lower
values indicating easier synthesis according to that scoring model.

**Intuition and motivation.** One lucky seed can make a method look better.
V9 fixes CT/CE crossed with temperatures 1.0 and 0.5, seeds 1600/1601, and
100 requests per seed before seeing these models' molecules. Every setting
uses EMA weights, 128 predictor evaluations and no nucleus truncation or
Gibbs corrector. Temperature 1.0 is the primary objective comparison because
it leaves the proved conversion untempered. Temperature 0.5 is a secondary
engineering choice informed by V5/V6, not independent confirmation.

**Mathematics, with every symbol defined.** Let $s$ index one of $S$ seeds,
$m\in\{CT,CE\}$ identify the method, $N$ be requests per seed, and $c_{m,s}$
be the count of first-occurrence unique valid molecules passing both quality
thresholds. Then $q_{m,s}=c_{m,s}/N$. At one fixed temperature, the paired
difference is $\Delta_s=q_{CE,s}-q_{CT,s}$, its mean is
$\bar\Delta=S^{-1}\sum_s\Delta_s$, and its sample standard deviation is
$s_\Delta=\sqrt{\sum_s(\Delta_s-\bar\Delta)^2/(S-1)}$. The standard deviation
describes seed-to-seed spread; it is not a confidence interval. With only
two seeds, uncertainty remains substantial. Pairing labels seeds consistently;
it does not make two different models generate identical trajectories.

**Small concrete example.** These counts are synthetic teaching data. At
$N=10$, CT has quality counts $[5,7]$ and CE has $[6,6]$. The first seed favors
CE by 10 percentage points, but the second favors CT by 10 points. Their mean
difference is zero and the paired sample standard deviation is about 14.14
percentage points. Selecting the first seed alone would suggest a gain that disappears in the paired mean.
Compute each decoding branch separately: repair may change molecular identity,
so strict and repaired quality counts cannot be combined into one numerator.

**Code below, shapes, and invariants.** The CPU-only cell accepts two maps
from seed to integer quality count and one common request count. It requires
identical seed sets, at least two seeds, and counts between zero and $N$.
The conceptual quality and difference arrays have shape $[S]$; no model or
GPU tensors are needed. Exact fractions keep the toy mean difference exactly
zero. The cell also checks the planned $2\times2\times2=8$ generation runs and
$8\times100=800$ requests. It reads no checkpoint, launches no job and asserts
no experimental result; actual results come from independently rescored raw
rows under the prospective V9 design.

**Differences from released implementations.** This is a local adaptation
study with two seeds of 100 requests per configuration. The historical local
MDLM comparator has three seeds of 1,000; its 85.8% quality is context, not a
matched sample-count or full-training-budget comparison. The two new arms
share the Stage 24 training setup. An infrastructure amendment uses a separately
audited V8 CT checkpoint plus a distinct V8b CE run: V8's original controller
failed during process teardown and remains failed. No molecular output informed
that amendment. The design and incident record retain the exact hashes and
caveats. Neither CT-versus-CE training-loss magnitudes nor a selected pilot
mean authorize a superiority claim; final seeds 0/1/2 remain reserved.

**Comprehension checkpoint.** Why divide quality by requests rather than valid
molecules? Expected reasoning: invalid outputs must still lower quality, and
the released definition also counts unique accepted molecules only. Why pair
the same seed labels? Expected reasoning: define an unambiguous per-seed
contrast without asserting identical samples. Does the synthetic first seed
prove CE improved? Expected reasoning: the second reverses its advantage and
the paired mean is zero. Why show strict results if repair is released behavior?
Expected reasoning: repair can rescue or alter structures and conceal raw
syntax failures. Can 800 exploratory requests beat a paper mean statistically?
Expected reasoning: they are spread over four settings; selection, sample
size, training budget and reserved final evaluation still matter.


In [ ]:
from fractions import Fraction as Stage25Fraction
from math import sqrt as stage25_sqrt

def stage25_paired_quality(ct_counts, ce_counts, requested):
    if type(requested) is not int or requested <= 0:
        raise ValueError("requested must be a positive integer")
    if set(ct_counts) != set(ce_counts) or len(ct_counts) < 2:
        raise ValueError("both methods require the same two or more seeds")
    seeds = sorted(ct_counts)
    for counts in (ct_counts, ce_counts):
        if any(type(counts[s]) is not int or not 0 <= counts[s] <= requested
               for s in seeds):
            raise ValueError("quality counts must be integers within requests")
    differences = {s: Stage25Fraction(ce_counts[s] - ct_counts[s], requested)
                   for s in seeds}
    mean = sum(differences.values()) / len(seeds)
    variance = sum((value - mean) ** 2 for value in differences.values()) / (len(seeds) - 1)
    return {"paired_differences": differences, "mean_difference": mean,
            "paired_sample_sd": stage25_sqrt(float(variance))}

stage25_toy = stage25_paired_quality({1600: 5, 1601: 7}, {1600: 6, 1601: 6}, 10)
assert stage25_toy["mean_difference"] == 0
assert stage25_toy["paired_differences"] == {
    1600: Stage25Fraction(1, 10), 1601: Stage25Fraction(-1, 10)}
stage25_design = {"methods": ["CT", "CE"], "temperatures": [1.0, 0.5],
                  "seeds": [1600, 1601], "requests_per_seed": 100,
                  "predictor_evaluations_per_molecule": 128}
stage25_runs = (len(stage25_design["methods"]) * len(stage25_design["temperatures"])
                * len(stage25_design["seeds"]))
assert stage25_runs == 8
assert stage25_runs * stage25_design["requests_per_seed"] == 800
print("Synthetic example only:", stage25_toy)
print("Planned runs:", stage25_runs, "planned requests:", 800)


# Stage 26 — A mask-rich stationary prior as a transfer hypothesis

**Paper correspondence.** GenMol's MDLM forward process replaces tokens with
an absorbing MASK. The released UDLM uses uniform categorical replacements.
Our schedule-consistent categorical process can use a positive nonuniform
stationary prior instead. Combining ordinary-token and MASK replacements is
already part of the D3PM design space; the local hypothesis here is whether a
mask-rich empirical prior improves transfer from GenMol's pretrained MDLM.
It is a new prior treatment, not a reproduction of official uniform UDLM.
See `docs/udlm_mask_rich_prior_hypothesis.md` for the derivation and primary
paper reference. Neither this mathematical construction nor its implementation
is evidence of improved molecular quality.

**Intuition and motivation.** MDLM learned to fill visible MASK positions.
Under our empirical prior, the MASK probability is only about $1.06\times10^{-7}$;
almost all corruption replaces tokens with ordinary-looking tokens. A larger
MASK probability makes more noisy inputs resemble the pretraining task while
retaining positive probabilities for reversible ordinary-token substitutions.
This may ease transfer, but can also limit correction or leave unresolved MASKs.
V9 did not establish a CE quality improvement; V10 separately tests finer
sampling steps. This stage does not select or launch another experiment.

**Mathematics, with every symbol defined.** Let $A$ be the active vocabulary
size, $k$ an active token ID, $f_k$ its normalized training frequency, and
$w\in(0,1)$ the uniform floor weight. The positive empirical base is
$b_k=(1-w)f_k+w/A$. Let $m$ identify MASK, $\delta_{km}$ be one for $k=m$
and zero otherwise, and $\lambda\in[0,1)$ be the new MASK mixture weight.
Define

$$\pi_k^{(\lambda)}=\lambda\delta_{km}+(1-\lambda)b_k.$$

The prior vector has shape $[A]$, sums to one and stays strictly positive.
In particular, a non-MASK token has at least $(1-\lambda)w/A$ probability.
For diffusion time $t\in(0,1]$ and residual-clean parameter $\epsilon\in(0,1)$,
write $\alpha_t=1-(1-\epsilon)t$. For clean token $j$, noisy token $k$ has
probability

$$q_t(k\mid j)=\alpha_t\mathbf1[k=j]+(1-\alpha_t)\pi_k^{(\lambda)},$$

where $\mathbf1$ is an indicator. The transition matrix has shape $[A,A]$.
For clean non-MASK $j$, $q_t(m\mid j)=(1-\alpha_t)\pi_m^{(\lambda)}$.
At $\lambda=0$ the numeric prior equals the empirical base. As $\lambda$
approaches one, the forward kernel approaches absorbing masking, but exact
$\lambda=1$ breaks the positive-prior bridge and CE-to-LOO inversion contracts.
There is a second limit to track: at time one, $\alpha_1=\epsilon$, so the
forward distribution still contains clean signal and is not exactly the prior.
Making MASK mass approach one at fixed $\epsilon$ can worsen this terminal
mismatch. An absorbing forward limit alone does not prove a valid sampler.

**Small concrete example.** Use the synthetic alphabet `[MASK, A, B]` and
base probabilities $(0.02,0.58,0.40)$. With $\lambda=0.9$, the new prior is
$(0.902,0.058,0.040)$. At $t=0.5$ and $\epsilon=0.001$, a clean B becomes
MASK with probability $0.4995\times0.902=0.450549$. All three outcomes remain
possible. These deliberately visible toy probabilities are not the project's
1,880-token empirical counts.

**Code below, shapes, and invariants.** The standalone CPU cell uses exact
fractions to build a length-three prior and a $3\times3$ forward matrix. It
checks strict positivity, row normalization, the MASK probability and that
propagating the stationary prior leaves it unchanged. No checkpoint, model,
GPU or filesystem is needed. Production tensors remain integer `[B,L]` token
IDs, Boolean `[B,L]` editable masks, float64 `[A]` stored priors and model
logits `[B,L,K]`, where $B$ is batch size, $L$ sequence length and $K$ the full
tokenizer vocabulary size. MASK is an active corruption value; clean special
tokens remain fixed targets. Editable positions are captured before noising,
so a sampled MASK at an editable position remains editable.

**Differences from released implementations.** This is an opt-in local
`prior_variant: mask_rich_empirical` with mandatory `mask_mixture_weight` and
the existing `empirical_uniform_mix` base-floor setting. A checkpoint must
record the mixture weight, MASK ID, base-prior hash and resulting-prior hash;
its new state marker prevents relabeling an old E checkpoint, even at zero
mixture weight where numeric priors coincide. A new trained prior starts from
the common MDLM EMA under a new protocol. The prior cannot be changed as a
sampling-only override. Existing R/S/E experiments retain their exact identity.

**Comprehension checkpoint.** Why require $\lambda<1$? Expected reasoning:
ordinary-token mass stays positive, keeping likelihood ratios and inverse
conversion defined. Is $\pi_m=0.902$ the probability that a clean B is MASK at
$t=0.5$? Expected reasoning: no; only the refreshed fraction uses the prior,
giving 0.450549 in the toy. Why can MASK be sampled without becoming a fixed
position? Expected reasoning: editability comes from the original template,
not the current token value. Why does matching the forward absorbing limit
not make the entire sampler MDLM? Expected reasoning: terminal initialization,
learned conditionals and the sampling rule also matter. Does lower loss under
more masking prove better molecules? Expected reasoning: corruption difficulty
changed; independently generated molecular benchmarks are still necessary.


In [ ]:
from fractions import Fraction as Stage26Fraction

def stage26_mix_prior(base, mixture_weight, mask_index):
    if (not base or any(value <= 0 for value in base) or sum(base) != 1
            or not 0 <= mixture_weight < 1):
        raise ValueError("positive normalized base and mixture in [0,1) required")
    if type(mask_index) is not int or not 0 <= mask_index < len(base):
        raise ValueError("MASK must be an active token index")
    return tuple((1 - mixture_weight) * value + mixture_weight * (k == mask_index)
                 for k, value in enumerate(base))

stage26_base = (Stage26Fraction(1, 50), Stage26Fraction(29, 50), Stage26Fraction(2, 5))
stage26_prior = stage26_mix_prior(stage26_base, Stage26Fraction(9, 10), 0)
assert stage26_prior == (Stage26Fraction(451, 500), Stage26Fraction(29, 500),
                        Stage26Fraction(1, 25))
assert sum(stage26_prior) == 1 and min(stage26_prior) > 0
assert stage26_mix_prior(stage26_base, Stage26Fraction(0), 0) == stage26_base
stage26_t, stage26_eps = Stage26Fraction(1, 2), Stage26Fraction(1, 1000)
stage26_alpha = 1 - (1 - stage26_eps) * stage26_t
stage26_forward = tuple(tuple(stage26_alpha * (j == k) + (1 - stage26_alpha) * p
                             for k, p in enumerate(stage26_prior)) for j in range(3))
assert all(sum(row) == 1 for row in stage26_forward)
assert tuple(sum(stage26_prior[j] * stage26_forward[j][k] for j in range(3))
             for k in range(3)) == stage26_prior
assert stage26_forward[2][0] == Stage26Fraction(450549, 1000000)
print("Synthetic prior:", tuple(map(float, stage26_prior)))
print("Synthetic P(MASK at t=0.5 | clean B):", float(stage26_forward[2][0]))


# Stage 27 — Where should temperature act on a clean denoiser?

**Paper correspondence.** GenMol adapts an MDLM that predicts clean tokens
from a masked sequence. Released UDLM uses leave-one-out predictions in a
categorical reverse process and applies sampling temperature to those logits.
Stage 23 introduced our local cross-entropy (CE) clean-denoiser alternative and
its conversion to the existing reverse bridge. This stage asks whether to
temper the clean prediction before conversion. It is a new inference
hypothesis, not a bug fix or a result claimed by either paper. See also
`docs/udlm_denoiser_temperature_hypothesis.md`.

**Intuition and motivation.** Temperature below one sharpens a distribution,
but the distribution being sharpened matters. Conversion divides by the local
likelihood of observing the noisy token. Sharpening after that division also
sharpens the likelihood correction. A model may strongly predict that a rare
visible token was clean while the converted prediction assigns substantial
weight elsewhere. Sharpening the clean prediction instead tests a different
preference. Either preference can retain incorrect tokens or reduce diversity.

**Mathematics, with every symbol defined.** At one editable sequence position,
let $A$ be the active vocabulary size, $j$ a candidate clean token, $k$ the
observed noisy token, and $i$ a candidate token at an earlier time. Let
$\pi_j>0$ be the stationary probability of token $j$, with $\sum_j\pi_j=1$.
For current time $t\in(0,1]$, earlier time $s\in[0,t)$, and residual-clean
parameter $\nu\in(0,1)$, define $\alpha_u=1-(1-\nu)u$ for any time $u$.
The indicator $\mathbf1[j=k]$ is one when the IDs agree and zero otherwise.
The local likelihood is

$$L_j=q_t(k\mid j)=\alpha_t\mathbf1[j=k]+(1-\alpha_t)\pi_k.$$

Let $D_j$ be the model's predicted clean-token probability conditioned on the
whole current noisy sequence. Define normalization of positive weights $v$ by
$N(v)_j=v_j/\sum_hv_h$, where $h$ also ranges over active tokens. Products,
quotients and powers below act coordinate by coordinate. The converted
leave-one-out weights are $R=N(D/L)$. For temperature $T>0$, the current rule
and its implied clean prediction are

$$R_{\rm old}=N((D/L)^{1/T}),\qquad
D_{\rm old}=N(R_{\rm old}L)=N(D^{1/T}L^{1-1/T}).$$

The proposed order instead uses

$$D_{\rm new}=N(D^{1/T}),\qquad R_{\rm new}=N(D_{\rm new}/L).$$

The bridge must consume $R_{\rm new}$ at temperature one: another temperature
operation would change this declared rule. At $T=1$ both rules agree exactly.
At $T=0.5$, $D_{\rm old}=N(D^2/L)$ while $D_{\rm new}=N(D^2)$. The
inverse-likelihood factor can counteract confidence in the current token.
For an observed MASK, $L_j$ is equal among non-MASK candidates, so their
relative clean weights agree between rules; the MASK candidate still differs.

To distinguish a clean prediction from one reverse transition, let $U$ be
either $D_{\rm old}$ or $D_{\rm new}$ and set $r=\alpha_t/\alpha_s$.
The mixture of normalized forward bridges is

$$P_U(i)=\sum_j U_j
\frac{[\alpha_s\mathbf1[i=j]+(1-\alpha_s)\pi_i]
      [r\mathbf1[i=k]+(1-r)\pi_k]}{L_j}.$$

Here $P_U(i)$ is the next, partly noisy token probability; it is not $U_i$.
All denominators are positive, and $\sum_iP_U(i)=1$. With the true clean
posterior and $T=1$, this is the coordinate reverse marginal. Learned
predictions, coordinate factorization of a joint sequence distribution,
initialization from the stationary prior instead of the exact time-one law,
and a positive final diffusion time remain approximations. Neither ordering
removes final MASK tokens or imposes a chemistry constraint.

**Small concrete example.** Use the invented alphabet `[MASK, A, B]`,
$\pi=(0.9,0.0999,0.0001)$, observed $k=2$, $t=0.5$, $s=0.4$,
$\nu=0.001$ and $D=(0.001,0.009,0.99)$. At $T=0.5$, the implied clean B
weight is **54.3949%** under the current rule and **99.9916%** under the new
rule. The actual reverse-step B probability is **84.8083%** versus
**99.9959%**. These are chosen three-state probabilities, not measured
molecular frequencies, and greater retention alone does not imply correctness.

**Code below, shapes, and invariants.** The standalone CPU cell imports only
`Fraction`. Its distributions and likelihoods have shape `[A]` with $A=3$;
the explicit bridge sums over the clean index for each earlier-state index.
It checks normalization, positivity, the two conversion identities and
temperature-one equivalence with rational arithmetic. Integer `power` is
$1/T$ in this tiny demonstration, restricted to positive integers for exact
powers; production temperatures can be any supported positive real value.
In the model, integer tokens and Boolean editable masks have shape `[B,L]`,
and logits have shape `[B,L,K]`, where $B$ is batch size, $L$ here denotes
sequence length (distinct from the indexed likelihood $L_j$), and $K$ is the
full model output vocabulary size. The active support has size $A\leq K$.
No production import, checkpoint, dataset, GPU or filesystem is used here.

**Differences from released implementations.** The proposed opt-in
`temperature_space: x0_denoiser` applies to CE-parameterized UDLM only;
absent/default `raw_loo` retains the released temperature convention and
historical configuration identities. Initially use top-p one and no Gibbs
corrector, with the new bridge temperature fixed to one. This is an inference
choice on a fixed checkpoint with its original trained prior, unlike the new
prior training in Stage 26. Saved configurations and independent rescoring
must bind the selected order. A molecular test needs the same checkpoint,
prior, temperature, NFE and fresh paired engineering seeds for both rules,
all declared outcomes and final editable control-token counts. V12 remains
unchanged; final seeds 0/1/2 remain reserved. No benchmark improvement is
established by this cell, and no experiment is launched.

**Comprehension checkpoint.** Why do the rules coincide at $T=1$? Expected
reasoning: the likelihood exponent in $D_{\rm old}$ is zero and both recover
$D$. Why is applying temperature twice wrong for the new rule? Expected
reasoning: its defined clean posterior is already $N(D^{1/T})$; another power
after conversion reintroduces temperature-dependent likelihood weighting.
Why is 54.3949% different from 84.8083%? Expected reasoning: the first is
an implied clean weight, while the second includes the normalized transition
to a still-noisy earlier time. Does an observed MASK make the rules identical?
Expected reasoning: only the ratios among non-MASK clean candidates coincide;
the MASK candidate has a different likelihood. Why is this not a benchmark
win? Expected reasoning: the example supplies probabilities by hand, and
greater retention can preserve errors or reduce molecular diversity.


In [ ]:
from fractions import Fraction as Stage27Fraction

def stage27_normalize(weights):
    assert weights and min(weights) > 0
    return tuple(value / sum(weights) for value in weights)

def stage27_rules(denoiser, likelihood, power):
    assert type(power) is int and power > 0
    assert len(denoiser) == len(likelihood) and sum(denoiser) == 1
    raw = stage27_normalize(tuple(d / l for d, l in zip(denoiser, likelihood)))
    old_raw = stage27_normalize(tuple(value ** power for value in raw))
    old_clean = stage27_normalize(tuple(r * l for r, l in zip(old_raw, likelihood)))
    assert old_clean == stage27_normalize(tuple(
        d ** power * l ** (1 - power) for d, l in zip(denoiser, likelihood)))
    new_clean = stage27_normalize(tuple(d ** power for d in denoiser))
    new_raw = stage27_normalize(tuple(d / l for d, l in zip(new_clean, likelihood)))
    assert new_clean == stage27_normalize(tuple(r * l for r, l in zip(new_raw, likelihood)))
    return old_clean, new_clean

def stage27_bridge(clean_weights, prior, current, alpha_t, alpha_s):
    ratio = alpha_t / alpha_s
    return tuple(sum(clean_weights[j]
        * (alpha_s * (i == j) + (1 - alpha_s) * prior[i])
        * (ratio * (i == current) + (1 - ratio) * prior[current])
        / (alpha_t * (j == current) + (1 - alpha_t) * prior[current])
        for j in range(len(prior))) for i in range(len(prior)))

stage27_prior = (Stage27Fraction(9, 10), Stage27Fraction(999, 10000), Stage27Fraction(1, 10000))
stage27_denoiser = (Stage27Fraction(1, 1000), Stage27Fraction(9, 1000), Stage27Fraction(99, 100))
stage27_current = 2
stage27_t, stage27_s, stage27_nu = Stage27Fraction(1, 2), Stage27Fraction(2, 5), Stage27Fraction(1, 1000)
stage27_alpha_t = 1 - (1 - stage27_nu) * stage27_t
stage27_alpha_s = 1 - (1 - stage27_nu) * stage27_s
stage27_likelihood = tuple(stage27_alpha_t * (j == stage27_current)
    + (1 - stage27_alpha_t) * stage27_prior[stage27_current] for j in range(3))
stage27_old, stage27_new = stage27_rules(stage27_denoiser, stage27_likelihood, 2)
assert stage27_rules(stage27_denoiser, stage27_likelihood, 1) == (stage27_denoiser,) * 2
stage27_old_step, stage27_new_step = tuple(stage27_bridge(weights, stage27_prior,
    stage27_current, stage27_alpha_t, stage27_alpha_s) for weights in (stage27_old, stage27_new))
assert all(sum(values) == 1 and min(values) > 0 for values in
           (stage27_old, stage27_new, stage27_old_step, stage27_new_step))
print("Synthetic implied clean B weights (old, new):", float(stage27_old[2]), float(stage27_new[2]))
print("Synthetic reverse-step B probabilities (old, new):", float(stage27_old_step[2]), float(stage27_new_step[2]))


# Stage 28 — Property optimization with fragment remasking

## 28.1 A fixed token context inside a changing search population

**Paper correspondence.** GenMol's fragment attaching/remasking method in
Sections 4.2 and 5.3 searches for high-scoring molecules by combining fragments,
regenerating part of a candidate and updating a fragment population. See the
[GenMol paper](https://arxiv.org/html/2501.06158v3). Our V14 engineering question
is whether an existing UDLM sampler helps this conditional search, even though
earlier de novo studies did not establish superiority. No PMO outcome is used
or reported in this stage.

**Intuition and motivation.** Keep most of a proposed SAFE token sequence as
context, replace one fragment span by MASK tokens, and generate only that span.
The neural generator proposes a molecule; a separate property oracle scores
the decoded molecule. The population policy can then reuse useful fragments.
Gamma zero disables molecular-context guidance, not the surrounding fixed
tokens or bidirectional attention to them. Property scores do not become a
gradient through this generator in the released population search.

**Mathematics, with every symbol defined.** Let $x\in\{0,\ldots,K-1\}^{B\times L}$
be input token IDs, with batch size $B$, padded length $L$ and vocabulary size
$K$. Let $e_{b\ell}\in\{0,1\}$ mark an editable MASK position in row $b$ and
column $\ell$. If $z^{(r)}$ is the token array after update $r$, every update
must satisfy $z^{(r)}_{b\ell}=x_{b\ell}$ wherever $e_{b\ell}=0$.
UDLM initialization draws only editable positions from the checkpoint's stationary
prior $\pi$, where $\pi_j>0$ and $\sum_{j=0}^{K-1}\pi_j=1$. UDLM reverse
updates can revise a token that was already filled; editability is the original
mask, not a fresh test for whether the current value equals MASK.

**Small concrete example.** The invented sequence below has two editable
positions between separators. We supply two updates by hand, including a final
MASK token. Context preservation still holds. This demonstrates an invariant;
the supplied updates are not draws from a diffusion model or molecular samples.

**Code below, shapes, and invariants.** Nested tuples represent `[B,L]=[1,9]`
IDs and Boolean masks. Each replacement row has two entries, one per editable
position. The assertion checks every immutable position after both updates,
including BOS/EOS/PAD framing. The final count is over editable positions only.
No model, tokenizer, chemistry package, dataset, file, or device is accessed.

**Differences from released implementations.** The long-input path retains
GenMol's random fragment choice and insertion of 5..15 MASK tokens, subject to
capacity. Short inputs retain completion with effective added length 18;
`mask_len` does not override that released completion path. UDLM changes the
sampling law inside the editable span. Immutable token context does **not**
guarantee graph-level fragment preservation after SAFE decoding, repair or
largest-component selection. PMO searches property-scored valid molecules;
it does not use the separate fragment-constraint success metric. Residual
control tokens also require token-level evidence, not decoded text inspection.

**Comprehension checkpoint.** Can an already-filled editable token change
again? Expected reasoning: yes; editability is frozen from the input mask.
Does gamma zero remove the context? Expected reasoning: no, the unchanged
tokens remain visible to attention. Does the invariant prove molecular
substructure preservation? Expected reasoning: no, it is a token-array fact
before the decoding/repair map. Does a final MASK violate context preservation?
Expected reasoning: only if it changed an immutable position; completion and
chemistry require additional checks.


In [ ]:
stage28_ids = ((1, 6, 5, 4, 4, 5, 7, 2, 3),)
stage28_editable = tuple(tuple(token == 4 for token in row) for row in stage28_ids)
stage28_states = []
for replacement in ((6, 7), (4, 8)):
    values = iter(replacement)
    state = tuple(tuple(next(values) if editable else original
        for original, editable in zip(row, mask))
        for row, mask in zip(stage28_ids, stage28_editable))
    assert all(state[b][position] == stage28_ids[b][position]
        for b, mask in enumerate(stage28_editable)
        for position, editable in enumerate(mask) if not editable)
    stage28_states.append(state)
stage28_editable_count = sum(sum(row) for row in stage28_editable)
stage28_final_mask_count = sum(token == 4 for row, mask in zip(stage28_states[-1], stage28_editable)
    for token, editable in zip(row, mask) if editable)
assert (stage28_final_mask_count, stage28_editable_count) == (1, 2)
print("Invented final editable MASK count / positions:", stage28_final_mask_count, "/", stage28_editable_count)


## 28.2 What does the Fexofenadine MPO score reward?

**Paper correspondence.** Goal-directed hit generation requires an explicit
property objective. V14 uses the existing `fexofenadine_mpo` task and released
gamma-zero setting. Its exact implementation is pinned in
`experiments/udlm/diagnostics/pmo_oracle_inputs_20260907/manifest.json`, including
PyTDC 0.4.1 and the RDKit source/native libraries. The formulas below were read
from the pinned `tdc/chem_utils/oracle/oracle.py`, not inferred from the task name.

**Intuition and motivation.** This synthetic benchmark objective rewards
similarity to a reference molecule together with a polarity/lipophilicity
profile. Its geometric mean penalizes a weak component. It is not a clinical
efficacy, toxicity or synthesizability assay. The provisional GSK3B task was
deferred after an oracle pickle/library incompatibility was found before any
optimization; no PMO output selected this replacement task.

**Mathematics, with every symbol defined.** For a valid molecule $m$, let
$s(m)\in[0,1]$ be RDKit's Tanimoto similarity of atom-pair count fingerprints
to the fixed Fexofenadine reference (maximum atom-pair path length 10).
Let $p(m)$ be topological polar surface area (TPSA, in square ångströms), and
$l(m)$ be RDKit's dimensionless MolLogP estimate. Define

$$C(s)=\min(1,\max(0,s/0.8)),\quad
H(p)=\exp\!\left[-\tfrac12\left(\tfrac{\max(90-p,0)}{10}\right)^2\right],$$
$$J(l)=\exp\!\left[-\tfrac12\left(\tfrac{\max(l-4,0)}{1}\right)^2\right],\quad
F(m)=[C(s(m))H(p(m))J(l(m))]^{1/3}.$$

Here $C,H,J$ are unitless component scores, $F$ is the unitless property score,
and $\exp$ is the exponential function. TPSA at least 90 and logP at most 4
receive no respective penalty; similarity saturates at 0.8. The reference
structure itself is fixed in the pinned TDC implementation. A legitimate zero
component yields zero score and still consumes a valid new oracle call.

**Small concrete example.** Supply invented descriptor values, not a molecule:
$s=0.1,p=90,l=4$ gives $C=1/8,H=J=1$, hence $F=1/2$.
With $s=0.8,p=80,l=5$, both Gaussian components equal $e^{-1/2}$, so
$F=e^{-1/3}\approx0.716531$. Raising TPSA above 90 or lowering logP below 4
cannot further improve its saturated component.

**Code below, shapes, and invariants.** The cell takes three finite scalar
descriptors and returns three scalar components plus a scalar score. It imports
only `math`; no fingerprint, molecule, or oracle is computed. The hand-derived
cases and saturation directions guard against accidentally reversing a modifier.

**Differences from released implementations.** We retain the actual TDC
evaluator and normalization. The opt-in adapter calls its singleton-list API
and requires exactly one finite real result, rejecting booleans. The installed
scalar API can silently replace evaluator exceptions by zero; that failure
must stop this study rather than masquerade as poor chemistry. The untouched
legacy runner retains its scalar path. Fail-fast scoring changes error handling,
not this objective's formula or a successful valid molecule's score.

**Comprehension checkpoint.** Why is the first score 0.5 rather than the
arithmetic mean 0.7083? Expected reasoning: the task uses a cube root of the
product. Which direction of TPSA is rewarded up to the target? Expected
reasoning: increasing it toward 90 removes a lower-side penalty. Is a zero
score always an error? Expected reasoning: no; distinguish a legitimate zero
from an exception that a library silently converted to zero. Does a high score
establish therapeutic benefit? Expected reasoning: no; this is a specified
computational benchmark objective.


In [ ]:
from math import exp as stage28_exp, isfinite as stage28_isfinite, isclose as stage28_isclose

def stage28_descriptor_score(similarity, tpsa, logp):
    assert all(stage28_isfinite(value) for value in (similarity, tpsa, logp))
    assert 0 <= similarity <= 1 and tpsa >= 0
    components = (min(1.0, similarity / 0.8),
        stage28_exp(-0.5 * (max(90 - tpsa, 0) / 10) ** 2),
        stage28_exp(-0.5 * max(logp - 4, 0) ** 2))
    score = (components[0] * components[1] * components[2]) ** (1 / 3)
    assert 0 <= score <= 1
    return components, score

stage28_half_components, stage28_half_score = stage28_descriptor_score(0.1, 90, 4)
assert stage28_half_components == (0.125, 1.0, 1.0)
assert stage28_isclose(stage28_half_score, 0.5, abs_tol=1e-15)
assert stage28_isclose(stage28_descriptor_score(0.8, 80, 5)[1], stage28_exp(-1 / 3), abs_tol=1e-15)
assert stage28_descriptor_score(0.8, 100, 3)[1] == 1
assert stage28_descriptor_score(0, 90, 4)[1] == 0
print("Invented descriptor-only score:", stage28_half_score)


## 28.3 A proposal is not an oracle call, and a final score is not AUC

**Paper correspondence.** PMO evaluates efficient goal-directed search under
a limited oracle-call budget. Our saved curves use the released top-10 AUC
convention. The production definitions are in
`scripts/exps/pmo/main/genmol/experiment_io.py`; V14 freezes its grid and budget.

**Intuition and motivation.** Count new canonical molecules whose property
score was actually obtained. Invalid proposals and cache hits do not consume
another oracle call; offline scores in the starting fragment vocabulary are
separate. A repeated cached child can still update the released fragment
population. AUC rewards finding good molecules earlier, even when two searches
end with the same collection. Rejected proposals may consume neural compute.

**Mathematics, with every symbol defined.** Let $q_1,\ldots,q_B$ be successful
new canonical molecule scores in charging order, with budget $B$. For $c>0$
charged calls, let $k_c=\min(10,c)$ and $m(c)$ be the sum of the $k_c$ highest
scores among $q_1,\ldots,q_c$, divided by $k_c$; define $m(0)=0$.
For increasing reporting counts $0=c_0<c_1<\cdots<c_n=B$, the normalized
trapezoidal score is

$$\mathrm{AUC}_{10}=\frac1B\sum_{i=1}^{n}
(c_i-c_{i-1})\frac{m(c_{i-1})+m(c_i)}2.$$

Here $i$ indexes grid intervals and $n$ is their count. V14 uses $B=2000$
and grid spacing 100. For an incomplete run, an optionally padded runner AUC
is not evidence of completing $B$ calls; retain the observed endpoint and status.
Separately, iteration $r$ is zero-based and warmup length parameter $W=1000$
uses the released condition $r>W$: remasking first occurs at $r=1001$.
Thus iterations 0 through 1000 are attaching-only, at most 1001 new calls;
duplicates can make the actual number smaller. Warmup requires zero generation
NFE (backbone forward evaluations), even though the checkpoint is loaded.

**Small concrete example.** Fictional already-canonical IDs `A,A,None,B,C,D,E`
produce four charged scores $1/5,4/5,1/2,9/10$ under a toy budget 4. The
second A is cached, None is invalid, and E is beyond budget. With spacing 2,
the curve is $(0,0),(2,1/2),(4,3/5)$ and AUC is exactly $2/5$.
Reordering the same scores to find $9/10,4/5$ first gives $23/40$ while the
terminal top-10 mean remains $3/5$. These are chosen scores, not oracle outputs.

**Code below, shapes, and invariants.** `Fraction` supplies exact arithmetic.
The cache maps invented canonical identifiers to scalar scores; the charging
axis is a length-four list, separate from the length-seven proposal list.
The toy does not canonicalize SMILES or fragment molecules. The generic top-10
function uses only available scores when fewer than ten exist and keeps the
zero origin. Warmup boundary assertions prevent an unnoticed off-by-one change.

**Differences from released implementations.** The adapter leaves canonical
caching, released duplicate updates and the `r>1000` warmup boundary intact.
It adds checkpoint/source bindings, observed NFE, strict failures and disabled
resume. Same-seed attaching-only warmup must be audited across arms: selected
fragments, molecules, scores and population evolution should agree. An accepted,
charged post-warmup remasking event with positive observed NFE is needed before
calling the run a diffusion comparison; compute spent only on rejected
proposals is insufficient. The toy omits the real population mechanics on purpose.

**Comprehension checkpoint.** Why does the second A not advance the call axis?
Expected reasoning: its canonical score is already cached. Can it still affect
the released population? Expected reasoning: yes, released duplicate updates
remain enabled. Why do identical final scores give different AUCs? Expected
reasoning: intermediate best-score means depend on discovery order. Does
`warmup=1000` mean exactly 1000 oracle calls? Expected reasoning: no, the
condition is an iteration boundary with 1001 attaching-only iterations and
possible duplicates. Does a completed warmup-only run test the diffusion law?
Expected reasoning: no, there must be accepted charged neural mutations.


In [ ]:
from fractions import Fraction as Stage28Fraction

stage28_lookup = dict(A=Stage28Fraction(1, 5), B=Stage28Fraction(4, 5),
    C=Stage28Fraction(1, 2), D=Stage28Fraction(9, 10), E=Stage28Fraction(1))
stage28_cache, stage28_charged, stage28_reasons = {}, [], []
for canonical in ("A", "A", None, "B", "C", "D", "E"):
    if canonical is None:
        reason = "invalid"
    elif canonical in stage28_cache:
        reason = "cache_hit"
    elif len(stage28_charged) == 4:
        reason = "budget_exhausted"
    else:
        stage28_cache[canonical] = stage28_lookup[canonical]
        stage28_charged.append(stage28_cache[canonical])
        reason = "charged"
    stage28_reasons.append(reason)
assert stage28_reasons == ["charged", "cache_hit", "invalid", "charged", "charged", "charged", "budget_exhausted"]

def stage28_top10(values):
    assert values
    selected = sorted(values, reverse=True)[:10]
    return sum(selected, Stage28Fraction(0)) / len(selected)

def stage28_exact_auc(scores, spacing):
    budget = len(scores)
    assert budget and type(spacing) is int and spacing > 0
    calls = sorted(set([0, *range(spacing, budget + 1, spacing), budget]))
    curve = [(c, stage28_top10(scores[:c]) if c else Stage28Fraction(0)) for c in calls]
    area = sum((right[0] - left[0]) * (left[1] + right[1]) / 2
        for left, right in zip(curve, curve[1:]))
    return curve, area / budget

stage28_curve, stage28_auc = stage28_exact_auc(stage28_charged, 2)
assert stage28_curve == [(0, Stage28Fraction(0)), (2, Stage28Fraction(1, 2)), (4, Stage28Fraction(3, 5))]
assert stage28_auc == Stage28Fraction(2, 5)
stage28_early = [stage28_charged[i] for i in (3, 1, 2, 0)]
assert stage28_exact_auc(stage28_early, 2)[1] == Stage28Fraction(23, 40)
assert stage28_top10(stage28_early) == stage28_top10(stage28_charged) == Stage28Fraction(3, 5)
stage28_remask_enabled = [iteration > 1000 for iteration in (999, 1000, 1001)]
assert stage28_remask_enabled == [False, False, True]
print("Invented exact AUC and terminal mean:", stage28_auc, stage28_top10(stage28_charged))


## 28.4 The prospective comparison and the evidence needed to interpret it

**Paper correspondence.** The paper's full PMO benchmark covers 23 tasks,
10,000 calls per run and three runs. V14 is a one-task, two-seed, 2,000-call
engineering pilot; it cannot reproduce a full-paper sum or establish that UDLM
beats GenMol. Its committed design is
`experiments/udlm/designs/engineering_v14_pmo_pilot.md` and executable panel is
`experiments/udlm/protocols/engineering_v14_pmo.json`.

**Intuition and motivation.** Fix the online oracle budget, initial scored
fragment vocabulary and released population policy, then compare three actual
checkpoint/sampler combinations. The selected candidates use earlier de novo
observations; these are prospective PMO evaluations, not independent selection
confirmation. No outcome table is embedded or read by this stage.

**Mathematics, with every symbol defined.** With arm index $a$, seed index $s$,
and measured full-budget AUC $A_{a,s}$, a paired treatment difference is
$d_s=A_{\mathrm{treatment},s}-A_{\mathrm{MDLM},s}$. The two-seed mean is
$\bar d=(d_{2300}+d_{2301})/2$ and sample SD is
$\sqrt{\sum_s(d_s-\bar d)^2/(2-1)}$. Neither statistic is a superiority test
with two adaptive engineering seeds. Primary treatment is MASK CE; secondary
is S CT. Each comparison requires both declared complete pairs and successful
integrity/warmup/charged-mutation checks, with failures and unlaunched runs shown.

**Small concrete example.** Three arms times two seeds gives six planned runs;
six budgets of 2000 permit at most 12,000 online oracle calls. This count is a
ceiling, not an observed exposure. A failure can leave fewer calls and stop
later waves. A hypothetical pair of differences $+0.02,-0.01$ has mean $+0.005$
but disagrees in sign; it fails V14's declared both-seeds-positive gate.

**Code below, shapes, and invariants.** The cell contains only the three frozen
checkpoint identifiers and small dictionaries. The two-element seed tuple and
three-element arm list describe plans, not results. It asserts complete hashes,
fixed controls and requested budgets, and computes the invented contrast with
`Fraction`. It performs no file reads, checkpoint loads, oracle calls or launches.

**Differences from released implementations.** MDLM uses EMA from 50k training,
temperature 1.2 and confidence randomness 2, with adaptive observed NFE. S CT
uses 1000 additional batch-16 updates, a schedule-consistent uniform prior and
128 predictor NFE. MASK CE uses 1000 batch-128 updates, clean-token CE and its
verified 0.9 MASK / 0.1 empirical mixture prior. CE first converts clean
probabilities by the local likelihood, then V14 applies temperature in raw-LOO
space. Both UDLM arms use temperature 0.5, top-p one, endpoint 1e-5, no Gibbs,
full active support and ignored randomness zero. The trained prior is never
swapped at inference. All arms preserve gamma zero, effective completion
length 18, population size 100, seeds 2300/2301 and the fixed scored vocabulary.
These training amounts, prior/loss choices and neural compute differ; the
experiment compares complete sampler configurations, not one isolated cause.

The controller enforces at most two freshly qualified GPU UUIDs, frozen source,
bounded child time, explicit process mapping and preserved terminal failures.
`PYTHONHASHSEED=0` and CPU thread limits one are common across arms. Scoring
errors propagate; actual generation calls, NFE, cache charges and runtime must
be saved. Shared-GPU timing does not establish a controlled speed advantage.
Independent score replay and a PDF with all arms, both signed contrasts and
paper-comparison caveats are required before interpretation. Final de novo
seeds 0/1/2 remain reserved. Nothing here changes the active campaign.

**Comprehension checkpoint.** Does equal oracle budget imply equal NFE or equal
training? Expected reasoning: no; those are separately recorded budgets.
Can one retain the better seed and omit the other? Expected reasoning: no,
both predeclared pairs and all failures remain visible. Does a positive mean
for the invented contrast pass the gate? Expected reasoning: no, one seed is
negative. Would passing the engineering gate complete the overall project
goal? Expected reasoning: no; a later independent, broader benchmark is needed.


In [ ]:
from fractions import Fraction as Stage28PanelFraction

stage28_planned_seeds = (2300, 2301)
stage28_planned_arms = (
    dict(name="MDLM", checkpoint_sha256="8d00aa47b02f64bf39ff6b0b2e786f213587366fc2c3d29712a00f3f84108dd6",
         temperature=1.2, randomness=2.0, prior="absorbing MASK", nfe=None),
    dict(name="S CT", checkpoint_sha256="100f467b94766f2c87cc734398c8590bd14a1c8e0722ce31dbca7b446d986e72",
         temperature=0.5, randomness=0.0, prior="schedule_uniform", nfe=128),
    dict(name="MASK CE", checkpoint_sha256="62299de99d8c003e3776215efce9643cca196091a6284f351de2b404a22c2e8d",
         temperature=0.5, randomness=0.0, prior="mask_rich_empirical", nfe=128),
)
assert all(len(arm["checkpoint_sha256"]) == 64 for arm in stage28_planned_arms)
stage28_requested_runs = len(stage28_planned_arms) * len(stage28_planned_seeds)
stage28_requested_call_ceiling = stage28_requested_runs * 2000
assert (stage28_requested_runs, stage28_requested_call_ceiling) == (6, 12000)
stage28_invented_differences = (Stage28PanelFraction(2, 100), Stage28PanelFraction(-1, 100))
stage28_invented_mean = sum(stage28_invented_differences) / 2
assert stage28_invented_mean == Stage28PanelFraction(1, 200)
assert not all(value > 0 for value in stage28_invented_differences)
print("Planned runs / online-call ceiling, not outcomes:", stage28_requested_runs, stage28_requested_call_ceiling)


# Stage 29 — A proposed context guidance rule in reverse-probability space

## 29.1 Which tokens count as context?

**Paper correspondence.** [GenMol Section 4.3](https://arxiv.org/html/2501.06158v3#S4.SS3)
uses molecular context guidance: compare predictions with more and less visible
context. [Discrete classifier-free guidance, Section 3.1](https://arxiv.org/html/2412.10193v3#S3.SS1)
instead motivates combining reverse probabilities. This chapter teaches the
[reviewed adaptation memo](https://github.com/aamenov/genmol-udlm/blob/a5026e91a1a8febe95e70f1de66f35b817a4cda5/docs/udlm_context_guidance_hypothesis.md),
using invented arrays and exact arithmetic. It invokes no production guidance
API and makes no molecular efficacy claim or change to the fixed V14 panel.

**Intuition and motivation.** Hide some original context from a second view of
the same noisy sequence. Keep the editable noisy observation identical in both
views, so their difference reflects visible context. A token filled during
denoising remains editable; it must not silently become fixed context.

**Mathematics, with every symbol defined.** Let $x_{\rm init}$ and $x_t$ be
original and current token arrays of shape $[B,L]$, with $B$ rows and $L$
positions. Indices $b$ and $\ell$ select a row and position. The fixed Boolean
mask $e_{b\ell}$ marks original editable positions. Let $S$ be all tokenizer
control IDs, including BOS/EOS/PAD/MASK/UNK. Define
$C_b=\{\ell:e_{b\ell}=0,\ x_{{\rm init},b\ell}\notin S\}$.
For masking fraction $\gamma\in[0,1]$, select a subset
$H_b\subseteq C_b$ of size $\lfloor\gamma|C_b|\rfloor$ and replace only those
positions by MASK in $x_t^{\rm poor}$. Thus
$x_t^{\rm poor}[e]=x_t[e]$. The original attention mask remains fixed.
After any proposed update, clamp $x_s[\neg e]=x_{\rm init}[\neg e]$,
including context hidden from the poor predictor. Here $s<t$ is the next time.

**Small concrete example.** The two rows below have editable positions 2 and 1,
respectively. Both have two eligible context positions, so $\gamma=1/2$ hides
one per row. We choose those subsets by hand; no random draw or model is used.
The current value 8 at row-zero position 2 is still editable.

**Code below, shapes, and invariants.** Nested tuples represent `[B,L]=[2,6]`.
We construct the per-row eligible positions and degraded view, check that
editable observations are unchanged, then clamp a hand-supplied update. The
printed arrays are artificial token IDs, not generated molecules.

**Differences from released implementations.** The
[released GenMol sampler](https://github.com/NVIDIA-BioNeMo/genmol/blob/add09fc83b7255bd09c797e527c0f4b51f5fb7c1/src/genmol/sampler.py#L64)
recomputes eligibility from current row-zero tokens, excluding BOS/EOS/MASK/PAD,
and shares chosen positions across rows. Our proposed eligibility is original,
per-row and excludes every declared control. Excluding controls from context
selection does not remove them from the checkpoint's corruption alphabet.
An immutable token invariant does not prove graph-level fragment preservation
after SAFE decoding or repair.

**Comprehension checkpoint.** Why must position 2 in row zero stay out of
$C_0$ after receiving token 8? Expected reasoning: editability is an original
coordinate property, not the current token value. Why restore hidden context
after sampling? Expected reasoning: hiding changes a predictor's view, not the
permitted output positions. Are the two rows allowed different $H_b$?
Expected reasoning: yes; each uses its own original eligible coordinates.


In [ ]:
stage29_initial = ((1, 5, 4, 7, 2, 3), (1, 4, 6, 8, 2, 3))
stage29_current = ((1, 5, 8, 7, 2, 3), (1, 5, 6, 8, 2, 3))
stage29_controls = {0, 1, 2, 3, 4}
stage29_editable = tuple(tuple(token == 4 for token in row) for row in stage29_initial)
stage29_eligible = tuple(tuple(i for i, (token, edit) in enumerate(zip(row, mask))
    if not edit and token not in stage29_controls)
    for row, mask in zip(stage29_initial, stage29_editable))
stage29_hidden = tuple(positions[:len(positions) // 2] for positions in stage29_eligible)
stage29_poor = tuple(tuple(4 if i in hidden else token for i, token in enumerate(row))
    for row, hidden in zip(stage29_current, stage29_hidden))
assert stage29_eligible == ((1, 3), (2, 3))
assert stage29_hidden == ((1,), (2,))
assert all(stage29_poor[b][i] == stage29_current[b][i]
    for b, row in enumerate(stage29_editable) for i, edit in enumerate(row) if edit)
stage29_release_row0_eligible = tuple(i for i, token in enumerate(stage29_current[0])
    if token not in {1, 2, 3, 4})
assert 2 in stage29_release_row0_eligible and 2 not in stage29_eligible[0]
stage29_proposed_ids = ((9, 9, 6, 9, 9, 9), (9, 8, 9, 9, 9, 9))
stage29_clamped = tuple(tuple(proposed if edit else original
    for proposed, edit, original in zip(proposal, mask, row))
    for proposal, mask, row in zip(stage29_proposed_ids, stage29_editable, stage29_initial))
assert all(stage29_clamped[b][i] == stage29_initial[b][i]
    for b, row in enumerate(stage29_editable) for i, edit in enumerate(row) if not edit)
print("Artificial context-degraded views:", stage29_poor)
print("Artificial update after original-context clamp:", stage29_clamped)


## 29.2 Convert each branch using the same noisy observation

**Paper correspondence.** The rank-one reverse conditional in the
[discrete-guidance paper, Section 2.1](https://arxiv.org/html/2412.10193v3#S2.SS1)
provides the bridge. Our CE-to-LOO conversion uses that bridge for the fixed
stationary prior. The two context views are a proposed molecular adaptation;
they are not the paper's learned class-label/null-label pair.

**Intuition and motivation.** A clean-token prediction answers which original
token is plausible. A reverse transition answers which token is plausible at
an earlier noise time. These are different distributions. Convert each
branch first, keeping its original editable observation and process fixed.

**Mathematics, with every symbol defined.** At one editable coordinate, let
$k$ be its current ID, $j$ a possible clean ID, $i$ an earlier ID, and $K$ the
active alphabet size. Indices $h$ below sum over that alphabet. The stationary
prior satisfies $\pi_i>0$ and $\sum_i\pi_i=1$. Retained-signal probabilities
satisfy $0<\alpha_t<\alpha_s<1$ for distinct interior times $s<t$.
Set $r=\alpha_t/\alpha_s$,
$L_j=\alpha_t\mathbf1[j=k]+(1-\alpha_t)\pi_k$, and
$A_i=r\mathbf1[i=k]+(1-r)\pi_k$; $\mathbf1$ is the indicator function.
For branch $v\in\{c,u\}$, $D_v(j)$ is a normalized clean CE prediction;
$c$ means full context and $u$ means degraded context. Define

$$R_v(j)=\frac{D_v(j)/L_j}{\sum_h D_v(h)/L_h},\qquad
P_v(i)=\frac{A_i[\alpha_s R_v(i)+(1-\alpha_s)\pi_i]}
{\sum_h A_h[\alpha_s R_v(h)+(1-\alpha_s)\pi_h]}.$$

A raw-LOO checkpoint already supplies $R_v$ after normalization; do not divide
it by $L_j$ again. Both branches use the same original $k$, times and prior.
For CE predictions the equivalent direct mixture is

$$P_v(i)=\sum_j D_v(j)\,
\frac{A_i[\alpha_s\mathbf1[i=j]+(1-\alpha_s)\pi_i]}{L_j}.$$

This coordinate-wise identity does not make independent token draws an exact
correlated sequence posterior.

**Small concrete example.** Use $K=3$, $\pi=(1/5,1/2,3/10)$,
$\alpha_s=2/3$, $\alpha_t=1/3$, $k=0$, and synthetic clean distributions
$D_c=(3/5,3/10,1/10)$, $D_u=(1/5,1/5,3/5)$. The resulting reverse laws are
$P_c=(24/35,31/140,13/140)$ and $P_u=(3/7,29/140,51/140)$.

**Code below, shapes, and invariants.** Each tuple has shape `[K]=[3]`;
the conceptual model output would have shape `[B,L,K]`. Two independent
expressions compute the same reverse vector using exact `Fraction` arithmetic.
Each vector is positive and sums exactly to one. No model logits are loaded.

**Differences from released implementations.** GenMol's MDLM guidance mixes
clean logits before its decoding step. Official UDLM guidance computes two
reverse laws from the same noisy input/time before combination. Here only the
original immutable context differs between inputs; a degraded input is not
demonstrated to be a true unconditional predictor.

**Comprehension checkpoint.** Which observation belongs in the poor branch's
$L_j$? Expected reasoning: the original editable $k$, which context degradation
must not change. Can a raw-LOO checkpoint reuse the CE division? Expected
reasoning: no, that would apply the likelihood correction twice. Why keep
$\pi$ and the time grid shared? Expected reasoning: changing either would
confound guidance with a different reverse process.


In [ ]:
from fractions import Fraction as Stage29ReverseFraction

def stage29_reverse(denoiser, prior, observed, alpha_s, alpha_t):
    ratio = alpha_t / alpha_s
    likelihood = tuple(alpha_t * (j == observed) + (1 - alpha_t) * prior[observed]
        for j in range(len(prior)))
    weights = tuple(d / likelihood[j] for j, d in enumerate(denoiser))
    loo = tuple(value / sum(weights) for value in weights)
    a = tuple(ratio * (i == observed) + (1 - ratio) * prior[observed]
        for i in range(len(prior)))
    bridge = tuple(a[i] * (alpha_s * loo[i] + (1 - alpha_s) * prior[i])
        for i in range(len(prior)))
    bridge = tuple(value / sum(bridge) for value in bridge)
    direct = tuple(sum(denoiser[j] * a[i]
        * (alpha_s * (i == j) + (1 - alpha_s) * prior[i]) / likelihood[j]
        for j in range(len(prior))) for i in range(len(prior)))
    assert bridge == direct and sum(bridge) == 1 and min(bridge) > 0
    return loo, bridge

stage29_pi = tuple(Stage29ReverseFraction(v) for v in ("1/5", "1/2", "3/10"))
stage29_dc = tuple(Stage29ReverseFraction(v) for v in ("3/5", "3/10", "1/10"))
stage29_du = tuple(Stage29ReverseFraction(v) for v in ("1/5", "1/5", "3/5"))
stage29_rc, stage29_pc = stage29_reverse(stage29_dc, stage29_pi, 0,
    Stage29ReverseFraction(2, 3), Stage29ReverseFraction(1, 3))
stage29_ru, stage29_pu = stage29_reverse(stage29_du, stage29_pi, 0,
    Stage29ReverseFraction(2, 3), Stage29ReverseFraction(1, 3))
assert stage29_pc == tuple(Stage29ReverseFraction(v) for v in ("24/35", "31/140", "13/140"))
assert stage29_pu == tuple(Stage29ReverseFraction(v) for v in ("3/7", "29/140", "51/140"))
print("Exact synthetic reverse laws:", stage29_pc, stage29_pu)


## 29.3 Combine reverse probabilities, then sample

**Paper correspondence.** Discrete classifier-free guidance motivates a
geometric combination of conditional and unconditional reverse laws. Our
adaptation replaces the latter with a context-degraded law. It borrows the
combination order; it does not reproduce the official null-label training.

**Intuition and motivation.** Prefer tokens that become more plausible with
the available context. A large probability alone is not enough: the relative
change between full and degraded context matters. No property oracle or
property gradient enters this proposed rule.

**Mathematics, with every symbol defined.** For positive normalized reverse
laws $P_c$ and $P_u$ over tokens $i$, use guidance scale $w\ge1$ and define

$$P_g(i)=\frac{P_c(i)^w P_u(i)^{1-w}}
{\sum_h P_c(h)^w P_u(h)^{1-w}}
=\operatorname{softmax}_i[w\log P_c(i)+(1-w)\log P_u(i)].$$

$P_g$ is the guided proposal, $h$ is a token summation index, and softmax
normalizes exponentiated log weights. For two tokens $i,j$, the guided odds
equal $P_c(i)/P_c(j)$ times
$[(P_c(i)/P_c(j))/(P_u(i)/P_u(j))]^{w-1}$.
Scale $w=1$ returns $P_c$. Equal branches also return $P_c$ for any admissible
$w$. Algebraically $w=0$ returns $P_u$, but zero is outside this proposed
extrapolation range and does not redefine the released CLI's zero-scale bypass.
The symbol $\gamma$ here is the context-masking fraction, not $w$; official
UDLM code uses its `guidance.gamma` name for the extrapolation coefficient.

**Small concrete example.** Reuse the preceding three-category tables at
$w=2$. Posterior combination yields
$(141984/175679,245055/1405432,24505/1405432)$.
Combining clean weights as $D_c^2/D_u$ and only then applying the bridge gives
$(1929/2380,73/476,43/1190)$ instead. Squaring $P_c$ alone also differs.
The nonlinear bridge and guidance operation therefore do not commute.

**Code below, shapes, and invariants.** This independent cell uses rational
three-entry tables, computes both operation orders, and checks exact fractions,
normalization and the identity laws. Its small bridge sums over all three
possible clean IDs. These tables are invented probabilities, not experimental
outcomes or predictions along an actual molecular trajectory.

**Differences from released implementations.** Clean-logit extrapolation,
reverse-law extrapolation and ordinary temperature are distinct controls.
The official UDLM conditional/unconditional pair is trained with label dropout;
masking molecular context supplies no such guarantee. A future benchmark must
explicitly declare this new law and include the appropriate unguided controls.

**Comprehension checkpoint.** Why does the negative exponent at $w>1$ matter?
Expected reasoning: it divides by the poor law, so mismatched zeros are unsafe.
Does $w=2$ mean temperature $1/2$? Expected reasoning: no, temperature lacks
the poor-branch denominator. Does this exact counterexample prove better
molecules? Expected reasoning: no; it establishes different categorical laws.


In [ ]:
from fractions import Fraction as Stage29BlendFraction

def stage29_normalize(values):
    values = tuple(values)
    return tuple(value / sum(values) for value in values)

def stage29_guide(conditional, poor, weight):
    assert min(conditional + poor) > 0
    return stage29_normalize(c ** weight * u ** (1 - weight)
        for c, u in zip(conditional, poor))

def stage29_example_bridge(denoiser):
    f = Stage29BlendFraction
    pi, a_s, a_t, k = (f(1, 5), f(1, 2), f(3, 10)), f(2, 3), f(1, 3), 0
    return tuple(sum(denoiser[j]
        * (a_t / a_s * (i == k) + (1 - a_t / a_s) * pi[k])
        * (a_s * (i == j) + (1 - a_s) * pi[i])
        / (a_t * (j == k) + (1 - a_t) * pi[k])
        for j in range(3)) for i in range(3))

stage29_c = tuple(Stage29BlendFraction(v) for v in ("3/5", "3/10", "1/10"))
stage29_u = tuple(Stage29BlendFraction(v) for v in ("1/5", "1/5", "3/5"))
stage29_conditional, stage29_degraded = stage29_example_bridge(stage29_c), stage29_example_bridge(stage29_u)
stage29_guided = stage29_guide(stage29_conditional, stage29_degraded, 2)
stage29_clean_first = stage29_example_bridge(stage29_guide(stage29_c, stage29_u, 2))
stage29_temperature_only = stage29_normalize(value ** 2 for value in stage29_conditional)
assert stage29_guided == tuple(Stage29BlendFraction(v)
    for v in ("141984/175679", "245055/1405432", "24505/1405432"))
assert stage29_clean_first == tuple(Stage29BlendFraction(v)
    for v in ("1929/2380", "73/476", "43/1190"))
assert stage29_guided != stage29_clean_first and stage29_guided != stage29_temperature_only
assert stage29_guide(stage29_conditional, stage29_degraded, 1) == stage29_conditional
assert stage29_guide(stage29_conditional, stage29_conditional, 3) == stage29_conditional
assert sum(stage29_guided) == sum(stage29_clean_first) == 1
print("Posterior guidance / clean-first guidance:", stage29_guided, stage29_clean_first)


## 29.4 Identity paths, actual work and finite precision

**Paper correspondence.** Turning a guidance equation into a sampler requires
explicit numerical and compute conventions. These are proposed engineering
requirements around the reviewed reverse-law rule, not new molecular results
from either paper. The existing studies and final-seed reservations remain
unchanged; this chapter authorizes no training or generation.

**Intuition and motivation.** An identity setting should reproduce the old
sampler without consuming extra randomness. Batching two predictor views may
reduce invocation overhead but still evaluates both views. Finally, a proof of
positive probabilities does not guarantee their floating-point representation.

**Mathematics, with every symbol defined.** Let $N$ be predictor transitions,
$B$ candidates, $J$ actual backbone invocations and $m_j$ the batch size of
invocation $j$. Candidate-equivalent evaluations are $E=\sum_{j=1}^J m_j$.
Active serial guidance has $J=2N$, $m_j=B$; packed guidance has $J=N$,
$m_j=2B$. Both cost $E=2BN$, or $2N$ evaluations per candidate. An identity
path costs $BN$. With $B=4,N=128$, active guidance costs 1,024 candidate
evaluations and the identity path 512. Invocation count alone is insufficient.

If $\gamma=0$, $w=1$, or every selected subset is empty, an implementation must
bypass the poor prediction **and context RNG draws**. It must preserve the old
sample IDs and caller RNG state for the same sampling seed. Identical laws found
after two predictions do not erase the work already performed. Active context
subsets should have a dedicated recorded generator, independent of categorical
sampling, with per-row/per-step subset policy explicitly declared.

On the interior grid, $A_i>0$ and $(1-\alpha_s)\pi_i>0$, so both reverse laws
are mathematically positive, even when some clean weights vanish. For finite
log weights $z_i$, log normalization computes
$\log P_i=z_i-m-\log\sum_h\exp(z_h-m)$ with $m=\max_i z_i$.
An exponential can still underflow to zero. A prospective implementation must
declare precision and reject nonfinite arithmetic or lost support; silently
flooring or clipping would define another law. Zero-support priors, truncation,
identical times and the exact clean endpoint are outside this positivity proof.

**Small concrete example.** The code compares serial, packed and identity work
for four candidates and 128 steps. A separate two-category example has finite
normalized logs but a zero represented exponential. It illustrates arithmetic,
without constructing a sampler, drawing randomness or calling a device.

**Code below, shapes, and invariants.** A list of hypothetical batch sizes has
length $J$; its sum is $E$. The two log-weight tuples have shape `[K]=[2]`.
`Fraction` checks the exact masking fraction, while `math` shows finite-precision
underflow. These are planned-work counts, not measured runtime or NFE receipts.

**Differences from released implementations.** A counter that records only
forward invocations undercounts packed guidance. Serial and packed neural
execution also need not yield bitwise-identical logits or sample IDs. Poor
context may be outside the checkpoint's useful training distribution, especially
when MASK had very low prior mass. Large $w$ can amplify calibration error.
Neither full support nor this coordinate-wise heuristic proves a globally
tilted molecular distribution, chemical constraint success or better quality.
Using the separate, independently reviewed experimental prototype requires its
explicit state and support checks; molecular efficacy still needs a prospective
benchmark. This chapter invokes no production API.

**Comprehension checkpoint.** Does packing halve the candidate evaluations?
Expected reasoning: no; batch size doubles. May an identity path draw an unused
context subset? Expected reasoning: no, that changes the RNG trajectory. Does a
finite log probability guarantee a positive materialized float? Expected
reasoning: no; exponentiation can underflow. Can the toy select a guidance scale
for molecules? Expected reasoning: no; it contains no molecular evidence.


In [ ]:
from fractions import Fraction as Stage29ComputeFraction
from math import exp as stage29_exp, isfinite as stage29_isfinite, log as stage29_log

def stage29_work(batch, steps, execution, *, gamma, weight, eligible_per_row):
    subset = tuple(int(gamma * count) for count in eligible_per_row)
    identity = gamma == 0 or weight == 1 or not any(subset)
    sizes = [batch] * steps if identity else ([batch] * (2 * steps)
        if execution == "serial" else [2 * batch] * steps)
    return {"forward_invocations": len(sizes), "batch_sizes": sizes,
        "candidate_evaluations": sum(sizes),
        "context_subset_draws": 0 if identity else steps * sum(count > 0 for count in subset)}

stage29_half = Stage29ComputeFraction(1, 2)
stage29_serial = stage29_work(4, 128, "serial", gamma=stage29_half, weight=2, eligible_per_row=(4,) * 4)
stage29_packed = stage29_work(4, 128, "packed", gamma=stage29_half, weight=2, eligible_per_row=(4,) * 4)
assert (stage29_serial["forward_invocations"], stage29_packed["forward_invocations"]) == (256, 128)
assert stage29_serial["candidate_evaluations"] == stage29_packed["candidate_evaluations"] == 1024
stage29_identity_ledgers = [stage29_work(4, 128, "serial", gamma=gamma, weight=weight,
    eligible_per_row=eligible) for gamma, weight, eligible in
    ((0, 2, (4,) * 4), (stage29_half, 1, (4,) * 4), (stage29_half, 2, (1,) * 4))]
assert all(row["candidate_evaluations"] == 512 and row["context_subset_draws"] == 0
    for row in stage29_identity_ledgers)
stage29_log_c, stage29_log_u = (0.0, -1000.0), (0.0, 0.0)
stage29_log_scores = tuple(2 * c - u for c, u in zip(stage29_log_c, stage29_log_u))
stage29_maximum = max(stage29_log_scores)
stage29_centered = tuple(value - stage29_maximum for value in stage29_log_scores)
stage29_log_total = stage29_log(sum(stage29_exp(value) for value in stage29_centered))
stage29_log_probabilities = tuple(value - stage29_log_total for value in stage29_centered)
assert all(stage29_isfinite(value) for value in stage29_log_probabilities)
stage29_float_probabilities = tuple(stage29_exp(value) for value in stage29_log_probabilities)
assert stage29_float_probabilities[1] == 0.0
print("Planned serial / packed candidate work:", stage29_serial["candidate_evaluations"], stage29_packed["candidate_evaluations"])
print("Finite log weights can underflow on exponentiation:", stage29_log_probabilities, stage29_float_probabilities)


# Final reporting stage - Automatic PDF benchmark report

## Why this stage exists

After Stages 0-18 and the UDLM Stage 20 checks have run, the final code cell
writes `output/pdf/genmol_benchmark_report.pdf`. Running report collection last
means the Stage 20 evidence and its reproduction-ledger row are present. The
benchmark suites remain the primary content:

- training provenance and checkpoint scale;
- de novo generation, including strict and repaired metrics;
- fragment-constrained generation;
- PMO, lead optimization, and real docking; and
- published MDLM ablations plus the bounded UDLM-prior comparison.

## Read-only reporting rule

This cell summarizes an explicit whitelist of results produced by earlier
cells. It never launches training, generation, an oracle, docking, a network
call, or new GPU work. Generating a report must not spend another oracle budget
or create a different stochastic sample.

## Status semantics

- `EXECUTED - PAPER SCALE` means the complete benchmark result object exists at
  the required scale.
- `EXECUTED - NUMERICAL ORACLE` or `EXECUTED - BOUNDED SMOKE TEST` means only a
  mechanism was checked.
- `NOT EXECUTED - DISABLED` means the full benchmark guard was false or absent.
- `INCOMPLETE - EXPECTED RESULT MISSING` means a benchmark was enabled without
  its required result object.
- `FAILED - INVARIANT VIOLATION` means a scientific check failed.

`None` is different from a measured score of zero.

## Benchmark result contracts

The PDF expects dedicated paper-scale summaries for training, de novo,
fragment-constrained, PMO, lead, docking, and published-ablation results. Rows
must retain run IDs, seeds, denominators, checkpoint identity, metric
definitions, and failures. The Stage 20 record instead contains bounded native
equation cross-checks, a revisability trace, validated CPU-artifact provenance,
the committed E-L1/E-A1 denoising selections, and explicit final success
thresholds; it cannot fill a paper-scale row.

## Relation to the paper and released code

The paper defines benchmark scale and scientific metrics. NVIDIA's release
resolves GenMol implementation details, and official UDLM revision
`edb0f8c28b7caeb4ea7a06a2fee8d74ab6da1661` anchors the uniform-diffusion
equations and architecture comparison. Paper,
released-code, and local bounded results remain separate fields.

## What the code below does

1. Collects Stages 0-18 and 20 plus paper-scale ledgers by explicit name.
2. Validates ordered coverage, status semantics, and finite summaries.
3. Builds a benchmark-first PDF with suite and ablation pages.
4. Reopens and renders every page, checks headings and nonblank pages, and
   deletes only its temporary renders.
5. Prints the absolute benchmark PDF path.

## Comprehension checkpoint

1. Why can a bounded UDLM CPU overfit not fill the 3 x 1,000 de-novo row?
2. Which denominators distinguish strict from repaired validity?
3. Why must a report record the checkpoint and all generation seeds?
4. What status applies when a full-run guard is true but evidence is absent?
5. Why is report generation intentionally last and read-only scientifically?


In [ ]:
from collections.abc import Mapping, Sequence
from datetime import datetime, timezone
from hashlib import sha256
from html import escape as report_escape
from pathlib import Path as ReportPath
import dataclasses as report_dataclasses
import importlib.metadata as report_metadata
import json as report_json
import math as report_math
import numbers as report_numbers
import platform as report_platform
import shutil as report_shutil
import subprocess as report_subprocess

import numpy as report_np
import pandas as report_pd
import torch as report_torch
from PIL import Image as ReportImage
from pypdf import PdfReader
from reportlab.lib import colors as report_colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import (
    KeepTogether,
    PageBreak,
    Paragraph,
    SimpleDocTemplate,
    Spacer,
    Table,
    TableStyle,
)


def collect_report_records(namespace):
    def compact_value(value, depth=0):
        if value is None:
            return "None"
        if isinstance(value, bool):
            return str(value)
        if isinstance(value, report_numbers.Integral):
            return str(int(value))
        if isinstance(value, report_numbers.Real):
            number = float(value)
            return f"{number:.6g}" if report_math.isfinite(number) else ("NaN" if report_math.isnan(number) else ("infinity" if number > 0 else "-infinity"))
        if isinstance(value, str):
            clean = " ".join(value.split())
            if len(clean) <= 180:
                return clean
            digest = sha256(clean.encode("utf-8")).hexdigest()[:12]
            return f"{clean[:160]}... [sha256:{digest}]"
        if report_dataclasses.is_dataclass(value) and not isinstance(value, type):
            return compact_value(report_dataclasses.asdict(value), depth + 1)
        if isinstance(value, Mapping):
            if depth >= 2:
                return f"mapping[{len(value)}]"
            parts = [
                f"{compact_value(key, depth + 1)}={compact_value(item, depth + 1)}"
                for key, item in list(value.items())[:8]
            ]
            suffix = ", ..." if len(value) > 8 else ""
            return "{" + ", ".join(parts) + suffix + "}"
        if isinstance(value, Sequence) and not isinstance(value, (str, bytes, bytearray)):
            if depth >= 2:
                return f"sequence[{len(value)}]"
            parts = [compact_value(item, depth + 1) for item in list(value)[:8]]
            suffix = ", ..." if len(value) > 8 else ""
            return "[" + ", ".join(parts) + suffix + "]"
        return f"<{type(value).__module__}.{type(value).__name__}>"

    def summarize(name, value):
        if value is None:
            return f"{name}: NOT RECORDED"
        if isinstance(value, report_torch.Tensor):
            tensor = value.detach().cpu()
            finite = bool(report_torch.isfinite(tensor).all()) if tensor.numel() else True
            stats = ""
            if tensor.numel() and tensor.dtype.is_floating_point:
                stats = (
                    f", min={tensor.min().item():.5g}, "
                    f"max={tensor.max().item():.5g}, mean={tensor.float().mean().item():.5g}"
                )
            return (
                f"{name}: tensor shape={tuple(tensor.shape)}, dtype={tensor.dtype}, "
                f"source_device={value.device}, requires_grad={value.requires_grad}, finite={finite}{stats}"
            )
        if isinstance(value, report_pd.DataFrame):
            numeric = value.select_dtypes(include="number")
            finite = bool(report_np.isfinite(numeric.to_numpy()).all()) if numeric.size else True
            preview = compact_value(value.head(2).to_dict(orient="records"))
            return (
                f"{name}: dataframe rows={len(value)}, columns={len(value.columns)}, "
                f"finite={finite}, preview={preview}"
            )
        if isinstance(value, report_pd.Series):
            numeric = report_pd.to_numeric(value, errors="coerce")
            finite = bool(report_np.isfinite(numeric.dropna().to_numpy()).all())
            return f"{name}: series length={len(value)}, finite={finite}, head={compact_value(value.head(5).tolist())}"
        return f"{name}: {compact_value(value)}"

    def size_hint(value):
        if value is None:
            return "None"
        if isinstance(value, report_pd.DataFrame):
            return f"{len(value)} rows"
        if isinstance(value, report_torch.Tensor):
            return f"{value.numel()} values"
        if isinstance(value, (Mapping, Sequence)) and not isinstance(value, (str, bytes, bytearray)):
            return f"{len(value)} items"
        return "1 value"

    stage_specs = [
        {
            "id": "stage0.environment",
            "stage": 0,
            "title": "Runtime, source, and GPU selection",
            "kind": "bounded_smoke",
            "evidence_names": ("SELECTED_PHYSICAL_GPU_IDS", "SELECTED_GPU_UUIDS", "versions"),
            "config_names": ("NUM_GPUS", "SEED"),
            "paper_source": "Experimental setup and reproducibility context",
            "repository_source": "README.md; pyproject.toml",
            "notes": "Discovers idle physical GPUs, then exposes exactly the user-selected count.",
            "checkpoint": "Explain physical GPU IDs versus process-local CUDA indices.",
        },
        {
            "id": "stage1.safe",
            "stage": 1,
            "title": "SAFE V1 representation and tokenizer",
            "kind": "bounded_smoke",
            "evidence_names": ("representation_df", "permutation_df", "token_df", "negative_df"),
            "config_names": ("TOKENIZER_REPO", "TOKENIZER_REVISION", "TOKENIZER_SHA256"),
            "paper_source": "SAFE representation section",
            "repository_source": "src/genmol/utils/utils_safe.py",
            "notes": "Checks round trips, fragment permutation, tokenization, padding, and strict failure paths.",
            "checkpoint": "Explain why SAFE reduces but does not eliminate long-range molecular dependencies.",
        },
        {
            "id": "stage2.forward_diffusion",
            "stage": 2,
            "title": "Absorbing-state forward diffusion",
            "kind": "oracle_test",
            "evidence_names": ("monte_carlo_df", "time_sampling_df", "trajectory_df"),
            "config_names": ("NOISE_EPS", "TIME_SAMPLING_EPS"),
            "paper_source": "Equation 1 and the log-linear noise schedule",
            "repository_source": "BioNeMo-MoCo MDLM schedule utilities",
            "notes": "Compares analytical mask probabilities with hard corruption and Monte Carlo frequencies.",
            "checkpoint": "Compute the expected masked-token count from 1 - alpha(t).",
        },
        {
            "id": "stage3.reverse_posterior",
            "stage": 3,
            "title": "Exact reverse posterior",
            "kind": "oracle_test",
            "evidence_names": ("toy_exact_table", "weight_table", "trajectory_table", "failure_table"),
            "config_names": ("MC_DRAWS", "MC_ALPHA_S", "MC_ALPHA_T"),
            "paper_source": "Equations 1-2",
            "repository_source": "src/genmol/sampler.py; BioNeMo-MoCo MDLM transition",
            "notes": "Checks scalar, matrix, batched, sampling, Bayes, and invalid-input paths.",
            "checkpoint": "Derive reveal and stay probabilities when z_t is MASK.",
        },
        {
            "id": "stage4.nelbo",
            "stage": 4,
            "title": "SUBS parameterization and continuous NELBO",
            "kind": "oracle_test",
            "evidence_names": ("stage4_toy_df", "stage4_schedule_df", "stage4_rows", "stage4_grad_logits"),
            "config_names": ("STAGE4_NOISE_EPS",),
            "paper_source": "Equation 3",
            "repository_source": "BioNeMo-MoCo MDLM loss",
            "notes": "Checks token loss, global reduction, rare-mask weighting, gradients, and repository parity.",
            "checkpoint": "Explain why only currently masked valid positions contribute token loss.",
        },
        {
            "id": "stage5.denoiser",
            "stage": 5,
            "title": "BERT denoiser and one differentiable batch",
            "kind": "bounded_smoke",
            "evidence_names": ("stage5_architecture_df", "stage5_batch_df", "stage5_loss_value", "stage5_gradient_norm"),
            "config_names": ("STAGE5_MODEL_CONFIG", "stage5_parameter_count"),
            "paper_source": "Denoising network description",
            "repository_source": "src/genmol/model.py",
            "notes": "Runs the released architecture shape through corruption, logits, NELBO, and backward.",
            "checkpoint": "Trace clean IDs to z_t, logits, loss, and gradients.",
        },
        {
            "id": "stage6.adamw",
            "stage": 6,
            "title": "AdamW transaction and clipping",
            "kind": "oracle_test",
            "evidence_names": ("stage6_results", "stage6_parameter_delta", "stage6_peak_memory_mib"),
            "config_names": (
                "stage6_optimizer_kwargs",
                "stage6_gradient_clip_norm",
                "stage6_global_batch_size",
                "stage6_paper_training_steps",
            ),
            "paper_source": "Training configuration",
            "repository_source": "configs; src/genmol/model.py",
            "notes": "Checks a scalar AdamW oracle and one real clipped optimizer update.",
            "checkpoint": "Explain parameter update, optimizer state, and zeroed gradients.",
        },
        {
            "id": "stage7.data",
            "stage": 7,
            "title": "Data loading, padding, and exact batch plans",
            "kind": "bounded_smoke",
            "evidence_names": ("stage7_summary", "stage7_batch", "stage7_times"),
            "config_names": ("stage7_plan", "stage7_repository_plan"),
            "paper_source": "Training-data and batch-size description",
            "repository_source": "src/genmol/utils/utils_data.py",
            "notes": "Separates the exact requested global batch from the released ceiling-based plan.",
            "checkpoint": "Explain microbatch x accumulation x world size.",
        },
        {
            "id": "stage8.training_system",
            "stage": 8,
            "title": "Scheduler, DDP reduction, accumulation, EMA, and resume",
            "kind": "bounded_smoke",
            "evidence_names": ("stage8_report", "stage8_checkpoint", "stage8_launch_command"),
            "config_names": ("STAGE8_BASE_LR", "STAGE8_WARMUP_STEPS", "STAGE8_EMA_DECAY"),
            "paper_source": "Training procedure; engineering details are not all paper-explicit",
            "repository_source": "src/genmol/utils/ema.py; src/genmol/model.py",
            "notes": "Validates a bounded transaction and restoration; it does not run 50,000 steps.",
            "checkpoint": "Explain why EMA inference and exact resume state matter.",
        },
        {
            "id": "stage9.ancestral_sampling",
            "stage": 9,
            "title": "Equation 2 ancestral transition",
            "kind": "oracle_test",
            "evidence_names": ("stage9_grid_checks", "stage9_probs", "stage9_draw"),
            "config_names": (),
            "paper_source": "Equation 2",
            "repository_source": "src/genmol/sampler.py",
            "notes": "Reuses the Stage 3 posterior and adds a categorical draw.",
            "checkpoint": "Explain why a visible z_t token cannot change under absorbing diffusion.",
        },
        {
            "id": "stage10.confidence_sampling",
            "stage": 10,
            "title": "Confidence sampling",
            "kind": "oracle_test",
            "evidence_names": ("stage10_trace_a", "stage10_result_a", "stage10_visible_result"),
            "config_names": ("stage10_targets",),
            "paper_source": "Confidence-sampling appendix",
            "repository_source": "src/genmol/sampler.py; external BioNeMo-MoCo sampler boundary",
            "notes": "Checks sampled-token confidence, Gumbel ranking, reveal counts, replay, and fixed context.",
            "checkpoint": "Distinguish temperature tau, randomness r, and reveals N.",
        },
        {
            "id": "stage11.de_novo",
            "stage": 11,
            "title": "De novo generation pipeline",
            "kind": "bounded_smoke",
            "evidence_names": ("stage11_trace", "stage11_recreated_safe", "stage11_smiles"),
            "config_names": ("stage11_preview_lengths",),
            "paper_source": "De novo generation experiment",
            "repository_source": "src/genmol/sampler.py; src/genmol/utils/utils_safe.py",
            "notes": "Uses a scripted denoiser to prove length, mask, decode, and validation mechanics only.",
            "checkpoint": "Explain why a valid scripted sample is not evidence of pretrained quality.",
        },
        {
            "id": "stage12.mcg",
            "stage": 12,
            "title": "Molecular Context Guidance",
            "kind": "oracle_test",
            "evidence_names": ("stage12_rows", "stage12_zero", "stage12_guided_logits"),
            "config_names": (),
            "paper_source": "Equation 4",
            "repository_source": "src/genmol/sampler.py; src/genmol/utils/utils_moco.py",
            "notes": "Compares good, poor, and guided logits from two passes through the same denoiser.",
            "checkpoint": "Explain w*L_good + (1-w)*L_poor without calling it a property oracle.",
        },
        {
            "id": "stage13.fragment_constraints",
            "stage": 13,
            "title": "Fragment-constrained generation",
            "kind": "bounded_smoke",
            "evidence_names": ("stage13_rows", "stage13_trace", "stage13_result"),
            "config_names": ("FRAGMENT_TASK_SETTINGS",),
            "paper_source": "Fragment-constrained generation experiments",
            "repository_source": "src/genmol/sampler.py",
            "notes": "Exercises linker, motif, scaffold, and superstructure mechanics with fixed-context checks.",
            "checkpoint": "Distinguish substructure containment from full attachment-point correctness.",
        },
        {
            "id": "stage14.fragment_policies",
            "stage": 14,
            "title": "R_vocab, R_remask, and fragment scoring",
            "kind": "bounded_smoke",
            "evidence_names": ("stage14_vocab_cut_sets", "stage14_vocab_safe", "stage14_remask_safe", "stage14_scores"),
            "config_names": (),
            "paper_source": "Equation 5 and vocabulary/remasking decomposition",
            "repository_source": "src/genmol/utils/utils_chem.py; src/genmol/utils/utils_safe.py",
            "notes": "Shows more dots and smaller fragments under R_remask while molecular identity is preserved.",
            "checkpoint": "Explain independent vocabulary cuts versus cutting every eligible bond.",
        },
        {
            "id": "stage15.fragment_optimization",
            "stage": 15,
            "title": "Attach-remask-update mechanics",
            "kind": "bounded_smoke",
            "evidence_names": ("stage15_empirical_fragment_lengths", "stage15_scripted_trace", "stage15_scripted_result"),
            "config_names": ("stage15_replacement_length",),
            "paper_source": "Gibbs-like fragment optimization description",
            "repository_source": "src/genmol/sampler.py; src/genmol/utils/utils_chem.py",
            "notes": "Separates the paper empirical length distribution from the repository 5-15 token path.",
            "checkpoint": "Explain why this is Gibbs-like rather than exact Gibbs sampling.",
        },
        {
            "id": "stage16.pmo",
            "stage": 16,
            "title": "Bounded PMO hit-generation loop",
            "kind": "bounded_smoke",
            "evidence_names": ("stage16_records", "stage16_ranked", "stage16_constant_curve_auc"),
            "config_names": ("PMO_GAMMA",),
            "paper_source": "Algorithm 1; PMO appendix",
            "repository_source": "src/genmol/sampler.py; src/genmol/utils/utils_moco.py",
            "notes": "Checks unique-call budgeting, caching, ranking, remask warmup, and repository AUC bookkeeping.",
            "checkpoint": "Explain why a toy oracle tests control flow but not molecular optimization quality.",
        },
        {
            "id": "stage17.lead",
            "stage": 17,
            "title": "Lead-optimization qualification gates",
            "kind": "oracle_test",
            "evidence_names": ("stage17_boundary_pass", "stage17_equal_docking_fail", "stage17_low_qed_fail", "stage17_best"),
            "config_names": ("LEAD_GAMMA_ROWS",),
            "paper_source": "Lead-optimization experiment and thresholds",
            "repository_source": "src/genmol/sampler.py; src/genmol/utils/utils_chem.py",
            "notes": "Tests docking improvement, QED, raw SA, similarity, and candidate selection without fake success claims.",
            "checkpoint": "List every condition required before declaring an optimized lead.",
        },
        {
            "id": "stage18.evaluation",
            "stage": 18,
            "title": "Generation metrics and reproduction ledger",
            "kind": "bounded_smoke",
            "evidence_names": ("stage18_metrics", "REPRODUCTION_LEDGER", "NOTEBOOK_V1_SMOKE_TESTS_PASSED"),
            "config_names": (),
            "paper_source": "Evaluation metrics and benchmark protocol",
            "repository_source": "src/genmol/utils/utils_chem.py",
            "notes": "Computes bounded validity/uniqueness/quality/diversity and records all missing paper-scale work.",
            "checkpoint": "Distinguish a smoke metric from a three-run paper benchmark.",
        },
        {
            "id": "stage20.udlm",
            "stage": 20,
            "title": "Uniform-diffusion extension",
            "kind": "bounded_smoke",
            "evidence_names": (
                "stage20_reference_checks",
                "stage20_time_checks",
                "stage20_sampling_trace",
                "stage20_cpu_evidence",
                "stage20_mdlm_baseline",
                "stage209_selection_bundle",
                "stage20_ledger_record",
                "STAGE20_UDLM_SMOKE_TESTS_PASSED",
            ),
            "config_names": (
                "UDLM_BASE_COMMIT",
                "stage20_success_criteria",
                "stage209_future_scale_up_contract",
            ),
            "paper_source": "UDLM Sections 4.1-4.2 and GenMol de-novo metrics",
            "repository_source": "official UDLM edb0f8c28b7caeb4ea7a06a2fee8d74ab6da1661; src/genmol/diffusion.py; src/genmol/backbone.py",
            "notes": "Cross-checks native equations, time conditioning, revisable sampling, two pinned CPU artifacts, and the frozen local MDLM manifest; it is not benchmark-scale UDLM evidence.",
            "checkpoint": "Explain why bounded optimization evidence cannot establish a de-novo win.",
        },
    ]

    paper_scale_specs = [
        {"id": "paper.training", "stage": 8, "title": "Full model training", "gate_name": "RUN_PAPER_SCALE_TRAINING", "result_names": ("paper_training_summary",), "paper_requirement": "50,000 optimizer steps, global batch 2048; separately verify the reported 8 A100 hardware."},
        {"id": "paper.de_novo", "stage": 11, "title": "De novo benchmark", "gate_name": "RUN_PAPER_SCALE_DE_NOVO", "result_names": ("paper_de_novo_summary",), "paper_requirement": "1,000 requested molecules x 3 independent runs with the trained checkpoint."},
        {"id": "paper.fragment_benchmark", "stage": 13, "title": "Fragment-constrained benchmark", "gate_name": "RUN_PAPER_SCALE_FRAGMENT_BENCHMARK", "result_names": ("paper_fragment_benchmark_summary",), "paper_requirement": "5 tasks x 10 drugs x 100 requests x 3 runs, including attachment-point evaluation."},
        {"id": "paper.pmo", "stage": 16, "title": "PMO benchmark", "gate_name": "RUN_PMO_BENCHMARK", "result_names": ("paper_pmo_summary",), "paper_requirement": "23 tasks x 3 runs x 10,000 unique oracle calls with task-specific gamma."},
        {"id": "paper.lead", "stage": 17, "title": "Lead-optimization benchmark", "gate_name": "RUN_PAPER_SCALE_LEAD_OPTIMIZATION", "result_names": ("paper_lead_summary",), "paper_requirement": "5 targets x 3 seeds x 2 similarity thresholds x 3 runs; 10 iterations x 100 proposals."},
        {"id": "paper.docking", "stage": 17, "title": "QuickVina docking", "gate_name": "RUN_QUICKVINA_DOCKING", "result_names": ("paper_docking_summary",), "paper_requirement": "Real QuickVina results for every required lead-optimization run."},
        {"id": "paper.ablations", "stage": 18, "title": "Published ablation suite", "gate_name": "RUN_PAPER_SCALE_ABLATIONS", "result_names": ("paper_ablation_summary",), "paper_requirement": "Every claimed method variant at complete benchmark scale with repeated runs."},
    ]

    records = []
    for spec in stage_specs:
        evidence = [summarize(name, namespace.get(name)) for name in spec["evidence_names"]]
        config = {name: summarize(name, namespace.get(name)) for name in spec["config_names"]}
        missing = [name for name in spec["evidence_names"] if name not in namespace]
        failed = (
            spec["stage"] == 18
            and not bool(namespace.get("NOTEBOOK_V1_SMOKE_TESTS_PASSED", False))
        ) or (
            spec["stage"] == 20
            and not bool(namespace.get("STAGE20_UDLM_SMOKE_TESTS_PASSED", False))
        )
        if failed:
            status = "FAILED - INVARIANT VIOLATION"
        elif missing:
            status = "INCOMPLETE - EXPECTED RESULT MISSING"
        elif spec["kind"] == "oracle_test":
            status = "EXECUTED - NUMERICAL ORACLE"
        else:
            status = "EXECUTED - BOUNDED SMOKE TEST"
        first_value = namespace.get(spec["evidence_names"][0])
        records.append(
            {
                **spec,
                "gate": True,
                "result": None if missing else {name: summarize(name, namespace.get(name)) for name in spec["evidence_names"]},
                "evidence": evidence,
                "config": config,
                "missing": missing,
                "reason": "" if not missing else "Required bounded result is absent.",
                "paper_requirement": "",
                "status": status,
                "sample_size": size_hint(first_value),
            }
        )

    for spec in paper_scale_specs:
        gate = bool(namespace.get(spec["gate_name"], False))
        present = all(namespace.get(name) is not None for name in spec["result_names"])
        if not gate:
            result = None
            status = "NOT EXECUTED - DISABLED"
            reason = f"{spec['gate_name']} is False or absent."
        elif not present:
            result = None
            status = "INCOMPLETE - EXPECTED RESULT MISSING"
            reason = f"{spec['gate_name']} is True, but required full-scale evidence is missing."
        else:
            result = {name: summarize(name, namespace.get(name)) for name in spec["result_names"]}
            status = "EXECUTED - PAPER SCALE"
            reason = "Full-scale evidence object is present."
        records.append(
            {
                **spec,
                "kind": "paper_scale",
                "paper_source": "Published GenMol benchmark protocol",
                "repository_source": "Official NVIDIA BioNeMo GenMol workflow",
                "config_names": (spec["gate_name"],),
                "config": {spec["gate_name"]: str(gate)},
                "evidence_names": spec["result_names"],
                "evidence": [] if result is None else list(result.values()),
                "gate": gate,
                "result": result,
                "missing": [] if present else list(spec["result_names"]),
                "reason": reason,
                "notes": "Paper-scale completion is derived from the guard and a dedicated full-scale evidence object.",
                "checkpoint": "Verify scale, seeds, checkpoint, and denominators before accepting this row.",
                "status": status,
                "sample_size": "paper requirement",
            }
        )

    project_root = ReportPath(namespace.get("PROJECT_ROOT", ReportPath.cwd())).resolve()
    notebook_path = project_root / "genmol_from_scratch.ipynb"
    notebook_data = report_json.loads(notebook_path.read_text(encoding="utf-8"))
    source_only = [
        {"cell_type": cell.get("cell_type"), "source": "".join(cell.get("source", []))}
        for cell in notebook_data.get("cells", [])
    ]
    source_sha256 = sha256(report_json.dumps(source_only, ensure_ascii=True, sort_keys=True).encode("utf-8")).hexdigest()

    package_versions = {}
    for package in ("torch", "transformers", "datasets", "rdkit", "safe-mol", "reportlab", "pypdf"):
        try:
            package_versions[package] = report_metadata.version(package)
        except report_metadata.PackageNotFoundError:
            package_versions[package] = "not installed"

    run_flag_names = (
        "RUN_PAPER_SCALE_TRAINING",
        "RUN_PAPER_SCALE_DE_NOVO",
        "RUN_PAPER_SCALE_FRAGMENT_BENCHMARK",
        "RUN_PMO_BENCHMARK",
        "RUN_PAPER_SCALE_LEAD_OPTIMIZATION",
        "RUN_QUICKVINA_DOCKING",
        "RUN_PAPER_SCALE_ABLATIONS",
        "RUN_EXTENDED_SAFE_V2",
    )
    run_flags = {name: bool(namespace.get(name, False)) for name in run_flag_names}
    by_id = {record["id"]: record for record in records}

    def evidence_for(record_id):
        evidence = by_id[record_id]["evidence"]
        return "; ".join(evidence[:2]) if evidence else "No local result."

    udlm_decision_gate = namespace.get("stage20_decision_gate_report_rows")
    assert isinstance(udlm_decision_gate, list) and udlm_decision_gate
    assert all(
        isinstance(row, tuple)
        and len(row) == 2
        and all(isinstance(value, str) and value for value in row)
        for row in udlm_decision_gate
    )
    udlm_decision_gate = [tuple(row) for row in udlm_decision_gate]

    ablations = [
        {"comparison": "UDLM full-vocabulary vs control-token-excluded prior", "paper_result": "Uniform over all K categories is the faithful UDLM prior; control-token exclusion is a labeled molecular ablation.", "local_result": compact_value(namespace.get("stage20_prior_ablation_summary")), "sample_size": compact_value(namespace.get("stage20_prior_ablation_sample_scope")), "status": by_id["stage20.udlm"]["status"], "interpretation": "The paired seed-1 CPU smoke verifies both pathways only; 5/16 versus 3/16 strict-valid counts do not rank priors or estimate a population effect."},
        {"comparison": "Equation 2 ancestral vs confidence sampling", "paper_result": "The paper uses confidence sampling as the principal high-quality path; no local paper bar is copied.", "local_result": evidence_for("stage9.ancestral_sampling") + "; " + evidence_for("stage10.confidence_sampling"), "sample_size": by_id["stage10.confidence_sampling"]["sample_size"], "status": "EXECUTED - NUMERICAL ORACLE", "interpretation": "Both mechanics and invariants ran; molecular-quality benchmarking did not."},
        {"comparison": "Confidence reveals N = 1, 2, 3", "paper_result": "Published variants must be compared at the same benchmark scale.", "local_result": "A complete N sweep was not executed.", "sample_size": "None", "status": "NOT EXECUTED - DISABLED", "interpretation": "One teaching configuration cannot stand in for an ablation sweep."},
        {"comparison": "Temperature tau and randomness r sweep", "paper_result": "The paper studies the confidence-sampling quality/diversity tradeoff.", "local_result": "A complete tau/r grid was not executed.", "sample_size": "None", "status": "NOT EXECUTED - DISABLED", "interpretation": "The Stage 10 oracle checks ranking mechanics only."},
        {"comparison": "MCG on vs off", "paper_result": "Equation 4 combines good and degraded context logits with guidance weight w.", "local_result": evidence_for("stage12.mcg"), "sample_size": by_id["stage12.mcg"]["sample_size"], "status": by_id["stage12.mcg"]["status"], "interpretation": "A numerical logit oracle ran; a full molecular benchmark did not."},
        {"comparison": "R_vocab vs R_remask fragment policies", "paper_result": "Vocabulary construction uses independent sparse cuts; remasking cuts every eligible bond.", "local_result": evidence_for("stage14.fragment_policies"), "sample_size": by_id["stage14.fragment_policies"]["sample_size"], "status": by_id["stage14.fragment_policies"]["status"], "interpretation": "The same molecule is decomposed into more, smaller fragments under R_remask."},
        {"comparison": "Strict vs repaired SAFE decoding", "paper_result": "Chemical validity must not be inflated by silently repairing malformed model output.", "local_result": evidence_for("stage1.safe"), "sample_size": by_id["stage1.safe"]["sample_size"], "status": "EXECUTED - NUMERICAL ORACLE", "interpretation": "The negative path preserves a strict validity denominator."},
        {"comparison": "SAFE V1 vs post-paper Extended SAFE V2", "paper_result": "The GenMol paper uses SAFE V1; V2 is outside the reproduction target.", "local_result": "Extended V2 appendix executed." if run_flags["RUN_EXTENDED_SAFE_V2"] else "Extended V2 appendix was not executed.", "sample_size": "appendix only", "status": "EXECUTED - BOUNDED SMOKE TEST" if run_flags["RUN_EXTENDED_SAFE_V2"] else "NOT EXECUTED - DISABLED", "interpretation": "V2 evidence is isolated and never mixed with V1 benchmark claims."},
    ]

    return {
        "records": records,
        "ablations": ablations,
        "udlm_decision_gate": udlm_decision_gate,
        "metadata": {
            "project_root": str(project_root),
            "notebook_path": str(notebook_path),
            "notebook_source_sha256": source_sha256,
            "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "host": report_platform.node(),
            "selected_gpu_count": len(namespace.get("SELECTED_PHYSICAL_GPU_IDS", [])),
            "selected_physical_gpu_ids": compact_value(namespace.get("SELECTED_PHYSICAL_GPU_IDS")),
            "selected_gpu_uuids": compact_value(namespace.get("SELECTED_GPU_UUIDS")),
            "configured_num_gpus": compact_value(namespace.get("NUM_GPUS")),
            "seed": compact_value(namespace.get("SEED")),
            "cuda_version": compact_value(report_torch.version.cuda),
            "torch_cuda_visible_count": report_torch.cuda.device_count(),
            "versions": compact_value(namespace.get("versions")),
            "package_versions": package_versions,
            "pushed_head_commit": compact_value(namespace.get("pushed_head_commit")),
            "active_branch": compact_value(namespace.get("branch")),
            "tracking_ref": compact_value(namespace.get("tracking_ref")),
            "udlm_base_commit": compact_value(namespace.get("UDLM_BASE_COMMIT")),
            "working_tree_status": compact_value(namespace.get("working_tree_status")),
            "run_flags": run_flags,
            "reproduction_ledger": summarize("REPRODUCTION_LEDGER", namespace.get("REPRODUCTION_LEDGER")),
        },
    }


def validate_report_records(payload):
    allowed_statuses = {
        "EXECUTED - NUMERICAL ORACLE",
        "EXECUTED - BOUNDED SMOKE TEST",
        "EXECUTED - PAPER SCALE",
        "NOT EXECUTED - DISABLED",
        "INCOMPLETE - EXPECTED RESULT MISSING",
        "FAILED - INVARIANT VIOLATION",
    }
    records = payload["records"]
    record_ids = [record["id"] for record in records]
    assert len(record_ids) == len(set(record_ids)), "Report record IDs must be unique."
    stage_records = [record for record in records if record["kind"] != "paper_scale"]
    assert [record["stage"] for record in stage_records] == [*range(19), 20], "Stages 0-18 and 20 must be ordered and complete."

    for record in records:
        assert record["status"] in allowed_statuses
        if record["status"].startswith("EXECUTED"):
            assert record["evidence"], f"{record['id']} has no executed evidence."
        if record["status"] == "NOT EXECUTED - DISABLED":
            assert record["result"] is None
            assert record["reason"]
            assert record["paper_requirement"]
        serialized = report_json.dumps(record, ensure_ascii=True, sort_keys=True)
        assert " at 0x" not in serialized
        assert "NONFINITE" not in serialized

    for ablation in payload["ablations"]:
        assert ablation["status"] in allowed_statuses
        assert ablation["local_result"] != ""

    gate_rows = payload["udlm_decision_gate"]
    expected_gate_labels = [
        "Comparator status",
        "Comparator exact means",
        "Comparator implementation provenance",
        "Comparator current-code rescore",
        *[f"Comparator caveat {index}" for index in range(1, 6)],
        "Point-estimate gate",
        "Validity uncertainty gate",
        "Uniqueness uncertainty gate",
        "Quality uncertainty gate",
        "Diversity uncertainty gate",
        "Final-candidate lock",
        "Registered pilot selector",
        "Final evaluation protocol",
        "Matching constraints",
        "Required candidate provenance",
        "Training-support fairness",
        "Forbidden inference",
        "Small-sample limitation",
        "Strict metrics",
        "Claim scope",
    ]
    assert [row[0] for row in gate_rows] == expected_gate_labels
    assert all(len(row) == 2 and all(value for value in row) for row in gate_rows)
    gate_text = " ".join(value for row in gate_rows for value in row)
    for required_text in (
        "not an exact paper reproduction",
        "dirty source tree",
        "Newcombe-Wilson",
        "Welch interval",
        "b474efc593b665489359425dbe1ed0873f8ae1d44b77478b871aff6d6b555904",
        "a77d7c84d8628403c6d7a11edebce5bf9b8405e3f8b79a326a6e06bd7f444cea",
        "3 seeds x 1000 requests",
        "128 NFE",
        "initialization checkpoint and raw-or-EMA choice",
        "launch-manifest path/hash/schema",
        "global lease",
        "Completed W=1 screens selected E-L1/E-A1; a separate selection-bound scale-up registry is required",
        "MDLM-matched content-only framing control",
        "continuation-system",
    ):
        assert required_text in gate_text

    paper_records = [record for record in records if record["kind"] == "paper_scale"]
    assert len(paper_records) == 7
    assert all(record["paper_requirement"] for record in paper_records)
    assert payload["metadata"]["selected_gpu_count"] == int(payload["metadata"]["configured_num_gpus"])
    assert (ReportPath(payload["metadata"]["project_root"]) / "src" / "genmol").is_dir()

    return [
        "Unique report record IDs",
        "Ordered Stage 0-18 and 20 coverage",
        "Executed evidence is nonempty",
        "Disabled results use None and an explicit paper requirement",
        "Finite bounded summaries with no object memory addresses",
        "GPU count comes from Stage 0 state",
        "Paper-scale and smoke-test statuses remain separate",
        "UDLM gate, comparator caveats, uncertainty methods, and claim scope are explicit",
        "Output root contains this checkout's src/genmol package",
    ]


def build_genmol_report_pdf(payload, output_path):
    styles = getSampleStyleSheet()
    navy = report_colors.HexColor("#17324D")
    blue = report_colors.HexColor("#2E6F9E")
    pale_green = report_colors.HexColor("#EAF6EF")
    pale_red = report_colors.HexColor("#FCEBEC")
    grey = report_colors.HexColor("#5B6570")
    line = report_colors.HexColor("#CAD3DB")

    styles.add(ParagraphStyle(name="GenMolTitle", parent=styles["Title"], fontName="Helvetica-Bold", fontSize=25, leading=30, textColor=navy, alignment=TA_CENTER, spaceAfter=10))
    styles.add(ParagraphStyle(name="GenMolSubtitle", parent=styles["Normal"], fontName="Helvetica", fontSize=11, leading=15, textColor=grey, alignment=TA_CENTER, spaceAfter=12))
    styles.add(ParagraphStyle(name="GenMolH1", parent=styles["Heading1"], fontName="Helvetica-Bold", fontSize=17, leading=21, textColor=navy, spaceAfter=9))
    styles.add(ParagraphStyle(name="GenMolH2", parent=styles["Heading2"], fontName="Helvetica-Bold", fontSize=11, leading=14, textColor=blue, spaceBefore=5, spaceAfter=5))
    styles.add(ParagraphStyle(name="GenMolBody", parent=styles["BodyText"], fontName="Helvetica", fontSize=8.8, leading=12.2, textColor=report_colors.HexColor("#20262C"), spaceAfter=6))
    styles.add(ParagraphStyle(name="GenMolSmall", parent=styles["BodyText"], fontName="Helvetica", fontSize=7.2, leading=9.3, textColor=grey, spaceAfter=3))
    styles.add(ParagraphStyle(name="GenMolTableHead", parent=styles["BodyText"], fontName="Helvetica-Bold", fontSize=6.5, leading=8, textColor=report_colors.white, alignment=TA_LEFT))
    styles.add(ParagraphStyle(name="GenMolTableBody", parent=styles["BodyText"], fontName="Helvetica", fontSize=6.2, leading=7.8, textColor=report_colors.HexColor("#20262C"), alignment=TA_LEFT))
    styles.add(ParagraphStyle(name="GenMolBanner", parent=styles["BodyText"], fontName="Helvetica-Bold", fontSize=11, leading=15, textColor=navy, alignment=TA_CENTER))
    styles.add(ParagraphStyle(name="GenMolFormula", parent=styles["BodyText"], fontName="Courier", fontSize=8, leading=11, textColor=navy, leftIndent=8, rightIndent=8, borderColor=line, borderWidth=0.5, borderPadding=6, backColor=report_colors.HexColor("#F7F9FB"), spaceAfter=6))

    output_path = ReportPath(output_path).resolve()
    project_root = ReportPath(payload["metadata"]["project_root"]).resolve()
    allowed_root = (project_root / "output" / "pdf").resolve()
    assert output_path.parent == allowed_root
    output_path.parent.mkdir(parents=True, exist_ok=True)

    doc = SimpleDocTemplate(str(output_path), pagesize=A4, leftMargin=20 * mm, rightMargin=20 * mm, topMargin=19 * mm, bottomMargin=17 * mm)
    usable_width = A4[0] - doc.leftMargin - doc.rightMargin

    def para(text, style="GenMolBody"):
        return Paragraph(report_escape(str(text)), styles[style])

    def table_cell(value, heading=False):
        return Paragraph(report_escape(str(value)), styles["GenMolTableHead" if heading else "GenMolTableBody"])

    def make_table(headers, rows, widths, font_size=6.2):
        assert abs(sum(widths) - usable_width) < 2
        body_style = styles["GenMolTableBody"].clone(f"GenMolTableBody{font_size}")
        body_style.fontSize = font_size
        body_style.leading = font_size + 1.5
        wrapped = [[table_cell(value, heading=True) for value in headers]]
        for row in rows:
            wrapped.append([Paragraph(report_escape(str(value)), body_style) for value in row])
        result = Table(wrapped, colWidths=widths, repeatRows=1, splitByRow=1, hAlign="LEFT")
        result.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), navy),
            ("TEXTCOLOR", (0, 0), (-1, 0), report_colors.white),
            ("GRID", (0, 0), (-1, -1), 0.35, line),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("LEFTPADDING", (0, 0), (-1, -1), 3.5),
            ("RIGHTPADDING", (0, 0), (-1, -1), 3.5),
            ("TOPPADDING", (0, 0), (-1, -1), 3),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [report_colors.white, report_colors.HexColor("#F7F9FB")]),
        ]))
        return result

    def decorate_page(canvas, document):
        canvas.saveState()
        canvas.setTitle("GenMol Benchmark Results - Experiment and Ablation Report")
        canvas.setAuthor("GenMol V2 teaching project")
        canvas.setStrokeColor(line)
        canvas.setLineWidth(0.4)
        canvas.line(doc.leftMargin, A4[1] - 12 * mm, A4[0] - doc.rightMargin, A4[1] - 12 * mm)
        canvas.setFont("Helvetica", 7)
        canvas.setFillColor(grey)
        canvas.drawString(doc.leftMargin, A4[1] - 9 * mm, "GenMol benchmark report")
        canvas.drawRightString(A4[0] - doc.rightMargin, 9 * mm, f"Page {document.page}")
        canvas.drawString(doc.leftMargin, 9 * mm, "Benchmark status, provenance, and bounded checks")
        canvas.restoreState()

    stage_records = [record for record in payload["records"] if record["kind"] != "paper_scale"]
    paper_records = [record for record in payload["records"] if record["kind"] == "paper_scale"]
    paper_done = sum(record["status"] == "EXECUTED - PAPER SCALE" for record in paper_records)
    metadata = payload["metadata"]
    story = []

    story.append(Spacer(1, 26 * mm))
    story.append(para("GenMol Benchmark Results", "GenMolTitle"))
    story.append(para("Automatic results report for every GenMol benchmark suite", "GenMolSubtitle"))
    banner = Table([[Paragraph(f"{paper_done}/{len(paper_records)} paper-scale experiment groups executed", styles["GenMolBanner"])]], colWidths=[usable_width])
    banner.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, -1), pale_green if paper_done == len(paper_records) else pale_red),
        ("BOX", (0, 0), (-1, -1), 0.8, blue),
        ("TOPPADDING", (0, 0), (-1, -1), 10),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
    ]))
    story.append(KeepTogether([banner, Spacer(1, 8 * mm)]))
    story.append(para("Scope: core GenMol algorithms and runnable pathways are implemented and smoke-tested. This document does not promote scripted-denoiser output, toy-oracle output, or a single optimizer step to the status of a pretrained molecular benchmark."))
    cover_rows = [
        ("Generated UTC", metadata["generated_at_utc"]),
        ("Notebook", metadata["notebook_path"]),
        ("Source-only notebook SHA-256", metadata["notebook_source_sha256"]),
        ("Official repository", "https://github.com/NVIDIA-BioNeMo/genmol"),
        ("Pushed HEAD commit", metadata["pushed_head_commit"]),
        ("Active branch and tracking ref", f"{metadata['active_branch']} -> {metadata['tracking_ref']}"),
        ("Required UDLM base ancestor", metadata["udlm_base_commit"]),
        ("Paper", "Attached GenMol paper; equations and benchmark requirements are cited by section."),
        ("Report behavior", "Reads existing variables only; no model, sampler, optimizer, oracle, docking, network, or GPU call."),
    ]
    story.append(make_table(("Provenance field", "Value"), cover_rows, (42 * mm, 128 * mm), font_size=7.0))
    story.append(Spacer(1, 5 * mm))
    story.append(para("Interpret status before numbers: EXECUTED - NUMERICAL ORACLE and EXECUTED - BOUNDED SMOKE TEST show correctness of mechanics. Only EXECUTED - PAPER SCALE denotes complete benchmark evidence."))
    story.append(PageBreak())

    story.append(para("Benchmark execution dashboard", "GenMolH1"))
    story.append(
        para(
            "Every GenMol benchmark suite appears here even when it was not run. Full benchmark status is derived "
            "from a dedicated evidence object; bounded notebook checks are supporting diagnostics only."
        )
    )
    stage_by_id = {record["id"]: record for record in stage_records}
    paper_by_id = {record["id"]: record for record in paper_records}
    bounded_links = {
        "paper.training": ("stage8.training_system",),
        "paper.de_novo": ("stage11.de_novo", "stage18.evaluation", "stage20.udlm"),
        "paper.fragment_benchmark": ("stage13.fragment_constraints",),
        "paper.pmo": ("stage16.pmo",),
        "paper.lead": ("stage17.lead",),
        "paper.docking": ("stage17.lead",),
        "paper.ablations": ("stage10.confidence_sampling", "stage12.mcg", "stage14.fragment_policies"),
    }
    benchmark_rows = []
    for paper_record in paper_records:
        linked = [stage_by_id[record_id] for record_id in bounded_links[paper_record["id"]]]
        bounded_status = " | ".join(
            f"Stage {record['stage']}: {record['status']}" for record in linked
        )
        full_result = "None" if paper_record["result"] is None else "; ".join(paper_record["evidence"][:2])
        benchmark_rows.append(
            (
                paper_record["title"],
                paper_record["status"],
                paper_record["paper_requirement"],
                full_result,
                bounded_status,
            )
        )
    story.append(
        make_table(
            ("Benchmark suite", "Full status", "Required scale", "Full-run result", "Bounded support"),
            benchmark_rows,
            (29 * mm, 31 * mm, 54 * mm, 25 * mm, 31 * mm),
            font_size=5.8,
        )
    )
    story.append(Spacer(1, 5 * mm))
    story.append(
        para(
            f"Complete paper-scale groups in this run: {paper_done}/{len(paper_records)}. "
            "A disabled benchmark is unmeasured, not a zero-valued result."
        )
    )
    story.append(PageBreak())

    story.append(para("De novo and fragment-constrained benchmarks", "GenMolH1"))
    de_novo_full = paper_by_id["paper.de_novo"]
    fragment_full = paper_by_id["paper.fragment_benchmark"]
    de_novo_local = stage_by_id["stage11.de_novo"]
    evaluation_local = stage_by_id["stage18.evaluation"]
    udlm_local = stage_by_id["stage20.udlm"]
    fragment_local = stage_by_id["stage13.fragment_constraints"]
    generation_rows = [
        (
            "De novo - full benchmark",
            de_novo_full["status"],
            de_novo_full["paper_requirement"],
            "None" if de_novo_full["result"] is None else "; ".join(de_novo_full["evidence"][:2]),
        ),
        (
            "De novo - bounded mechanism",
            de_novo_local["status"],
            "Scripted denoiser; mechanics only",
            "; ".join(de_novo_local["evidence"][:3]),
        ),
        (
            "Generation metrics - bounded",
            evaluation_local["status"],
            "Small local denominator",
            "; ".join(evaluation_local["evidence"][:2]),
        ),
        (
            "UDLM - bounded implementation evidence",
            udlm_local["status"],
            "Native equation parity, a revisability trace, and two hash-validated 16-request CPU smoke artifacts",
            "; ".join((udlm_local["evidence"][0], udlm_local["evidence"][2], udlm_local["evidence"][3])),
        ),
        (
            "Fragment constrained - full benchmark",
            fragment_full["status"],
            fragment_full["paper_requirement"],
            "None" if fragment_full["result"] is None else "; ".join(fragment_full["evidence"][:2]),
        ),
        (
            "Fragment constrained - bounded mechanism",
            fragment_local["status"],
            "Task mechanics and context checks",
            "; ".join(fragment_local["evidence"][:2]),
        ),
    ]
    story.append(
        make_table(
            ("Result group", "Status", "Scale or scope", "Recorded result"),
            generation_rows,
            (43 * mm, 36 * mm, 43 * mm, 48 * mm),
            font_size=6.0,
        )
    )
    story.append(Spacer(1, 4 * mm))
    story.append(para("Registered UDLM decision gate", "GenMolH2"))
    story.append(
        make_table(
            ("Gate or provenance field", "Exact registered requirement"),
            payload["udlm_decision_gate"],
            (48 * mm, 122 * mm),
            font_size=6.2,
        )
    )
    story.append(Spacer(1, 5 * mm))
    story.append(para("Required de novo result columns", "GenMolH2"))
    de_novo_schema = [
        ("Identity", "run_id, seed, checkpoint hash, requested count"),
        ("Denominators", "requested, decoded, strict-valid, unique-valid"),
        ("Metrics", "validity, uniqueness, quality, internal diversity, fragment distance"),
        ("Aggregation", "per-run values plus mean and standard deviation across three independent runs"),
        ("Failure accounting", "invalid SAFE, decode failure, duplicate, property failure, and empty-set policy"),
    ]
    story.append(make_table(("Group", "Required fields"), de_novo_schema, (42 * mm, 128 * mm), font_size=6.8))
    story.append(Spacer(1, 5 * mm))
    story.append(para("Required fragment-benchmark result columns", "GenMolH2"))
    fragment_schema = [
        ("Identity", "task, drug, run_id, seed, checkpoint hash"),
        ("Counts", "requests, decoded, strict-valid, constraint-valid, attachment-valid, unique"),
        ("Task metrics", "linker, motif, scaffold, superstructure, and every fifth configured task"),
        ("Aggregation", "5 tasks x 10 drugs x 100 requests x 3 runs with mean and standard deviation"),
    ]
    story.append(make_table(("Group", "Required fields"), fragment_schema, (42 * mm, 128 * mm), font_size=6.8))
    story.append(PageBreak())

    story.append(para("PMO benchmark", "GenMolH1"))
    pmo_full = paper_by_id["paper.pmo"]
    pmo_local = stage_by_id["stage16.pmo"]
    pmo_rows = [
        ("Paper-scale PMO", pmo_full["status"], pmo_full["paper_requirement"], "None" if pmo_full["result"] is None else "; ".join(pmo_full["evidence"][:2])),
        ("Bounded cache/budget loop", pmo_local["status"], "Toy oracle; control-flow evidence", "; ".join(pmo_local["evidence"][:3])),
    ]
    story.append(make_table(("Result group", "Status", "Scale or scope", "Recorded result"), pmo_rows, (43 * mm, 36 * mm, 43 * mm, 48 * mm), font_size=6.2))
    story.append(Spacer(1, 6 * mm))
    story.append(
        para(
            "Comparable PMO reporting requires canonical unique molecules in oracle-call order. For each of the 23 tasks "
            "and each of three runs, the summary must retain every 100-call top-10 mean, trapezoidal AUC convention, "
            "final extension to 10,000 calls, task-specific gamma, seed, duplicates, failures, and exact unique-call count."
        )
    )
    pmo_schema = [
        ("Run identity", "task, run_id, seed, checkpoint hash, gamma"),
        ("Budget", "10,000 unique canonical oracle calls; duplicate cache hits reported separately"),
        ("Trajectory", "call indices and mean top-10 score every 100 unique calls"),
        ("Primary result", "repository-compatible top-10 AUC plus final top-10 mean"),
        ("Aggregation", "three-run mean and standard deviation for each task and aggregate ranking"),
    ]
    story.append(make_table(("Group", "Required PMO fields"), pmo_schema, (42 * mm, 128 * mm), font_size=6.8))
    story.append(PageBreak())

    story.append(para("Lead optimization and docking benchmark", "GenMolH1"))
    lead_full = paper_by_id["paper.lead"]
    docking_full = paper_by_id["paper.docking"]
    lead_local = stage_by_id["stage17.lead"]
    lead_rows = [
        ("Paper-scale lead optimization", lead_full["status"], lead_full["paper_requirement"], "None" if lead_full["result"] is None else "; ".join(lead_full["evidence"][:2])),
        ("Paper-scale QuickVina docking", docking_full["status"], docking_full["paper_requirement"], "None" if docking_full["result"] is None else "; ".join(docking_full["evidence"][:2])),
        ("Bounded qualification gates", lead_local["status"], "Numerical boundary fixtures only", "; ".join(lead_local["evidence"][:3])),
    ]
    story.append(make_table(("Result group", "Status", "Scale or scope", "Recorded result"), lead_rows, (43 * mm, 36 * mm, 43 * mm, 48 * mm), font_size=6.1))
    story.append(Spacer(1, 6 * mm))
    story.append(
        para(
            "A successful lead row requires strict docking improvement and QED >= 0.6, raw SA <= 4, and Tanimoto "
            "similarity above delta in {0.4, 0.6}. Fake or scripted docking values can test Boolean gates but cannot "
            "appear in the successful paper-scale table."
        )
    )
    lead_schema = [
        ("Run identity", "target, seed molecule, run_id, repetition, similarity threshold, checkpoint hash"),
        ("Search scale", "10 iterations x 100 proposals, with the complete target/seed/threshold/repetition grid"),
        ("Molecule metrics", "candidate SMILES/SAFE hash, QED, raw SA, similarity, strict validity"),
        ("Docking metrics", "seed QuickVina score, candidate QuickVina score, sign convention, strict improvement"),
        ("Outcome", "qualification Boolean, failure reason, selected best lead, and no-candidate policy"),
        ("Aggregation", "success rate and score distributions across all required runs"),
    ]
    story.append(make_table(("Group", "Required lead fields"), lead_schema, (42 * mm, 128 * mm), font_size=6.8))
    story.append(PageBreak())

    story.append(para("Training provenance and implementation checks", "GenMolH1"))
    training_full = paper_by_id["paper.training"]
    training_local = stage_by_id["stage8.training_system"]
    training_rows = [
        ("Paper-scale training", training_full["status"], training_full["paper_requirement"], "None" if training_full["result"] is None else "; ".join(training_full["evidence"][:2])),
        ("Bounded training transaction", training_local["status"], "One small resumable transaction", "; ".join(training_local["evidence"][:2])),
    ]
    story.append(make_table(("Result group", "Status", "Scale or scope", "Recorded result"), training_rows, (43 * mm, 36 * mm, 43 * mm, 48 * mm), font_size=6.2))
    story.append(Spacer(1, 5 * mm))
    provenance_rows = [
        ("Remote host", metadata["host"]),
        ("Configured GPU count", metadata["configured_num_gpus"]),
        ("Selected physical GPU IDs", metadata["selected_physical_gpu_ids"]),
        ("Selected GPU UUIDs", metadata["selected_gpu_uuids"]),
        ("Process-visible CUDA devices", metadata["torch_cuda_visible_count"]),
        ("CUDA version", metadata["cuda_version"]),
        ("Seed", metadata["seed"]),
        ("Pushed HEAD commit", metadata["pushed_head_commit"]),
        ("Active branch", metadata["active_branch"]),
        ("Tracking ref", metadata["tracking_ref"]),
        ("UDLM base ancestor", metadata["udlm_base_commit"]),
        ("Uncommitted paths at provenance check", metadata["working_tree_status"]),
        ("Notebook source SHA-256", metadata["notebook_source_sha256"]),
    ]
    story.append(make_table(("Provenance field", "Recorded value"), provenance_rows, (48 * mm, 122 * mm), font_size=6.6))
    story.append(Spacer(1, 5 * mm))
    story.append(
        para(
            "Implementation checks below are provenance for the benchmark pipeline. They are not additional benchmark scores."
        )
    )
    implementation_rows = [
        (
            record["stage"],
            record["title"],
            record["status"],
            f"{len(record['evidence'])} evidence fields; {record['sample_size']}",
            f"{record['paper_source']} | {record['repository_source']}",
        )
        for record in stage_records
    ]
    story.append(
        make_table(
            ("Stage", "Supporting implementation", "Status", "Evidence", "Paper and released-code source"),
            implementation_rows,
            (10 * mm, 35 * mm, 33 * mm, 32 * mm, 60 * mm),
            font_size=5.6,
        )
    )
    story.append(PageBreak())

    story.append(para("Ablations and experiments", "GenMolH1"))
    story.append(para("Paper observations and local results occupy different columns. A local numerical oracle is evidence about mechanics, not agreement with a published benchmark. Missing sweeps are reported as disabled, never as zero."))
    ablation_rows = [(item["comparison"], item["paper_result"], item["local_result"], item["sample_size"], item["status"], item["interpretation"]) for item in payload["ablations"]]
    story.append(make_table(("Comparison", "Paper result or requirement", "Local result", "Sample size", "Status", "Interpretation"), ablation_rows, (30 * mm, 32 * mm, 38 * mm, 17 * mm, 26 * mm, 27 * mm), font_size=5.5))
    story.append(Spacer(1, 5 * mm))
    story.append(para("The complete published ablation suite remains a paper-scale ledger item until every variant is evaluated with matched data, checkpoint, sample count, seeds, and metric definitions."))
    story.append(PageBreak())

    story.append(para("Paper-scale execution ledger", "GenMolH1"))
    story.append(para("This page is mandatory even when every expensive guard is false. Completion requires both an enabled guard and a dedicated evidence object at the stated scale."))
    ledger_rows = [(record["title"], record["status"], record["paper_requirement"], record["reason"], "None" if record["result"] is None else "; ".join(record["evidence"][:2])) for record in paper_records]
    story.append(make_table(("Experiment", "Status", "Paper completion requirement", "Reason", "Executed evidence"), ledger_rows, (28 * mm, 31 * mm, 52 * mm, 31 * mm, 28 * mm), font_size=5.8))
    story.append(Spacer(1, 5 * mm))
    story.append(para(f"Paper-scale completion for this run: {paper_done}/{len(paper_records)} experiment groups. A false guard is an intentional safety boundary, not a failed scientific result."))
    story.append(PageBreak())

    story.append(para("Fidelity decisions and known deviations", "GenMolH1"))
    fidelity_rows = [
        ("Paper-explicit vs repository-resolved", "Each stage separates the equation or benchmark requirement from implementation details supplied by released code."),
        ("UserDataset repair", "The teaching path fixes the released dataset edge case while preserving released behavior as a labeled comparison."),
        ("Exact batch planning", "The notebook can choose exact microbatch x accumulation x world-size products; the released ceiling plan remains visible."),
        ("BERT time input", "The paper notation includes t, but the released BERT has no explicit time embedding."),
        ("UDLM schedule mismatch", "The release uses residual-clean alpha for corruption/sampling but ideal alpha=1-t in its loss; prior loss is therefore not exactly zero for the implemented forward endpoint."),
        ("Distributed loss weighting", "Released global_mean is process-local. DDP and gradient accumulation average local token ratios rather than forming one token ratio across all ranks and microbatches; an exact-global repair must be a labeled ablation."),
        ("Confidence sampler boundary", "The notebook tests ranking mechanics while labeling the external BioNeMo-MoCo integration boundary."),
        ("Remask length", "The paper samples an empirical fragment length; released code uses a 5-15 token randint path."),
        ("Fragment interpretation", "Attach-remask-update is described as Gibbs-like, not mathematically exact Gibbs sampling."),
        ("Lead docking", "No real QuickVina claim is made unless the guard and full evidence object are present."),
        ("SAFE V2", "Extended SAFE V2 is post-paper and isolated from the V1 reproduction target."),
    ]
    story.append(make_table(("Decision", "Report treatment"), fidelity_rows, (43 * mm, 127 * mm), font_size=6.6))
    story.append(Spacer(1, 5 * mm))
    story.append(para("Noise exclusion: the PDF contains summaries, shapes, counts, and short previews. It excludes raw logits, gradients, optimizer tensors, model state dictionaries, long token arrays, and arbitrary notebook stdout."))
    story.append(PageBreak())

    story.append(para("Verification and limitations", "GenMolH1"))
    story.append(para("Pre-build record checks", "GenMolH2"))
    validation_rows = [(index + 1, check, "PASS") for index, check in enumerate(payload["validation"])]
    story.append(make_table(("No.", "Invariant", "Result"), validation_rows, (12 * mm, 132 * mm, 26 * mm), font_size=7.0))
    story.append(Spacer(1, 6 * mm))
    story.append(para("Scientific limitations", "GenMolH2"))
    for limitation in (
        "Scripted denoisers verify sampling and decoding mechanics, not learned molecular quality.",
        "Toy property oracles verify cache and budget behavior, not PMO benchmark performance.",
        "A single optimizer transaction verifies engineering invariants, not convergence.",
        "Substructure containment alone does not prove every task-specific attachment-point condition.",
        "Bounded validity, uniqueness, quality, and diversity do not replace repeated paper-scale evaluation.",
        "The frozen local MDLM comparator is not an exact paper reproduction; its five historical caveats are rendered verbatim in the registered gate table.",
        "Process-local token-ratio averaging across DDP ranks and accumulation windows is not an exact globally pooled token mean.",
        "The current UDLM content-only training support excludes BOS/EOS while faithful MDLM uses the full attention mask; without an MDLM-matched framing control this remains a method-causality confound.",
        "Fragment distance, full PMO, real docking, and three-run mean/std remain absent unless their ledger rows say EXECUTED - PAPER SCALE.",
    ):
        story.append(para(f"- {limitation}"))
    story.append(Spacer(1, 5 * mm))
    story.append(para("Final interpretation", "GenMolH2"))
    story.append(para("Core algorithms and runnable pathways were reimplemented and smoke-tested against the paper equations and released code structure. Experiments marked NOT EXECUTED - DISABLED were not run and are not numerical reproductions of the published benchmarks."))

    doc.build(story, onFirstPage=decorate_page, onLaterPages=decorate_page)
    return output_path


def validate_rendered_report(output_path, render_dir):
    output_path = ReportPath(output_path).resolve()
    project_root = ReportPath(PROJECT_ROOT).resolve()
    render_root = (project_root / "tmp" / "pdfs").resolve()
    render_dir = ReportPath(render_dir).resolve()
    assert render_dir.parent == render_root

    reader = PdfReader(str(output_path))
    assert not reader.is_encrypted
    page_count = len(reader.pages)
    extracted = "\n".join(page.extract_text() or "" for page in reader.pages)
    required_headings = (
        "GenMol Benchmark Results",
        "Benchmark execution dashboard",
        "De novo and fragment-constrained benchmarks",
        "PMO benchmark",
        "Lead optimization and docking benchmark",
        "Ablations and experiments",
        "Paper-scale execution ledger",
        "Registered UDLM decision gate",
        "Verification and limitations",
    )


    if render_dir.exists():
        report_shutil.rmtree(render_dir)
    render_dir.mkdir(parents=True, exist_ok=True)
    prefix = render_dir / "page"
    report_subprocess.run(["pdftoppm", "-png", "-r", "110", str(output_path), str(prefix)], check=True, capture_output=True, text=True)
    rendered_pages = sorted(render_dir.glob("page-*.png"))
    nonwhite_fractions = []
    for page_path in rendered_pages:
        with ReportImage.open(page_path) as page_image:
            grayscale = page_image.convert("L")
            histogram = grayscale.histogram()
            nonwhite = sum(histogram[:245])
            nonwhite_fractions.append(nonwhite / (grayscale.width * grayscale.height))

    disabled_expected = sum(record["status"] == "NOT EXECUTED - DISABLED" for record in REPORT_RECORDS["records"] if record["kind"] == "paper_scale")
    file_size = output_path.stat().st_size
    audit = {
        "page_count": page_count,
        "file_size_bytes": file_size,
        "all_pages_rendered": len(rendered_pages) == page_count,
        "all_pages_nonblank": bool(nonwhite_fractions) and min(nonwhite_fractions) > 0.001,
        "all_required_headings_present": all(heading in extracted for heading in required_headings),
        "udlm_claim_language_present": all(
            fragment in extracted
            for fragment in (
                "Newcombe-Wilson",
                "dirty source tree",
                "continuation-system",
                "metric runner",
                "128 NFE",
                "raw-or-EMA",
                "device UUID",
                "launch-manifest",
                "global lease",
                "MDLM-matched",
            )
        ),
        "status_labels_consistent": extracted.count("NOT EXECUTED - DISABLED") >= disabled_expected,
        "page_count_in_bounds": 8 <= page_count <= 30,
        "file_size_in_bounds": 10_000 <= file_size <= 15_000_000,
        "minimum_nonwhite_fraction": min(nonwhite_fractions) if nonwhite_fractions else 0.0,
    }
    audit["summary_rows"] = [{"check": key, "result": value} for key, value in audit.items() if key != "summary_rows"]
    return audit


REPORT_RECORDS = collect_report_records(globals())
REPORT_VALIDATION = validate_report_records(REPORT_RECORDS)
REPORT_RECORDS["validation"] = REPORT_VALIDATION
REPORT_PATH = PROJECT_ROOT / "output" / "pdf" / "genmol_benchmark_report.pdf"
REPORT_RENDER_DIR = PROJECT_ROOT / "tmp" / "pdfs" / "genmol_benchmark_report"

build_genmol_report_pdf(REPORT_RECORDS, REPORT_PATH)
REPORT_AUDIT = validate_rendered_report(REPORT_PATH, REPORT_RENDER_DIR)

assert REPORT_AUDIT["all_pages_rendered"]
assert REPORT_AUDIT["all_pages_nonblank"]
assert REPORT_AUDIT["all_required_headings_present"]
assert REPORT_AUDIT["udlm_claim_language_present"]
assert REPORT_AUDIT["status_labels_consistent"]
assert REPORT_AUDIT["page_count_in_bounds"]
assert REPORT_AUDIT["file_size_in_bounds"]

display(report_pd.DataFrame(REPORT_AUDIT["summary_rows"]))
report_shutil.rmtree(REPORT_RENDER_DIR)
print(f"PDF report: {REPORT_PATH.resolve()}")


# Completion gate - bounded implementation stages are in place

The notebook now contains an executable teaching reimplementation with bounded smoke tests for the GenMol V1 path: reproducible GPU-count selection; SAFE; forward and reverse diffusion; weighted MDLM loss; BERT denoising; optimizer, accumulation, DDP semantics, EMA, and checkpoints; exact and confidence sampling; de novo generation; MCG; fragment constraints; R_vocab and R_remask; attachment; PMO hit generation; lead gates; and evaluation.

Every executable section is preceded by an explanation, intuition, equations where relevant, a concrete example, a Paper-vs-GitHub comparison, a statement of what the test proves, and comprehension questions.

## Scientific status

- Implemented and bounded-smoke-tested: GenMol Stages 0-18 and UDLM Stage 20.
- Not claimed: paper-scale numerical reproduction.
- Still required for paper tables: the full 945,455,307-example training split, validation/test splits, a 50,000-step trained checkpoint, repeated large-sample generation, the 23 PMO oracles with 10,000 unique calls each, and the exact docking assets/protocol.
- Expensive flags remain false, so opening or running this teaching notebook cannot accidentally launch those jobs.
- Extended SAFE V2 remains isolated because it is not the representation used in the paper.

## Final comprehension checkpoint

1. Can you now trace one molecule from clean SAFE tokens through q(z_t|x), the denoiser, reverse sampling, and strict decoding?
2. Can you explain why R_vocab and R_remask differ while representing the same molecule?
3. Can you distinguish MCG from property-oracle optimization?
4. Can you distinguish an algorithmic smoke test from a reproduced paper result?


## Audited implementation distinctions

- The released BERT has no explicit time embedding even though the paper writes x_theta(z_t, t); corruption level and loss weighting still depend on t.
- The paper idealizes alpha at t=1 as zero, while the numerical repository endpoint uses epsilon 0.001.
- Exact repository training can corrupt BOS/EOS because it excludes only padding; the molecule-view teaching policy is labeled separately.
- Stage 8 teaches an exact-global numerator/denominator reduction as a labeled repair; the faithful/current training path keeps released process-local token ratios, which DDP and accumulation average. Inference uses EMA weights, and MCG logit-normalization equivalence is stated.
- Fragment scores are labeled as a decomposition-occurrence estimator rather than an exhaustive subgraph census.

## UDLM extension status

- Stage 20 derives the uniform forward law, normalized reverse posterior, and
  full content-position objective, then cross-checks notebook-native functions
  against production diffusion and time-conditioning code.
- The released residual-clean corruption schedule and idealized loss schedule
  are deliberately distinguished; the paper's zero-prior-loss statement does
  not hold exactly for the implemented residual-clean endpoint.
- Two pinned seed-1 CPU toy artifacts at implementation base
  `02595d1ecf994bbd432ec67ef21b0d56ed97918d` are linked, hashed, and schema-validated. They are bounded
  integration evidence, not evidence that UDLM beats GenMol.
- A clean-source CPU audit of ordered training rows 10,001--30,000 recommends
  empirical floor 0.0002 for reviewed pilots. The rule was formalized after
  inspecting those blocks, so it is retrospective training-only engineering,
  not confirmatory or molecular-quality evidence; historical/manual artifacts
  remain at 0.01.
- Candidate training binds launch-manifest schema 2, runtime schema 2, summary
  schema 5, and receipt schema 5, including exact UUIDs, final-idle telemetry,
  the training lease, and exact R/S/E predecessor artifacts. The corrected
  W=1 lineage completed all three 1,000-update members and protocol v4 binds its
  independently validated terminal E chain; those training artifacts contain
  no molecule-level superiority result.
- `launch_health_panel.py` fixes the exact seed-1, full-vocabulary, MDLM-EMA
  10-update contract and deterministic `health-w{W}-{r,s,e}-{H}` names;
  `validate_health_panel.py` grants screen authorization only. Its terminal-E
  receipt is required independently before config materialization and registry
  freezing and cannot satisfy the later candidate lock.
- The W=1 E-only 100-update scheduler screen selected E-L1; two fresh
  500-update conditioning arms selected E-A1. Both decisions used only a frozen
  denoising panel. The later matched scale-up independently restarted R/S/E
  from verified MDLM EMA and completed 1,000 updates per arm.
- V4 freezes a 36-setting operating-point universe and a small-first staged
  campaign: D uses seed 1100 x 32 only for structural diagnosis; A/B/C use
  seeds 1101/1102/1103 with 32/64/96 requests; the eligible stage alone uses
  seeds 1000/1001 x 256 at 128 NFE. Failures remain visible and unrankable;
  retries, substitution, and cross-stage score pooling are forbidden. Ranked
  metrics come from fresh CPU independent rescores before advancement.
- The historical resume sequence was `--through-stage D`, then separate A, B, C, and
  `eligible` invocations after each predecessor decision. D concurrency is one;
  the frozen protocol maximum was three. Current work is capped at two GPUs.
  D terminated with a launcher configuration-identity failure; v4 is archived,
  not eligible for continuation or promotion. Stage 21 previews the separately
  specified engineering-v5 experiment. The historical workflow required all 43 outcomes
  become an addition-only exact-G child through
  `materialize_candidate_evidence.py --stage-published-evidence`, followed by
  separate clean pushed decision, deterministic ledger, and deterministically
  built schema-2 lock commits. There is no hand-authored lock draft.
- Only after all three training receipts pass may seed 1100 x 32 requests run;
  even then D is structural-only and cannot rank any checkpoint or setting.
- Candidate benchmark schema 8 embeds bounded sampler-input IDs, final IDs,
  editable bits, and control-token counts. `raw_samples.csv` is published
  before completion member `summary.json` using exclusive, identity-owned
  rollback; the generation lease and retained output-directory descriptor are
  revalidated before model import and publication.
- Every launch rechecks the full GPU inventory. A qualifying card has
  utilization strictly below 10% (exactly 10% is rejected), at least 30,000 MiB
  free, and non-prohibited compute mode. The active authorization is at most
  two GPUs without another request; UUID mapping never assumes physical GPU
  0 and active processes are recorded but never interrupted.
- Completed selection seeds publish schema-2 reference-only envelopes after
  structural validation and fresh-process raw-text rescoring. Failed seeds use
  launcher-authored schema-1 receipts; a partial failure retains the completed
  sibling while making the attempt ineligible. All outcomes must predate the
  lock and descend into the final benchmark revision.
- The completed screen registry, E-L1 decision, and E-A1 decision are frozen in
  the H/R0/R1/R2/R3 Git chain. L1 is an optimizer-schedule bundle with about
  37.57x L0's cumulative learning-rate exposure over updates 0--99, not an
  isolated cosine-shape ablation. Both 500-update A0/A1 arms independently
  reloaded the same MDLM EMA and reseeded after initialization; A1 preserved
  initial logits and passed its staged conditioner-gradient contract. These
  selections alone cannot authorize scale-up or support molecular superiority.
- The final success gate is three matched 1,000-request seeds with repaired and
  strict metrics, checkpoint/seeds/device/runtime provenance, independent
  raw-text rescoring of all three CSVs, and thresholds recorded in
  `stage20_success_criteria`.
- Final checkpoint: why is clean-logit interpolation not valid UDLM
  classifier-free guidance? Expected reasoning: clean probabilities are
  transformed nonlinearly into the reverse posterior, so guidance must combine
  conditional and unconditional reverse-posterior log probabilities.
